# Supplementary Figures S1–S15

Run this notebook from the `figures/` folder after populating `data/` and `results/`
(see the repository README). Each section below reruns the relevant part of an analysis
notebook from the saved models; panels are written to `figures/panels/`
(`paths.FIGURES_DIR`).

Run the 1 µM section before the 10 µM section: panel d compares the two doses using the per-dose scores written by each section.

In [ ]:
import os
import sys

# Repository root, so that paths.py (DATA_DIR / RESULTS_DIR / FIGURES_DIR) can be imported.
sys.path.insert(0, os.path.abspath('..'))
import paths

for d in (paths.DATA_DIR, paths.RESULTS_DIR, paths.FIGURES_DIR):
    print(d, '(found)' if os.path.isdir(d) else '(missing)')

## Replogle: reproducibility

From [`scripts/replogle/2_benchmarking.ipynb`](../scripts/replogle/2_benchmarking.ipynb).

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import torch
from matplotlib.colors import to_hex
from scipy.optimize import linear_sum_assignment

import sherlock as slk

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib as mpl

# ── Journal figure style ────────────────────────────────────────────────
RC = {
    'font.family':           'sans-serif',
    'font.sans-serif':       ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size':             8,
    'axes.labelsize':        8,
    'axes.titlesize':        9,
    'xtick.labelsize':       7,
    'ytick.labelsize':       7,
    'legend.fontsize':       6,
    'legend.title_fontsize': 7,
    'axes.spines.top':       False,
    'axes.spines.right':     False,
    'axes.linewidth':        0.75,
    'xtick.major.width':     0.75,
    'ytick.major.width':     0.75,
    'xtick.major.size':      3,
    'ytick.major.size':      3,
    'figure.dpi':            150,
    'savefig.dpi':           300,
    'pdf.fonttype':          42,
    'ps.fonttype':           42,
    'svg.fonttype': 'none',
}
mpl.rcParams.update(RC)

In [ ]:
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_RUNS      = 5
NUM_EPOCHS  = 200
VALIDATE_EVERY = 50
BATCH_SIZE  = 4096
LATENT_DIM  = 16
PATIENCE    = 20
RESULTS_DIR = f"{paths.RESULTS_DIR}/benchmarking"
N_CLUSTERS  = 8

# Metrics tracked for scoring and box plots
METRIC_COLS = [
    'cf_ate_pearson_r',
    'cluster_f1',
    'cluster_ari',
    'cluster_ami',
    'cluster_silhouette',
    'cluster_linear_probe',
    'cluster_knn_purity',
]

# (model_key, display_name, method-specific kwargs)
# sherlock (vae) kwargs mirror the tuned hyperparameters in "demo copy.ipynb"
# (cov_lambda / H_lambda / gate_row_repulsion_lambda disabled, ce_lambda raised to 20)
METHODS = [
    ("vae",           "sherlock",      {
        "l0_lambda": 1e-1,
        "ce_lambda": 20.,
        "n_epochs_l0_warmup": 300,
    },  500),
    ("svae",          "sVAE",          {"sparse_mask_penalty": 10.0},  NUM_EPOCHS),
    ("cvae",          "cVAE",          {},  NUM_EPOCHS),
    ("contrastivevi", "contrastiveVI", {},  NUM_EPOCHS),
    ("scgen",         "scgen",         {},  NUM_EPOCHS),
]

####TODO CHECK THIS#####
METHODS = [
    ("vae",           "sherlock",      {
        "l0_lambda": 1e-1,
        "ce_lambda": 20.,
        "n_epochs_l0_warmup": 300,
    }, 500),
    ("svae",          "sVAE",          {
        "sparse_mask_penalty": 4.366473592979633,
        "beta": 0.614252574393396,
        "n_hidden": 251,
        "n_layers": 3,
        "dropout_rate": 0.28184968246925673,
        "lr": 0.00331348361550895,
    }, NUM_EPOCHS),
    ("cvae",          "cVAE",          {
        "beta": 3.2226582291306802,
        "n_hidden": 67,
        "n_layers": 1,
        "dropout_rate": 0.29147206450401036,
        "lr": 0.006205861500406846,
    }, NUM_EPOCHS),
    ("contrastivevi", "contrastiveVI", {
        "wasserstein_penalty": 4.399492045282824,
        "n_hidden": 94,
        "n_layers": 2,
        "dropout_rate": 0.2902495045654139,
        "lr": 0.0005218479707033748,
    }, NUM_EPOCHS),
    ("scgen",         "scgen",         {}, NUM_EPOCHS),
]

os.makedirs(f"{RESULTS_DIR}/models",   exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures",  exist_ok=True)

print(f"Device: {DEVICE}  Results: {RESULTS_DIR}")

## Data loading and ground-truth ATE

In [ ]:
data = slk.datasets.replogle()

slk.pp.treat_effect(
    data,
    label_col='pert',
    control_label='NTC',
    method='perturbseq',
    inplace=True,
    key='treat_effect',
)
treat_effect_adata = data.uns['treat_effect']
print(f"Cells: {data.n_obs}  Genes: {data.n_vars}  Perturbations: {data.obs['pert'].nunique()}")

## Pathway annotations for clustering evaluation

In [ ]:
pathways = data.uns['pathways']

all_genes_filtered = [
    gene
    for genes in pathways.values()
    for gene in genes
    if gene != 'LSM5'
]

gene_to_pathway = {
    gene: pathway
    for pathway, genes in pathways.items()
    for gene in genes
    if gene != 'LSM5'
}
pathway_df_indexed = pd.DataFrame.from_dict(
    gene_to_pathway, orient='index', columns=['pathway']
)
pathway_df_indexed.index.name = 'gene'

print(f"Pathway genes: {len(all_genes_filtered)}  Pathways: {len(pathways)}")

## Benchmark loop

Each method is trained `N_RUNS` times independently. The best-scoring run
per method is saved to `results/models/`. Metrics are computed after training
using the proper counterfactual ATE protocol; the clustering cut is chosen
automatically via the elbow method.

In [ ]:
records         = []
best_run_out    = {}   # method → out dict of best single run
best_run_model  = {}   # method → model of best single run
best_run_score  = {}   # method → avg metric score of best single run
best_run_path   = {}   # method → run_x.pth of best single run

for model_key, display_name, extra_kwargs, EPOCHS, in METHODS:
    print(f"\n{'='*60}\n{display_name}  ({N_RUNS} runs)\n{'='*60}")
    best_run_score[display_name] = -np.inf

    # Each method gets its own folder; individual runs are checkpointed there
    # so a re-executed notebook reloads finished runs instead of retraining.
    model_dir = f"{RESULTS_DIR}/models/{display_name}"
    os.makedirs(model_dir, exist_ok=True)

    for run in range(N_RUNS):
        print(f"\n  -- {display_name} run {run + 1}/{N_RUNS} --")

        run_path = f"{model_dir}/run_{run}.pth"

        if os.path.exists(run_path):
            print(f"    Found checkpoint → {run_path}, loading instead of retraining")
            model = slk.tl.load_results(run_path).to(DEVICE)
            out = {"model": model, "param_store": None}
        else:
            out = slk.tl.run(
                data,
                model=model_key,
                batch_size=BATCH_SIZE,
                num_epochs=EPOCHS,
                device=DEVICE,
                latent_dim=LATENT_DIM,
                patience=PATIENCE,
                validate_every=VALIDATE_EVERY,
                seed=run,
                **extra_kwargs,
            )
            model = out['model']
            slk.tl.save_results(out, run_path)
            print(f"    Saved → {run_path}")

        metrics = slk.tl.evaluate_model(
            model,
            data,
            treat_effect_adata,
            pathway_df_indexed,
            all_genes_filtered,
            n_clusters=N_CLUSTERS,
            device=DEVICE,
            use_rho=False,
        )

        # Track best run for this method
        run_avg = float(metrics['cf_ate_pearson_r'])
        if run_avg > best_run_score[display_name]:
            best_run_score[display_name] = run_avg
            best_run_out[display_name]   = out
            best_run_model[display_name] = model
            best_run_path[display_name]  = run_path

        metrics['method'] = display_name
        metrics['run']    = run
        records.append(metrics)

        for k, v in metrics.items():
            if k not in ('method', 'run'):
                val_str = f"{v:.4f}" if np.isfinite(float(v)) else "NaN"
                print(f"    {k:30s} = {val_str}")

    # Save best run model as an exact copy of its own run_x.pth. Copying (vs.
    # re-saving from memory) guarantees best.pth carries the best run's
    # state_dict AND its Pyro param store (sigma_fac, …). The store is not in
    # state_dict, and the *global* Pyro store reflects whichever run ran last,
    # not necessarily the best — so we never rely on it.
    import shutil
    save_path = f"{model_dir}/best.pth"
    shutil.copyfile(best_run_path[display_name], save_path)
    print(f"\n  Saved best {display_name} model → {save_path} "
          f"(copied from {os.path.basename(best_run_path[display_name])})")

results_df = pd.DataFrame(records)
print("\nDone.")

## Results summary

In [ ]:
summary = (
    results_df
    .groupby('method')[METRIC_COLS]
    .agg(['mean', 'std'])
    .round(4)
)
# Preserve METHODS order
method_order = [m[1] for m in METHODS]
summary = summary.reindex([m for m in method_order if m in summary.index])
print(summary.to_string())

### Fig. S1: reproducibility across five runs (Replogle)

In [ ]:
# ── SHERLOCK robustness / reproducibility across N runs ───────────────────
# Same three metrics (and same style) as norman_figure.ipynb Panel B, computed
# on the cached SHERLOCK benchmark runs:
#   * rho_corr    cross-run r : leave-one-out Spearman r of the learned
#                               perturbation-covariance matrix vs. the mean of
#                               the other runs' matrices.
#   * W coact.    cross-run r : same, on the (P, P) cosine-normalized gate
#                               co-activation matrix (permutation-invariant to
#                               the K gate dims, so comparable across runs).
#   * CF effect   size corr.  : per-run Pearson r between predicted and observed
#                               counterfactual effect magnitudes (|ATE| norms).
from scipy.stats import pearsonr, spearmanr, bootstrap

SHERLOCK_DIR = f"{RESULTS_DIR}/models/sherlock"


def _effect_size_r(pred_df, obs_df):
    common = pred_df.index.intersection(obs_df.index)
    if len(common) < 3:
        return float('nan')
    p_norms = pred_df.loc[common].apply(np.linalg.norm, axis=1).values
    o_norms = obs_df.loc[common].apply(np.linalg.norm, axis=1).values
    v = np.isfinite(p_norms) & np.isfinite(o_norms)
    return float(pearsonr(p_norms[v], o_norms[v])[0]) if v.sum() > 2 else float('nan')


def _w_coact_matrix(W, threshold=0.5):
    """(P, P) cosine-normalized perturbation co-activation matrix. Invariant to
    permutation of the K gate dimensions, so it can be compared directly across
    independently trained runs (unlike the raw W)."""
    W_bin = (W > threshold).astype(float)
    dot   = W_bin @ W_bin.T
    norms = np.sqrt(np.diag(dot))
    return dot / (np.outer(norms, norms) + 1e-8)


def _flat_triu(M):
    return M[np.triu_indices(M.shape[0], k=1)]


def _loo_spearman_df(mats_by_run, metric_name):
    """Spearman r between each run's (P, P) matrix and the elementwise mean of
    every OTHER run's matrix (leave-one-out) — one row per run. These n points
    each depend on a distinct held-out run, so they behave like an i.i.d.
    sample (unlike the n(n-1)/2 non-independent all-pairs comparisons)."""
    rows = []
    keys = sorted(mats_by_run)
    for k in keys:
        others = [mats_by_run[j] for j in keys if j != k]
        if not others:
            continue
        mean_other = np.mean(others, axis=0)
        vi, vj = _flat_triu(mats_by_run[k]), _flat_triu(mean_other)
        valid  = np.isfinite(vi) & np.isfinite(vj)
        r = float(spearmanr(vi[valid], vj[valid])[0]) if valid.sum() > 2 else float('nan')
        rows.append({'metric': metric_name, 'run': k, 'r': r})
    return pd.DataFrame(rows)


# ── Compute (cached) per-run matrices + effect-size correlations ──────────
ROBUST_MATS_PATH = f"{RESULTS_DIR}/sherlock_robustness_matrices.npz"
ROBUST_EFF_PATH  = f"{RESULTS_DIR}/sherlock_robustness_effect.csv"
run_ckpts = {run: f"{SHERLOCK_DIR}/run_{run}.pth" for run in range(N_RUNS)
             if os.path.exists(f"{SHERLOCK_DIR}/run_{run}.pth")}


def _cache_fresh(cache_path, source_paths):
    """True if cache exists and is newer than every source checkpoint, so a
    retrained run automatically invalidates the cached matrices."""
    if not os.path.exists(cache_path):
        return False
    cm = os.path.getmtime(cache_path)
    return all(os.path.exists(s) and os.path.getmtime(s) <= cm for s in source_paths)


_fresh = _cache_fresh(ROBUST_MATS_PATH, run_ckpts.values()) and \
         _cache_fresh(ROBUST_EFF_PATH, run_ckpts.values())

if _fresh:
    _saved       = np.load(ROBUST_MATS_PATH)
    rho_by_run   = {int(k.split('_')[-1]): _saved[k] for k in _saved.files if k.startswith('rho_corr_')}
    wcoact_by_run = {int(k.split('_')[-1]): _saved[k] for k in _saved.files if k.startswith('wcoact_')}
    eff_df       = pd.read_csv(ROBUST_EFF_PATH)
    print(f"Loaded cached robustness matrices for {len(rho_by_run)} runs.")
else:
    rho_by_run, wcoact_by_run, eff_records = {}, {}, []
    for run, ckpt in run_ckpts.items():
        print(f"  run {run}: eval + counterfactual effects...")
        model = slk.tl.load_results(ckpt).to(DEVICE)
        slk.tl.eval_single(model, data, obsm_key='_z_robust', uns_key='_robust', device=DEVICE)
        tmp = data.uns['_robust']
        if tmp.get('rho_corr') is not None:
            rho_by_run[run] = np.asarray(tmp['rho_corr'], dtype=float)
        if tmp.get('W') is not None:
            wcoact_by_run[run] = _w_coact_matrix(np.asarray(tmp['W'], dtype=float))
        ate  = model.predict_counterfactual_effects(
            data, n_centroids=25, observed_effect=data.uns['treat_effect'], approximate=False,
        )
        eff_records.append({'run': run, 'effect_size_r': _effect_size_r(ate['predicted_df'], ate['observed_df'])})
    eff_df = pd.DataFrame(eff_records)
    np.savez(
        ROBUST_MATS_PATH,
        **{f'rho_corr_{k}': v for k, v in rho_by_run.items()},
        **{f'wcoact_{k}':   v for k, v in wcoact_by_run.items()},
    )
    eff_df.to_csv(ROBUST_EFF_PATH, index=False)
    print(f"Computed and cached robustness matrices for {len(rho_by_run)} runs.")

# ── Per-run counterfactual effect matrices (cached) ───────────────────────
# Needed for TWO different columns of the robustness figure:
#   * cross-run reproducibility of the predicted effect profiles
#   * accuracy of the predicted effect sizes against the measured effects
CF_PATH = f"{RESULTS_DIR}/sherlock_cf_effects.npz"
if os.path.exists(CF_PATH):
    _cf = np.load(CF_PATH, allow_pickle=True)
else:
    _store = {}
    _obs = None
    for run, ckpt in run_ckpts.items():
        _m = slk.tl.load_results(ckpt).to(DEVICE)
        _ate = _m.predict_counterfactual_effects(
            data, n_centroids=25, observed_effect=data.uns['treat_effect'], approximate=False,
        )
        _p, _o = _ate['predicted_df'], _ate['observed_df']
        _common = _p.index.intersection(_o.index)
        _store[f'pred_{run}'] = _p.loc[_common].values.astype(np.float32)
        if _obs is None:
            _obs = _o.loc[_common].values.astype(np.float32)
            _store['perts'] = np.array(_common, dtype=object)
        del _m
    _store['obs'] = _obs
    np.savez_compressed(CF_PATH, **_store)
    _cf = np.load(CF_PATH, allow_pickle=True)

pred_by_run = {int(k.split('_')[-1]): _cf[k] for k in _cf.files if k.startswith('pred_')}
obs_mat     = _cf['obs']


def _loo_spearman_flat(mats_by_run, metric_name):
    """Leave-one-out Spearman for non-symmetric (perturbation x gene) matrices:
    each run against the elementwise mean of the other runs."""
    rows, keys = [], sorted(mats_by_run)
    for k in keys:
        mean_other = np.mean([mats_by_run[j] for j in keys if j != k], axis=0)
        a, b = mats_by_run[k].ravel(), mean_other.ravel()
        v = np.isfinite(a) & np.isfinite(b)
        rows.append({'metric': metric_name, 'run': k, 'r': float(spearmanr(a[v], b[v])[0])})
    return pd.DataFrame(rows)


# ── Assemble the long-form robustness table ───────────────────────────────
# Three cross-run reproducibility measures + one accuracy measure against data.
L_RHO = 'Perturbation\ncorrelation'
L_GATE = 'Gate\nco-activation'
L_CFP = 'Counterfactual\neffect profile'
L_CFS = 'Counterfactual\neffect size'
ROBUST_LABELS = {'rho_corr_r': L_RHO, 'w_coact_r': L_GATE,
                 'cf_effect_r': L_CFP, 'effect_size_r': L_CFS}

pairwise_df = pd.concat([
    _loo_spearman_df(rho_by_run, 'rho_corr_r'),
    _loo_spearman_df(wcoact_by_run, 'w_coact_r'),
    _loo_spearman_flat(pred_by_run, 'cf_effect_r'),
], ignore_index=True)

df_robust = pd.concat([
    pairwise_df.rename(columns={'r': 'Score'})[['metric', 'Score']],
    eff_df[['effect_size_r']].rename(columns={'effect_size_r': 'Score'}).assign(metric='effect_size_r'),
], ignore_index=True)
df_robust['Metric'] = pd.Categorical(
    df_robust['metric'].map(ROBUST_LABELS),
    categories=[ROBUST_LABELS[k] for k in ROBUST_LABELS], ordered=True,
)

print(df_robust.groupby('metric', observed=True)['Score'].agg(['mean', 'std']).round(3).to_string())

# ── Plot: reproducibility columns (blue) vs accuracy column (grey) ────────
CLR_ACC = '#7f7f7f'
_pal = {L_RHO: CLR_SHERLOCK, L_GATE: CLR_SHERLOCK, L_CFP: CLR_SHERLOCK, L_CFS: CLR_ACC}

fig_r, ax_r = plt.subplots(figsize=(4.4, 3.0))
sns.stripplot(data=df_robust, x='Metric', y='Score', hue='Metric', palette=_pal, legend=False,
              size=5, jitter=0.15, linewidth=0.4, edgecolor='white', ax=ax_r)
sns.boxplot(data=df_robust, x='Metric', y='Score', color='white', width=0.45, linewidth=0.75,
            fliersize=0, ax=ax_r, boxprops=dict(edgecolor='0.3'),
            whiskerprops=dict(color='0.3'), capprops=dict(color='0.3'),
            medianprops=dict(color='0.15', linewidth=1.5))
ax_r.set_ylim(0, 1.18)
ax_r.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax_r.set_xlabel('')
ax_r.set_ylabel('Correlation')
ax_r.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.2f'))
ax_r.axvline(2.5, color='0.75', lw=0.6, ls='--')
ax_r.plot([-0.4, 2.4], [1.12, 1.12], color=CLR_SHERLOCK, lw=1, clip_on=False)
ax_r.text(1.0, 1.14, 'reproducibility across runs', ha='center', va='bottom',
          fontsize=7, color=CLR_SHERLOCK)
ax_r.plot([2.6, 3.4], [1.12, 1.12], color=CLR_ACC, lw=1, clip_on=False)
ax_r.text(3.0, 1.14, 'accuracy vs data', ha='center', va='bottom', fontsize=7, color=CLR_ACC)
ax_r.text(0.99, 0.02, f'$n$ = {len(run_ckpts)} runs', transform=ax_r.transAxes,
          ha='right', va='bottom', fontsize=6, color='0.4')
plt.tight_layout()
for _ext in ('png', 'svg', 'pdf'):
    fig_r.savefig(f"{RESULTS_DIR}/figures/sherlock_robustness.{_ext}",
                  dpi=300 if _ext == 'png' else None, bbox_inches='tight')
plt.show()
print('SHERLOCK robustness figure saved.')


## EGFR inhibitor screen

From [`scripts/egfr_drug_screen/2_egfr_analysis.ipynb`](../scripts/egfr_drug_screen/2_egfr_analysis.ipynb).

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path

import seaborn as sns
import matplotlib.pyplot as plt

import torch

import sherlock as slk

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({'font.size': 12, 'svg.fonttype': 'none'})
mpl.rcParams['axes.titlesize'] = 16
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['lines.linewidth'] = 2
figure_dir = f"{paths.FIGURES_DIR}/egfr_drug_screen/"
os.makedirs(figure_dir, exist_ok=True)

In [ ]:
pert_type = 'drug_egfr'  # 'crispri' or 'crisprko'

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
slk.configs.show()
slk.configs.set_config('ntc_label', 'DMSO')
slk.configs.set_config('pert_key', 'drug')
slk.configs.show()
#slk.configs.reset_config()
slk.configs.get_config('ntc_label')


In [ ]:
slk.configs.get_config('ntc_label')

In [ ]:
concentration = 10000
data = sc.read_h5ad(f"{paths.DATA_DIR}/gbm/bt333_egfr_degs_{concentration}.h5ad")

data.obs

In [ ]:
# remove na from drug column
data = data[~data.obs['drug'].isna()].copy()
data = data[
    (data.obs['dose'] == concentration) | (data.obs['drug'] == 'DMSO')
].copy()

In [ ]:
# filter for drugs has more than 100 cells
drug_counts = data.obs['drug'].value_counts()
drugs_to_keep = drug_counts[drug_counts > 100].index
data = data[data.obs['drug'].isin(drugs_to_keep)].copy()

print(f"Kept {len(drugs_to_keep)} drugs with > 100 cells.")
print(data.obs['drug'].value_counts())

In [ ]:
slk.pp.slk_prepare_data(data, pert_key="drug", ntc_label="DMSO", inplace=True)

In [ ]:
slk.pp.treat_effect(
    data,
    label_col = "drug",
    control_label = 'DMSO',
    method = "perturbseq",
    inplace=True,
    key='treat_effect'
)

In [ ]:
data

In [ ]:
data.X.max()

In [ ]:
# Check for NaN/Inf in expression matrix
print(f"Data shape: {data.shape}")
X = data.X.toarray() if hasattr(data.X, 'toarray') else data.X

n_nan = np.isnan(X).sum()
n_inf = np.isinf(X).sum()
print(f"NaN values in X: {n_nan}")
print(f"Inf values in X: {n_inf}")

if n_nan > 0 or n_inf > 0:
    print("Cleaning expression matrix...")
    # Replace NaN with 0 and clip Inf values
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    data.X = X
    print("Expression matrix cleaned!")

# Check data stats
print(f"X min: {data.X.min()}, max: {data.X.max()}")
print(f"Unique drugs: {data.obs['drug'].nunique()}")

In [ ]:
data.obs

In [ ]:
data

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# results = slk.tl.run_single(data, 
#                             batch_size=1024,
#                             num_workers=0, 
#                             device=device, 
#                             use_conditions=False,
#                             l0_lambda = 10,
#                             num_epochs=1000,
#                             #gate_init_p=0.4,
#                             H_lambda=0.01,
#                             ce_lambda=0.1,
#                             latent_dim= 8,
#                             )

In [ ]:
# import datetime
    
# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# slk.tl.save_results(results, f'{paths.RESULTS_DIR}/models/{pert_type}_model_{timestamp}.pth')

In [ ]:
NUM_EPOCHS     = 1000
VALIDATE_EVERY = 50
BATCH_SIZE     = 4096
LATENT_DIM     = 16
PATIENCE       = 350

In [ ]:
SHERLOCK_KWARGS = {
    # "tau_end": 0.3,
    # # "tau_init": 1.5,
    # "cov_lambda": 1e-5,
    # "H_lambda": 1e-1,
    # "gate_row_repulsion_lambda": 1e-2,
    # "ce_lambda": 10,
    # "shift": "linear",
    "l0_lambda":50,
    "debug": True,
    # "lr": 1e-3,
    # "ntc_lambda": 1e-5,       
    # "pert_lambda": 1,
    "n_epochs_l0_warmup": 100
}

In [ ]:
seed = 1

In [ ]:
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# results = slk.tl.run(
#         data,
#         model="vae",
#         batch_size=BATCH_SIZE,
#         num_epochs=NUM_EPOCHS,
#         device=DEVICE,
#         latent_dim=LATENT_DIM,
#         patience=PATIENCE,
#         validate_every=VALIDATE_EVERY,
#         seed=seed,
#         # debug=True,
#         **SHERLOCK_KWARGS,
#     )

In [ ]:
# results

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# results = slk.tl.run_single(data, batch_size=4096, shuffle=True, num_workers=9, device=device)


In [ ]:
# slk.tl.save_results(results, f'{paths.RESULTS_DIR}/models/replogle_model.pth')


In [ ]:
# import datetime
    
# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# slk.tl.save_results(results, f'{paths.RESULTS_DIR}/models/{pert_type}_model_{timestamp}_{seed}.pth')

# print(f'Saved results to ../data/{pert_type}_model_{timestamp}_{seed}.pth')

In [ ]:
# model = results['model']

In [ ]:
# This run Ross Saved results to ../data/drug_egfr_model_20260529_163520_1.pth

In [ ]:
# # this is a great run
# #model = slk.tl.load_results(f'{paths.RESULTS_DIR}/models/crispri_model_20251105_103826.pth')
# # {paths.RESULTS_DIR}/models/crispri_model_20251110_215304.pth
# #{paths.RESULTS_DIR}/models/crispri_model_20251111_004834.pth


# # accuray is not 100%
model = slk.tl.load_results(f'{paths.RESULTS_DIR}/models/drug_egfr_model_20260529_163520_1.pth')
# #model = results['model']

In [ ]:
slk.tl.eval_single(model, data, obsm_key="z", uns_key="results")

In [ ]:
data.uns['results']

In [ ]:
data.X = data.X.toarray()
slk.pl.plot_r2(data)

In [ ]:
slk.pl.plot_corr(data)

In [ ]:
p_key = slk.configs.get_config('pert_key')
NTC = slk.configs.get_config('ntc_label')

sub = data[data.obs[p_key] != NTC]

sc.pp.neighbors(sub, n_neighbors=15, use_rep='z')
sc.tl.umap(sub)

import random
n_cats = sub.obs[p_key].nunique()
palette = sns.color_palette("hsv", n_colors=n_cats)
random.shuffle(palette)
sc.pl.umap(sub, color='drug', palette=palette, frameon=False)

In [ ]:
zrc = data.uns['results']['z_rho_corr']

print(f"Spearman ρ = {zrc['spearman_r']:.3f}  (p = {zrc['spearman_p']:.1e})")
print(f" Pearson  r = {zrc['pearson_r']:.3f}  (p = {zrc['pearson_p']:.1e})") 

In [ ]:
slk.pl.plot_zcorr(data)

In [ ]:
W = data.uns['results']['W']
plt.hist(W.flatten(), bins=30)
plt.show()

In [ ]:
W_bin = (W > 0.5).astype(float)
sns.heatmap(W, cmap="coolwarm")

In [ ]:

dataset = slk.tl.PerturbMatchingDataset(data)
all_genes = dataset.perturbation_dict.keys()

In [ ]:

dataset = slk.tl.PerturbMatchingDataset(data)
all_genes = dataset.perturbation_dict.keys()

In [ ]:
new_colormap = [
    'darkslateblue',  # 0
    'firebrick',      # 1
    'darkorange',     # 2
    'forestgreen',    # 3
    'gold',           # 4
    'dodgerblue',     # 5
    'mediumorchid',   # 6
    'deeppink',       # 7
    'teal',           # 8
    'black'           # 9
]
t = 2.2

In [ ]:
# 1.5 previously
groups = slk.tl.cluster_rho(data, t=t
                            , show=True)

In [ ]:
#for a non subsetted data, pass in genes corresponding to idx ie dataset.perturbation_dict.keys()
named_groups = slk.tl.group2name(groups, all_genes)
named_groups

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, set_link_color_palette
from matplotlib.colors import to_hex

def cluster_rho(adata, t=1.5, uns_key='results', rho_corr=None, show=True, pert_key='perturbation', figure_dir=None, save_name=None):
    if rho_corr is None:
        C = adata.uns[uns_key]['rho_corr']
    else:
        C = rho_corr
    
    # Get perturbation names - try multiple possible locations
    uns = adata.uns.get(uns_key, {})
    if 'rho_perts' in uns:
        pert_names = np.asarray(uns['rho_perts'])
    elif 'pert_names' in uns:
        pert_names = np.asarray(uns['pert_names'])
    elif 'perts' in uns:
        pert_names = np.asarray(uns['perts'])
    else:
        # Fall back to unique perturbation values from obs
        _pk = slk.configs.get_config('pert_key')
        ntc = slk.configs.get_config('ntc_label')
        pert_names = np.asarray([p for p in adata.obs[_pk].unique() if p != ntc])
    
    D = 1.0 - C  # distance
    Z_link = linkage(D[np.triu_indices_from(D, 1)], method='ward')
    
    # Cluster labels at this cut, and the exact colors scipy will draw for them —
    # so plot_zcorr_colored (same linkage, same t, same palette) matches this dendrogram
    groups = fcluster(Z_link, t=t, criterion='distance')
    unique_groups = sorted(set(groups))
    palette_hex = [to_hex(new_colormap[i % len(new_colormap)]) for i in range(len(unique_groups))]
    set_link_color_palette(palette_hex)
    
    if show:
        plt.figure(figsize=(max(5, len(pert_names) * 0.15), 4))
        dendrogram(
            Z_link,
            color_threshold=t,
            above_threshold_color="gray",
            labels=pert_names,  # Use actual names
            leaf_rotation=90,
            leaf_font_size=8,
        )
        plt.axhline(y=t, color="red", linestyle="--", linewidth=1)
        #plt.title(f"Covariance hierarchical clustering t={t}")
        plt.xlabel("Perturbation")
        plt.ylabel("Distance")
        plt.tight_layout()
        
        # Export the figures
        if figure_dir and save_name:
            plt.savefig(os.path.join(figure_dir, f"{save_name}_dendrogram.svg"), dpi=300, bbox_inches='tight')
            plt.savefig(os.path.join(figure_dir, f"{save_name}_dendrogram.png"), dpi=300, bbox_inches='tight')
            plt.savefig(os.path.join(figure_dir, f"{save_name}_dendrogram.pdf"), dpi=300, bbox_inches='tight')
            print(f"Saved figures to {figure_dir}/{save_name}_dendrogram.[svg|png|pdf]")
        
        plt.show()
    
    # Record the group -> color mapping scipy actually used, for reference/reuse
    ddata = dendrogram(Z_link, color_threshold=t, above_threshold_color='gray', no_plot=True)
    group_to_color = {}
    for leaf_idx, color in zip(ddata['leaves'], ddata.get('leaves_color_list', [])):
        g = groups[leaf_idx]
        if g not in group_to_color:
            group_to_color[g] = color
    adata.uns['_dendrogram_colors'] = {str(k): v for k, v in group_to_color.items()}
    
    # Return as a dictionary mapping perturbation names to cluster IDs
    cluster_assignments = dict(zip(pert_names, groups))
    
    return groups, cluster_assignments

In [ ]:
groups = cluster_rho(data, t=t, show=True, figure_dir=figure_dir, save_name=f"{pert_type}_{concentration}uM")

In [ ]:
#named_groups.to_csv(f'{paths.RESULTS_DIR}/gbm/{pert_type}_drug_groups_{concentration}uM_final.csv')

In [ ]:
# map perturbation -> group using named_groups (index=gene, column='group')
gene_to_group = named_groups['group'].to_dict()

# assign group to obs based on slk pert key
sub.obs['pert_group'] = sub.obs[p_key].map(gene_to_group)

# mark NTC / unknown perturbations as -1 and ensure integer dtype
sub.obs['pert_group'] = sub.obs['pert_group'].fillna(-1).astype('category')

# quick summary
print(sub.obs['pert_group'].value_counts().sort_index())

sc.pl.umap(sub, color=['pert_group'], title='Perturbation Groups', palette='tab20', size=30)

In [ ]:
new_colormap = [
    'darkslateblue',  # 0
    'firebrick',      # 1
    'darkorange',     # 2
    'forestgreen',    # 3
    'gold',           # 4
    'dodgerblue',     # 5
    'mediumorchid',   # 6
    'deeppink',       # 7
    'teal',           # 8
    'black'           # 9
]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

sc.pl.umap(
    sub,
    color='drug',
    title='Drugs',
    palette=palette,
    size=12,
    frameon=False,
    ax=axes[0],
    show=False,
    legend_fontsize=6,
)

sc.pl.umap(sub, color='pert_group', title='Perturbation groups', 
           palette=new_colormap, size=12, frameon=False, ax=axes[1], show=False)

for ax in axes:
    ax.set_title(ax.get_title(), fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f"{pert_type}_{concentration}_sherlock_zspace.svg"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{pert_type}_{concentration}_sherlock_zspace.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{pert_type}_{concentration}_sherlock_zspace.pdf"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# assign group to obs based on slk pert key
data.obs['pert_group'] = data.obs[p_key].map(gene_to_group)

# mark NTC / unknown perturbations as -1 and ensure integer dtype
data.obs['pert_group'] = data.obs['pert_group'].fillna(-1).astype('category')

# quick summary
print(data.obs['pert_group'].value_counts().sort_index())

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster, dendrogram, set_link_color_palette
from matplotlib.colors import to_hex
from matplotlib import gridspec
from matplotlib.patches import Patch


def plot_zcorr_colored(adata, t=1.5, uns_key='results', group_color=new_colormap):
    """
    Plot correlation matrices with a color bar for each perturbation group.
    Uses the same linkage/threshold/palette as cluster_rho, so the bar colors
    match what cluster_rho's dendrogram draws for the same cut t.
    """
    
    Sigma_P_posterior = adata.uns[uns_key]['rho_corr']
    Sigma_z = adata.uns[uns_key]['z_corr']
    pert_names = adata.uns[uns_key]['rho_perts']

    # Same distance + linkage as cluster_rho
    D = 1.0 - Sigma_P_posterior
    Z_link = linkage(D[np.triu_indices_from(D, 1)], method='ward')
    order = leaves_list(Z_link)

    # Apply the same permutation to both matrices
    SigmaP_ord = Sigma_P_posterior[order][:, order]
    Sigmaz_ord = Sigma_z[order][:, order]
    pert_names_ord = pert_names[order]

    # Same cut as cluster_rho(..., t=t)
    groups = fcluster(Z_link, t=t, criterion='distance')
    groups_ord = groups[order]
    unique_groups = sorted(set(groups))

    # Same palette + same scipy color-assignment logic as cluster_rho, so colors match the dendrogram
    palette_hex = [to_hex(group_color[i % len(group_color)]) for i in range(len(unique_groups))]
    set_link_color_palette(palette_hex)
    ddata = dendrogram(Z_link, color_threshold=t, above_threshold_color='gray', no_plot=True)
    group_to_color = {}
    for leaf_idx, color in zip(ddata['leaves'], ddata.get('leaves_color_list', [])):
        g = groups[leaf_idx]
        if g not in group_to_color:
            group_to_color[g] = color

    row_colors = [group_to_color[grp] for grp in groups_ord]
    n_perts = len(pert_names_ord)

    # Bar width and its gap from the heatmap, scaled to stay proportional to heatmap
    # size (reference notebook used width=3, gap=1 for 71 perturbations)
    side_bar_width = n_perts * (3 / 71)
    bar_gap = n_perts * (1 / 71)

    # Plot single heatmap
    fig = plt.figure(figsize=(4, 3))
    ax0 = fig.add_subplot(111)
    
    sns.heatmap(SigmaP_ord, cmap="coolwarm", square=True, ax=ax0, 
                xticklabels=False,
                yticklabels=False,
                cbar_kws={
                    'label': 'Correlation',
                    'shrink': 0.6,
                    'aspect': 20
                })
    ax0.set_title("")
    ax0.set_xlabel("Perturbations")
    ax0.set_ylabel("Perturbations")
    
    # Add vertical color bar on the right, with a small gap from the heatmap
    for i, color in enumerate(row_colors):
        ax0.add_patch(plt.Rectangle((n_perts + bar_gap, i), side_bar_width, 1, color=color, 
                                    transform=ax0.transData, clip_on=False))

    # Create legend outside with transparent background
    legend_elements = [Patch(facecolor=group_to_color[grp], label=f'{int(grp)}')
                      for grp in unique_groups]
    fig.legend(
        handles=legend_elements,
        loc='center left',
        bbox_to_anchor=(1.0, 0.5),
        title='Perturbation Groups',
        fontsize=8,
        handlelength=1.5,
        handleheight=1.5,
        frameon=False
    )

    plt.tight_layout()
    plt.savefig(os.path.join(figure_dir, f"{pert_type}_correlation_color.svg"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{pert_type}_correlation_color.png"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{pert_type}_correlation_color.pdf"), dpi=300, bbox_inches='tight')
    plt.show()

plot_zcorr_colored(data, t=t)

In [ ]:
# Load drug annotation CSV
drug_anno = pd.read_csv(f'{paths.DATA_DIR}/gbm/EGFRi_annotations_final_RG.csv')

# The h5ad 'drug' column uses short names matching the CSV 'drug' column
# Try matching on short name first; fall back to full 'name' if needed
anno_cols = ['class_broad', 'reversible', 'specificity_RG', 'target_amino_acid_or_binding_site'	]

drug_meta_short = drug_anno.set_index('drug')[anno_cols]
drug_meta_full  = drug_anno.set_index('name')[anno_cols]

obs_drugs = sub.obs['drug'].unique()
n_short = sum(d in drug_meta_short.index for d in obs_drugs)
n_full  = sum(d in drug_meta_full.index  for d in obs_drugs)
print(f"Short-name matches: {n_short}/{len(obs_drugs)}")
print(f"Full-name  matches: {n_full}/{len(obs_drugs)}")

drug_meta = drug_meta_short if n_short >= n_full else drug_meta_full

for col in anno_cols:
    sub.obs[col] = sub.obs['drug'].map(drug_meta[col])
    # fill missing with 'Unknown' so scanpy treats it as a proper category
    sub.obs[col] = sub.obs[col].fillna('Unknown').astype('category')

print("\nAnnotation columns added to sub.obs:")
for col in anno_cols:
    n_unknown = (sub.obs[col] == 'Unknown').sum()
    print(f"  {col}: {sub.obs[col].nunique()} categories, {n_unknown} unmatched cells")

sub.obs[['drug'] + anno_cols].drop_duplicates('drug').sort_values('drug')


In [ ]:
for col in ['class_broad', 'reversible']:
    if hasattr(sub.obs[col], 'cat'):
        sub.obs[col] = sub.obs[col].cat.rename_categories({'Unknown': 'Unannotated'})
    else:
        sub.obs[col] = sub.obs[col].replace('Unknown', 'Unannotated')
        
    if col == 'reversible':
        if hasattr(sub.obs[col], 'cat'):
            sub.obs[col] = sub.obs[col].cat.rename_categories({
                'Yes': 'Reversible',
                'No': 'Covalent',
            })

### Fig. S2b: SHERLOCK latent space, by chemical class

In [ ]:
sc.pl.umap(
    sub,
    color='class_broad',
    title='Chemical class',
    size=12,
    frameon=False,
    show=False,
    legend_fontsize=10
)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir,'umap_pert_class_broad.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir,'umap_pert_class_broad.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir,'umap_pert_class_broad.svg'), dpi=300, bbox_inches='tight')
plt.show()

### Fig. S2b: SHERLOCK latent space, by reversibility

In [ ]:
sc.pl.umap(
    sub,
    color='reversible',
    title='Reversibility',
    size=12,
    frameon=False,
    show=False,
    legend_fontsize=14
)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir,'umap_pert_reversible.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir,'umap_pert_reversible.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir,'umap_pert_reversible.svg'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot UMAP coloured by each drug-annotation column
import math
ncols = 2
nrows = math.ceil(len(anno_cols) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows))
axes = axes.flatten()

for ax, col in zip(axes, anno_cols):
    sc.pl.umap(
        sub,
        color=col,
        title=col.replace('_', ' '),
        frameon=False,
        ax=ax,
        show=False,
        size=12,
        legend_fontsize=9,
    )

# hide unused axes
for ax in axes[len(anno_cols):]:
    ax.set_visible(False)

plt.tight_layout(pad=0.1)
for ext in ('svg', 'png', 'pdf'):
    plt.savefig(
        os.path.join(figure_dir, f'{pert_type}_{concentration}_drug_annotations.{ext}'),
        dpi=300, bbox_inches='tight'
    )
plt.show()


In [ ]:
# Overview: key grouping columns side-by-side
overview_cols = ['class_broad', 'reversible', 'specificity_RG', 'target_amino_acid_or_binding_site']

ncols = 2
nrows = math.ceil(len(overview_cols) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows))
axes = axes.flatten()

for ax, col in zip(axes, overview_cols):
    sc.pl.umap(
        sub,
        color=col,
        title=col.replace('_', ' '),
        frameon=False,
        ax=ax,
        show=False,
        size=10,
        legend_fontsize=9,
    )

for ax in axes[len(overview_cols):]:
    ax.set_visible(False)

plt.tight_layout(pad=0.1)
for ext in ('svg', 'png', 'pdf'):
    plt.savefig(
        os.path.join(figure_dir, f'{pert_type}_{concentration}_drug_anno_overview.{ext}'),
        dpi=300, bbox_inches='tight'
    )
plt.show()


In [ ]:
# Highlight Covalent and Tyrphostin drugs on the UMAP, rest in gray
umap_coords = sub.obsm['X_umap']
is_covalent = (sub.obs['reversible'] == 'Covalent').values
is_tyrphostin = (sub.obs['class_broad'] == 'tyrphostin').values
is_other = ~(is_covalent | is_tyrphostin)

fig, ax = plt.subplots(figsize=(2, 2))

# background cells: small, light, drawn first
ax.scatter(
    umap_coords[is_other, 0], umap_coords[is_other, 1],
    s=0.5, c='lightgray', linewidths=0, alpha=0.6, zorder=1,
)

# highlighted cells: larger, saturated, dark-outlined so they stay legible even shrunk down
ax.scatter(
    umap_coords[is_covalent, 0], umap_coords[is_covalent, 1],
    s=2, c='#ef476f', alpha=0.95, zorder=2, label='Covalent',
)
ax.scatter(
    umap_coords[is_tyrphostin, 0], umap_coords[is_tyrphostin, 1],
    s=2, c='#26547c', alpha=0.95, zorder=3, label='Tyrphostin',
)

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker='o', linestyle='none', markerfacecolor='#ef476f', markeredgewidth=0, markersize=6, label='Covalent'),
    Line2D([0], [0], marker='o', linestyle='none', markerfacecolor='#26547c', markeredgewidth=0, markersize=6, label='Tyrphostin'),
]
ax.legend(
    handles=legend_handles, loc='lower right', bbox_to_anchor=(1.2, 0.0),
    frameon=False, fontsize=13, handletextpad=0.3,
)

plt.tight_layout(pad=0.2)
for ext in ('svg', 'png', 'pdf'):
    plt.savefig(
        os.path.join(figure_dir, f'{pert_type}_{concentration}_covalent_tyrphostin_highlight.{ext}'),
        dpi=300, bbox_inches='tight',
    )
plt.show()


### Enrichment of drug classes in Sherlock perturbation groups (Fisher's exact test)

In [ ]:
overview_cols = ['class_broad',  'reversible']

In [ ]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import itertools

# ── build per-drug annotation table ──────────────────────────────────────────
# use one row per drug (not per cell) to avoid pseudo-replication
drug_obs = (
    sub.obs[['drug', 'pert_group'] + overview_cols]
    .drop_duplicates('drug')
    .dropna(subset=['pert_group'])
    .copy()
)
drug_obs['pert_group'] = drug_obs['pert_group'].astype(str)

groups = sorted(drug_obs['pert_group'].unique())
n_total = len(drug_obs)

rows = []
for col in overview_cols:
    cats = [c for c in drug_obs[col].dropna().unique() if c != 'Unknown']
    for cat in cats:
        in_cat = drug_obs[col] == cat
        for grp in groups:
            in_grp = drug_obs['pert_group'] == grp
            a = (in_cat &  in_grp).sum()   # in cat AND in group
            b = (in_cat & ~in_grp).sum()   # in cat, NOT in group
            c = (~in_cat &  in_grp).sum()  # NOT in cat, in group
            d = (~in_cat & ~in_grp).sum()  # neither
            _, p = fisher_exact([[a, b], [c, d]], alternative='greater')
            rows.append({'column': col, 'category': cat, 'group': grp,
                         'n_overlap': a, 'p_value': p})

df_enrich = pd.DataFrame(rows)

# FDR correction (Benjamini-Hochberg) across all tests
_, df_enrich['padj'], _, _ = multipletests(df_enrich['p_value'], method='fdr_bh')
df_enrich['neg_log10_padj'] = -np.log10(df_enrich['padj'].clip(1e-10))

sig = df_enrich[df_enrich['padj'] < 0.05].sort_values('padj')
print(f"Significant enrichments (FDR < 0.05): {len(sig)}")
sig


### Fig. S2c: enrichment of drug classes in SHERLOCK groups

In [ ]:
# ── heatmap: one panel per annotation column ─────────────────────────────────
# Color = -log10(FDR-adjusted p-value): brighter = more significant enrichment
# * marks FDR < 0.05

import matplotlib.cm as cm
import matplotlib.colors as mcolors

vmax = df_enrich['neg_log10_padj'].replace([np.inf], 10).max()
norm = mcolors.Normalize(vmin=0, vmax=vmax)
cmap = plt.get_cmap('YlOrRd')

fig, axes = plt.subplots(1, len(overview_cols),
                         figsize=(2.8 * len(overview_cols), 3.5))

for ax, col in zip(axes, overview_cols):
    pivot = (
        df_enrich[df_enrich['column'] == col]
        .pivot(index='category', columns='group', values='neg_log10_padj')
        .fillna(0)
    )
    pivot = pivot.loc[pivot.max(axis=1).sort_values(ascending=False).index]

    sns.heatmap(
        pivot,
        ax=ax,
        cmap='YlOrRd',
        vmin=0,
        vmax=vmax,
        linewidths=0,
        cbar=False,
        xticklabels=True,
        yticklabels=True,
    )

    # star significant cells
    for r, row_cat in enumerate(pivot.index):
        for c, grp in enumerate(pivot.columns):
            if pivot.loc[row_cat, grp] > 1.3:
                ax.text(c + 0.5, r + 0.5, '*', ha='center', va='center',
                        fontsize=10, color='black')

    ax.set_title(col.replace('_', ' '), fontsize=10)
    ax.set_xlabel('Sherlock group', fontsize=9)
    ax.set_ylabel('')
    ax.tick_params(axis='x', labelsize=8, rotation=90)
    ax.tick_params(axis='y', labelsize=8)

# shared colorbar on the right
fig.subplots_adjust(right=0.82)
cbar_ax = fig.add_axes([0.90, 0.15, 0.018, 0.65])
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('−log₁₀(FDR-adjusted p-value)', fontsize=9)
cbar.ax.tick_params(labelsize=8)
# mark the FDR < 0.05 threshold line
# cbar.ax.axhline(y=1.3, color='black', linewidth=1.2, linestyle='--')
# cbar.ax.text(2.8, 1.3, 'FDR=0.05', va='center', fontsize=7, color='black',
#              transform=cbar.ax.get_yaxis_transform())

for ext in ('svg', 'png', 'pdf'):
    plt.savefig(
        os.path.join(figure_dir,
                     f'{pert_type}_{concentration}_enrichment_heatmap.{ext}'),
        dpi=300, bbox_inches='tight'
    )
plt.show()


In [ ]:
sc.pp.normalize_total(sub)
sc.pp.log1p(sub)
sc.pp.pca(sub, random_state=0)

sc.pp.neighbors(sub, random_state=0)
sc.tl.umap(sub, random_state=0)

### Fig. S2a: UMAP of the expression space, by drug

In [ ]:
# Alternative: Save manually with more control
sc.pl.umap(
    sub,
    color='drug',
    title='drug',
    palette=palette,
    size=10,
    frameon=False,
    show=False,
    legend_fontsize=5,
)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir,'umap_pathway_comparison_pert_presherlock.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir,'umap_pathway_comparison_pert_presherlock.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir,'umap_pathway_comparison_pert_presherlock.svg'), dpi=300, bbox_inches='tight')
plt.show()

### Fig. S2a: UMAP of the expression space, by perturbation group

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

sc.pl.umap(
    sub,
    color='drug',
    title='Drug',
    palette=palette,
    size=12,
    frameon=False,
    ax=axes[0],
    show=False,
    legend_fontsize=6,
)

sc.pl.umap(sub, color='pert_group', title='Perturbation group', 
           palette=new_colormap, size=12, frameon=False, ax=axes[1], show=False)

plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f"{pert_type}_{concentration}_sherlock_zspace_presherlock.svg"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{pert_type}_{concentration}_sherlock_zspace_presherlock.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{pert_type}_{concentration}_sherlock_zspace_presherlock.pdf"), dpi=300, bbox_inches='tight')
plt.show()

## Fig. S3a–b: CRISPRa kinome screen

Drawn from the best CRISPRa run saved by [`scripts/kinome_crispri_crispra/2b_reproducibility_crispra.ipynb`](../scripts/kinome_crispri_crispra/2b_reproducibility_crispra.ipynb), as for Fig. 2i–j; the plotting cells are not yet part of that notebook.

## CRISPRi kinome screen: reproducibility

From [`scripts/kinome_crispri_crispra/2a_reproducibility_crispri.ipynb`](../scripts/kinome_crispri_crispra/2a_reproducibility_crispri.ipynb).

Reloads the five saved runs from `results/reproducibility_meta_bootstrap_only_v1_noconditions_v2/`; missing runs are trained.

# Running Sherlock on low-quality / low-cell-count data

This notebook runs Sherlock on a dataset where per-perturbation cell counts are too
low for a standard run. The tutorial below summarizes how the input data was
prepared and how the training hyperparameters were adjusted to compensate.

## 1. Creating metacells

- **Metacell size**: cells are aggregated in groups of **4** (`bootstrap_4`). This
  size was chosen empirically — it gave the most reasonable number of DEGs per
  perturbation without introducing too much noise.
- **Cells per perturbation/condition**: aim for **~30 cells per perturbation per
  condition** before aggregation, so each condition has a reasonable number of
  metacells to work with.
- **Perturbation selection**: only perturbations with **enough detectable DEGs**
  are kept — perturbations with too weak a signal are dropped, since the model
  can't learn a meaningful effect from noise.
- **Gene selection**: take the **top 10 DEGs per perturbation**, preferring genes
  that are **mostly unique to a single perturbation** (rather than DEGs shared
  across many perturbations). This keeps the gene space informative without
  diluting per-perturbation signal.

## 2. Parameter tuning for small data

- **More epochs**: with small/low-quality data, increase `num_epochs` — the model
  needs more iterations to converge on a weaker signal.
- **Don't overfit the classification head**: watch the auxiliary perturbation
  classification accuracy (`cls_head` / CE loss) during training — it can overfit
  quickly on small data.
- **Lower `ce_lambda` if accuracy rises too fast**: if classification accuracy
  climbs too quickly early in training, reduce `ce_lambda` so the auxiliary CE
  loss doesn't dominate/distort the latent space before the main reconstruction
  signal has a chance to shape it.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import scanpy as sc
import datetime
from itertools import combinations
from scipy.optimize import linear_sum_assignment
from scipy.stats import pearsonr, spearmanr

import sherlock as slk

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
pert_type = 'crispri'  # 'crispri' or 'crisprko'

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({'font.size': 12, 'svg.fonttype': 'none'})
mpl.rcParams['axes.titlesize'] = 16
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12
mpl.rcParams['axes.labelsize']  = 12
mpl.rcParams['lines.linewidth'] = 2

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_RUNS      = 5
N_CLUSTERS  = 6   # number of clusters for reproducibility analysis
RESULTS_DIR = f'{paths.RESULTS_DIR}/reproducibility_meta_bootstrap_only_v1_noconditions_v2'
figure_dir  = RESULTS_DIR + '/figures'

os.makedirs(figure_dir,              exist_ok=True)
os.makedirs(RESULTS_DIR + '/matrices', exist_ok=True)

# SHERLOCK_KWARGS = dict(
#     batch_size     = 4096,
#     #num_workers    = 0,
#     num_epochs     = 1000,
#     use_conditions = True,
#     validate_every = 20,
#     l0_lambda      = 20,
#     #l1_lambda      = 0.5,
#     #lr             = 1e-3,
#     patience       = 500,
#     #rho_init       = False,
#     #ntc_lambda     = 1.0,
#     latent_dim     = 16,
#     rank           = 6,
#     #n_epochs_kl_warmup = 10,
#     n_epochs_l0_warmup = 200
# ) 

print(f'Device: {DEVICE}')
print(f'Results: {RESULTS_DIR}')
print(f'N_RUNS={N_RUNS}  N_CLUSTERS={N_CLUSTERS}')

In [ ]:
SHERLOCK_KWARGS = dict(
    batch_size     = 4096,
    #num_workers    = 0,
    num_epochs     = 2500,
    use_conditions = False,
    validate_every = 20,
    l0_lambda      = 35,
    #l1_lambda      = 0.5,
    #lr             = 1e-3,
    patience       = 500,
    #rho_init       = False,
    #ntc_lambda     = 1.0,
    latent_dim     = 16,
    rank           = 6,
    ce_lambda       = 1,
    n_epochs_kl_warmup = 20,
    n_epochs_l0_warmup = 500
) 

In [ ]:
slk.configs.reset_config()
slk.configs.set_config('ntc_label', 'non-targeting')
slk.configs.show()

## Data loading & preprocessing

In [ ]:

adata_path = f'{paths.DATA_DIR}/gbm/filtered_gbm_{pert_type}_crop_top_bootstrap_4.h5ad'
adata = sc.read(adata_path)
print(adata)

In [ ]:


# Remap dose to treatment labels
mapping = {1.: '1to1', 0.5: '1to0.5', 0.25: '1to0.25', 0.: 'Untreated'}
adata.obs['treatment'] = adata.obs['dose'].replace(mapping)

print(f'After filtering: {adata.n_obs} cells, {adata.n_vars} genes')
print(f'Treatment labels: {sorted(adata.obs["treatment"].unique())}')

In [ ]:
adata = adata[adata.obs['treatment'] != 'Untreated']

In [ ]:
data = adata.copy()
data.obs.dose = data.obs.dose.astype(str)

slk.pp.slk_prepare_data(
    data, pert_key='gene_id', ntc_label='NTC', inplace=True, force=True
)

p_key     = slk.configs.get_config('pert_key')
NTC       = slk.configs.get_config('ntc_label')
c_key     = slk.configs.get_config('treatment_key')

print(f'Cells: {data.n_obs}   Genes: {data.n_vars}')
print(f'Perturbations (non-NTC): {data.obs[p_key].nunique() - 1}')
print(f'Conditions: {sorted(data.obs[c_key].unique())}')

In [ ]:
slk.pp.treat_effect(
    data, label_col='gene_id', control_label='NTC',
    method='perturbseq', inplace=True, key='treat_effect',
)
print('treat_effect computed.')

## Part 1 — Reproducibility loop

Train `N_RUNS` independent models with the same hyperparameters but different random seeds.
Set `use_rerun = True` to skip runs that have already been saved.

In [ ]:
use_rerun = True

for run in range(N_RUNS):
    out_path = f'{RESULTS_DIR}/matrices/run_{run}.npz'
    if os.path.exists(out_path) and use_rerun:
        print(f'Run {run+1}: already saved, skipping.')
        continue

    print(f"\n{'='*50}\nDeg-only run {run+1}/{N_RUNS}\n{'='*50}")
    out   = slk.tl.run_single(data, device=DEVICE, seed=run, **SHERLOCK_KWARGS, debug=True)
    model = out['model']

    slk.tl.eval_single(model, data, obsm_key='z', uns_key='results', device=DEVICE)
    res = data.uns['results']

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    slk.tl.save_results(out, f'{paths.RESULTS_DIR}/models/{pert_type}_degonly_model_{timestamp}_{run}.pth')
    print(f'Saved model to ../data/{pert_type}_degonly_model_{timestamp}_{run}.pth')

    W          = np.asarray(res['W'],         dtype=float)
    rho_corr   = np.asarray(res['rho_corr'],  dtype=float)
    z_corr     = np.asarray(res['z_corr'],    dtype=float)
    rho_perts  = np.asarray(res['rho_perts'])
    cond_names = list(res['conditions'])
    cf_sizes   = [np.asarray(cf, dtype=float) for cf in res['counterfactual_effect_size']]
    cfs_mat    = np.asarray(res['condition_perturbation_interaction'], dtype=float)

    # ATE: predicted vs observed treatment effect correlation
    ate    = model.predict_counterfactual_effects(
        data, n_centroids=25, observed_effect=data.uns['treat_effect'],
    )
    ate_r  = float(ate['corr'])
    pred_df = ate['predicted_df']

    # ev_sig qvals per condition
    for ci in range(len(cond_names)):
        slk.tl.ev_sig(data, cond_idx=ci)
        qvals = np.asarray(data.uns['results']['ev_results'][ci]['qvals'], dtype=float)
        np.save(f'{RESULTS_DIR}/matrices/run_{run}_evqvals_{ci}.npy', qvals)

    # Save matrices
    np.save(f'{RESULTS_DIR}/matrices/run_{run}_W.npy',        W)
    np.save(f'{RESULTS_DIR}/matrices/run_{run}_rho_corr.npy', rho_corr)
    np.save(f'{RESULTS_DIR}/matrices/run_{run}_z_corr.npy',   z_corr)
    np.savetxt(f'{RESULTS_DIR}/matrices/run_{run}_perts.txt',  rho_perts,  fmt='%s')
    np.savetxt(f'{RESULTS_DIR}/matrices/run_{run}_conds.txt',  cond_names, fmt='%s')
    np.save(f'{RESULTS_DIR}/matrices/run_{run}_cfs_mat.npy',  cfs_mat)
    np.savetxt(f'{RESULTS_DIR}/matrices/run_{run}_ate_r.txt',  [ate_r],    fmt='%.6f')
    np.save(f'{RESULTS_DIR}/matrices/run_{run}_ate_pred.npy', pred_df.values)
    np.savetxt(f'{RESULTS_DIR}/matrices/run_{run}_ate_perts.txt', list(pred_df.index), fmt='%s')
    for ci, cf in enumerate(cf_sizes):
        np.save(f'{RESULTS_DIR}/matrices/run_{run}_cf_{ci}.npy', cf)
    np.savez(out_path)  # sentinel

    # Keep the best-ATE model
    best_r_path = f'{RESULTS_DIR}/matrices/best_ate_r.txt'
    current_best = float(np.loadtxt(best_r_path)) if os.path.exists(best_r_path) else -np.inf
    if ate_r > current_best:
        slk.tl.save_results(out, f'{RESULTS_DIR}/matrices/best_model.pth')
        np.savetxt(best_r_path, [ate_r], fmt='%.6f')
        np.savetxt(f'{RESULTS_DIR}/matrices/best_run.txt', [run], fmt='%d')
        print(f'  New best model: ATE r={ate_r:.4f}')

    print(f'  Saved run {run+1} (ATE r={ate_r:.4f})')

print('\nTraining complete.')

## Part 2 — Load saved matrices

Re-entry point: run from here if matrices already exist.

In [ ]:
run_results = []
_ntc_labels = {'ntc', 'control', 'ctrl', 'non-targeting', 'dmso', 'vehicle'}

for run in range(N_RUNS):
    W          = np.load(f'{RESULTS_DIR}/matrices/run_{run}_W.npy')
    rho_corr   = np.load(f'{RESULTS_DIR}/matrices/run_{run}_rho_corr.npy')
    z_corr     = np.load(f'{RESULTS_DIR}/matrices/run_{run}_z_corr.npy')
    rho_perts  = np.loadtxt(f'{RESULTS_DIR}/matrices/run_{run}_perts.txt', dtype=str, ndmin=1)
    # Filter NTC from saved perts
    rho_perts  = np.array([p for p in rho_perts if p.lower() not in _ntc_labels])
    cond_names = list(np.loadtxt(f'{RESULTS_DIR}/matrices/run_{run}_conds.txt', dtype=str, ndmin=1))
    cfs_mat    = np.load(f'{RESULTS_DIR}/matrices/run_{run}_cfs_mat.npy')
    ate_r      = float(np.loadtxt(f'{RESULTS_DIR}/matrices/run_{run}_ate_r.txt'))
    ate_pred   = np.load(f'{RESULTS_DIR}/matrices/run_{run}_ate_pred.npy')
    ate_perts  = list(np.loadtxt(f'{RESULTS_DIR}/matrices/run_{run}_ate_perts.txt', dtype=str, ndmin=1))
    n_conds    = len(cond_names)
    cf_sizes   = [np.load(f'{RESULTS_DIR}/matrices/run_{run}_cf_{ci}.npy') for ci in range(n_conds)]
    ev_qvals   = [np.load(f'{RESULTS_DIR}/matrices/run_{run}_evqvals_{ci}.npy') for ci in range(n_conds)]

    run_results.append(dict(
        run=run, W=W, rho_corr=rho_corr, z_corr=z_corr,
        rho_perts=rho_perts, cond_names=cond_names,
        cfs_mat=cfs_mat, ate_r=ate_r, ate_pred=ate_pred,
        ate_perts=ate_perts, cf_sizes=cf_sizes, ev_qvals=ev_qvals,
    ))
    print(f'Run {run+1}: ATE r={ate_r:.4f}  W {W.shape}  perts {len(rho_perts)}')

all_perts   = sorted(run_results[0]['rho_perts'].tolist())
cond_names  = run_results[0]['cond_names']
all_ate_r   = [r['ate_r'] for r in run_results]
best_run_id = int(np.argmax(all_ate_r))

print(f'\nATE r per run: {[f"{r:.4f}" for r in all_ate_r]}')
print(f'Mean +/- Std: {np.mean(all_ate_r):.4f} +/- {np.std(all_ate_r):.4f}')
print(f'Best run: {best_run_id+1} (ATE r={all_ate_r[best_run_id]:.4f})')
print(f'Conditions: {cond_names}')

## Helper functions

In [ ]:
def align_groups_to_reference(ref_ng, tgt_ng):
    """Hungarian-match tgt group labels to ref group labels by shared membership."""
    common = set(ref_ng.index) & set(tgt_ng.index)
    if not common:
        return {}
    ref_labels = sorted(ref_ng['group'].unique())
    tgt_labels = sorted(tgt_ng['group'].unique())
    C = np.zeros((len(ref_labels), len(tgt_labels)), dtype=int)
    for gene in common:
        ri = ref_labels.index(ref_ng.loc[gene, 'group'])
        ti = tgt_labels.index(tgt_ng.loc[gene, 'group'])
        C[ri, ti] += 1
    row_ind, col_ind = linear_sum_assignment(C.max() - C)
    mapping = {tgt_labels[j]: ref_labels[i] for i, j in zip(row_ind, col_ind)}
    for tl in tgt_labels:
        if tl not in mapping:
            genes_in_tl = [g for g in tgt_ng.index if tgt_ng.loc[g, 'group'] == tl and g in common]
            if genes_in_tl:
                vote = {}
                for g in genes_in_tl:
                    rg = ref_ng.loc[g, 'group']
                    vote[rg] = vote.get(rg, 0) + 1
                mapping[tl] = max(vote, key=vote.get)
    return mapping


def cluster_all_runs(run_results, n_clusters, all_perts):
    clustered = []
    for rr in run_results:
        groups = slk.tl.cluster_rho(
            data, t=n_clusters, rho_corr=rr['z_corr'],
            criterion='maxclust', show=False,
        )
        named_groups = slk.tl.group2name(groups, list(rr['rho_perts']))
        clustered.append({**rr, 'groups': groups, 'named_groups': named_groups})
    return clustered


def compute_confidence(clustered_runs, all_perts):
    ref_ng     = clustered_runs[0]['named_groups']
    ref_groups = sorted(ref_ng['group'].unique())
    n_runs     = len(clustered_runs)

    aligned = {p: [] for p in all_perts}
    for rr in clustered_runs:
        mapping = ({g: g for g in ref_groups} if rr['run'] == 0
                   else align_groups_to_reference(ref_ng, rr['named_groups']))
        ng = rr['named_groups']
        for p in all_perts:
            if p in ng.index:
                aligned[p].append(mapping.get(ng.loc[p, 'group'], -1))
            else:
                aligned[p].append(-1)

    group_cols = [f'Group {g}' for g in ref_groups]
    rows = []
    for p in all_perts:
        asn  = aligned[p]
        row  = {f'Group {g}': asn.count(g) / n_runs for g in ref_groups}
        row['modal_group'] = f'Group {max(ref_groups, key=lambda g: asn.count(g))}'
        row['max_conf']    = max(asn.count(g) for g in ref_groups) / n_runs
        row['unassigned']  = asn.count(-1) / n_runs
        rows.append({'perturbation': p, **row})

    df = pd.DataFrame(rows).set_index('perturbation')
    return df.sort_values(['modal_group', 'max_conf'], ascending=[True, False]), group_cols


def jaccard(labels_i, labels_j):
    n      = len(labels_i)
    same_i = labels_i[:, None] == labels_i[None, :]
    same_j = labels_j[:, None] == labels_j[None, :]
    triu   = np.triu_indices(n, k=1)
    a, b   = same_i[triu], same_j[triu]
    union  = int((a | b).sum())
    return int((a & b).sum()) / union if union > 0 else np.nan


def compute_pairwise_similarity(run_results, clustered_runs, all_perts):
    """W co-activation, rho, and cluster Jaccard for every run pair."""
    n_runs = len(run_results)
    pairs  = list(combinations(range(n_runs), 2))
    w_corrs, rho_corrs, jac_raw, jac_aln = [], [], [], []

    for i, j in pairs:
        ri, rj = run_results[i], run_results[j]

        Ci   = ri['W'] @ ri['W'].T
        Cj   = rj['W'] @ rj['W'].T
        triu = np.triu_indices(Ci.shape[0], k=1)
        w_corrs.append(pearsonr(Ci[triu], Cj[triu])[0])

        fi, fj = ri['z_corr'][triu], rj['z_corr'][triu]
        valid  = np.isfinite(fi) & np.isfinite(fj)
        rho_corrs.append(pearsonr(fi[valid], fj[valid])[0])

        ci_r, cj_r = clustered_runs[i], clustered_runs[j]
        jac_raw.append(jaccard(ci_r['groups'].astype(object), cj_r['groups'].astype(object)))

        ng_i, ng_j = ci_r['named_groups'], cj_r['named_groups']
        common     = [p for p in all_perts if p in ng_i.index and p in ng_j.index]
        mapping_ij = align_groups_to_reference(ng_i, ng_j)
        mp_i = np.array([ng_i.loc[p, 'group'] for p in common], dtype=object)
        mp_j = np.array([mapping_ij.get(ng_j.loc[p, 'group'], -1) for p in common], dtype=object)
        valid_mask = mp_j != -1
        jac_aln.append(jaccard(mp_i[valid_mask], mp_j[valid_mask]))

    return pairs, w_corrs, rho_corrs, jac_raw, jac_aln

## Part 3 — W and rho reproducibility

For every run pair:
- **W co-activation Pearson r**: correlation of upper triangle of W @ W^T matrices
- **rho (z_corr) Pearson r**: correlation of perturbation correlation matrices

In [ ]:
clustered_runs = cluster_all_runs(run_results, N_CLUSTERS, all_perts)
pairs, w_corrs, rho_corrs, jac_raw, jac_aln = compute_pairwise_similarity(
    run_results, clustered_runs, all_perts
)

print('Pairwise W co-activation Pearson r:')
print(f'  Mean +/- Std: {np.mean(w_corrs):.4f} +/- {np.std(w_corrs):.4f}')
print(f'  Min / Max : {np.min(w_corrs):.4f} / {np.max(w_corrs):.4f}')

print('\nPairwise rho (z_corr) Pearson r:')
valid_rho = [v for v in rho_corrs if np.isfinite(v)]
print(f'  Mean +/- Std: {np.mean(valid_rho):.4f} +/- {np.std(valid_rho):.4f}')
print(f'  Min / Max : {np.min(valid_rho):.4f} / {np.max(valid_rho):.4f}')

## Part 4 — Clustering reproducibility (N_CLUSTERS = 6)

In [ ]:
confidence_df, group_cols = compute_confidence(clustered_runs, all_perts)
P_n      = len(all_perts)
pert_idx = {p: i for i, p in enumerate(all_perts)}

# Co-membership matrix
comembership = np.zeros((P_n, P_n))
for rr in clustered_runs:
    ng           = rr['named_groups']
    perts_in_run = [p for p in all_perts if p in ng.index]
    for pi in perts_in_run:
        gi = ng.loc[pi, 'group']
        for pj in perts_in_run:
            if ng.loc[pj, 'group'] == gi:
                comembership[pert_idx[pi], pert_idx[pj]] += 1
comembership /= N_RUNS
np.fill_diagonal(comembership, 1.0)

print(f'Clustering with N_CLUSTERS={N_CLUSTERS}:')
print(f'  Mean max-confidence : {confidence_df["max_conf"].mean():.3f} +/- {confidence_df["max_conf"].std():.3f}')
print(f'  Stable (>=0.8)      : {(confidence_df["max_conf"] >= 0.8).sum()} / {len(confidence_df)}')
print(f'  Pairwise Jaccard (aligned) : {np.nanmean(jac_aln):.3f} +/- {np.nanstd(jac_aln):.3f}')
print()
print(confidence_df.round(2).to_string())

### Fig. S3c (CRISPRi): group confidence across five runs

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, max(4, P_n * 0.12 + 2)))

# Confidence heatmap
sns.heatmap(
    confidence_df[group_cols], ax=axes[0],
    cmap='YlOrRd', vmin=0, vmax=1,
    linewidths=0.3, linecolor='lightgray',
    cbar_kws={'label': 'Fraction of runs'},
    annot=False, yticklabels=True,
)
axes[0].set_title(f'Group confidence\n(N_CLUSTERS={N_CLUSTERS})', fontsize=10)
axes[0].set_ylabel('Perturbation')
axes[0].tick_params(axis='y', labelsize=6)

# Co-membership heatmap
sort_order = confidence_df.index.tolist()
cm_plot    = pd.DataFrame(comembership, index=all_perts, columns=all_perts).loc[sort_order, sort_order]
sns.heatmap(
    cm_plot, ax=axes[1],
    cmap='Blues', vmin=0, vmax=1,
    xticklabels=False, yticklabels=False,
    cbar_kws={'label': 'Co-membership fraction'},
    square=True,
)
axes[1].set_title('Co-membership\nacross runs', fontsize=10)

# Reproducibility boxplot
for pos, (vals, title) in enumerate([
    (w_corrs,  'W co-act.\nPearson r'),
    (rho_corrs,'rho\nPearson r'),
    (jac_raw,  'Jaccard\n(raw)'),
    (jac_aln,  'Jaccard\n(aligned)'),
], 1):
    clean = [v for v in vals if np.isfinite(v)]
    axes[2].boxplot(clean, positions=[pos], widths=0.5, patch_artist=True,
                    boxprops=dict(facecolor='lightsteelblue', alpha=0.7))
    axes[2].scatter([pos]*len(clean), clean, color='steelblue', s=30, zorder=3, alpha=0.8)

axes[2].set_xticks(range(1, 5))
axes[2].set_xticklabels(['W co-act.\nPearson r', 'rho\nPearson r',
                          'Jaccard\n(raw)', 'Jaccard\n(aligned)'], fontsize=8)
axes[2].set_title('Pairwise reproducibility\nmetrics', fontsize=10)
axes[2].set_ylim(0, 1.05)
axes[2].grid(True, alpha=0.3)

fig.suptitle(f'Clustering reproducibility — Deg-only ({N_RUNS} runs, N_CLUSTERS={N_CLUSTERS})',
             y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/figures/clustering_reproducibility_k{N_CLUSTERS}.png',
            bbox_inches='tight', dpi=150)
plt.show()

# Save the confidence heatmap separately
fig_left, ax_left = plt.subplots(figsize=(6, max(4, P_n * 0.12 + 2)))
sns.heatmap(
    confidence_df[group_cols], ax=ax_left,
    cmap='YlOrRd', vmin=0, vmax=1,
    linewidths=0.3, linecolor='lightgray',
    cbar_kws={'label': 'Fraction of runs'},
    annot=False, yticklabels=True,
)
ax_left.set_title(f'Group confidence(N_CLUSTERS={N_CLUSTERS})', fontsize=10)
ax_left.set_ylabel('Perturbation')
ax_left.tick_params(axis='y', labelsize=6)
plt.tight_layout()
for ext in ['png', 'svg', 'pdf']:
    plt.savefig(f'{RESULTS_DIR}/figures/crisri_clustering_confidence_heatmap_k{N_CLUSTERS}.{ext}',
                bbox_inches='tight', dpi=300)
plt.show()


## CRISPRa kinome screen: reproducibility

From [`scripts/kinome_crispri_crispra/2b_reproducibility_crispra.ipynb`](../scripts/kinome_crispri_crispra/2b_reproducibility_crispra.ipynb).

# Running Sherlock on low-quality / low-cell-count data

This notebook runs Sherlock on a dataset where per-perturbation cell counts are too
low for a standard run. The tutorial below summarizes how the input data was
prepared and how the training hyperparameters were adjusted to compensate.

## 1. Creating metacells

- **Metacell size**: cells are aggregated in groups of **4** (`bootstrap_4`). This
  size was chosen empirically — it gave the most reasonable number of DEGs per
  perturbation without introducing too much noise.
- **Cells per perturbation/condition**: aim for **~30 cells per perturbation per
  condition** before aggregation, so each condition has a reasonable number of
  metacells to work with.
- **Perturbation selection**: only perturbations with **enough detectable DEGs**
  are kept — perturbations with too weak a signal are dropped, since the model
  can't learn a meaningful effect from noise.
- **Gene selection**: take the **top 10 DEGs per perturbation**, preferring genes
  that are **mostly unique to a single perturbation** (rather than DEGs shared
  across many perturbations). This keeps the gene space informative without
  diluting per-perturbation signal.

## 2. Parameter tuning for small data

- **More epochs**: with small/low-quality data, increase `num_epochs` — the model
  needs more iterations to converge on a weaker signal.
- **Don't overfit the classification head**: watch the auxiliary perturbation
  classification accuracy (`cls_head` / CE loss) during training — it can overfit
  quickly on small data.
- **Lower `ce_lambda` if accuracy rises too fast**: if classification accuracy
  climbs too quickly early in training, reduce `ce_lambda` so the auxiliary CE
  loss doesn't dominate/distort the latent space before the main reconstruction
  signal has a chance to shape it.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import scanpy as sc
import datetime
from itertools import combinations
from scipy.optimize import linear_sum_assignment
from scipy.stats import pearsonr, spearmanr

import sherlock as slk

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
pert_type = 'crispra'  # 'crispri' or 'crisprko'

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({'font.size': 12, 'svg.fonttype': 'none'})
mpl.rcParams['axes.titlesize'] = 16
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12
mpl.rcParams['axes.labelsize']  = 12
mpl.rcParams['lines.linewidth'] = 2

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_RUNS      = 5
N_CLUSTERS  = 6   # number of clusters for reproducibility analysis
RESULTS_DIR = f'{paths.RESULTS_DIR}/reproducibility_meta_bootstrap_only_v1_noconditions_v2_treated_crispra'
figure_dir  = RESULTS_DIR + '/figures'

os.makedirs(figure_dir,              exist_ok=True)
os.makedirs(RESULTS_DIR + '/matrices', exist_ok=True)

# SHERLOCK_KWARGS = dict(
#     batch_size     = 4096,
#     #num_workers    = 0,
#     num_epochs     = 1000,
#     use_conditions = True,
#     validate_every = 20,
#     l0_lambda      = 20,
#     #l1_lambda      = 0.5,
#     #lr             = 1e-3,
#     patience       = 500,
#     #rho_init       = False,
#     #ntc_lambda     = 1.0,
#     latent_dim     = 16,
#     rank           = 6,
#     #n_epochs_kl_warmup = 10,
#     n_epochs_l0_warmup = 200
# ) 

print(f'Device: {DEVICE}')
print(f'Results: {RESULTS_DIR}')
print(f'N_RUNS={N_RUNS}  N_CLUSTERS={N_CLUSTERS}')

In [ ]:
SHERLOCK_KWARGS = dict(
    batch_size     = 4096,
    #num_workers    = 0,
    num_epochs     = 2500,
    use_conditions = False,
    validate_every = 20,
    l0_lambda      = 35,
    #l1_lambda      = 0.5,
    #lr             = 1e-3,
    patience       = 500,
    #rho_init       = False,
    #ntc_lambda     = 1.0,
    latent_dim     = 16,
    rank           = 6,
    ce_lambda       = 1,
    n_epochs_kl_warmup = 20,
    n_epochs_l0_warmup = 500
)

In [ ]:
slk.configs.reset_config()
slk.configs.set_config('ntc_label', 'non-targeting')
slk.configs.show()

## Data loading & preprocessing

In [ ]:

adata_path = f'{paths.DATA_DIR}/gbm/filtered_gbm_{pert_type}_crop_top_bootstrap_6.h5ad'
adata = sc.read(adata_path)
print(adata)

In [ ]:


# Remap dose to treatment labels
mapping = {1.: '1to1', 0.5: '1to0.5', 0.25: '1to0.25', 0.: 'Untreated'}
adata.obs['treatment'] = adata.obs['dose'].replace(mapping)

print(f'After filtering: {adata.n_obs} cells, {adata.n_vars} genes')
print(f'Treatment labels: {sorted(adata.obs["treatment"].unique())}')

In [ ]:
adata = adata[adata.obs['treatment'] != 'Untreated']

In [ ]:
data = adata.copy()
data.obs.dose = data.obs.dose.astype(str)

slk.pp.slk_prepare_data(
    data, pert_key='gene_id', ntc_label='NTC', inplace=True, force=True
)

p_key     = slk.configs.get_config('pert_key')
NTC       = slk.configs.get_config('ntc_label')
c_key     = slk.configs.get_config('treatment_key')

print(f'Cells: {data.n_obs}   Genes: {data.n_vars}')
print(f'Perturbations (non-NTC): {data.obs[p_key].nunique() - 1}')
print(f'Conditions: {sorted(data.obs[c_key].unique())}')

In [ ]:
slk.pp.treat_effect(
    data, label_col='gene_id', control_label='NTC',
    method='perturbseq', inplace=True, key='treat_effect',
)
print('treat_effect computed.')

## Part 1 — Reproducibility loop

Train `N_RUNS` independent models with the same hyperparameters but different random seeds.
Set `use_rerun = True` to skip runs that have already been saved.

In [ ]:
use_rerun = True

for run in range(N_RUNS):
    out_path = f'{RESULTS_DIR}/matrices/run_{run}.npz'
    if os.path.exists(out_path) and use_rerun:
        print(f'Run {run+1}: already saved, skipping.')
        continue

    print(f"\n{'='*50}\nDeg-only run {run+1}/{N_RUNS}\n{'='*50}")
    out   = slk.tl.run_single(data, device=DEVICE, seed=run, **SHERLOCK_KWARGS, debug=True)
    model = out['model']

    slk.tl.eval_single(model, data, obsm_key='z', uns_key='results', device=DEVICE)
    res = data.uns['results']

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    slk.tl.save_results(out, f'{paths.RESULTS_DIR}/models/{pert_type}_degonly_model_{timestamp}_{run}.pth')
    print(f'Saved model to ../data/{pert_type}_degonly_model_{timestamp}_{run}.pth')

    W          = np.asarray(res['W'],         dtype=float)
    rho_corr   = np.asarray(res['rho_corr'],  dtype=float)
    z_corr     = np.asarray(res['z_corr'],    dtype=float)
    rho_perts  = np.asarray(res['rho_perts'])
    cond_names = list(res['conditions'])
    cf_sizes   = [np.asarray(cf, dtype=float) for cf in res['counterfactual_effect_size']]
    cfs_mat    = np.asarray(res['condition_perturbation_interaction'], dtype=float)

    # ATE: predicted vs observed treatment effect correlation
    ate    = model.predict_counterfactual_effects(
        data, n_centroids=25, observed_effect=data.uns['treat_effect'],
    )
    ate_r  = float(ate['corr'])
    pred_df = ate['predicted_df']

    # ev_sig qvals per condition
    for ci in range(len(cond_names)):
        slk.tl.ev_sig(data, cond_idx=ci)
        qvals = np.asarray(data.uns['results']['ev_results'][ci]['qvals'], dtype=float)
        np.save(f'{RESULTS_DIR}/matrices/run_{run}_evqvals_{ci}.npy', qvals)

    # Save matrices
    np.save(f'{RESULTS_DIR}/matrices/run_{run}_W.npy',        W)
    np.save(f'{RESULTS_DIR}/matrices/run_{run}_rho_corr.npy', rho_corr)
    np.save(f'{RESULTS_DIR}/matrices/run_{run}_z_corr.npy',   z_corr)
    np.savetxt(f'{RESULTS_DIR}/matrices/run_{run}_perts.txt',  rho_perts,  fmt='%s')
    np.savetxt(f'{RESULTS_DIR}/matrices/run_{run}_conds.txt',  cond_names, fmt='%s')
    np.save(f'{RESULTS_DIR}/matrices/run_{run}_cfs_mat.npy',  cfs_mat)
    np.savetxt(f'{RESULTS_DIR}/matrices/run_{run}_ate_r.txt',  [ate_r],    fmt='%.6f')
    np.save(f'{RESULTS_DIR}/matrices/run_{run}_ate_pred.npy', pred_df.values)
    np.savetxt(f'{RESULTS_DIR}/matrices/run_{run}_ate_perts.txt', list(pred_df.index), fmt='%s')
    for ci, cf in enumerate(cf_sizes):
        np.save(f'{RESULTS_DIR}/matrices/run_{run}_cf_{ci}.npy', cf)
    np.savez(out_path)  # sentinel

    # Keep the best-ATE model
    best_r_path = f'{RESULTS_DIR}/matrices/best_ate_r.txt'
    current_best = float(np.loadtxt(best_r_path)) if os.path.exists(best_r_path) else -np.inf
    if ate_r > current_best:
        slk.tl.save_results(out, f'{RESULTS_DIR}/matrices/best_model.pth')
        np.savetxt(best_r_path, [ate_r], fmt='%.6f')
        np.savetxt(f'{RESULTS_DIR}/matrices/best_run.txt', [run], fmt='%d')
        print(f'  New best model: ATE r={ate_r:.4f}')

    print(f'  Saved run {run+1} (ATE r={ate_r:.4f})')

print('\nTraining complete.')

## Part 2 — Load saved matrices

Re-entry point: run from here if matrices already exist.

In [ ]:
run_results = []
_ntc_labels = {'ntc', 'control', 'ctrl', 'non-targeting', 'dmso', 'vehicle'}

for run in range(N_RUNS):
    W          = np.load(f'{RESULTS_DIR}/matrices/run_{run}_W.npy')
    rho_corr   = np.load(f'{RESULTS_DIR}/matrices/run_{run}_rho_corr.npy')
    z_corr     = np.load(f'{RESULTS_DIR}/matrices/run_{run}_z_corr.npy')
    rho_perts  = np.loadtxt(f'{RESULTS_DIR}/matrices/run_{run}_perts.txt', dtype=str, ndmin=1)
    # Filter NTC from saved perts
    rho_perts  = np.array([p for p in rho_perts if p.lower() not in _ntc_labels])
    cond_names = list(np.loadtxt(f'{RESULTS_DIR}/matrices/run_{run}_conds.txt', dtype=str, ndmin=1))
    cfs_mat    = np.load(f'{RESULTS_DIR}/matrices/run_{run}_cfs_mat.npy')
    ate_r      = float(np.loadtxt(f'{RESULTS_DIR}/matrices/run_{run}_ate_r.txt'))
    ate_pred   = np.load(f'{RESULTS_DIR}/matrices/run_{run}_ate_pred.npy')
    ate_perts  = list(np.loadtxt(f'{RESULTS_DIR}/matrices/run_{run}_ate_perts.txt', dtype=str, ndmin=1))
    n_conds    = len(cond_names)
    cf_sizes   = [np.load(f'{RESULTS_DIR}/matrices/run_{run}_cf_{ci}.npy') for ci in range(n_conds)]
    ev_qvals   = [np.load(f'{RESULTS_DIR}/matrices/run_{run}_evqvals_{ci}.npy') for ci in range(n_conds)]

    run_results.append(dict(
        run=run, W=W, rho_corr=rho_corr, z_corr=z_corr,
        rho_perts=rho_perts, cond_names=cond_names,
        cfs_mat=cfs_mat, ate_r=ate_r, ate_pred=ate_pred,
        ate_perts=ate_perts, cf_sizes=cf_sizes, ev_qvals=ev_qvals,
    ))
    print(f'Run {run+1}: ATE r={ate_r:.4f}  W {W.shape}  perts {len(rho_perts)}')

all_perts   = sorted(run_results[0]['rho_perts'].tolist())
cond_names  = run_results[0]['cond_names']
all_ate_r   = [r['ate_r'] for r in run_results]
best_run_id = int(np.argmax(all_ate_r))

print(f'\nATE r per run: {[f"{r:.4f}" for r in all_ate_r]}')
print(f'Mean +/- Std: {np.mean(all_ate_r):.4f} +/- {np.std(all_ate_r):.4f}')
print(f'Best run: {best_run_id+1} (ATE r={all_ate_r[best_run_id]:.4f})')
print(f'Conditions: {cond_names}')

## Helper functions

In [ ]:
def align_groups_to_reference(ref_ng, tgt_ng):
    """Hungarian-match tgt group labels to ref group labels by shared membership."""
    common = set(ref_ng.index) & set(tgt_ng.index)
    if not common:
        return {}
    ref_labels = sorted(ref_ng['group'].unique())
    tgt_labels = sorted(tgt_ng['group'].unique())
    C = np.zeros((len(ref_labels), len(tgt_labels)), dtype=int)
    for gene in common:
        ri = ref_labels.index(ref_ng.loc[gene, 'group'])
        ti = tgt_labels.index(tgt_ng.loc[gene, 'group'])
        C[ri, ti] += 1
    row_ind, col_ind = linear_sum_assignment(C.max() - C)
    mapping = {tgt_labels[j]: ref_labels[i] for i, j in zip(row_ind, col_ind)}
    for tl in tgt_labels:
        if tl not in mapping:
            genes_in_tl = [g for g in tgt_ng.index if tgt_ng.loc[g, 'group'] == tl and g in common]
            if genes_in_tl:
                vote = {}
                for g in genes_in_tl:
                    rg = ref_ng.loc[g, 'group']
                    vote[rg] = vote.get(rg, 0) + 1
                mapping[tl] = max(vote, key=vote.get)
    return mapping


def cluster_all_runs(run_results, n_clusters, all_perts):
    clustered = []
    for rr in run_results:
        groups = slk.tl.cluster_rho(
            data, t=n_clusters, rho_corr=rr['z_corr'],
            criterion='maxclust', show=False,
        )
        named_groups = slk.tl.group2name(groups, list(rr['rho_perts']))
        clustered.append({**rr, 'groups': groups, 'named_groups': named_groups})
    return clustered


def compute_confidence(clustered_runs, all_perts):
    ref_ng     = clustered_runs[0]['named_groups']
    ref_groups = sorted(ref_ng['group'].unique())
    n_runs     = len(clustered_runs)

    aligned = {p: [] for p in all_perts}
    for rr in clustered_runs:
        mapping = ({g: g for g in ref_groups} if rr['run'] == 0
                   else align_groups_to_reference(ref_ng, rr['named_groups']))
        ng = rr['named_groups']
        for p in all_perts:
            if p in ng.index:
                aligned[p].append(mapping.get(ng.loc[p, 'group'], -1))
            else:
                aligned[p].append(-1)

    group_cols = [f'Group {g}' for g in ref_groups]
    rows = []
    for p in all_perts:
        asn  = aligned[p]
        row  = {f'Group {g}': asn.count(g) / n_runs for g in ref_groups}
        row['modal_group'] = f'Group {max(ref_groups, key=lambda g: asn.count(g))}'
        row['max_conf']    = max(asn.count(g) for g in ref_groups) / n_runs
        row['unassigned']  = asn.count(-1) / n_runs
        rows.append({'perturbation': p, **row})

    df = pd.DataFrame(rows).set_index('perturbation')
    return df.sort_values(['modal_group', 'max_conf'], ascending=[True, False]), group_cols


def jaccard(labels_i, labels_j):
    n      = len(labels_i)
    same_i = labels_i[:, None] == labels_i[None, :]
    same_j = labels_j[:, None] == labels_j[None, :]
    triu   = np.triu_indices(n, k=1)
    a, b   = same_i[triu], same_j[triu]
    union  = int((a | b).sum())
    return int((a & b).sum()) / union if union > 0 else np.nan


def compute_pairwise_similarity(run_results, clustered_runs, all_perts):
    """W co-activation, rho, and cluster Jaccard for every run pair."""
    n_runs = len(run_results)
    pairs  = list(combinations(range(n_runs), 2))
    w_corrs, rho_corrs, jac_raw, jac_aln = [], [], [], []

    for i, j in pairs:
        ri, rj = run_results[i], run_results[j]

        Ci   = ri['W'] @ ri['W'].T
        Cj   = rj['W'] @ rj['W'].T
        triu = np.triu_indices(Ci.shape[0], k=1)
        w_corrs.append(pearsonr(Ci[triu], Cj[triu])[0])

        fi, fj = ri['z_corr'][triu], rj['z_corr'][triu]
        valid  = np.isfinite(fi) & np.isfinite(fj)
        rho_corrs.append(pearsonr(fi[valid], fj[valid])[0])

        ci_r, cj_r = clustered_runs[i], clustered_runs[j]
        jac_raw.append(jaccard(ci_r['groups'].astype(object), cj_r['groups'].astype(object)))

        ng_i, ng_j = ci_r['named_groups'], cj_r['named_groups']
        common     = [p for p in all_perts if p in ng_i.index and p in ng_j.index]
        mapping_ij = align_groups_to_reference(ng_i, ng_j)
        mp_i = np.array([ng_i.loc[p, 'group'] for p in common], dtype=object)
        mp_j = np.array([mapping_ij.get(ng_j.loc[p, 'group'], -1) for p in common], dtype=object)
        valid_mask = mp_j != -1
        jac_aln.append(jaccard(mp_i[valid_mask], mp_j[valid_mask]))

    return pairs, w_corrs, rho_corrs, jac_raw, jac_aln

## Part 3 — W and rho reproducibility

For every run pair:
- **W co-activation Pearson r**: correlation of upper triangle of W @ W^T matrices
- **rho (z_corr) Pearson r**: correlation of perturbation correlation matrices

In [ ]:
clustered_runs = cluster_all_runs(run_results, N_CLUSTERS, all_perts)
pairs, w_corrs, rho_corrs, jac_raw, jac_aln = compute_pairwise_similarity(
    run_results, clustered_runs, all_perts
)

print('Pairwise W co-activation Pearson r:')
print(f'  Mean +/- Std: {np.mean(w_corrs):.4f} +/- {np.std(w_corrs):.4f}')
print(f'  Min / Max : {np.min(w_corrs):.4f} / {np.max(w_corrs):.4f}')

print('\nPairwise rho (z_corr) Pearson r:')
valid_rho = [v for v in rho_corrs if np.isfinite(v)]
print(f'  Mean +/- Std: {np.mean(valid_rho):.4f} +/- {np.std(valid_rho):.4f}')
print(f'  Min / Max : {np.min(valid_rho):.4f} / {np.max(valid_rho):.4f}')

## Part 4 — Clustering reproducibility (N_CLUSTERS = 6)

In [ ]:
confidence_df, group_cols = compute_confidence(clustered_runs, all_perts)
P_n      = len(all_perts)
pert_idx = {p: i for i, p in enumerate(all_perts)}

# Co-membership matrix
comembership = np.zeros((P_n, P_n))
for rr in clustered_runs:
    ng           = rr['named_groups']
    perts_in_run = [p for p in all_perts if p in ng.index]
    for pi in perts_in_run:
        gi = ng.loc[pi, 'group']
        for pj in perts_in_run:
            if ng.loc[pj, 'group'] == gi:
                comembership[pert_idx[pi], pert_idx[pj]] += 1
comembership /= N_RUNS
np.fill_diagonal(comembership, 1.0)

print(f'Clustering with N_CLUSTERS={N_CLUSTERS}:')
print(f'  Mean max-confidence : {confidence_df["max_conf"].mean():.3f} +/- {confidence_df["max_conf"].std():.3f}')
print(f'  Stable (>=0.8)      : {(confidence_df["max_conf"] >= 0.8).sum()} / {len(confidence_df)}')
print(f'  Pairwise Jaccard (aligned) : {np.nanmean(jac_aln):.3f} +/- {np.nanstd(jac_aln):.3f}')
print()
print(confidence_df.round(2).to_string())

### Fig. S3c (CRISPRa): group confidence across five runs

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, max(4, P_n * 0.12 + 2)))

# Confidence heatmap
sns.heatmap(
    confidence_df[group_cols], ax=axes[0],
    cmap='YlOrRd', vmin=0, vmax=1,
    linewidths=0.3, linecolor='lightgray',
    cbar_kws={'label': 'Fraction of runs'},
    annot=False, yticklabels=True,
)
axes[0].set_title(f'Group confidence\n(N_CLUSTERS={N_CLUSTERS})', fontsize=10)
axes[0].set_ylabel('Perturbation')
axes[0].tick_params(axis='y', labelsize=6)

# Co-membership heatmap
sort_order = confidence_df.index.tolist()
cm_plot    = pd.DataFrame(comembership, index=all_perts, columns=all_perts).loc[sort_order, sort_order]
sns.heatmap(
    cm_plot, ax=axes[1],
    cmap='Blues', vmin=0, vmax=1,
    xticklabels=False, yticklabels=False,
    cbar_kws={'label': 'Co-membership fraction'},
    square=True,
)
axes[1].set_title('Co-membership\nacross runs', fontsize=10)

# Reproducibility boxplot
for pos, (vals, title) in enumerate([
    (w_corrs,  'W co-act.\nPearson r'),
    (rho_corrs,'rho\nPearson r'),
    (jac_raw,  'Jaccard\n(raw)'),
    (jac_aln,  'Jaccard\n(aligned)'),
], 1):
    clean = [v for v in vals if np.isfinite(v)]
    axes[2].boxplot(clean, positions=[pos], widths=0.5, patch_artist=True,
                    boxprops=dict(facecolor='lightsteelblue', alpha=0.7))
    axes[2].scatter([pos]*len(clean), clean, color='steelblue', s=30, zorder=3, alpha=0.8)

axes[2].set_xticks(range(1, 5))
axes[2].set_xticklabels(['W co-act.\nPearson r', 'rho\nPearson r',
                          'Jaccard\n(raw)', 'Jaccard\n(aligned)'], fontsize=8)
axes[2].set_title('Pairwise reproducibility\nmetrics', fontsize=10)
axes[2].set_ylim(0, 1.05)
axes[2].grid(True, alpha=0.3)

fig.suptitle(f'Clustering reproducibility — Deg-only ({N_RUNS} runs, N_CLUSTERS={N_CLUSTERS})',
             y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/figures/clustering_reproducibility_k{N_CLUSTERS}.png',
            bbox_inches='tight', dpi=150)
plt.show()

# Save the confidence heatmap separately
fig_left, ax_left = plt.subplots(figsize=(6, max(4, P_n * 0.12 + 2)))
sns.heatmap(
    confidence_df[group_cols], ax=ax_left,
    cmap='YlOrRd', vmin=0, vmax=1,
    linewidths=0.3, linecolor='lightgray',
    cbar_kws={'label': 'Fraction of runs'},
    annot=False, yticklabels=True,
)
ax_left.set_title(f'Group confidence(N_CLUSTERS={N_CLUSTERS})', fontsize=10)
ax_left.set_ylabel('Perturbation')
ax_left.tick_params(axis='y', labelsize=6)
plt.tight_layout()
for ext in ['png', 'svg', 'pdf']:
    plt.savefig(f'{RESULTS_DIR}/figures/crisra_clustering_confidence_heatmap_k{N_CLUSTERS}.{ext}',
                bbox_inches='tight', dpi=300)
plt.show()


## 11 kinase inhibitors, 1 µM

From [`scripts/kinase_inhibitors_tcell/2_analysis_1uM.ipynb`](../scripts/kinase_inhibitors_tcell/2_analysis_1uM.ipynb).

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path

import seaborn as sns
import matplotlib.pyplot as plt

import torch

import sherlock as slk

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({'font.size': 12, 'svg.fonttype': 'none'})
mpl.rcParams['axes.titlesize'] = 16
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['lines.linewidth'] = 2
figure_dir = f"{paths.FIGURES_DIR}/kinase_inhibitors_tcell/"
os.makedirs(figure_dir, exist_ok=True)

In [ ]:
pert_type = 'drug'  # 'crispri' or 'crisprko'

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
slk.configs.show()
slk.configs.set_config('ntc_label', 'DMSO')
slk.configs.set_config('pert_key', 'drug')
slk.configs.set_config('treatment_key', 'tcell_treatment')
slk.configs.set_config('untreated_label', 'NoT')
slk.configs.show()
#slk.configs.reset_config()
slk.configs.get_config('ntc_label')


In [ ]:
slk.configs.get_config('ntc_label')

In [ ]:
concentration  = '1'
tcdg = 1000
run = 12

In [ ]:
data = sc.read_h5ad(f"{paths.DATA_DIR}/gbm/bt333_degs_tcell_{concentration}_{tcdg}.h5ad")

data.obs

In [ ]:
data.obs['concentration']

In [ ]:
# remove na from drug column
data = data[~data.obs['drug'].isna()].copy()
data = data[
    (data.obs['concentration'] == concentration) | (data.obs['drug'] == 'DMSO')
].copy()

In [ ]:
data.obs

In [ ]:
slk.pp.slk_prepare_data(data, pert_key="drug", ntc_label="DMSO", inplace=True, treatment_key="tcell_treatment", untreated_label="NoT")

In [ ]:
data.obs

In [ ]:
slk.pp.treat_effect(
    data,
    label_col = "drug",
    control_label = 'DMSO',
    method = "perturbseq",
    inplace=True,
    key='treat_effect'
)

In [ ]:
data

In [ ]:
data.X.max()

In [ ]:
# Check for NaN/Inf in expression matrix
print(f"Data shape: {data.shape}")
X = data.X.toarray() if hasattr(data.X, 'toarray') else data.X

n_nan = np.isnan(X).sum()
n_inf = np.isinf(X).sum()
print(f"NaN values in X: {n_nan}")
print(f"Inf values in X: {n_inf}")

if n_nan > 0 or n_inf > 0:
    print("Cleaning expression matrix...")
    # Replace NaN with 0 and clip Inf values
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    data.X = X
    print("Expression matrix cleaned!")

# Check data stats
print(f"X min: {data.X.min()}, max: {data.X.max()}")
print(f"Unique drugs: {data.obs['drug'].nunique()}")

In [ ]:
# DIAGNOSE: Check condition distribution
print("=== Condition Distribution ===")
print(f"\nTotal cells: {len(data)}")
print(f"\nUnique values in 'tcell_treatment':")
print(data.obs['tcell_treatment'].value_counts())

print(f"\n\nDMSO cells by condition:")
dmso_data = data[data.obs['drug']=='DMSO']
print(dmso_data.obs['tcell_treatment'].value_counts())

print(f"\n\nNon-DMSO cells by condition:")
non_dmso_data = data[data.obs['drug']!='DMSO']
print(non_dmso_data.obs['tcell_treatment'].value_counts())

# what the count for all drugs including DMSO
print(f"\n\nAll cells by drug:")
print(data.obs['drug'].value_counts())  

print(f"\n\n=== CRITICAL CHECK ===")
if len(dmso_data) == 0:
    print("❌ ERROR: NO DMSO CELLS FOUND!")
    print("   The model cannot train without control cells.")
else:
    print(f"✅ Found {len(dmso_data)} DMSO cells")
    
# Check if all conditions have DMSO
all_conditions = data.obs['tcell_treatment'].unique()
for cond in all_conditions:
    dmso_count = len(data[(data.obs['drug']=='DMSO') & (data.obs['tcell_treatment']==cond)])
    print(f"   Condition '{cond}': {dmso_count} DMSO cells")

In [ ]:
data

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# results = slk.tl.run_single(data, 
#                             batch_size=2048,
#                             num_workers=0, 
#                             device=device, 
#                             use_conditions=True,
#                             l0_lambda = 100,
#                             num_epochs=1000,
#                             #gate_init_p=0.4,
#                             H_lambda=0.01,
#                             ce_lambda=0.1,
#                             latent_dim= 4,
#                             rank = 6
#                             )

In [ ]:
# SHERLOCK_KWARGS = dict(
#     batch_size     = 4096,
#     #num_workers    = 0,
#     num_epochs     = 1000,
#     use_conditions = True,
#     validate_every = 20,
#     l0_lambda      = 200,
#     #l1_lambda      = 0.5,
#     #lr             = 1e-3,
#     patience       = 500,
#     #rho_init       = False,
#     #ntc_lambda     = 1.0,
#     latent_dim     = 8,
#     rank           = 3,
#     #n_epochs_kl_warmup = 10,
#     n_epochs_l0_warmup = 200
    
# )  # sweep winner: ATE=0.660, spearman=0.774, pearson=0.766, al_cp_rho=0.570, spec=1.29
# IF it drops to quickly lower l0, or icnrease n_epochs_l0_warmup, 
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SHERLOCK_KWARGS = dict(
    batch_size     = 4096,
    #num_workers    = 0,
    num_epochs     = 1000,
    use_conditions = True,
    validate_every = 20,
    l0_lambda      = 200,
    #l1_lambda      = 0.5,
    #lr             = 1e-3,
    patience       = 500,
    #rho_init       = False,
    #ntc_lambda     = 1.0,
    latent_dim     = 8,
    rank           = 3,
    #n_epochs_kl_warmup = 10,
    n_epochs_l0_warmup = 200
) 




In [ ]:
seed = 0

In [ ]:
# results = slk.tl.run_single(data, device=DEVICE, seed=seed, debug=True, **SHERLOCK_KWARGS)

In [ ]:
# import datetime
    
# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# slk.tl.save_results(results, f'{paths.RESULTS_DIR}/models/{pert_type}_model_{timestamp}_{seed}.pth')

# print(f'Saved results to ../data/{pert_type}_model_{timestamp}_{seed}.pth')

In [ ]:
#model = results['model']

In [ ]:
# Saved results to ../data/drug_model_20260525_185605_0.pth

In [ ]:
model = slk.tl.load_results(f'{paths.RESULTS_DIR}/models/drug_model_20260525_185605_0.pth')


In [ ]:
model

In [ ]:
slk.tl.eval_single(model, data, obsm_key="z", uns_key="results")

In [ ]:
data.uns['results']

In [ ]:
#data.X = data.X.toarray()
slk.pl.plot_r2(data)

In [ ]:
slk.pl.plot_corr(data)

In [ ]:
slk.pl.plot_ev(data, cmax=2, top_n=50)

In [ ]:
slk.tl.ev_sig(data, cond_idx=0)

In [ ]:
slk.tl.ev_sig(data, cond_idx=1)

In [ ]:
# Useful dataframe that plot_ev_sig uses internally
slk.pl.make_ev_long_df(data, cond_idx=0, uns_key="results")

In [ ]:
# Useful dataframe that plot_ev_sig uses internally
slk.pl.make_ev_long_df(data, cond_idx=1, uns_key="results")

In [ ]:
# ── AL vs CP cross-condition target gene correlation ────────────────────────
from scipy.stats import pearsonr, spearmanr

AL, CP      = 'ALWII4127', 'CP673451'
_ntc_labels = {'control', 'ctrl', 'non-targeting', 'dmso', 'vehicle'}

# counterfactual_effect_size shape: (C, P, G)
cfs      = np.asarray(data.uns['results']['counterfactual_effect_size'], dtype=float)
cond_names = list(data.uns['results']['conditions'])

# pert_names in model order (sorted, NTC removed)
_pnames = [p for p in np.unique(data.obs[slk.configs.get_config('pert_key')].values)
           if p.lower() not in _ntc_labels]
al_idx = _pnames.index(AL)
cp_idx = _pnames.index(CP)

print(f"counterfactual_effect_size shape: {cfs.shape}  (C={cfs.shape[0]}, P={cfs.shape[1]}, G={cfs.shape[2]})")
print(f"AL={AL} index={al_idx}   CP={CP} index={cp_idx}")
print(f"Conditions: {cond_names}\n")

# ── Per-condition Pearson/Spearman between AL and CP gene effect vectors ──
n_conds = len(cond_names)
fig, axes = plt.subplots(1, n_conds, figsize=(5 * n_conds, 4.5))
if n_conds == 1:
    axes = [axes]

for ci, (ax, cname) in enumerate(zip(axes, cond_names)):
    al_vec = cfs[ci, al_idx, :]   # (G,)
    cp_vec = cfs[ci, cp_idx, :]   # (G,)

    valid = np.isfinite(al_vec) & np.isfinite(cp_vec)
    r_p, p_p = pearsonr(al_vec[valid],  cp_vec[valid])
    r_s, p_s = spearmanr(al_vec[valid], cp_vec[valid])
    print(f"Condition: {cname}")
    print(f"  Pearson  r = {r_p:.3f}  (p = {p_p:.2e})")
    print(f"  Spearman r = {r_s:.3f}  (p = {p_s:.2e})\n")

    # scatter: one dot per gene
    ax.scatter(al_vec[valid], cp_vec[valid], s=6, alpha=0.3, color='steelblue', rasterized=True)

    # highlight genes significant for both (q < 0.05 in both AL and CP)
    if 'ev_results' in data.uns['results']:
        ev = data.uns['results']['ev_results']
        if ci < len(ev):
            qv = np.asarray(ev[ci]['qvals'], dtype=float)   # (P, G)
            both_sig = (qv[al_idx] < 0.05) & (qv[cp_idx] < 0.05) & valid
            ax.scatter(al_vec[both_sig], cp_vec[both_sig], s=20, alpha=0.8,
                       color='tomato', zorder=3, label=f'sig both (n={both_sig.sum()})')
            ax.legend(fontsize=8)

    # diagonal reference
    lim = max(np.abs(al_vec[valid]).max(), np.abs(cp_vec[valid]).max()) * 1.05
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.axvline(0, color='gray', lw=0.5, ls='--')
    ax.plot([-lim, lim], [-lim, lim], color='gray', lw=0.8, ls=':')
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel(f'{AL} effect size', fontsize=10)
    ax.set_ylabel(f'{CP} effect size', fontsize=10)
    ax.set_title(f'{cname}\nPearson r={r_p:.3f}  Spearman r={r_s:.3f}', fontsize=10)
    ax.set_aspect('equal')

plt.suptitle('AL vs CP counterfactual gene effects per condition', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f'AL_CP_cross_condition_correlation.svg'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f'AL_CP_cross_condition_correlation.png'), dpi=300, bbox_inches='tight')
plt.show()

# ── Heatmap: all drug-pair Pearson r per condition ────────────────────────
fig, axes = plt.subplots(1, n_conds, figsize=(5 * n_conds, 4.5))
if n_conds == 1:
    axes = [axes]

for ci, (ax, cname) in enumerate(zip(axes, cond_names)):
    P = len(_pnames)
    r_mat = np.full((P, P), np.nan)
    for pi in range(P):
        for pj in range(pi, P):
            vi = cfs[ci, pi, :]
            vj = cfs[ci, pj, :]
            v  = np.isfinite(vi) & np.isfinite(vj)
            if v.sum() > 2:
                r, _ = pearsonr(vi[v], vj[v])
                r_mat[pi, pj] = r_mat[pj, pi] = r
    np.fill_diagonal(r_mat, 1.0)

    im = ax.imshow(r_mat, cmap='coolwarm', vmin=-1, vmax=1)
    ax.set_xticks(range(P)); ax.set_xticklabels(_pnames, rotation=90, fontsize=7)
    ax.set_yticks(range(P)); ax.set_yticklabels(_pnames, fontsize=7)
    ax.set_title(f'{cname}\nPearson r of CF gene effects', fontsize=10)
    plt.colorbar(im, ax=ax, label='Pearson r', fraction=0.046, pad=0.04)
    # highlight AL-CP cell
    ax.add_patch(plt.Rectangle((cp_idx - 0.5, al_idx - 0.5), 1, 1,
                                fill=False, edgecolor='black', lw=2))
    ax.add_patch(plt.Rectangle((al_idx - 0.5, cp_idx - 0.5), 1, 1,
                                fill=False, edgecolor='black', lw=2))

plt.suptitle('All-drug CF gene effect correlation per condition', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f'drug_pair_CF_correlation_heatmap.svg'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f'drug_pair_CF_correlation_heatmap.png'), dpi=300, bbox_inches='tight')
plt.show()


### Fig. S5a: significance of downstream effects per condition (1 µM)

In [ ]:
# --- Publication-ready wrapper: all conditions ---
# figure_dir is defined in cell 1
FIGSIZE = (2, 16)
DENDRO_RATIO = 0.03

from unittest.mock import patch

cond_labels = data.uns['results']['conditions']
n_conds = len(data.uns['results']['counterfactual_effect_size'])

_orig_clustermap = sns.clustermap

def _make_patched(captured):
    def _patched_clustermap(*args, **kwargs):
        data_T = args[0].T
        args = (data_T,) + args[1:]
        row_cluster = kwargs.pop('row_cluster', True)
        col_cluster = kwargs.pop('col_cluster', False)
        kwargs['row_cluster'] = col_cluster
        kwargs['col_cluster'] = row_cluster
        kwargs['figsize'] = FIGSIZE
        kwargs['dendrogram_ratio'] = (0.1, DENDRO_RATIO)
        kwargs['cmap'] = 'viridis'
        kwargs['linewidths'] = 0
        kwargs['cbar_pos'] = (1.38, 0.15, 0.05, 0.3)
        g = _orig_clustermap(*args, **kwargs)
        captured['g'] = g
        return g
    return _patched_clustermap

for cond_idx in range(n_conds):
    cond_label = cond_labels[cond_idx]
    print(f'Plotting condition {cond_idx}: {cond_label}')
    _captured = {}
    with patch.object(sns, 'clustermap', _make_patched(_captured)):
        with patch.object(plt, 'show', lambda: None):
            slk.pl.plot_ev_sig(
                data,
                cond_idx=cond_idx,
                uns_key='results',
                alpha=0.05,
                top_n_genes=100,
                min_sig_genes=3,
                cluster_rows=True,
                cluster_cols=False,
                use_clustermap=True,
            )
    if 'g' not in _captured:
        print(f'  -> No significant results, skipping.')
        continue
    g = _captured['g']
    n_genes = g.data2d.shape[0]
    g.ax_heatmap.set_title('')
    g.ax_heatmap.set_xticks(range(g.data2d.shape[1]))
    g.ax_heatmap.set_xticklabels(list(g.data2d.columns), fontsize=8, rotation=90)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=10, rotation=0)
    g.ax_heatmap.set_xlabel(f'{concentration}um | {cond_label}', fontsize=10)
    g.ax_heatmap.set_ylabel(f'{n_genes} genes', fontsize=10)
    g.cax.set_ylabel('-log10(q-value)', fontsize=9)
    pos = g.ax_heatmap.get_position()
    new_x0 = 0.0
    new_width = 0.88
    g.ax_heatmap.set_position([new_x0, pos.y0, new_width, pos.height])
    dcol = g.ax_col_dendrogram.get_position()
    g.ax_col_dendrogram.set_position([new_x0, dcol.y0, new_width, dcol.height])
    save_name = f'figure_ev_sig_{concentration}um_{tcdg}_{cond_label}'
    for ext in ('svg', 'png', 'pdf'):
        g.fig.savefig(os.path.join(figure_dir, f'{save_name}.{ext}'), dpi=300, bbox_inches='tight')
    # Save q-value matrix as CSV (perturbation x gene)
    qval_matrix = 10 ** (-g.data2d)
    qval_matrix.index.name = 'perturbation'
    qval_matrix.columns.name = 'gene'
    qval_matrix.to_csv(os.path.join(figure_dir, f'{save_name}_qvalues.csv'))
    print(f'  -> Saved to {figure_dir}: {save_name}.svg/png/pdf + _qvalues.csv')
    plt.show()
# cluster on effect size, and this top 100 genes. Cluster on the whole gene list. 

In [ ]:
# Get condition information
cond_labels = data.uns['results']['conditions']
n_conds = len(data.uns['results']['counterfactual_effect_size'])

# Save filtered significant results per condition
for cond_idx in range(n_conds):
    cond_label = cond_labels[cond_idx]
    slk.tl.ev_sig(data, cond_idx=cond_idx)
    # Get long-form data for this condition
    df_sig = slk.pl.make_ev_long_df(data, cond_idx=cond_idx)
    
    # Filter for significant q-values only
    df_sig = df_sig[df_sig["qval"] < 0.05]
    
    # Skip if no significant results
    if df_sig.empty:
        print(f"Condition {cond_idx} ({cond_label}): No significant results (q < 0.05)")
        continue
    
    # Create filename
    save_name = f'filtered_significant_{pert_type}_{concentration}_{cond_label}.csv'
    save_path = os.path.join(figure_dir, save_name)
    
    # Save to CSV
    df_sig.to_csv(save_path, index=False)
    
    print(f"Condition {cond_idx} ({cond_label}): {len(df_sig)} significant pairs saved")
    print(f"  → {save_path}\n")

In [ ]:
# Genes significantly regulated by both ALWII4127 (AL) and CP673451 (CP)
# Uses the ev_sig results already computed for cond_idx=1

alpha = 0.05
cond_idx = 1

df_ev = slk.pl.make_ev_long_df(data, cond_idx=cond_idx, uns_key='results')
df_ev_sig = df_ev[df_ev['qval'] < alpha]

genes_AL = set(df_ev_sig[df_ev_sig['pert'] == 'ALWII4127']['gene'])
genes_CP = set(df_ev_sig[df_ev_sig['pert'] == 'CP673451']['gene'])

shared_genes = genes_AL & genes_CP

print(f'Significant genes in ALWII4127 : {len(genes_AL)}')
print(f'Significant genes in CP673451  : {len(genes_CP)}')
print(f'Shared genes                   : {len(shared_genes)}')
print()

# Show shared genes with EV and q-value for each drug
df_shared = df_ev_sig[df_ev_sig['gene'].isin(shared_genes) &
                       df_ev_sig['pert'].isin(['ALWII4127', 'CP673451'])]
df_shared = (df_shared[['pert', 'gene', 'ev', 'qval']]
             .pivot(index='gene', columns='pert', values=['ev', 'qval'])
             .sort_values(('ev', 'ALWII4127'), ascending=False))
df_shared.columns = ['ev_AL', 'ev_CP', 'qval_AL', 'qval_CP']
print(df_shared.to_string())
df_shared.to_csv(os.path.join(figure_dir, f'shared_AL_CP_{concentration}um_with_T_cell_conditions.csv'))


In [ ]:
# Genes significantly regulated by BOTH AL and CP, but NO other drug

# Significant genes per drug (all drugs)
genes_per_drug = df_ev_sig.groupby('pert')['gene'].apply(set).to_dict()

genes_other = set()
for drug, genes in genes_per_drug.items():
    if drug not in ('ALWII4127', 'CP673451'):
        genes_other |= genes

exclusive_genes = shared_genes - genes_other

print(f'Shared AL+CP genes          : {len(shared_genes)}')
print(f'Also in other drugs         : {len(shared_genes & genes_other)}')
print(f'Exclusive to AL+CP only     : {len(exclusive_genes)}')
print()

df_exclusive = df_ev_sig[df_ev_sig['gene'].isin(exclusive_genes) &
                          df_ev_sig['pert'].isin(['ALWII4127', 'CP673451'])]
df_exclusive = (df_exclusive[['pert', 'gene', 'ev', 'qval']]
                .pivot(index='gene', columns='pert', values=['ev', 'qval'])
                .sort_values(('ev', 'ALWII4127'), ascending=False))
df_exclusive.columns = ['ev_AL', 'ev_CP', 'qval_AL', 'qval_CP']
print(df_exclusive.to_string())
df_exclusive.to_csv(os.path.join(figure_dir, f'exclusive_AL_CP_{concentration}um_with_T_cell_conditions.csv'))


#immune / antigen-presentation genes
These are the easiest to interpret and strongest for your story.
STAT1 — core interferon-response transcription factor
HLA-E — nonclassical MHC-I, immune recognition
UBE2L6 — interferon-inducible ubiquitin-conjugating enzyme
LY6E — interferon-stimulated / immune-modulatory gene
PSME1 — immunoproteasome/antigen-processing related
DDX60 — antiviral innate immune sensor/ISG
HLA-DMA — MHC-II peptide loading
ERAP1 — trims peptides for MHC-I presentation
PSMB9 — immunoproteasome subunit
HLA-C — classical MHC-I
PSMB8 — immunoproteasome subunit

#Adhesion / membrane / tumor–immune interface
These are also very relevant biologically.
EMP1 — membrane protein, often linked to invasive/mesenchymal states
ITGA6 — integrin alpha-6, adhesion / stem-like and invasive behavior
ACTN1 — actin cytoskeleton / adhesion
LOXL1 — extracellular matrix remodeling
SVIL — actin-associated cytoskeletal protein
SH3D19 — membrane trafficking / signaling scaffold
LSAMP — cell adhesion molecule
COL6A1 — ECM / mesenchymal matrix
CAVIN1 — caveolae / membrane organization
AHNAK — large scaffold protein, membrane/cytoskeleton
ITGAV — integrin alpha-V, ECM interaction / migration
ANXA2 — membrane organization, invasion, fibrinolysis

#Cytoskeleton / trafficking / migration
CLASP2 — microtubule stabilization
CLIP1 — microtubule-binding trafficking protein
NAV2 — cytoskeletal guidance / migration-related
EPN2 — endocytosis
TMSB10 — actin dynamics
ARHGEF7 — Rho GTPase signaling / cytoskeleton
ARHGEF9 — Rho signaling scaffold
CALCOCO2 — autophagy cargo receptor, also stress response

#Neural / lineage / glioma-state genes
These are important because they suggest tumor-state remodeling, not just immune activation.
APBB2 — neuronal adaptor-related
PBX1 — developmental transcription factor
KANTR — lncRNA near developmental programs
SNHG14 — lncRNA often linked to neural/glioma contexts
FRMPD3 — neuronal/synaptic-associated scaffold
LDLRAD3 — neuronal membrane-associated receptor family
LRRC4C — neuronal adhesion molecule
DNPEP — aminopeptidase, less specific but seen in neural contexts
GUK1 — nucleotide metabolism, not lineage-specific
S100A16 / S100A13 — stress / calcium-binding proteins, can mark reactive states

In [ ]:
p_key = slk.configs.get_config('pert_key')
NTC = slk.configs.get_config('ntc_label')

sub = data[data.obs[p_key] != NTC]

sc.pp.neighbors(sub, n_neighbors=15, use_rep='z', random_state=0)
sc.tl.umap(sub, random_state=0)

import random
n_cats = sub.obs[p_key].nunique()
palette = sns.color_palette("hsv", n_colors=n_cats)
random.seed(0)
random.shuffle(palette)
sc.pl.umap(sub, color='drug', palette=palette, frameon=False)

# save figures pdf, png, svg

In [ ]:
palette = sns.color_palette("hsv", n_colors=2)

In [ ]:
sc.pl.umap(sub, color='tcell_treatment', palette=palette, frameon=False)

In [ ]:
zrc = data.uns['results']['z_rho_corr']

print(f"Spearman ρ = {zrc['spearman_r']:.3f}  (p = {zrc['spearman_p']:.1e})")
print(f" Pearson  r = {zrc['pearson_r']:.3f}  (p = {zrc['pearson_p']:.1e})") 

In [ ]:
slk.pl.plot_zcorr(data)

In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list

def plot_zcorr_with_names(adata, uns_key='results'):
    # Get config values
    pert_key = slk.configs.get_config('pert_key')
    ntc_label = slk.configs.get_config('ntc_label')
    
    # Get perturbation names in the same order as the model (sorted unique, excluding NTC)
    # This matches the logic in PerturbMatchingDataset
    pert_vals = adata.obs[pert_key].values
    pert_vals = pert_vals[pert_vals != ntc_label]
    pert_names = np.unique(pert_vals)
    
    # Get matrices
    Sigma_P_posterior = adata.uns[uns_key]['rho_corr']
    Sigma_z = adata.uns[uns_key]['z_corr']

    # Compute clustering order
    link = linkage(Sigma_P_posterior, method="average", metric="euclidean")
    order = leaves_list(link)              # permutation indices, shape (P,)

    # Reorder matrices
    SigmaP_ord = Sigma_P_posterior[order][:, order]
    Sigmaz_ord = Sigma_z[order][:, order]
    
    # Reorder names
    pert_names_ord = pert_names[order]

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

    sns.heatmap(SigmaP_ord, cmap="coolwarm", square=True, ax=axes[0], 
                xticklabels=pert_names_ord, yticklabels=pert_names_ord)
    axes[0].set_title(r"$\Sigma_P$  (posterior row covariance)")
    axes[0].set_xlabel("perturbation")
    axes[0].set_ylabel("perturbation")

    sns.heatmap(Sigmaz_ord, cmap="coolwarm", square=True, ax=axes[1],
                xticklabels=pert_names_ord, yticklabels=pert_names_ord)
    axes[1].set_title("Correlation of mean $z$")
    axes[1].set_xlabel("perturbation")

    plt.tight_layout()
    plt.savefig(os.path.join(figure_dir, f"{concentration}_correlation.svg"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{concentration}_correlation.png"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{concentration}_correlation.pdf"), dpi=300, bbox_inches='tight')
    plt.show()

plot_zcorr_with_names(data)

In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list

def plot_zcorr_with_names(adata, uns_key='results'):
    # Get config values
    pert_key = slk.configs.get_config('pert_key')
    ntc_label = slk.configs.get_config('ntc_label')
    
    # Get perturbation names in the same order as the model (sorted unique, excluding NTC)
    # This matches the logic in PerturbMatchingDataset
    pert_vals = adata.obs[pert_key].values
    pert_vals = pert_vals[pert_vals != ntc_label]
    pert_names = np.unique(pert_vals)
    
    # Get matrices
    Sigma_P_posterior = adata.uns[uns_key]['rho_corr']

    # Compute clustering order
    link = linkage(Sigma_P_posterior, method="average", metric="euclidean")
    order = leaves_list(link)              # permutation indices, shape (P,)

    # Reorder matrices
    SigmaP_ord = Sigma_P_posterior[order][:, order]
    
    # Reorder names
    pert_names_ord = pert_names[order]

    # Plot only left heatmap
    fig, ax = plt.subplots(figsize=(8, 6))

    sns.heatmap(SigmaP_ord, cmap="coolwarm", square=True, ax=ax, 
                xticklabels=pert_names_ord, yticklabels=pert_names_ord)
    ax.set_title(r"$\Sigma_P$  (posterior row covariance)")
    #ax.set_xlabel("perturbation")
    #ax.set_ylabel("perturbation")

    plt.tight_layout()
    plt.savefig(os.path.join(figure_dir, f"{concentration}_correlation.svg"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{concentration}_correlation.png"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{concentration}_correlation.pdf"), dpi=300, bbox_inches='tight')
    plt.show()

plot_zcorr_with_names(data)

In [ ]:
W = data.uns['results']['W']
plt.hist(W.flatten(), bins=30)

# save this figure
plt.savefig(os.path.join(figure_dir, f"{concentration}_W_histogram.svg"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_W_histogram.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_W_histogram.pdf"), dpi=300, bbox_inches='tight')     
plt.show()

In [ ]:
W_bin = (W > 0.5).astype(float)
sns.heatmap(W, cmap="coolwarm")

# save this figure
plt.savefig(os.path.join(figure_dir, f"{concentration}_W_heatmap.svg"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_W_heatmap.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_W_heatmap.pdf"), dpi=300, bbox_inches='tight')     

In [ ]:
dataset = slk.tl.PerturbMatchingDataset(data)
all_genes = dataset.perturbation_dict.keys()

In [ ]:
groups = slk.tl.cluster_rho(data, t=1, show=True)

In [ ]:
new_colormap = [
    'darkslateblue',  # 0
    'firebrick',      # 1
    'darkorange',     # 2
    'forestgreen',    # 3
    'gold',           # 4
    'dodgerblue',     # 5
    'mediumorchid',   # 6
    'deeppink',       # 7
    'teal',           # 8
    'black'           # 9
]

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, set_link_color_palette
from matplotlib.colors import to_hex

def cluster_rho(adata, t=1.5, uns_key='results', rho_corr=None, show=True, pert_key=None,
                group_together=None, figure_dir=None, save_name=None):
    if pert_key is None:
        pert_key = slk.configs.get_config('pert_key')
    if rho_corr is None:
        C = adata.uns[uns_key]['rho_corr']
    else:
        C = rho_corr

    # Get perturbation names - try multiple possible locations
    if 'pert_names' in adata.uns.get(uns_key, {}):
        pert_names = adata.uns[uns_key]['pert_names']
    elif 'perts' in adata.uns.get(uns_key, {}):
        pert_names = adata.uns[uns_key]['perts']
    else:
        pert_names = np.unique(adata.obs[pert_key].values).tolist()
        _ntc_labels = {'control', 'ctrl', 'non-targeting', 'dmso', 'vehicle'}
        pert_names = [p for p in pert_names if p.lower() not in _ntc_labels]

    D = 1.0 - C  # distance
    Z_link = linkage(D[np.triu_indices_from(D, 1)], method='ward')

    # Auto-raise t so that all group_together perturbations land in the same cluster
    if group_together:
        name_to_idx = {name: i for i, name in enumerate(pert_names)}
        target_idxs = {name_to_idx[n] for n in group_together if n in name_to_idx}
        if len(target_idxs) >= 2:
            n = len(pert_names)
            clusters = {i: {i} for i in range(n)}
            merge_height = t
            for k, (left, right, dist, _) in enumerate(Z_link):
                left, right = int(left), int(right)
                merged = clusters[left] | clusters[right]
                clusters[n + k] = merged
                if target_idxs.issubset(merged):
                    merge_height = float(dist)
                    break
            if merge_height > t:
                print(f'Adjusting t: {t} -> {merge_height:.4f} to group {group_together} together')
                t = merge_height + 1e-6

    if show:
        plt.figure(figsize=(max(3, len(pert_names) * 0.15), 4))
        _ug = sorted(set(fcluster(Z_link, t=t, criterion='distance')))
        _pal = [to_hex(new_colormap[i % len(new_colormap)]) for i in range(len(_ug))]
        set_link_color_palette(_pal)
        ddata_vis = dendrogram(
            Z_link,
            color_threshold=t,
            above_threshold_color="gray",
            labels=pert_names,
            leaf_rotation=90,
            leaf_font_size=8,
        )
        plt.axhline(y=t, color="#2CA02C", linestyle="--", linewidth=1)
        if 'Tyrphostin9' in list(pert_names):
            _t9_pos = ddata_vis['ivl'].index('Tyrphostin9') if 'Tyrphostin9' in ddata_vis['ivl'] else None
            if _t9_pos is not None:
                _t9_x = 5 + _t9_pos * 10
                for line in plt.gca().get_lines():
                    if any(abs(x - _t9_x) < 0.1 for x in line.get_xdata()):
                        line.set_color('#2CA02C')
        plt.xlabel('Perturbations')
        plt.ylabel('Distance')
        plt.tight_layout()
        if figure_dir and save_name:
            plt.savefig(os.path.join(figure_dir, f'{save_name}_dendrogram.svg'), dpi=300, bbox_inches='tight')
            plt.savefig(os.path.join(figure_dir, f'{save_name}_dendrogram.png'), dpi=300, bbox_inches='tight')
            plt.savefig(os.path.join(figure_dir, f'{save_name}_dendrogram.pdf'), dpi=300, bbox_inches='tight')
            print(f'Saved figures to {figure_dir}/{save_name}_dendrogram.[svg|png|pdf]')
        plt.show()

    groups = fcluster(Z_link, t=t, criterion='distance')

    unique_groups = sorted(set(groups))
    palette_hex = [to_hex(new_colormap[i % len(new_colormap)]) for i in range(len(unique_groups))]
    set_link_color_palette(palette_hex)
    ddata = dendrogram(Z_link, color_threshold=t, above_threshold_color='gray', no_plot=True)
    leaf_colors = ddata.get('leaves_color_list', [])
    group_to_color = {}
    for leaf_idx, color in zip(ddata['leaves'], leaf_colors):
        g = groups[leaf_idx]
        if g not in group_to_color:
            group_to_color[g] = color
    adata.uns['_dendrogram_colors'] = {str(k): v for k, v in group_to_color.items()}

    cluster_assignments = dict(zip(pert_names, groups))
    return groups, cluster_assignments


### Fig. S4a (1 µM): hierarchical clustering of drugs

In [ ]:
cluster_rho(data, t=1, show=True, figure_dir=figure_dir, save_name=f"{pert_type}_{concentration}uM")

In [ ]:
#for a non subsetted data, pass in genes corresponding to idx ie dataset.perturbation_dict.keys()
named_groups = slk.tl.group2name(groups, all_genes)
named_groups

In [ ]:
# map perturbation -> group using named_groups (index=gene, column='group')
gene_to_group = named_groups['group'].to_dict()

# assign group to obs based on slk pert key
sub.obs['pert_group'] = sub.obs[p_key].map(gene_to_group)

# mark NTC / unknown perturbations as -1 and ensure integer dtype
sub.obs['pert_group'] = sub.obs['pert_group'].fillna(-1).astype('category')

# quick summary
print(sub.obs['pert_group'].value_counts().sort_index())

sc.pl.umap(sub, color=['pert_group', 'tcell_treatment','drug'], title='Perturbation Groups', size=30)

In [ ]:
from __future__ import annotations

from scipy.cluster.hierarchy import linkage, leaves_list
from matplotlib import gridspec
from matplotlib.patches import Patch

def plot_zcorr_colored(adata, t=1.5, uns_key='results', show_pert_labels=True, group_color=new_colormap, pert_key=None):
    """
    Plot correlation matrices with color bars for each perturbation group.
    
    Parameters
    ----------
    adata : AnnData
        AnnData object with results in adata.uns[uns_key].
    uns_key : str, default 'results'
        Key in adata.uns where results are stored.
    show_pert_labels : bool, default True
        Whether to show perturbation labels on axes.
    group_col : str, default 'pert_group'
        Column in adata.obs containing perturbation groups.
    """
    
    if pert_key is None:
        pert_key = slk.configs.get_config('pert_key')
    Sigma_P_posterior = adata.uns[uns_key]['rho_corr']
    Sigma_z = adata.uns[uns_key]['z_corr']
    if 'pert_names' in adata.uns.get(uns_key, {}):
        pert_names = np.array(adata.uns[uns_key]['pert_names'])
    elif 'perts' in adata.uns.get(uns_key, {}):
        pert_names = np.array(adata.uns[uns_key]['perts'])
    else:
        _ntc_labels = {'control', 'ctrl', 'non-targeting', 'dmso', 'vehicle'}
        pert_names = np.array([p for p in np.unique(adata.obs[pert_key].values)
                               if p.lower() not in _ntc_labels])

    # Same linkage as cluster_rho: ward on distance matrix
    from scipy.cluster.hierarchy import fcluster
    D = 1.0 - Sigma_P_posterior
    Z_link = linkage(D[np.triu_indices_from(D, 1)], method='ward')
    order = leaves_list(Z_link)

    # Apply ordering to both matrices
    SigmaP_ord = Sigma_P_posterior[order][:, order]
    Sigmaz_ord = Sigma_z[order][:, order]
    pert_names_ord = pert_names[order]

    # Same color logic as cluster_rho
    from scipy.cluster.hierarchy import fcluster, set_link_color_palette, dendrogram
    from matplotlib.colors import to_hex
    groups = fcluster(Z_link, t=t, criterion='distance')
    groups_ord = groups[order]
    unique_groups = sorted(set(groups_ord))
    # Set same palette as cluster_rho
    palette_hex = [to_hex(group_color[i % len(group_color)]) for i in range(len(unique_groups))]
    set_link_color_palette(palette_hex)
    # Extract leaf→color via dendrogram (no_plot)
    ddata = dendrogram(Z_link, color_threshold=t, above_threshold_color='#2CA02C', no_plot=True)
    leaf_colors = ddata.get('leaves_color_list', [])
    group_to_color = {}
    for leaf_idx, color in zip(ddata['leaves'], leaf_colors):
        g = groups[leaf_idx]
        if g not in group_to_color:
            group_to_color[g] = color
    # Fallback if leaves_color_list unavailable
    for grp in unique_groups:
        if grp not in group_to_color:
            group_to_color[grp] = palette_hex[unique_groups.index(grp)]
    row_colors = [group_to_color[grp] for grp in groups_ord]
    
    n_perts = len(pert_names_ord)

    # Thickness controls for color bars
    #top_bar_height =0   # was 0.5
    side_bar_width = 0.8   # was 0.5

    # Plot single heatmap
    fig = plt.figure(figsize=(4, 3))
    ax0 = fig.add_subplot(111)
    
    sns.heatmap(SigmaP_ord, cmap="coolwarm", square=True, ax=ax0, 
                xticklabels=pert_names_ord if show_pert_labels else False,
                yticklabels=pert_names_ord if show_pert_labels else False,
                cbar_kws={'label': 'Correlation', 'pad': 0.15})
    ax0.set_title("")
    ax0.set_xlabel("Perturbations")
    ax0.set_ylabel("Perturbations")

    # Make font size smaller
    if show_pert_labels:
        ax0.set_xticklabels(ax0.get_xticklabels(), fontsize=7)
        ax0.set_yticklabels(ax0.get_yticklabels(), fontsize=7)
    
    gap = 0.5  # space between heatmap and group color strip

    # Add color bar on right (thicker)
    for i, color in enumerate(row_colors):
        # Top
        #ax0.add_patch(plt.Rectangle((i, -0.5), 1, top_bar_height, color=color, 
                                    #transform=ax0.transData, clip_on=False))
        # Right
        ax0.add_patch(plt.Rectangle((n_perts + gap, i), side_bar_width, 1, color=color, 
                                    transform=ax0.transData, clip_on=False))

    # Create legend outside with transparent background
    legend_elements = [Patch(facecolor=group_to_color[grp], label=f'Group {grp}') 
                      for grp in unique_groups]
    fig.legend(
        handles=legend_elements,
        loc='center left',
        bbox_to_anchor=(1.0, 0.5),
        title='Perturbation Groups',
        fontsize=8,
        frameon=False
    )

    plt.tight_layout()
    plt.savefig(os.path.join(figure_dir, f"{pert_type}_{concentration}_correlation_color.svg"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{pert_type}_{concentration}_correlation_color.png"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{pert_type}_{concentration}_correlation_color.pdf"), dpi=300, bbox_inches='tight')
    plt.show()

plot_zcorr_colored(sub, t=1)



In [ ]:
palette1 = [
"#FFD166",  
"#023047",  
]

palette2 = [
    "#EF476F", "#0077B6", "#8338EC", "#FB5607",  "#F7A1C4",
    "#00D9FF", "#38A169", "#8B5E3C", "#4A5568", "#2EC4B6", "#9B2226"
]

palette2_very_light = [
    "#EF476F",  # keep
    "#A8D8F0",  # very light blue
    "#D6C7FA",  # very light purple
    "#FB5607",  # keep
    "#FDD9E7",  # very light pink
    "#CFF5FF",  # very light cyan
    "#CDEEDD",  # very light green
    "#E3D2C4",  # very light brown
    "#D6DEE8",  # very light gray-blue
    "#CFF1EA",  # very light teal
    "#E8B3B3"   # very light dark red
]



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

sc.pl.umap(sub, color='tcell_treatment', title='tcell_treatment', 
           palette=palette1, size=12, frameon=False, ax=axes[0], show=False,legend_loc=None)

sc.pl.umap(sub, color='drug', title='drug', 
           palette=palette2_very_light, size=12, frameon=False, ax=axes[1], show=False,legend_loc=None)

plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f"{concentration}_sherlock_zspace.svg"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_sherlock_zspace.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_sherlock_zspace.pdf"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

sc.pl.umap(sub, color='tcell_treatment', title='tcell_treatment', 
           palette=palette1, size=12, frameon=False, ax=axes[0], show=False)

sc.pl.umap(sub, color='drug', title='drug', 
           palette=palette2_very_light, size=12, frameon=False, ax=axes[1], show=False)

plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f"{concentration}_sherlock_zspace_with_legends.svg"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_sherlock_zspace_with_legends.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_sherlock_zspace_with_legends.pdf"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Print gene symbols per group, one gene per line (no quotation marks)
grouped = named_groups.groupby('group').apply(lambda df: df.index.tolist())

for grp in sorted(grouped.index):
    print(f"Group {grp}:")
    for gene in grouped.loc[grp]:
        print(gene)
    print()

In [ ]:
pert_key="drug"
ntc_label="DMSO"
treatment_key='tcell_treatment'

In [ ]:
# Get perturbation and condition names
# Extract names
pert_names = (list(data.uns['results']['perts']) if 'perts' in data.uns.get('results', {})
           else [p for p in np.unique(data.obs[slk.configs.get_config('pert_key')].values)
                 if p.lower() not in {'control','ctrl','non-targeting','dmso','vehicle'}])    # matches cfs_mat axis 0
cond_names = list(data.uns['results']['conditions'])  # matches cfs_mat axes 1 and 2

# Get the matrix
cfs_mat = data.uns['results']['condition_perturbation_interaction']  # shape: (P, C, C)

# Verify dimensions match
assert cfs_mat.shape[0] == len(pert_names)
assert cfs_mat.shape[1] == len(cond_names)
assert cfs_mat.shape[2] == len(cond_names)

In [ ]:
# Average shift per perturbation (across all condition pairs)
mask = np.triu(np.ones((len(cond_names), len(cond_names))), k=1).astype(bool)
avg_shift_per_pert = np.array([cfs_mat[p][mask].mean() for p in range(len(pert_names))])

# Rank perturbations by context-dependency
df_pert_sensitivity = pd.DataFrame({
    'perturbation': pert_names,
    'condition_effect_on_perturbation': avg_shift_per_pert,
}).sort_values('condition_effect_on_perturbation', ascending=False)

In [ ]:
# Save df_pert_sensitivity for cross-concentration comparison
df_pert_sensitivity.to_csv(
    os.path.join(figure_dir, f'{concentration}um_pert_sensitivity.csv'),
    index=False
)
print(f'Saved {concentration}um_pert_sensitivity.csv')

### Fig. S4b (1 µM): condition dependence of each drug

In [ ]:
# plot bar plot of condition effect on perturbation
plt.figure(figsize=(4, 2))

if hasattr(data.obs['drug'], 'cat'):
    drug_order = [drug for drug in data.obs['drug'].cat.categories if drug != 'DMSO']
else:
    drug_order = sorted([drug for drug in data.obs['drug'].unique() if drug != 'DMSO'])

drug_to_color = dict(zip(drug_order, palette2_very_light))
bar_colors = [drug_to_color.get(drug, 'gray') for drug in df_pert_sensitivity['perturbation']]

sns.barplot(data=df_pert_sensitivity, x='perturbation', y='condition_effect_on_perturbation', palette=bar_colors)
plt.xticks(rotation=45, ha='right')
plt.xlabel('Drug')
plt.ylabel('Conditional\nperturbation response')
#plt.title('Condition Effect on \nPerturbation by Drug')
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f"{concentration}_condition_effect_on_perturbation.svg"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_condition_effect_on_perturbation.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_condition_effect_on_perturbation.pdf"), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
df_pert_sensitivity

In [ ]:
slk.pl.plot_cf_bipartite(
    data,
    cond_idx=0,          
    pert_name="CP673451",     
    top_n=15,            # top N target genes by effect size
)

In [ ]:
# --- Publication-ready save for plot_cf_bipartite (figure_1 style) — all conditions ---
from matplotlib.collections import LineCollection
from unittest.mock import patch
import sherlock.plotting._simple as _slk_simple

PERT_X_MARGIN = 0.25  # 0=full spread, 0.5=fully centered
_orig_draw_bipartite = _slk_simple._draw_bipartite_nx

def _centered_draw_bipartite(pert_nodes, gene_nodes, edges, ax, **kwargs):
    n_p, n_g = len(pert_nodes), len(gene_nodes)
    if n_p == 1:
        orig_pert_xs = [0.5]
        new_pert_xs  = [0.5]
    else:
        orig_pert_xs = [i / (n_p - 1) for i in range(n_p)]
        new_pert_xs  = [PERT_X_MARGIN + i * (1 - 2*PERT_X_MARGIN) / (n_p - 1) for i in range(n_p)]
    x_remap = dict(zip(orig_pert_xs, new_pert_xs))
    _orig_draw_bipartite(pert_nodes, gene_nodes, edges, ax,
                         pert_color='#26547c',
                         gene_color='#D0D0D0',
                         gene_overlap_color='#1E90FF',
                         **kwargs)
    for coll in ax.collections:
        offsets = coll.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue
        if np.all(offsets[:, 1] == 1.0) and len(offsets) == n_p:
            coll.set_offsets(np.column_stack([new_pert_xs, np.ones(n_p)]))
    for txt, x in zip(ax.texts[:n_p], new_pert_xs):
        _, old_y = txt.get_position()
        txt.set_position((x, old_y))
    for coll in ax.collections:
        if not isinstance(coll, LineCollection):
            continue
        segs = coll.get_segments()
        new_segs = []
        for seg in segs:
            new_seg = []
            for pt in seg:
                x, y = pt
                if y == 1.0:
                    closest = min(orig_pert_xs, key=lambda ox: abs(ox - x))
                    x = x_remap.get(closest, x)
                new_seg.append([x, y])
            new_segs.append(new_seg)
        coll.set_segments(new_segs)

# Build pert→group color mapping (same clustering as plot_zcorr_colored)
from scipy.cluster.hierarchy import fcluster, set_link_color_palette, dendrogram
from matplotlib.colors import to_hex
_rho = data.uns['results']['rho_corr']
_pnames = (list(data.uns['results']['perts']) if 'perts' in data.uns.get('results', {})
           else [p for p in np.unique(data.obs[slk.configs.get_config('pert_key')].values)
                 if p.lower() not in {'control','ctrl','non-targeting','dmso','vehicle'}])
_D = 1.0 - _rho
_Z = linkage(_D[np.triu_indices_from(_D, 1)], method='ward')
_groups = fcluster(_Z, t=1, criterion='distance')
_ugrps = sorted(set(_groups))
_pal = [to_hex(new_colormap[i % len(new_colormap)]) for i in range(len(_ugrps))]
set_link_color_palette(_pal)
_ddata = dendrogram(_Z, color_threshold=1, above_threshold_color='#2CA02C', no_plot=True)
_g2c = {}
for _li, _lc in zip(_ddata['leaves'], _ddata.get('leaves_color_list', [])):
    if _groups[_li] not in _g2c:
        _g2c[_groups[_li]] = _lc
for _grp, _col in zip(_ugrps, _pal):
    _g2c.setdefault(_grp, _col)
pert_to_color = {n: _g2c[g] for n, g in zip(_pnames, _groups)}

cond_labels = data.uns['results']['conditions']
_orig_get_cf_data = _slk_simple._get_cf_data

def _make_qval_masked_cf(qv, cidx):
    def _fn(adata, uns_key='results', cf_key='counterfactual_effect_size'):
        cf, perts, genes = _orig_get_cf_data(adata, uns_key=uns_key, cf_key=cf_key)
        cf = cf.copy()
        cf[cidx][qv >= 0.05] = 0.0
        n_sig = int((qv < 0.05).sum())
        print(f'  cond {cidx}: {n_sig}/{qv.size} (pert,gene) pairs pass qval<0.05')
        return cf, perts, genes
    return _fn

for cond_idx in range(len(cond_labels)):
    cond_label = cond_labels[cond_idx]
    _qvals = np.asarray(data.uns['results']['ev_results'][cond_idx]['qvals'], dtype=float)  # (P, G)
    _cf_fig = {}
    def _show_capture():
        _cf_fig['fig'] = plt.gcf()
    with patch.object(_slk_simple, '_draw_bipartite_nx', _centered_draw_bipartite), \
         patch.object(_slk_simple, '_get_cf_data', _make_qval_masked_cf(_qvals, cond_idx)), \
         patch.object(plt, 'show', _show_capture):
        slk.pl.plot_cf_bipartite(
            data,
            cond_idx=cond_idx,
            pert_name=["CP673451", "ALWII4127"],
            top_n=15,
            min_effect=1e-10,
            figsize=(6, 2.8),
        )
    fig = _cf_fig['fig']
    ax = fig.axes[0]
    ax.set_title('')  # Remove title
    for coll in ax.collections:
        if isinstance(coll, LineCollection):
            coll.set_color('#BBBBBB')
            coll.set_linewidth(1.8)
            coll.set_alpha(0.7)
    for txt in ax.texts:
        txt.set_fontsize(8)
        txt.set_color('#444444')
        txt.set_fontweight('bold')
    # Shrink node circles
    for coll in ax.collections:
        if not isinstance(coll, LineCollection):
            sizes = coll.get_sizes()
            if len(sizes) > 0:
                coll.set_sizes([200])
    legend = ax.get_legend()
    if legend:
        legend.remove()
    # Color pert nodes by their group color
    _pert_list = ["CP673451", "ALWII4127"]
    _node_colors = [pert_to_color.get(p, '#26547c') for p in _pert_list]
    for coll in ax.collections:
        offsets = coll.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue
        if np.all(offsets[:, 1] == 1.0) and len(offsets) == len(_pert_list):
            coll.set_facecolor(_node_colors)
            coll.set_edgecolor(_node_colors)
    plt.tight_layout(pad=0.3)
    save_name = f"{concentration}_cf_bipartite_CP673451_ALWII4127_{cond_label}"
    for ext in ('svg', 'png', 'pdf'):
        fig.savefig(os.path.join(figure_dir, f'{save_name}.{ext}'), dpi=300, bbox_inches='tight', transparent=True)
    print(f'Saved {save_name}.svg/png/pdf')
    plt.show()


In [ ]:
# --- Publication-ready save for plot_cf_bipartite (figure_1 style) — all conditions ---
from matplotlib.collections import LineCollection
from unittest.mock import patch
import sherlock.plotting._simple as _slk_simple

PERT_X_MARGIN = 0.25  # 0=full spread, 0.5=fully centered
_orig_draw_bipartite = _slk_simple._draw_bipartite_nx

def _centered_draw_bipartite(pert_nodes, gene_nodes, edges, ax, **kwargs):
    n_p, n_g = len(pert_nodes), len(gene_nodes)
    if n_p == 1:
        orig_pert_xs = [0.5]
        new_pert_xs  = [0.5]
    else:
        orig_pert_xs = [i / (n_p - 1) for i in range(n_p)]
        new_pert_xs  = [PERT_X_MARGIN + i * (1 - 2*PERT_X_MARGIN) / (n_p - 1) for i in range(n_p)]
    x_remap = dict(zip(orig_pert_xs, new_pert_xs))
    _orig_draw_bipartite(pert_nodes, gene_nodes, edges, ax,
                         pert_color='#26547c',
                         gene_color='#D0D0D0',
                         gene_overlap_color='#1E90FF',
                         **kwargs)
    for coll in ax.collections:
        offsets = coll.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue
        if np.all(offsets[:, 1] == 1.0) and len(offsets) == n_p:
            coll.set_offsets(np.column_stack([new_pert_xs, np.ones(n_p)]))
    for txt, x in zip(ax.texts[:n_p], new_pert_xs):
        _, old_y = txt.get_position()
        txt.set_position((x, old_y))
    for coll in ax.collections:
        if not isinstance(coll, LineCollection):
            continue
        segs = coll.get_segments()
        new_segs = []
        for seg in segs:
            new_seg = []
            for pt in seg:
                x, y = pt
                if y == 1.0:
                    closest = min(orig_pert_xs, key=lambda ox: abs(ox - x))
                    x = x_remap.get(closest, x)
                new_seg.append([x, y])
            new_segs.append(new_seg)
        coll.set_segments(new_segs)

# Build pert→group color mapping (same clustering as plot_zcorr_colored)
from scipy.cluster.hierarchy import fcluster, set_link_color_palette, dendrogram
from matplotlib.colors import to_hex
_rho = data.uns['results']['rho_corr']
_pnames = (list(data.uns['results']['perts']) if 'perts' in data.uns.get('results', {})
           else [p for p in np.unique(data.obs[slk.configs.get_config('pert_key')].values)
                 if p.lower() not in {'control','ctrl','non-targeting','dmso','vehicle'}])
_D = 1.0 - _rho
_Z = linkage(_D[np.triu_indices_from(_D, 1)], method='ward')
_groups = fcluster(_Z, t=1, criterion='distance')
_ugrps = sorted(set(_groups))
_pal = [to_hex(new_colormap[i % len(new_colormap)]) for i in range(len(_ugrps))]
set_link_color_palette(_pal)
_ddata = dendrogram(_Z, color_threshold=1, above_threshold_color='#2CA02C', no_plot=True)
_g2c = {}
for _li, _lc in zip(_ddata['leaves'], _ddata.get('leaves_color_list', [])):
    if _groups[_li] not in _g2c:
        _g2c[_groups[_li]] = _lc
for _grp, _col in zip(_ugrps, _pal):
    _g2c.setdefault(_grp, _col)
pert_to_color = {n: _g2c[g] for n, g in zip(_pnames, _groups)}

cond_labels = data.uns['results']['conditions']
_orig_get_cf_data = _slk_simple._get_cf_data

def _make_qval_masked_cf(qv, cidx):
    def _fn(adata, uns_key='results', cf_key='counterfactual_effect_size'):
        cf, perts, genes = _orig_get_cf_data(adata, uns_key=uns_key, cf_key=cf_key)
        cf = cf.copy()
        cf[cidx][qv >= 0.05] = 0.0
        n_sig = int((qv < 0.05).sum())
        print(f'  cond {cidx}: {n_sig}/{qv.size} (pert,gene) pairs pass qval<0.05')
        return cf, perts, genes
    return _fn

for cond_idx in range(len(cond_labels)):
    cond_label = cond_labels[cond_idx]
    _qvals = np.asarray(data.uns['results']['ev_results'][cond_idx]['qvals'], dtype=float)  # (P, G)
    _cf_fig = {}
    def _show_capture():
        _cf_fig['fig'] = plt.gcf()
    with patch.object(_slk_simple, '_draw_bipartite_nx', _centered_draw_bipartite), \
         patch.object(_slk_simple, '_get_cf_data', _make_qval_masked_cf(_qvals, cond_idx)), \
         patch.object(plt, 'show', _show_capture):
        slk.pl.plot_cf_bipartite(
            data,
            cond_idx=cond_idx,
            pert_name=["CP673451", "BAY1217389"],
            top_n=15,
            min_effect=1e-10,
            figsize=(6, 2.8),
        )
    fig = _cf_fig['fig']
    ax = fig.axes[0]
    ax.set_title('')  # Remove title
    for coll in ax.collections:
        if isinstance(coll, LineCollection):
            coll.set_color('#BBBBBB')
            coll.set_linewidth(1.8)
            coll.set_alpha(0.7)
    for txt in ax.texts:
        txt.set_fontsize(8)
        txt.set_color('#444444')
        txt.set_fontweight('bold')
    # Shrink node circles
    for coll in ax.collections:
        if not isinstance(coll, LineCollection):
            sizes = coll.get_sizes()
            if len(sizes) > 0:
                coll.set_sizes([200])
    legend = ax.get_legend()
    if legend:
        legend.remove()
    # Color pert nodes by their group color
    _pert_list = ["CP673451", "BAY1217389"]
    _node_colors = [pert_to_color.get(p, '#26547c') for p in _pert_list]
    for coll in ax.collections:
        offsets = coll.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue
        if np.all(offsets[:, 1] == 1.0) and len(offsets) == len(_pert_list):
            coll.set_facecolor(_node_colors)
            coll.set_edgecolor(_node_colors)
    plt.tight_layout(pad=0.3)
    save_name = f"{concentration}_cf_bipartite_CP673451_BAY1217389_{cond_label}"
    for ext in ('svg', 'png', 'pdf'):
        fig.savefig(os.path.join(figure_dir, f'{save_name}.{ext}'), dpi=300, bbox_inches='tight', transparent=True)
    print(f'Saved {save_name}.svg/png/pdf')
    plt.show()


### Fig. S4c (1 µM): downstream genes of CP673451 and STK16IN1 per condition

In [ ]:
# --- Publication-ready save for plot_cf_bipartite (figure_1 style) — all conditions ---
from matplotlib.collections import LineCollection
from unittest.mock import patch
import sherlock.plotting._simple as _slk_simple

PERT_X_MARGIN = 0.25  # 0=full spread, 0.5=fully centered
_orig_draw_bipartite = _slk_simple._draw_bipartite_nx

def _centered_draw_bipartite(pert_nodes, gene_nodes, edges, ax, **kwargs):
    n_p, n_g = len(pert_nodes), len(gene_nodes)
    if n_p == 1:
        orig_pert_xs = [0.5]
        new_pert_xs  = [0.5]
    else:
        orig_pert_xs = [i / (n_p - 1) for i in range(n_p)]
        new_pert_xs  = [PERT_X_MARGIN + i * (1 - 2*PERT_X_MARGIN) / (n_p - 1) for i in range(n_p)]
    x_remap = dict(zip(orig_pert_xs, new_pert_xs))
    _orig_draw_bipartite(pert_nodes, gene_nodes, edges, ax,
                         pert_color='#26547c',
                         gene_color='#D0D0D0',
                         gene_overlap_color='#1E90FF',
                         **kwargs)
    for coll in ax.collections:
        offsets = coll.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue
        if np.all(offsets[:, 1] == 1.0) and len(offsets) == n_p:
            coll.set_offsets(np.column_stack([new_pert_xs, np.ones(n_p)]))
    for txt, x in zip(ax.texts[:n_p], new_pert_xs):
        _, old_y = txt.get_position()
        txt.set_position((x, old_y))
    for coll in ax.collections:
        if not isinstance(coll, LineCollection):
            continue
        segs = coll.get_segments()
        new_segs = []
        for seg in segs:
            new_seg = []
            for pt in seg:
                x, y = pt
                if y == 1.0:
                    closest = min(orig_pert_xs, key=lambda ox: abs(ox - x))
                    x = x_remap.get(closest, x)
                new_seg.append([x, y])
            new_segs.append(new_seg)
        coll.set_segments(new_segs)

# Build pert→group color mapping (same clustering as plot_zcorr_colored)
from scipy.cluster.hierarchy import fcluster, set_link_color_palette, dendrogram
from matplotlib.colors import to_hex
_rho = data.uns['results']['rho_corr']
_pnames = (list(data.uns['results']['perts']) if 'perts' in data.uns.get('results', {})
           else [p for p in np.unique(data.obs[slk.configs.get_config('pert_key')].values)
                 if p.lower() not in {'control','ctrl','non-targeting','dmso','vehicle'}])
_D = 1.0 - _rho
_Z = linkage(_D[np.triu_indices_from(_D, 1)], method='ward')
_groups = fcluster(_Z, t=1, criterion='distance')
_ugrps = sorted(set(_groups))
_pal = [to_hex(new_colormap[i % len(new_colormap)]) for i in range(len(_ugrps))]
set_link_color_palette(_pal)
_ddata = dendrogram(_Z, color_threshold=1, above_threshold_color='#2CA02C', no_plot=True)
_g2c = {}
for _li, _lc in zip(_ddata['leaves'], _ddata.get('leaves_color_list', [])):
    if _groups[_li] not in _g2c:
        _g2c[_groups[_li]] = _lc
for _grp, _col in zip(_ugrps, _pal):
    _g2c.setdefault(_grp, _col)
pert_to_color = {n: _g2c[g] for n, g in zip(_pnames, _groups)}

cond_labels = data.uns['results']['conditions']
_orig_get_cf_data = _slk_simple._get_cf_data

def _make_qval_masked_cf(qv, cidx):
    def _fn(adata, uns_key='results', cf_key='counterfactual_effect_size'):
        cf, perts, genes = _orig_get_cf_data(adata, uns_key=uns_key, cf_key=cf_key)
        cf = cf.copy()
        cf[cidx][qv >= 0.05] = 0.0
        n_sig = int((qv < 0.05).sum())
        print(f'  cond {cidx}: {n_sig}/{qv.size} (pert,gene) pairs pass qval<0.05')
        return cf, perts, genes
    return _fn

for cond_idx in range(len(cond_labels)):
    cond_label = cond_labels[cond_idx]
    _qvals = np.asarray(data.uns['results']['ev_results'][cond_idx]['qvals'], dtype=float)  # (P, G)
    _cf_fig = {}
    def _show_capture():
        _cf_fig['fig'] = plt.gcf()
    with patch.object(_slk_simple, '_draw_bipartite_nx', _centered_draw_bipartite), \
         patch.object(_slk_simple, '_get_cf_data', _make_qval_masked_cf(_qvals, cond_idx)), \
         patch.object(plt, 'show', _show_capture):
        slk.pl.plot_cf_bipartite(
            data,
            cond_idx=cond_idx,
            pert_name=["CP673451", "STK16IN1"],
            top_n=15,
            min_effect=1e-10,
            figsize=(6, 2.8),
        )
    fig = _cf_fig['fig']
    ax = fig.axes[0]
    ax.set_title('')  # Remove title
    for coll in ax.collections:
        if isinstance(coll, LineCollection):
            coll.set_color('#BBBBBB')
            coll.set_linewidth(1.8)
            coll.set_alpha(0.7)
    for txt in ax.texts:
        txt.set_fontsize(8)
        txt.set_color('#444444')
        txt.set_fontweight('bold')
    # Shrink node circles
    for coll in ax.collections:
        if not isinstance(coll, LineCollection):
            sizes = coll.get_sizes()
            if len(sizes) > 0:
                coll.set_sizes([200])
    legend = ax.get_legend()
    if legend:
        legend.remove()
    # Color pert nodes by their group color
    _pert_list = ["CP673451", "STK16IN1"]
    _node_colors = [pert_to_color.get(p, '#26547c') for p in _pert_list]
    for coll in ax.collections:
        offsets = coll.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue
        if np.all(offsets[:, 1] == 1.0) and len(offsets) == len(_pert_list):
            coll.set_facecolor(_node_colors)
            coll.set_edgecolor(_node_colors)
    plt.tight_layout(pad=0.3)
    save_name = f"{concentration}_cf_bipartite_CP673451_STK16IN1_{cond_label}"
    for ext in ('svg', 'png', 'pdf'):
        fig.savefig(os.path.join(figure_dir, f'{save_name}.{ext}'), dpi=300, bbox_inches='tight', transparent=True)
    print(f'Saved {save_name}.svg/png/pdf')
    plt.show()


## 11 kinase inhibitors, 10 µM

From [`scripts/kinase_inhibitors_tcell/3_analysis_10uM.ipynb`](../scripts/kinase_inhibitors_tcell/3_analysis_10uM.ipynb).

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path

import seaborn as sns
import matplotlib.pyplot as plt

import torch

import sherlock as slk

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({'font.size': 12, 'svg.fonttype': 'none'})
mpl.rcParams['axes.titlesize'] = 16
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['lines.linewidth'] = 2
figure_dir = f"{paths.FIGURES_DIR}/kinase_inhibitors_tcell/"
os.makedirs(figure_dir, exist_ok=True)

In [ ]:
pert_type = 'drug'  # 'crispri' or 'crisprko'

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
slk.configs.show()
slk.configs.set_config('ntc_label', 'DMSO')
slk.configs.set_config('pert_key', 'drug')
slk.configs.set_config('treatment_key', 'tcell_treatment')
slk.configs.set_config('untreated_label', 'NoT')
slk.configs.show()
#slk.configs.reset_config()
slk.configs.get_config('ntc_label')


In [ ]:
slk.configs.get_config('ntc_label')

In [ ]:
concentration  = '10'
tcdg = 1000
run = 12

In [ ]:
data = sc.read_h5ad(f"{paths.DATA_DIR}/gbm/bt333_degs_tcell_{concentration}_{tcdg}.h5ad")

data.obs

In [ ]:
data.obs['concentration']

In [ ]:
# remove na from drug column
data = data[~data.obs['drug'].isna()].copy()
data = data[
    (data.obs['concentration'] == concentration) | (data.obs['drug'] == 'DMSO')
].copy()

In [ ]:
data.obs

In [ ]:
slk.pp.slk_prepare_data(data, pert_key="drug", ntc_label="DMSO", inplace=True, treatment_key="tcell_treatment", untreated_label="NoT")

In [ ]:
data.obs

In [ ]:
slk.pp.treat_effect(
    data,
    label_col = "drug",
    control_label = 'DMSO',
    method = "perturbseq",
    inplace=True,
    key='treat_effect'
)

In [ ]:
data

In [ ]:
data.X.max()

In [ ]:
# Check for NaN/Inf in expression matrix
print(f"Data shape: {data.shape}")
X = data.X.toarray() if hasattr(data.X, 'toarray') else data.X

n_nan = np.isnan(X).sum()
n_inf = np.isinf(X).sum()
print(f"NaN values in X: {n_nan}")
print(f"Inf values in X: {n_inf}")

if n_nan > 0 or n_inf > 0:
    print("Cleaning expression matrix...")
    # Replace NaN with 0 and clip Inf values
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    data.X = X
    print("Expression matrix cleaned!")

# Check data stats
print(f"X min: {data.X.min()}, max: {data.X.max()}")
print(f"Unique drugs: {data.obs['drug'].nunique()}")

In [ ]:
# DIAGNOSE: Check condition distribution
print("=== Condition Distribution ===")
print(f"\nTotal cells: {len(data)}")
print(f"\nUnique values in 'tcell_treatment':")
print(data.obs['tcell_treatment'].value_counts())

print(f"\n\nDMSO cells by condition:")
dmso_data = data[data.obs['drug']=='DMSO']
print(dmso_data.obs['tcell_treatment'].value_counts())

print(f"\n\nNon-DMSO cells by condition:")
non_dmso_data = data[data.obs['drug']!='DMSO']
print(non_dmso_data.obs['tcell_treatment'].value_counts())

# what the count for all drugs including DMSO
print(f"\n\nAll cells by drug:")
print(data.obs['drug'].value_counts())  

print(f"\n\n=== CRITICAL CHECK ===")
if len(dmso_data) == 0:
    print("❌ ERROR: NO DMSO CELLS FOUND!")
    print("   The model cannot train without control cells.")
else:
    print(f"✅ Found {len(dmso_data)} DMSO cells")
    
# Check if all conditions have DMSO
all_conditions = data.obs['tcell_treatment'].unique()
for cond in all_conditions:
    dmso_count = len(data[(data.obs['drug']=='DMSO') & (data.obs['tcell_treatment']==cond)])
    print(f"   Condition '{cond}': {dmso_count} DMSO cells")

In [ ]:
data

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# results = slk.tl.run_single(data, 
#                             batch_size=2048,
#                             num_workers=0, 
#                             device=device, 
#                             use_conditions=True,
#                             l0_lambda = 100,
#                             num_epochs=1000,
#                             #gate_init_p=0.4,
#                             H_lambda=0.01,
#                             ce_lambda=0.1,
#                             latent_dim= 4,
#                             rank = 6
#                             )

In [ ]:
# SHERLOCK_KWARGS = dict(
#     batch_size     = 4096,
#     #num_workers    = 0,
#     num_epochs     = 1000,
#     use_conditions = True,
#     validate_every = 20,
#     l0_lambda      = 200,
#     #l1_lambda      = 0.5,
#     #lr             = 1e-3,
#     patience       = 500,
#     #rho_init       = False,
#     #ntc_lambda     = 1.0,
#     latent_dim     = 8,
#     rank           = 3,
#     #n_epochs_kl_warmup = 10,
#     n_epochs_l0_warmup = 200
    
# )  # sweep winner: ATE=0.660, spearman=0.774, pearson=0.766, al_cp_rho=0.570, spec=1.29
# IF it drops to quickly lower l0, or icnrease n_epochs_l0_warmup, 
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# SHERLOCK_KWARGS = dict(
#     batch_size     = 4096,
#     #num_workers    = 0,
#     num_epochs     = 1000,
#     use_conditions = True,
#     validate_every = 20,
#     l0_lambda      = 200,
#     #l1_lambda      = 0.5,
#     #lr             = 1e-3,
#     patience       = 500,
#     #rho_init       = False,
#     #ntc_lambda     = 1.0,
#     latent_dim     = 8,
#     rank           = 3,
#     #n_epochs_kl_warmup = 10,
#     n_epochs_l0_warmup = 200
# ) 




In [ ]:
seed = 7

In [ ]:
#results = slk.tl.run_single(data, device=DEVICE, seed=seed, debug=True, **SHERLOCK_KWARGS)

In [ ]:
# import datetime
    
# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# slk.tl.save_results(results, f'{paths.RESULTS_DIR}/models/{pert_type}_model_{timestamp}_{seed}.pth')

# print(f'Saved results to ../data/{pert_type}_model_{timestamp}_{seed}.pth')

In [ ]:
#model = results['model']

In [ ]:
# Saved results to ../data/drug_model_20260519_143354_1.pth
#Saved results to ../data/drug_model_20260519_183741_0.pth
# Saved results to ../data/drug_model_20260519_192528_2.pth best
#Saved results to ../data/drug_model_20260520_153913_4.pth
# {paths.RESULTS_DIR}/models/drug_model_20260520_181644_5.pth
# f'{paths.RESULTS_DIR}/gbm/results/reproducibility_drug/matrices/best_model.pth'#

In [ ]:
model = slk.tl.load_results(f'{paths.RESULTS_DIR}/models/drug_model_20260519_192528_2.pth')


In [ ]:
model

In [ ]:
slk.tl.eval_single(model, data, obsm_key="z", uns_key="results")

In [ ]:
data.uns['results']

In [ ]:
#data.X = data.X.toarray()
slk.pl.plot_r2(data)

In [ ]:
slk.pl.plot_corr(data)

In [ ]:
slk.pl.plot_ev(data, cmax=2, top_n=50)

In [ ]:
slk.tl.ev_sig(data, cond_idx=0)

In [ ]:
slk.tl.ev_sig(data, cond_idx=1)

In [ ]:
# Useful dataframe that plot_ev_sig uses internally
slk.pl.make_ev_long_df(data, cond_idx=0, uns_key="results")

In [ ]:
# Useful dataframe that plot_ev_sig uses internally
slk.pl.make_ev_long_df(data, cond_idx=1, uns_key="results")

In [ ]:
# ── AL vs CP cross-condition target gene correlation ────────────────────────
from scipy.stats import pearsonr, spearmanr

AL, CP      = 'ALWII4127', 'CP673451'
_ntc_labels = {'control', 'ctrl', 'non-targeting', 'dmso', 'vehicle'}

# counterfactual_effect_size shape: (C, P, G)
cfs      = np.asarray(data.uns['results']['counterfactual_effect_size'], dtype=float)
cond_names = list(data.uns['results']['conditions'])

# pert_names in model order (sorted, NTC removed)
_pnames = [p for p in np.unique(data.obs[slk.configs.get_config('pert_key')].values)
           if p.lower() not in _ntc_labels]
al_idx = _pnames.index(AL)
cp_idx = _pnames.index(CP)

print(f"counterfactual_effect_size shape: {cfs.shape}  (C={cfs.shape[0]}, P={cfs.shape[1]}, G={cfs.shape[2]})")
print(f"AL={AL} index={al_idx}   CP={CP} index={cp_idx}")
print(f"Conditions: {cond_names}\n")

# ── Per-condition Pearson/Spearman between AL and CP gene effect vectors ──
n_conds = len(cond_names)
fig, axes = plt.subplots(1, n_conds, figsize=(5 * n_conds, 4.5))
if n_conds == 1:
    axes = [axes]

for ci, (ax, cname) in enumerate(zip(axes, cond_names)):
    al_vec = cfs[ci, al_idx, :]   # (G,)
    cp_vec = cfs[ci, cp_idx, :]   # (G,)

    valid = np.isfinite(al_vec) & np.isfinite(cp_vec)
    r_p, p_p = pearsonr(al_vec[valid],  cp_vec[valid])
    r_s, p_s = spearmanr(al_vec[valid], cp_vec[valid])
    print(f"Condition: {cname}")
    print(f"  Pearson  r = {r_p:.3f}  (p = {p_p:.2e})")
    print(f"  Spearman r = {r_s:.3f}  (p = {p_s:.2e})\n")

    # scatter: one dot per gene
    ax.scatter(al_vec[valid], cp_vec[valid], s=6, alpha=0.3, color='steelblue', rasterized=True)

    # highlight genes significant for both (q < 0.05 in both AL and CP)
    if 'ev_results' in data.uns['results']:
        ev = data.uns['results']['ev_results']
        if ci < len(ev):
            qv = np.asarray(ev[ci]['qvals'], dtype=float)   # (P, G)
            both_sig = (qv[al_idx] < 0.05) & (qv[cp_idx] < 0.05) & valid
            ax.scatter(al_vec[both_sig], cp_vec[both_sig], s=20, alpha=0.8,
                       color='tomato', zorder=3, label=f'sig both (n={both_sig.sum()})')
            ax.legend(fontsize=8)

    # diagonal reference
    lim = max(np.abs(al_vec[valid]).max(), np.abs(cp_vec[valid]).max()) * 1.05
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.axvline(0, color='gray', lw=0.5, ls='--')
    ax.plot([-lim, lim], [-lim, lim], color='gray', lw=0.8, ls=':')
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel(f'{AL} effect size', fontsize=10)
    ax.set_ylabel(f'{CP} effect size', fontsize=10)
    ax.set_title(f'{cname}\nPearson r={r_p:.3f}  Spearman r={r_s:.3f}', fontsize=10)
    ax.set_aspect('equal')

plt.suptitle('AL vs CP counterfactual gene effects per condition', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f'AL_CP_cross_condition_correlation.svg'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f'AL_CP_cross_condition_correlation.png'), dpi=300, bbox_inches='tight')
plt.show()

# ── Heatmap: all drug-pair Pearson r per condition ────────────────────────
fig, axes = plt.subplots(1, n_conds, figsize=(5 * n_conds, 4.5))
if n_conds == 1:
    axes = [axes]

for ci, (ax, cname) in enumerate(zip(axes, cond_names)):
    P = len(_pnames)
    r_mat = np.full((P, P), np.nan)
    for pi in range(P):
        for pj in range(pi, P):
            vi = cfs[ci, pi, :]
            vj = cfs[ci, pj, :]
            v  = np.isfinite(vi) & np.isfinite(vj)
            if v.sum() > 2:
                r, _ = pearsonr(vi[v], vj[v])
                r_mat[pi, pj] = r_mat[pj, pi] = r
    np.fill_diagonal(r_mat, 1.0)

    im = ax.imshow(r_mat, cmap='coolwarm', vmin=-1, vmax=1)
    ax.set_xticks(range(P)); ax.set_xticklabels(_pnames, rotation=90, fontsize=7)
    ax.set_yticks(range(P)); ax.set_yticklabels(_pnames, fontsize=7)
    ax.set_title(f'{cname}\nPearson r of CF gene effects', fontsize=10)
    plt.colorbar(im, ax=ax, label='Pearson r', fraction=0.046, pad=0.04)
    # highlight AL-CP cell
    ax.add_patch(plt.Rectangle((cp_idx - 0.5, al_idx - 0.5), 1, 1,
                                fill=False, edgecolor='black', lw=2))
    ax.add_patch(plt.Rectangle((al_idx - 0.5, cp_idx - 0.5), 1, 1,
                                fill=False, edgecolor='black', lw=2))

plt.suptitle('All-drug CF gene effect correlation per condition', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f'drug_pair_CF_correlation_heatmap.svg'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f'drug_pair_CF_correlation_heatmap.png'), dpi=300, bbox_inches='tight')
plt.show()


### Fig. S5b: significance of downstream effects per condition (10 µM)

In [ ]:
# --- Publication-ready wrapper: all conditions ---
# figure_dir is defined in cell 1
FIGSIZE = (2, 16)
DENDRO_RATIO = 0.03

from unittest.mock import patch

cond_labels = data.uns['results']['conditions']
n_conds = len(data.uns['results']['counterfactual_effect_size'])

_orig_clustermap = sns.clustermap

def _make_patched(captured):
    def _patched_clustermap(*args, **kwargs):
        data_T = args[0].T
        args = (data_T,) + args[1:]
        row_cluster = kwargs.pop('row_cluster', True)
        col_cluster = kwargs.pop('col_cluster', False)
        kwargs['row_cluster'] = col_cluster
        kwargs['col_cluster'] = row_cluster
        kwargs['figsize'] = FIGSIZE
        kwargs['dendrogram_ratio'] = (0.1, DENDRO_RATIO)
        kwargs['cmap'] = 'viridis'
        kwargs['linewidths'] = 0
        kwargs['cbar_pos'] = (1.38, 0.15, 0.05, 0.3)
        g = _orig_clustermap(*args, **kwargs)
        captured['g'] = g
        return g
    return _patched_clustermap

for cond_idx in range(n_conds):
    cond_label = cond_labels[cond_idx]
    print(f'Plotting condition {cond_idx}: {cond_label}')
    _captured = {}
    with patch.object(sns, 'clustermap', _make_patched(_captured)):
        with patch.object(plt, 'show', lambda: None):
            slk.pl.plot_ev_sig(
                data,
                cond_idx=cond_idx,
                uns_key='results',
                alpha=0.05,
                top_n_genes=100,
                min_sig_genes=3,
                cluster_rows=True,
                cluster_cols=False,
                use_clustermap=True,
            )
    if 'g' not in _captured:
        print(f'  -> No significant results, skipping.')
        continue
    g = _captured['g']
    n_genes = g.data2d.shape[0]
    g.ax_heatmap.set_title('')
    g.ax_heatmap.set_xticks(range(g.data2d.shape[1]))
    g.ax_heatmap.set_xticklabels(list(g.data2d.columns), fontsize=8, rotation=90)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=10, rotation=0)
    g.ax_heatmap.set_xlabel(f'{concentration}um | {cond_label}', fontsize=10)
    g.ax_heatmap.set_ylabel(f'{n_genes} genes', fontsize=10)
    g.cax.set_ylabel('-log10(q-value)', fontsize=9)
    pos = g.ax_heatmap.get_position()
    new_x0 = 0.0
    new_width = 0.88
    g.ax_heatmap.set_position([new_x0, pos.y0, new_width, pos.height])
    dcol = g.ax_col_dendrogram.get_position()
    g.ax_col_dendrogram.set_position([new_x0, dcol.y0, new_width, dcol.height])
    save_name = f'figure_ev_sig_{concentration}um_{tcdg}_{cond_label}'
    for ext in ('svg', 'png', 'pdf'):
        g.fig.savefig(os.path.join(figure_dir, f'{save_name}.{ext}'), dpi=300, bbox_inches='tight')
    # Save q-value matrix as CSV (perturbation x gene)
    qval_matrix = 10 ** (-g.data2d)
    qval_matrix.index.name = 'perturbation'
    qval_matrix.columns.name = 'gene'
    qval_matrix.to_csv(os.path.join(figure_dir, f'{save_name}_qvalues.csv'))
    print(f'  -> Saved to {figure_dir}: {save_name}.svg/png/pdf + _qvalues.csv')
    plt.show()
# cluster on effect size, and this top 100 genes. Cluster on the whole gene list. 

In [ ]:
# Get condition information
cond_labels = data.uns['results']['conditions']
n_conds = len(data.uns['results']['counterfactual_effect_size'])

# Save filtered significant results per condition
for cond_idx in range(n_conds):
    cond_label = cond_labels[cond_idx]
    slk.tl.ev_sig(data, cond_idx=cond_idx)
    # Get long-form data for this condition
    df_sig = slk.pl.make_ev_long_df(data, cond_idx=cond_idx)
    
    # Filter for significant q-values only
    df_sig = df_sig[df_sig["qval"] < 0.05]
    
    # Skip if no significant results
    if df_sig.empty:
        print(f"Condition {cond_idx} ({cond_label}): No significant results (q < 0.05)")
        continue
    
    # Create filename
    save_name = f'filtered_significant_{pert_type}_{concentration}_{cond_label}.csv'
    save_path = os.path.join(figure_dir, save_name)
    
    # Save to CSV
    df_sig.to_csv(save_path, index=False)
    
    print(f"Condition {cond_idx} ({cond_label}): {len(df_sig)} significant pairs saved")
    print(f"  → {save_path}\n")

In [ ]:
# Genes significantly regulated by both ALWII4127 (AL) and CP673451 (CP)
# Uses the ev_sig results already computed for cond_idx=1

alpha = 0.05
cond_idx = 1

df_ev = slk.pl.make_ev_long_df(data, cond_idx=cond_idx, uns_key='results')
df_ev_sig = df_ev[df_ev['qval'] < alpha]

genes_AL = set(df_ev_sig[df_ev_sig['pert'] == 'ALWII4127']['gene'])
genes_CP = set(df_ev_sig[df_ev_sig['pert'] == 'CP673451']['gene'])

shared_genes = genes_AL & genes_CP

print(f'Significant genes in ALWII4127 : {len(genes_AL)}')
print(f'Significant genes in CP673451  : {len(genes_CP)}')
print(f'Shared genes                   : {len(shared_genes)}')
print()

# Show shared genes with EV and q-value for each drug
df_shared = df_ev_sig[df_ev_sig['gene'].isin(shared_genes) &
                       df_ev_sig['pert'].isin(['ALWII4127', 'CP673451'])]
df_shared = (df_shared[['pert', 'gene', 'ev', 'qval']]
             .pivot(index='gene', columns='pert', values=['ev', 'qval'])
             .sort_values(('ev', 'ALWII4127'), ascending=False))
df_shared.columns = ['ev_AL', 'ev_CP', 'qval_AL', 'qval_CP']
print(df_shared.to_string())
df_shared.to_csv(os.path.join(figure_dir, f'shared_AL_CP_{concentration}um_with_T_cell_conditions.csv'))


In [ ]:
# Genes significantly regulated by BOTH AL and CP, but NO other drug

# Significant genes per drug (all drugs)
genes_per_drug = df_ev_sig.groupby('pert')['gene'].apply(set).to_dict()

genes_other = set()
for drug, genes in genes_per_drug.items():
    if drug not in ('ALWII4127', 'CP673451'):
        genes_other |= genes

exclusive_genes = shared_genes - genes_other

print(f'Shared AL+CP genes          : {len(shared_genes)}')
print(f'Also in other drugs         : {len(shared_genes & genes_other)}')
print(f'Exclusive to AL+CP only     : {len(exclusive_genes)}')
print()

df_exclusive = df_ev_sig[df_ev_sig['gene'].isin(exclusive_genes) &
                          df_ev_sig['pert'].isin(['ALWII4127', 'CP673451'])]
df_exclusive = (df_exclusive[['pert', 'gene', 'ev', 'qval']]
                .pivot(index='gene', columns='pert', values=['ev', 'qval'])
                .sort_values(('ev', 'ALWII4127'), ascending=False))
df_exclusive.columns = ['ev_AL', 'ev_CP', 'qval_AL', 'qval_CP']
print(df_exclusive.to_string())
df_exclusive.to_csv(os.path.join(figure_dir, f'exclusive_AL_CP_{concentration}um_with_T_cell_conditions.csv'))


#immune / antigen-presentation genes
These are the easiest to interpret and strongest for your story.
STAT1 — core interferon-response transcription factor
HLA-E — nonclassical MHC-I, immune recognition
UBE2L6 — interferon-inducible ubiquitin-conjugating enzyme
LY6E — interferon-stimulated / immune-modulatory gene
PSME1 — immunoproteasome/antigen-processing related
DDX60 — antiviral innate immune sensor/ISG
HLA-DMA — MHC-II peptide loading
ERAP1 — trims peptides for MHC-I presentation
PSMB9 — immunoproteasome subunit
HLA-C — classical MHC-I
PSMB8 — immunoproteasome subunit

#Adhesion / membrane / tumor–immune interface
These are also very relevant biologically.
EMP1 — membrane protein, often linked to invasive/mesenchymal states
ITGA6 — integrin alpha-6, adhesion / stem-like and invasive behavior
ACTN1 — actin cytoskeleton / adhesion
LOXL1 — extracellular matrix remodeling
SVIL — actin-associated cytoskeletal protein
SH3D19 — membrane trafficking / signaling scaffold
LSAMP — cell adhesion molecule
COL6A1 — ECM / mesenchymal matrix
CAVIN1 — caveolae / membrane organization
AHNAK — large scaffold protein, membrane/cytoskeleton
ITGAV — integrin alpha-V, ECM interaction / migration
ANXA2 — membrane organization, invasion, fibrinolysis

#Cytoskeleton / trafficking / migration
CLASP2 — microtubule stabilization
CLIP1 — microtubule-binding trafficking protein
NAV2 — cytoskeletal guidance / migration-related
EPN2 — endocytosis
TMSB10 — actin dynamics
ARHGEF7 — Rho GTPase signaling / cytoskeleton
ARHGEF9 — Rho signaling scaffold
CALCOCO2 — autophagy cargo receptor, also stress response

#Neural / lineage / glioma-state genes
These are important because they suggest tumor-state remodeling, not just immune activation.
APBB2 — neuronal adaptor-related
PBX1 — developmental transcription factor
KANTR — lncRNA near developmental programs
SNHG14 — lncRNA often linked to neural/glioma contexts
FRMPD3 — neuronal/synaptic-associated scaffold
LDLRAD3 — neuronal membrane-associated receptor family
LRRC4C — neuronal adhesion molecule
DNPEP — aminopeptidase, less specific but seen in neural contexts
GUK1 — nucleotide metabolism, not lineage-specific
S100A16 / S100A13 — stress / calcium-binding proteins, can mark reactive states

In [ ]:
p_key = slk.configs.get_config('pert_key')
NTC = slk.configs.get_config('ntc_label')

sub = data[data.obs[p_key] != NTC]

sc.pp.neighbors(sub, n_neighbors=15, use_rep='z', random_state=0)
sc.tl.umap(sub, random_state=0)

import random
n_cats = sub.obs[p_key].nunique()
palette = sns.color_palette("hsv", n_colors=n_cats)
random.seed(0)
random.shuffle(palette)
sc.pl.umap(sub, color='drug', palette=palette, frameon=False)

# save figures pdf, png, svg

In [ ]:
palette = sns.color_palette("hsv", n_colors=2)

In [ ]:
sc.pl.umap(sub, color='tcell_treatment', palette=palette, frameon=False)

In [ ]:
zrc = data.uns['results']['z_rho_corr']

print(f"Spearman ρ = {zrc['spearman_r']:.3f}  (p = {zrc['spearman_p']:.1e})")
print(f" Pearson  r = {zrc['pearson_r']:.3f}  (p = {zrc['pearson_p']:.1e})") 

In [ ]:
slk.pl.plot_zcorr(data)

In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list

def plot_zcorr_with_names(adata, uns_key='results'):
    # Get config values
    pert_key = slk.configs.get_config('pert_key')
    ntc_label = slk.configs.get_config('ntc_label')
    
    # Get perturbation names in the same order as the model (sorted unique, excluding NTC)
    # This matches the logic in PerturbMatchingDataset
    pert_vals = adata.obs[pert_key].values
    pert_vals = pert_vals[pert_vals != ntc_label]
    pert_names = np.unique(pert_vals)
    
    # Get matrices
    Sigma_P_posterior = adata.uns[uns_key]['rho_corr']
    Sigma_z = adata.uns[uns_key]['z_corr']

    # Compute clustering order
    link = linkage(Sigma_P_posterior, method="average", metric="euclidean")
    order = leaves_list(link)              # permutation indices, shape (P,)

    # Reorder matrices
    SigmaP_ord = Sigma_P_posterior[order][:, order]
    Sigmaz_ord = Sigma_z[order][:, order]
    
    # Reorder names
    pert_names_ord = pert_names[order]

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

    sns.heatmap(SigmaP_ord, cmap="coolwarm", square=True, ax=axes[0], 
                xticklabels=pert_names_ord, yticklabels=pert_names_ord)
    axes[0].set_title(r"$\Sigma_P$  (posterior row covariance)")
    axes[0].set_xlabel("perturbation")
    axes[0].set_ylabel("perturbation")

    sns.heatmap(Sigmaz_ord, cmap="coolwarm", square=True, ax=axes[1],
                xticklabels=pert_names_ord, yticklabels=pert_names_ord)
    axes[1].set_title("Correlation of mean $z$")
    axes[1].set_xlabel("perturbation")

    plt.tight_layout()
    plt.savefig(os.path.join(figure_dir, f"{concentration}_correlation.svg"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{concentration}_correlation.png"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{concentration}_correlation.pdf"), dpi=300, bbox_inches='tight')
    plt.show()

plot_zcorr_with_names(data)

In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list

def plot_zcorr_with_names(adata, uns_key='results'):
    # Get config values
    pert_key = slk.configs.get_config('pert_key')
    ntc_label = slk.configs.get_config('ntc_label')
    
    # Get perturbation names in the same order as the model (sorted unique, excluding NTC)
    # This matches the logic in PerturbMatchingDataset
    pert_vals = adata.obs[pert_key].values
    pert_vals = pert_vals[pert_vals != ntc_label]
    pert_names = np.unique(pert_vals)
    
    # Get matrices
    Sigma_P_posterior = adata.uns[uns_key]['rho_corr']

    # Compute clustering order
    link = linkage(Sigma_P_posterior, method="average", metric="euclidean")
    order = leaves_list(link)              # permutation indices, shape (P,)

    # Reorder matrices
    SigmaP_ord = Sigma_P_posterior[order][:, order]
    
    # Reorder names
    pert_names_ord = pert_names[order]

    # Plot only left heatmap
    fig, ax = plt.subplots(figsize=(8, 6))

    sns.heatmap(SigmaP_ord, cmap="coolwarm", square=True, ax=ax, 
                xticklabels=pert_names_ord, yticklabels=pert_names_ord)
    ax.set_title(r"$\Sigma_P$  (posterior row covariance)")
    #ax.set_xlabel("perturbation")
    #ax.set_ylabel("perturbation")

    plt.tight_layout()
    plt.savefig(os.path.join(figure_dir, f"{concentration}_correlation.svg"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{concentration}_correlation.png"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{concentration}_correlation.pdf"), dpi=300, bbox_inches='tight')
    plt.show()

plot_zcorr_with_names(data)

In [ ]:
W = data.uns['results']['W']
plt.hist(W.flatten(), bins=30)

# save this figure
plt.savefig(os.path.join(figure_dir, f"{concentration}_W_histogram.svg"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_W_histogram.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_W_histogram.pdf"), dpi=300, bbox_inches='tight')     
plt.show()

In [ ]:
W_bin = (W > 0.5).astype(float)
sns.heatmap(W, cmap="coolwarm")

# save this figure
plt.savefig(os.path.join(figure_dir, f"{concentration}_W_heatmap.svg"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_W_heatmap.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_W_heatmap.pdf"), dpi=300, bbox_inches='tight')     

In [ ]:
dataset = slk.tl.PerturbMatchingDataset(data)
all_genes = dataset.perturbation_dict.keys()

In [ ]:
CLUSTER_T = 0.7  # single source of truth for the clustering cut used by the dendrogram, heatmap, and bipartite-graph coloring below
groups = slk.tl.cluster_rho(data, t=CLUSTER_T, show=True)

In [ ]:
new_colormap = [
    'darkslateblue',  # 0
    'firebrick',      # 1
    'darkorange',     # 2
    'forestgreen',    # 3
    'gold',           # 4
    'dodgerblue',     # 5
    'mediumorchid',   # 6
    'deeppink',       # 7
    'teal',           # 8
    'black'           # 9
]

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, set_link_color_palette
from matplotlib.colors import to_hex

def cluster_rho(adata, t=1.5, uns_key='results', rho_corr=None, show=True, pert_key=None,
                group_together=None, figure_dir=None, save_name=None):
    if pert_key is None:
        pert_key = slk.configs.get_config('pert_key')
    if rho_corr is None:
        C = adata.uns[uns_key]['rho_corr']
    else:
        C = rho_corr

    # Get perturbation names - try multiple possible locations
    if 'pert_names' in adata.uns.get(uns_key, {}):
        pert_names = adata.uns[uns_key]['pert_names']
    elif 'perts' in adata.uns.get(uns_key, {}):
        pert_names = adata.uns[uns_key]['perts']
    else:
        pert_names = np.unique(adata.obs[pert_key].values).tolist()
        _ntc_labels = {'control', 'ctrl', 'non-targeting', 'dmso', 'vehicle'}
        pert_names = [p for p in pert_names if p.lower() not in _ntc_labels]

    D = 1.0 - C  # distance
    Z_link = linkage(D[np.triu_indices_from(D, 1)], method='ward')

    # Auto-raise t so that all group_together perturbations land in the same cluster
    if group_together:
        name_to_idx = {name: i for i, name in enumerate(pert_names)}
        target_idxs = {name_to_idx[n] for n in group_together if n in name_to_idx}
        if len(target_idxs) >= 2:
            n = len(pert_names)
            clusters = {i: {i} for i in range(n)}
            merge_height = t
            for k, (left, right, dist, _) in enumerate(Z_link):
                left, right = int(left), int(right)
                merged = clusters[left] | clusters[right]
                clusters[n + k] = merged
                if target_idxs.issubset(merged):
                    merge_height = float(dist)
                    break
            if merge_height > t:
                print(f'Adjusting t: {t} -> {merge_height:.4f} to group {group_together} together')
                t = merge_height + 1e-6

    if show:
        plt.figure(figsize=(max(3, len(pert_names) * 0.15), 4))
        _ug = sorted(set(fcluster(Z_link, t=t, criterion='distance')))
        _pal = [to_hex(new_colormap[i % len(new_colormap)]) for i in range(len(_ug))]
        set_link_color_palette(_pal)
        ddata_vis = dendrogram(
            Z_link,
            color_threshold=t,
            above_threshold_color="gray",
            labels=pert_names,
            leaf_rotation=90,
            leaf_font_size=8,
        )
        plt.axhline(y=t, color="#2CA02C", linestyle="--", linewidth=1)
        if 'Tyrphostin9' in list(pert_names):
            _t9_pos = ddata_vis['ivl'].index('Tyrphostin9') if 'Tyrphostin9' in ddata_vis['ivl'] else None
            if _t9_pos is not None:
                _t9_x = 5 + _t9_pos * 10
                for line in plt.gca().get_lines():
                    if any(abs(x - _t9_x) < 0.1 for x in line.get_xdata()):
                        line.set_color('#2CA02C')
        plt.xlabel('Perturbations')
        plt.ylabel('Distance')
        plt.tight_layout()
        if figure_dir and save_name:
            plt.savefig(os.path.join(figure_dir, f'{save_name}_dendrogram.svg'), dpi=300, bbox_inches='tight')
            plt.savefig(os.path.join(figure_dir, f'{save_name}_dendrogram.png'), dpi=300, bbox_inches='tight')
            plt.savefig(os.path.join(figure_dir, f'{save_name}_dendrogram.pdf'), dpi=300, bbox_inches='tight')
            print(f'Saved figures to {figure_dir}/{save_name}_dendrogram.[svg|png|pdf]')
        plt.show()

    groups = fcluster(Z_link, t=t, criterion='distance')

    unique_groups = sorted(set(groups))
    palette_hex = [to_hex(new_colormap[i % len(new_colormap)]) for i in range(len(unique_groups))]
    set_link_color_palette(palette_hex)
    ddata = dendrogram(Z_link, color_threshold=t, above_threshold_color='gray', no_plot=True)
    leaf_colors = ddata.get('leaves_color_list', [])
    group_to_color = {}
    for leaf_idx, color in zip(ddata['leaves'], leaf_colors):
        g = groups[leaf_idx]
        if g not in group_to_color:
            group_to_color[g] = color
    adata.uns['_dendrogram_colors'] = {str(k): v for k, v in group_to_color.items()}

    cluster_assignments = dict(zip(pert_names, groups))
    return groups, cluster_assignments


### Fig. S4a (10 µM): hierarchical clustering of drugs

In [ ]:
cluster_rho(data, t=CLUSTER_T, show=True, figure_dir=figure_dir, save_name=f"{pert_type}_{concentration}uM")

In [ ]:
#for a non subsetted data, pass in genes corresponding to idx ie dataset.perturbation_dict.keys()
named_groups = slk.tl.group2name(groups, all_genes)
named_groups

In [ ]:
# map perturbation -> group using named_groups (index=gene, column='group')
gene_to_group = named_groups['group'].to_dict()

# assign group to obs based on slk pert key
sub.obs['pert_group'] = sub.obs[p_key].map(gene_to_group)

# mark NTC / unknown perturbations as -1 and ensure integer dtype
sub.obs['pert_group'] = sub.obs['pert_group'].fillna(-1).astype('category')

# quick summary
print(sub.obs['pert_group'].value_counts().sort_index())

sc.pl.umap(sub, color=['pert_group', 'tcell_treatment','drug'], title='Perturbation Groups', size=30)

In [ ]:
from __future__ import annotations

from scipy.cluster.hierarchy import linkage, leaves_list
from matplotlib import gridspec
from matplotlib.patches import Patch

def plot_zcorr_colored(adata, t=1.5, uns_key='results', show_pert_labels=True, group_color=new_colormap, pert_key=None):
    """
    Plot correlation matrices with color bars for each perturbation group.
    
    Parameters
    ----------
    adata : AnnData
        AnnData object with results in adata.uns[uns_key].
    uns_key : str, default 'results'
        Key in adata.uns where results are stored.
    show_pert_labels : bool, default True
        Whether to show perturbation labels on axes.
    group_col : str, default 'pert_group'
        Column in adata.obs containing perturbation groups.
    """
    
    if pert_key is None:
        pert_key = slk.configs.get_config('pert_key')
    Sigma_P_posterior = adata.uns[uns_key]['rho_corr']
    Sigma_z = adata.uns[uns_key]['z_corr']
    if 'pert_names' in adata.uns.get(uns_key, {}):
        pert_names = np.array(adata.uns[uns_key]['pert_names'])
    elif 'perts' in adata.uns.get(uns_key, {}):
        pert_names = np.array(adata.uns[uns_key]['perts'])
    else:
        _ntc_labels = {'control', 'ctrl', 'non-targeting', 'dmso', 'vehicle'}
        pert_names = np.array([p for p in np.unique(adata.obs[pert_key].values)
                               if p.lower() not in _ntc_labels])

    # Same linkage as cluster_rho: ward on distance matrix
    from scipy.cluster.hierarchy import fcluster
    D = 1.0 - Sigma_P_posterior
    Z_link = linkage(D[np.triu_indices_from(D, 1)], method='ward')
    order = leaves_list(Z_link)

    # Apply ordering to both matrices
    SigmaP_ord = Sigma_P_posterior[order][:, order]
    Sigmaz_ord = Sigma_z[order][:, order]
    pert_names_ord = pert_names[order]

    # Same color logic as cluster_rho
    from scipy.cluster.hierarchy import fcluster, set_link_color_palette, dendrogram
    from matplotlib.colors import to_hex
    groups = fcluster(Z_link, t=t, criterion='distance')
    groups_ord = groups[order]
    unique_groups = sorted(set(groups_ord))
    # Set same palette as cluster_rho
    palette_hex = [to_hex(group_color[i % len(group_color)]) for i in range(len(unique_groups))]
    set_link_color_palette(palette_hex)
    # Extract leaf→color via dendrogram (no_plot)
    ddata = dendrogram(Z_link, color_threshold=t, above_threshold_color='#2CA02C', no_plot=True)
    leaf_colors = ddata.get('leaves_color_list', [])
    group_to_color = {}
    for leaf_idx, color in zip(ddata['leaves'], leaf_colors):
        g = groups[leaf_idx]
        if g not in group_to_color:
            group_to_color[g] = color
    # Fallback if leaves_color_list unavailable
    for grp in unique_groups:
        if grp not in group_to_color:
            group_to_color[grp] = palette_hex[unique_groups.index(grp)]
    row_colors = [group_to_color[grp] for grp in groups_ord]
    
    n_perts = len(pert_names_ord)

    # Thickness controls for color bars
    #top_bar_height =0   # was 0.5
    side_bar_width = 0.8   # was 0.5

    # Plot single heatmap
    fig = plt.figure(figsize=(4, 3))
    ax0 = fig.add_subplot(111)
    
    sns.heatmap(SigmaP_ord, cmap="coolwarm", square=True, ax=ax0, 
                xticklabels=pert_names_ord if show_pert_labels else False,
                yticklabels=pert_names_ord if show_pert_labels else False,
                cbar_kws={'label': 'Correlation', 'pad': 0.15})
    ax0.set_title("")
    ax0.set_xlabel("Perturbations")
    ax0.set_ylabel("Perturbations")

    # Make font size smaller
    if show_pert_labels:
        ax0.set_xticklabels(ax0.get_xticklabels(), fontsize=7)
        ax0.set_yticklabels(ax0.get_yticklabels(), fontsize=7)
    
    gap = 0.5  # space between heatmap and group color strip

    # Add color bar on right (thicker)
    for i, color in enumerate(row_colors):
        # Top
        #ax0.add_patch(plt.Rectangle((i, -0.5), 1, top_bar_height, color=color, 
                                    #transform=ax0.transData, clip_on=False))
        # Right
        ax0.add_patch(plt.Rectangle((n_perts + gap, i), side_bar_width, 1, color=color, 
                                    transform=ax0.transData, clip_on=False))

    # Create legend outside with transparent background
    legend_elements = [Patch(facecolor=group_to_color[grp], label=f'Group {grp}') 
                      for grp in unique_groups]
    fig.legend(
        handles=legend_elements,
        loc='center left',
        bbox_to_anchor=(1.0, 0.5),
        title='Perturbation Groups',
        fontsize=8,
        frameon=False
    )

    plt.tight_layout()
    plt.savefig(os.path.join(figure_dir, f"{pert_type}_{concentration}_correlation_color.svg"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{pert_type}_{concentration}_correlation_color.png"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{pert_type}_{concentration}_correlation_color.pdf"), dpi=300, bbox_inches='tight')
    plt.show()

plot_zcorr_colored(sub, t=CLUSTER_T)



In [ ]:
palette1 = [
"#FFD166",  
"#023047",  
]

palette2 = [
    "#EF476F", "#0077B6", "#8338EC", "#FB5607",  "#F7A1C4",
    "#00D9FF", "#38A169", "#8B5E3C", "#4A5568", "#2EC4B6", "#9B2226"
]

palette2_very_light = [
    "#EF476F",  # keep
    "#A8D8F0",  # very light blue
    "#D6C7FA",  # very light purple
    "#FB5607",  # keep
    "#FDD9E7",  # very light pink
    "#CFF5FF",  # very light cyan
    "#CDEEDD",  # very light green
    "#E3D2C4",  # very light brown
    "#D6DEE8",  # very light gray-blue
    "#CFF1EA",  # very light teal
    "#E8B3B3"   # very light dark red
]



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

sc.pl.umap(sub, color='tcell_treatment', title='tcell_treatment', 
           palette=palette1, size=12, frameon=False, ax=axes[0], show=False,legend_loc=None)

sc.pl.umap(sub, color='drug', title='drug', 
           palette=palette2_very_light, size=12, frameon=False, ax=axes[1], show=False,legend_loc=None)

plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f"{concentration}_sherlock_zspace.svg"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_sherlock_zspace.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_sherlock_zspace.pdf"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

sc.pl.umap(sub, color='tcell_treatment', title='tcell_treatment', 
           palette=palette1, size=12, frameon=False, ax=axes[0], show=False)

sc.pl.umap(sub, color='drug', title='drug', 
           palette=palette2_very_light, size=12, frameon=False, ax=axes[1], show=False)

plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f"{concentration}_sherlock_zspace_with_legends.svg"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_sherlock_zspace_with_legends.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_sherlock_zspace_with_legends.pdf"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Print gene symbols per group, one gene per line (no quotation marks)
grouped = named_groups.groupby('group').apply(lambda df: df.index.tolist())

for grp in sorted(grouped.index):
    print(f"Group {grp}:")
    for gene in grouped.loc[grp]:
        print(gene)
    print()

In [ ]:
pert_key="drug"
ntc_label="DMSO"
treatment_key='tcell_treatment'

In [ ]:
# Get perturbation and condition names
# Extract names
pert_names = (list(data.uns['results']['perts']) if 'perts' in data.uns.get('results', {})
           else [p for p in np.unique(data.obs[slk.configs.get_config('pert_key')].values)
                 if p.lower() not in {'control','ctrl','non-targeting','dmso','vehicle'}])    # matches cfs_mat axis 0
cond_names = list(data.uns['results']['conditions'])  # matches cfs_mat axes 1 and 2

# Get the matrix
cfs_mat = data.uns['results']['condition_perturbation_interaction']  # shape: (P, C, C)

# Verify dimensions match
assert cfs_mat.shape[0] == len(pert_names)
assert cfs_mat.shape[1] == len(cond_names)
assert cfs_mat.shape[2] == len(cond_names)

In [ ]:
# Average shift per perturbation (across all condition pairs)
mask = np.triu(np.ones((len(cond_names), len(cond_names))), k=1).astype(bool)
avg_shift_per_pert = np.array([cfs_mat[p][mask].mean() for p in range(len(pert_names))])

# Rank perturbations by context-dependency
df_pert_sensitivity = pd.DataFrame({
    'perturbation': pert_names,
    'condition_effect_on_perturbation': avg_shift_per_pert,
}).sort_values('condition_effect_on_perturbation', ascending=False)

In [ ]:
# Save df_pert_sensitivity for cross-concentration comparison
df_pert_sensitivity.to_csv(
    os.path.join(figure_dir, f'{concentration}um_pert_sensitivity.csv'),
    index=False
)
print(f'Saved {concentration}um_pert_sensitivity.csv')

### Fig. S4b (10 µM): condition dependence of each drug

In [ ]:
# plot bar plot of condition effect on perturbation
plt.figure(figsize=(4, 2))

if hasattr(data.obs['drug'], 'cat'):
    drug_order = [drug for drug in data.obs['drug'].cat.categories if drug != 'DMSO']
else:
    drug_order = sorted([drug for drug in data.obs['drug'].unique() if drug != 'DMSO'])

drug_to_color = dict(zip(drug_order, palette2_very_light))
bar_colors = [drug_to_color.get(drug, 'gray') for drug in df_pert_sensitivity['perturbation']]

sns.barplot(data=df_pert_sensitivity, x='perturbation', y='condition_effect_on_perturbation', palette=bar_colors)
plt.xticks(rotation=45, ha='right')
plt.xlabel('Drug')
plt.ylabel('Conditional\nperturbation response')
#plt.title('Condition Effect on \nPerturbation by Drug')
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f"{concentration}_condition_effect_on_perturbation.svg"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_condition_effect_on_perturbation.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{concentration}_condition_effect_on_perturbation.pdf"), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
df_pert_sensitivity

In [ ]:
slk.pl.plot_cf_bipartite(
    data,
    cond_idx=0,          
    pert_name="CP673451",     
    top_n=15,            # top N target genes by effect size
)

In [ ]:
# --- Publication-ready save for plot_cf_bipartite (figure_1 style) — all conditions ---
from matplotlib.collections import LineCollection
from unittest.mock import patch
import sherlock.plotting._simple as _slk_simple

PERT_X_MARGIN = 0.25  # 0=full spread, 0.5=fully centered
_orig_draw_bipartite = _slk_simple._draw_bipartite_nx

def _centered_draw_bipartite(pert_nodes, gene_nodes, edges, ax, **kwargs):
    n_p, n_g = len(pert_nodes), len(gene_nodes)
    if n_p == 1:
        orig_pert_xs = [0.5]
        new_pert_xs  = [0.5]
    else:
        orig_pert_xs = [i / (n_p - 1) for i in range(n_p)]
        new_pert_xs  = [PERT_X_MARGIN + i * (1 - 2*PERT_X_MARGIN) / (n_p - 1) for i in range(n_p)]
    x_remap = dict(zip(orig_pert_xs, new_pert_xs))
    _orig_draw_bipartite(pert_nodes, gene_nodes, edges, ax,
                         pert_color='#26547c',
                         gene_color='#D0D0D0',
                         gene_overlap_color='#1E90FF',
                         **kwargs)
    for coll in ax.collections:
        offsets = coll.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue
        if np.all(offsets[:, 1] == 1.0) and len(offsets) == n_p:
            coll.set_offsets(np.column_stack([new_pert_xs, np.ones(n_p)]))
    for txt, x in zip(ax.texts[:n_p], new_pert_xs):
        _, old_y = txt.get_position()
        txt.set_position((x, old_y))
    for coll in ax.collections:
        if not isinstance(coll, LineCollection):
            continue
        segs = coll.get_segments()
        new_segs = []
        for seg in segs:
            new_seg = []
            for pt in seg:
                x, y = pt
                if y == 1.0:
                    closest = min(orig_pert_xs, key=lambda ox: abs(ox - x))
                    x = x_remap.get(closest, x)
                new_seg.append([x, y])
            new_segs.append(new_seg)
        coll.set_segments(new_segs)

# Build pert→group color mapping (same clustering as plot_zcorr_colored)
from scipy.cluster.hierarchy import fcluster, set_link_color_palette, dendrogram
from matplotlib.colors import to_hex
_rho = data.uns['results']['rho_corr']
_pnames = (list(data.uns['results']['perts']) if 'perts' in data.uns.get('results', {})
           else [p for p in np.unique(data.obs[slk.configs.get_config('pert_key')].values)
                 if p.lower() not in {'control','ctrl','non-targeting','dmso','vehicle'}])
_D = 1.0 - _rho
_Z = linkage(_D[np.triu_indices_from(_D, 1)], method='ward')
_groups = fcluster(_Z, t=CLUSTER_T, criterion='distance')
_ugrps = sorted(set(_groups))
_pal = [to_hex(new_colormap[i % len(new_colormap)]) for i in range(len(_ugrps))]
set_link_color_palette(_pal)
_ddata = dendrogram(_Z, color_threshold=CLUSTER_T, above_threshold_color='#2CA02C', no_plot=True)
_g2c = {}
for _li, _lc in zip(_ddata['leaves'], _ddata.get('leaves_color_list', [])):
    if _groups[_li] not in _g2c:
        _g2c[_groups[_li]] = _lc
for _grp, _col in zip(_ugrps, _pal):
    _g2c.setdefault(_grp, _col)
pert_to_color = {n: _g2c[g] for n, g in zip(_pnames, _groups)}

cond_labels = data.uns['results']['conditions']
_orig_get_cf_data = _slk_simple._get_cf_data

def _make_qval_masked_cf(qv, cidx):
    def _fn(adata, uns_key='results', cf_key='counterfactual_effect_size'):
        cf, perts, genes = _orig_get_cf_data(adata, uns_key=uns_key, cf_key=cf_key)
        cf = cf.copy()
        cf[cidx][qv >= 0.05] = 0.0
        n_sig = int((qv < 0.05).sum())
        print(f'  cond {cidx}: {n_sig}/{qv.size} (pert,gene) pairs pass qval<0.05')
        return cf, perts, genes
    return _fn

for cond_idx in range(len(cond_labels)):
    cond_label = cond_labels[cond_idx]
    _qvals = np.asarray(data.uns['results']['ev_results'][cond_idx]['qvals'], dtype=float)  # (P, G)
    _cf_fig = {}
    def _show_capture():
        _cf_fig['fig'] = plt.gcf()
    with patch.object(_slk_simple, '_draw_bipartite_nx', _centered_draw_bipartite), \
         patch.object(_slk_simple, '_get_cf_data', _make_qval_masked_cf(_qvals, cond_idx)), \
         patch.object(plt, 'show', _show_capture):
        slk.pl.plot_cf_bipartite(
            data,
            cond_idx=cond_idx,
            pert_name=["CP673451", "ALWII4127"],
            top_n=15,
            min_effect=1e-10,
            figsize=(6, 2.8),
        )
    fig = _cf_fig['fig']
    ax = fig.axes[0]
    ax.set_title('')  # Remove title
    for coll in ax.collections:
        if isinstance(coll, LineCollection):
            coll.set_color('#BBBBBB')
            coll.set_linewidth(1.8)
            coll.set_alpha(0.7)
    for txt in ax.texts:
        txt.set_fontsize(8)
        txt.set_color('#444444')
        txt.set_fontweight('bold')
    # Shrink node circles
    for coll in ax.collections:
        if not isinstance(coll, LineCollection):
            sizes = coll.get_sizes()
            if len(sizes) > 0:
                coll.set_sizes([200])
    legend = ax.get_legend()
    if legend:
        legend.remove()
    # Color pert nodes by their group color
    _pert_list = ["CP673451", "ALWII4127"]
    _node_colors = [pert_to_color.get(p, '#26547c') for p in _pert_list]
    for coll in ax.collections:
        offsets = coll.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue
        if np.all(offsets[:, 1] == 1.0) and len(offsets) == len(_pert_list):
            coll.set_facecolor(_node_colors)
            coll.set_edgecolor(_node_colors)
    plt.tight_layout(pad=0.3)
    save_name = f"{concentration}_cf_bipartite_CP673451_ALWII4127_{cond_label}"
    for ext in ('svg', 'png', 'pdf'):
        fig.savefig(os.path.join(figure_dir, f'{save_name}.{ext}'), dpi=300, bbox_inches='tight', transparent=True)
    print(f'Saved {save_name}.svg/png/pdf')
    plt.show()


In [ ]:
# --- Publication-ready save for plot_cf_bipartite (figure_1 style) — all conditions ---
from matplotlib.collections import LineCollection
from unittest.mock import patch
import sherlock.plotting._simple as _slk_simple

PERT_X_MARGIN = 0.25  # 0=full spread, 0.5=fully centered
_orig_draw_bipartite = _slk_simple._draw_bipartite_nx

def _centered_draw_bipartite(pert_nodes, gene_nodes, edges, ax, **kwargs):
    n_p, n_g = len(pert_nodes), len(gene_nodes)
    if n_p == 1:
        orig_pert_xs = [0.5]
        new_pert_xs  = [0.5]
    else:
        orig_pert_xs = [i / (n_p - 1) for i in range(n_p)]
        new_pert_xs  = [PERT_X_MARGIN + i * (1 - 2*PERT_X_MARGIN) / (n_p - 1) for i in range(n_p)]
    x_remap = dict(zip(orig_pert_xs, new_pert_xs))
    _orig_draw_bipartite(pert_nodes, gene_nodes, edges, ax,
                         pert_color='#26547c',
                         gene_color='#D0D0D0',
                         gene_overlap_color='#1E90FF',
                         **kwargs)
    for coll in ax.collections:
        offsets = coll.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue
        if np.all(offsets[:, 1] == 1.0) and len(offsets) == n_p:
            coll.set_offsets(np.column_stack([new_pert_xs, np.ones(n_p)]))
    for txt, x in zip(ax.texts[:n_p], new_pert_xs):
        _, old_y = txt.get_position()
        txt.set_position((x, old_y))
    for coll in ax.collections:
        if not isinstance(coll, LineCollection):
            continue
        segs = coll.get_segments()
        new_segs = []
        for seg in segs:
            new_seg = []
            for pt in seg:
                x, y = pt
                if y == 1.0:
                    closest = min(orig_pert_xs, key=lambda ox: abs(ox - x))
                    x = x_remap.get(closest, x)
                new_seg.append([x, y])
            new_segs.append(new_seg)
        coll.set_segments(new_segs)

# Build pert→group color mapping (same clustering as plot_zcorr_colored)
from scipy.cluster.hierarchy import fcluster, set_link_color_palette, dendrogram
from matplotlib.colors import to_hex
_rho = data.uns['results']['rho_corr']
_pnames = (list(data.uns['results']['perts']) if 'perts' in data.uns.get('results', {})
           else [p for p in np.unique(data.obs[slk.configs.get_config('pert_key')].values)
                 if p.lower() not in {'control','ctrl','non-targeting','dmso','vehicle'}])
_D = 1.0 - _rho
_Z = linkage(_D[np.triu_indices_from(_D, 1)], method='ward')
_groups = fcluster(_Z, t=CLUSTER_T, criterion='distance')
_ugrps = sorted(set(_groups))
_pal = [to_hex(new_colormap[i % len(new_colormap)]) for i in range(len(_ugrps))]
set_link_color_palette(_pal)
_ddata = dendrogram(_Z, color_threshold=CLUSTER_T, above_threshold_color='#2CA02C', no_plot=True)
_g2c = {}
for _li, _lc in zip(_ddata['leaves'], _ddata.get('leaves_color_list', [])):
    if _groups[_li] not in _g2c:
        _g2c[_groups[_li]] = _lc
for _grp, _col in zip(_ugrps, _pal):
    _g2c.setdefault(_grp, _col)
pert_to_color = {n: _g2c[g] for n, g in zip(_pnames, _groups)}

cond_labels = data.uns['results']['conditions']
_orig_get_cf_data = _slk_simple._get_cf_data

def _make_qval_masked_cf(qv, cidx):
    def _fn(adata, uns_key='results', cf_key='counterfactual_effect_size'):
        cf, perts, genes = _orig_get_cf_data(adata, uns_key=uns_key, cf_key=cf_key)
        cf = cf.copy()
        cf[cidx][qv >= 0.05] = 0.0
        n_sig = int((qv < 0.05).sum())
        print(f'  cond {cidx}: {n_sig}/{qv.size} (pert,gene) pairs pass qval<0.05')
        return cf, perts, genes
    return _fn

for cond_idx in range(len(cond_labels)):
    cond_label = cond_labels[cond_idx]
    _qvals = np.asarray(data.uns['results']['ev_results'][cond_idx]['qvals'], dtype=float)  # (P, G)
    _cf_fig = {}
    def _show_capture():
        _cf_fig['fig'] = plt.gcf()
    with patch.object(_slk_simple, '_draw_bipartite_nx', _centered_draw_bipartite), \
         patch.object(_slk_simple, '_get_cf_data', _make_qval_masked_cf(_qvals, cond_idx)), \
         patch.object(plt, 'show', _show_capture):
        slk.pl.plot_cf_bipartite(
            data,
            cond_idx=cond_idx,
            pert_name=["CP673451", "BAY1217389"],
            top_n=15,
            min_effect=1e-10,
            figsize=(6, 2.8),
        )
    fig = _cf_fig['fig']
    ax = fig.axes[0]
    ax.set_title('')  # Remove title
    for coll in ax.collections:
        if isinstance(coll, LineCollection):
            coll.set_color('#BBBBBB')
            coll.set_linewidth(1.8)
            coll.set_alpha(0.7)
    for txt in ax.texts:
        txt.set_fontsize(8)
        txt.set_color('#444444')
        txt.set_fontweight('bold')
    # Shrink node circles
    for coll in ax.collections:
        if not isinstance(coll, LineCollection):
            sizes = coll.get_sizes()
            if len(sizes) > 0:
                coll.set_sizes([200])
    legend = ax.get_legend()
    if legend:
        legend.remove()
    # Color pert nodes by their group color
    _pert_list = ["CP673451", "BAY1217389"]
    _node_colors = [pert_to_color.get(p, '#26547c') for p in _pert_list]
    for coll in ax.collections:
        offsets = coll.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue
        if np.all(offsets[:, 1] == 1.0) and len(offsets) == len(_pert_list):
            coll.set_facecolor(_node_colors)
            coll.set_edgecolor(_node_colors)
    plt.tight_layout(pad=0.3)
    save_name = f"{concentration}_cf_bipartite_CP673451_BAY1217389_{cond_label}"
    for ext in ('svg', 'png', 'pdf'):
        fig.savefig(os.path.join(figure_dir, f'{save_name}.{ext}'), dpi=300, bbox_inches='tight', transparent=True)
    print(f'Saved {save_name}.svg/png/pdf')
    plt.show()


In [ ]:

# --- Publication-ready save for plot_cf_bipartite (figure_1 style) — all conditions ---
from matplotlib.collections import LineCollection
from unittest.mock import patch
import sherlock.plotting._simple as _slk_simple

PERT_X_MARGIN = 0.25  # 0=full spread, 0.5=fully centered
_orig_draw_bipartite = _slk_simple._draw_bipartite_nx

def _centered_draw_bipartite(pert_nodes, gene_nodes, edges, ax, **kwargs):
    n_p, n_g = len(pert_nodes), len(gene_nodes)
    if n_p == 1:
        orig_pert_xs = [0.5]
        new_pert_xs  = [0.5]
    else:
        orig_pert_xs = [i / (n_p - 1) for i in range(n_p)]
        new_pert_xs  = [PERT_X_MARGIN + i * (1 - 2*PERT_X_MARGIN) / (n_p - 1) for i in range(n_p)]
    x_remap = dict(zip(orig_pert_xs, new_pert_xs))
    _orig_draw_bipartite(pert_nodes, gene_nodes, edges, ax,
                         pert_color='#26547c',
                         gene_color='#D0D0D0',
                         gene_overlap_color='#1E90FF',
                         **kwargs)
    for coll in ax.collections:
        offsets = coll.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue
        if np.all(offsets[:, 1] == 1.0) and len(offsets) == n_p:
            coll.set_offsets(np.column_stack([new_pert_xs, np.ones(n_p)]))
    for txt, x in zip(ax.texts[:n_p], new_pert_xs):
        _, old_y = txt.get_position()
        txt.set_position((x, old_y))
    for coll in ax.collections:
        if not isinstance(coll, LineCollection):
            continue
        segs = coll.get_segments()
        new_segs = []
        for seg in segs:
            new_seg = []
            for pt in seg:
                x, y = pt
                if y == 1.0:
                    closest = min(orig_pert_xs, key=lambda ox: abs(ox - x))
                    x = x_remap.get(closest, x)
                new_seg.append([x, y])
            new_segs.append(new_seg)
        coll.set_segments(new_segs)

# Build pert→group color mapping (same clustering as plot_zcorr_colored)
from scipy.cluster.hierarchy import fcluster, set_link_color_palette, dendrogram
from matplotlib.colors import to_hex
_rho = data.uns['results']['rho_corr']
_pnames = (list(data.uns['results']['perts']) if 'perts' in data.uns.get('results', {})
           else [p for p in np.unique(data.obs[slk.configs.get_config('pert_key')].values)
                 if p.lower() not in {'control','ctrl','non-targeting','dmso','vehicle'}])
_D = 1.0 - _rho
_Z = linkage(_D[np.triu_indices_from(_D, 1)], method='ward')
_groups = fcluster(_Z, t=CLUSTER_T, criterion='distance')
_ugrps = sorted(set(_groups))
_pal = [to_hex(new_colormap[i % len(new_colormap)]) for i in range(len(_ugrps))]
set_link_color_palette(_pal)
_ddata = dendrogram(_Z, color_threshold=CLUSTER_T, above_threshold_color='#2CA02C', no_plot=True)
_g2c = {}
for _li, _lc in zip(_ddata['leaves'], _ddata.get('leaves_color_list', [])):
    if _groups[_li] not in _g2c:
        _g2c[_groups[_li]] = _lc
for _grp, _col in zip(_ugrps, _pal):
    _g2c.setdefault(_grp, _col)
pert_to_color = {n: _g2c[g] for n, g in zip(_pnames, _groups)}

cond_labels = data.uns['results']['conditions']
_orig_get_cf_data = _slk_simple._get_cf_data

def _make_qval_masked_cf(qv, cidx):
    def _fn(adata, uns_key='results', cf_key='counterfactual_effect_size'):
        cf, perts, genes = _orig_get_cf_data(adata, uns_key=uns_key, cf_key=cf_key)
        cf = cf.copy()
        cf[cidx][qv >= 0.05] = 0.0
        n_sig = int((qv < 0.05).sum())
        print(f'  cond {cidx}: {n_sig}/{qv.size} (pert,gene) pairs pass qval<0.05')
        return cf, perts, genes
    return _fn

for cond_idx in range(len(cond_labels)):
    cond_label = cond_labels[cond_idx]
    _qvals = np.asarray(data.uns['results']['ev_results'][cond_idx]['qvals'], dtype=float)  # (P, G)
    _cf_fig = {}
    def _show_capture():
        _cf_fig['fig'] = plt.gcf()
    with patch.object(_slk_simple, '_draw_bipartite_nx', _centered_draw_bipartite), \
         patch.object(_slk_simple, '_get_cf_data', _make_qval_masked_cf(_qvals, cond_idx)), \
         patch.object(plt, 'show', _show_capture):
        slk.pl.plot_cf_bipartite(
            data,
            cond_idx=cond_idx,
            pert_name=["CP673451", "Tyrphostin9"],
            top_n=15,
            min_effect=1e-10,
            figsize=(6, 2.8),
        )
    fig = _cf_fig['fig']
    ax = fig.axes[0]
    ax.set_title('')  # Remove title
    for coll in ax.collections:
        if isinstance(coll, LineCollection):
            coll.set_color('#BBBBBB')
            coll.set_linewidth(1.8)
            coll.set_alpha(0.7)
    for txt in ax.texts:
        txt.set_fontsize(8)
        txt.set_color('#444444')
        txt.set_fontweight('bold')
    # Shrink node circles
    for coll in ax.collections:
        if not isinstance(coll, LineCollection):
            sizes = coll.get_sizes()
            if len(sizes) > 0:
                coll.set_sizes([200])
    legend = ax.get_legend()
    if legend:
        legend.remove()
    # Color pert nodes by their group color
    _pert_list = ["CP673451", "Tyrphostin9"]
    _node_colors = [pert_to_color.get(p, '#26547c') for p in _pert_list]
    for coll in ax.collections:
        offsets = coll.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue
        if np.all(offsets[:, 1] == 1.0) and len(offsets) == len(_pert_list):
            coll.set_facecolor(_node_colors)
            coll.set_edgecolor(_node_colors)
    plt.tight_layout(pad=0.3)
    save_name = f"{concentration}_cf_bipartite_CP673451_Tyrphostin9_{cond_label}"
    for ext in ('svg', 'png', 'pdf'):
        fig.savefig(os.path.join(figure_dir, f'{save_name}.{ext}'), dpi=300, bbox_inches='tight', transparent=True)
    print(f'Saved {save_name}.svg/png/pdf')
    plt.show()


In [ ]:

# --- Publication-ready save for plot_cf_bipartite (figure_1 style) — all conditions ---
from matplotlib.collections import LineCollection
from unittest.mock import patch
import sherlock.plotting._simple as _slk_simple

PERT_X_MARGIN = 0.25  # 0=full spread, 0.5=fully centered
_orig_draw_bipartite = _slk_simple._draw_bipartite_nx

def _centered_draw_bipartite(pert_nodes, gene_nodes, edges, ax, **kwargs):
    n_p, n_g = len(pert_nodes), len(gene_nodes)
    if n_p == 1:
        orig_pert_xs = [0.5]
        new_pert_xs  = [0.5]
    else:
        orig_pert_xs = [i / (n_p - 1) for i in range(n_p)]
        new_pert_xs  = [PERT_X_MARGIN + i * (1 - 2*PERT_X_MARGIN) / (n_p - 1) for i in range(n_p)]
    x_remap = dict(zip(orig_pert_xs, new_pert_xs))
    _orig_draw_bipartite(pert_nodes, gene_nodes, edges, ax,
                         pert_color='#26547c',
                         gene_color='#D0D0D0',
                         gene_overlap_color='#1E90FF',
                         **kwargs)
    for coll in ax.collections:
        offsets = coll.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue
        if np.all(offsets[:, 1] == 1.0) and len(offsets) == n_p:
            coll.set_offsets(np.column_stack([new_pert_xs, np.ones(n_p)]))
    for txt, x in zip(ax.texts[:n_p], new_pert_xs):
        _, old_y = txt.get_position()
        txt.set_position((x, old_y))
    for coll in ax.collections:
        if not isinstance(coll, LineCollection):
            continue
        segs = coll.get_segments()
        new_segs = []
        for seg in segs:
            new_seg = []
            for pt in seg:
                x, y = pt
                if y == 1.0:
                    closest = min(orig_pert_xs, key=lambda ox: abs(ox - x))
                    x = x_remap.get(closest, x)
                new_seg.append([x, y])
            new_segs.append(new_seg)
        coll.set_segments(new_segs)

# Build pert→group color mapping (same clustering as plot_zcorr_colored)
from scipy.cluster.hierarchy import fcluster, set_link_color_palette, dendrogram
from matplotlib.colors import to_hex
_rho = data.uns['results']['rho_corr']
_pnames = (list(data.uns['results']['perts']) if 'perts' in data.uns.get('results', {})
           else [p for p in np.unique(data.obs[slk.configs.get_config('pert_key')].values)
                 if p.lower() not in {'control','ctrl','non-targeting','dmso','vehicle'}])
_D = 1.0 - _rho
_Z = linkage(_D[np.triu_indices_from(_D, 1)], method='ward')
_groups = fcluster(_Z, t=CLUSTER_T, criterion='distance')
_ugrps = sorted(set(_groups))
_pal = [to_hex(new_colormap[i % len(new_colormap)]) for i in range(len(_ugrps))]
set_link_color_palette(_pal)
_ddata = dendrogram(_Z, color_threshold=CLUSTER_T, above_threshold_color='#2CA02C', no_plot=True)
_g2c = {}
for _li, _lc in zip(_ddata['leaves'], _ddata.get('leaves_color_list', [])):
    if _groups[_li] not in _g2c:
        _g2c[_groups[_li]] = _lc
for _grp, _col in zip(_ugrps, _pal):
    _g2c.setdefault(_grp, _col)
pert_to_color = {n: _g2c[g] for n, g in zip(_pnames, _groups)}

cond_labels = data.uns['results']['conditions']
_orig_get_cf_data = _slk_simple._get_cf_data

def _make_qval_masked_cf(qv, cidx):
    def _fn(adata, uns_key='results', cf_key='counterfactual_effect_size'):
        cf, perts, genes = _orig_get_cf_data(adata, uns_key=uns_key, cf_key=cf_key)
        cf = cf.copy()
        cf[cidx][qv >= 0.05] = 0.0
        n_sig = int((qv < 0.05).sum())
        print(f'  cond {cidx}: {n_sig}/{qv.size} (pert,gene) pairs pass qval<0.05')
        return cf, perts, genes
    return _fn

for cond_idx in range(len(cond_labels)):
    cond_label = cond_labels[cond_idx]
    _qvals = np.asarray(data.uns['results']['ev_results'][cond_idx]['qvals'], dtype=float)  # (P, G)
    _cf_fig = {}
    def _show_capture():
        _cf_fig['fig'] = plt.gcf()
    with patch.object(_slk_simple, '_draw_bipartite_nx', _centered_draw_bipartite), \
         patch.object(_slk_simple, '_get_cf_data', _make_qval_masked_cf(_qvals, cond_idx)), \
         patch.object(plt, 'show', _show_capture):
        slk.pl.plot_cf_bipartite(
            data,
            cond_idx=cond_idx,
            pert_name=["ALWII4127", "Tyrphostin9"],
            top_n=15,
            min_effect=1e-10,
            figsize=(6, 2.8),
        )
    fig = _cf_fig['fig']
    ax = fig.axes[0]
    ax.set_title('')  # Remove title
    for coll in ax.collections:
        if isinstance(coll, LineCollection):
            coll.set_color('#BBBBBB')
            coll.set_linewidth(1.8)
            coll.set_alpha(0.7)
    for txt in ax.texts:
        txt.set_fontsize(8)
        txt.set_color('#444444')
        txt.set_fontweight('bold')
    # Shrink node circles
    for coll in ax.collections:
        if not isinstance(coll, LineCollection):
            sizes = coll.get_sizes()
            if len(sizes) > 0:
                coll.set_sizes([200])
    legend = ax.get_legend()
    if legend:
        legend.remove()
    # Color pert nodes by their group color
    _pert_list = ["ALWII4127", "Tyrphostin9"]
    _node_colors = [pert_to_color.get(p, '#26547c') for p in _pert_list]
    for coll in ax.collections:
        offsets = coll.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue
        if np.all(offsets[:, 1] == 1.0) and len(offsets) == len(_pert_list):
            coll.set_facecolor(_node_colors)
            coll.set_edgecolor(_node_colors)
    plt.tight_layout(pad=0.3)
    save_name = f"{concentration}_cf_bipartite_ALWII4127_Tyrphostin9_{cond_label}"
    for ext in ('svg', 'png', 'pdf'):
        fig.savefig(os.path.join(figure_dir, f'{save_name}.{ext}'), dpi=300, bbox_inches='tight', transparent=True)
    print(f'Saved {save_name}.svg/png/pdf')
    plt.show()


### Fig. S4c (10 µM): downstream genes of CP673451 and STK16IN1 per condition

In [ ]:

# --- Publication-ready save for plot_cf_bipartite (figure_1 style) — all conditions ---
from matplotlib.collections import LineCollection
from unittest.mock import patch
import sherlock.plotting._simple as _slk_simple

PERT_X_MARGIN = 0.25  # 0=full spread, 0.5=fully centered
_orig_draw_bipartite = _slk_simple._draw_bipartite_nx

def _centered_draw_bipartite(pert_nodes, gene_nodes, edges, ax, **kwargs):
    n_p, n_g = len(pert_nodes), len(gene_nodes)
    if n_p == 1:
        orig_pert_xs = [0.5]
        new_pert_xs  = [0.5]
    else:
        orig_pert_xs = [i / (n_p - 1) for i in range(n_p)]
        new_pert_xs  = [PERT_X_MARGIN + i * (1 - 2*PERT_X_MARGIN) / (n_p - 1) for i in range(n_p)]
    x_remap = dict(zip(orig_pert_xs, new_pert_xs))
    _orig_draw_bipartite(pert_nodes, gene_nodes, edges, ax,
                         pert_color='#26547c',
                         gene_color='#D0D0D0',
                         gene_overlap_color='#1E90FF',
                         **kwargs)
    for coll in ax.collections:
        offsets = coll.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue
        if np.all(offsets[:, 1] == 1.0) and len(offsets) == n_p:
            coll.set_offsets(np.column_stack([new_pert_xs, np.ones(n_p)]))
    for txt, x in zip(ax.texts[:n_p], new_pert_xs):
        _, old_y = txt.get_position()
        txt.set_position((x, old_y))
    for coll in ax.collections:
        if not isinstance(coll, LineCollection):
            continue
        segs = coll.get_segments()
        new_segs = []
        for seg in segs:
            new_seg = []
            for pt in seg:
                x, y = pt
                if y == 1.0:
                    closest = min(orig_pert_xs, key=lambda ox: abs(ox - x))
                    x = x_remap.get(closest, x)
                new_seg.append([x, y])
            new_segs.append(new_seg)
        coll.set_segments(new_segs)

# Build pert→group color mapping (same clustering as plot_zcorr_colored)
from scipy.cluster.hierarchy import fcluster, set_link_color_palette, dendrogram
from matplotlib.colors import to_hex
_rho = data.uns['results']['rho_corr']
_pnames = (list(data.uns['results']['perts']) if 'perts' in data.uns.get('results', {})
           else [p for p in np.unique(data.obs[slk.configs.get_config('pert_key')].values)
                 if p.lower() not in {'control','ctrl','non-targeting','dmso','vehicle'}])
_D = 1.0 - _rho
_Z = linkage(_D[np.triu_indices_from(_D, 1)], method='ward')
_groups = fcluster(_Z, t=CLUSTER_T, criterion='distance')
_ugrps = sorted(set(_groups))
_pal = [to_hex(new_colormap[i % len(new_colormap)]) for i in range(len(_ugrps))]
set_link_color_palette(_pal)
_ddata = dendrogram(_Z, color_threshold=CLUSTER_T, above_threshold_color='#2CA02C', no_plot=True)
_g2c = {}
for _li, _lc in zip(_ddata['leaves'], _ddata.get('leaves_color_list', [])):
    if _groups[_li] not in _g2c:
        _g2c[_groups[_li]] = _lc
for _grp, _col in zip(_ugrps, _pal):
    _g2c.setdefault(_grp, _col)
pert_to_color = {n: _g2c[g] for n, g in zip(_pnames, _groups)}

cond_labels = data.uns['results']['conditions']
_orig_get_cf_data = _slk_simple._get_cf_data

def _make_qval_masked_cf(qv, cidx):
    def _fn(adata, uns_key='results', cf_key='counterfactual_effect_size'):
        cf, perts, genes = _orig_get_cf_data(adata, uns_key=uns_key, cf_key=cf_key)
        cf = cf.copy()
        cf[cidx][qv >= 0.05] = 0.0
        n_sig = int((qv < 0.05).sum())
        print(f'  cond {cidx}: {n_sig}/{qv.size} (pert,gene) pairs pass qval<0.05')
        return cf, perts, genes
    return _fn

for cond_idx in range(len(cond_labels)):
    cond_label = cond_labels[cond_idx]
    _qvals = np.asarray(data.uns['results']['ev_results'][cond_idx]['qvals'], dtype=float)  # (P, G)
    _cf_fig = {}
    def _show_capture():
        _cf_fig['fig'] = plt.gcf()
    with patch.object(_slk_simple, '_draw_bipartite_nx', _centered_draw_bipartite), \
         patch.object(_slk_simple, '_get_cf_data', _make_qval_masked_cf(_qvals, cond_idx)), \
         patch.object(plt, 'show', _show_capture):
        slk.pl.plot_cf_bipartite(
            data,
            cond_idx=cond_idx,
            pert_name=["CP673451", "STK16IN1"],
            top_n=15,
            min_effect=1e-10,
            figsize=(6, 2.8),
        )
    fig = _cf_fig['fig']
    ax = fig.axes[0]
    ax.set_title('')  # Remove title
    for coll in ax.collections:
        if isinstance(coll, LineCollection):
            coll.set_color('#BBBBBB')
            coll.set_linewidth(1.8)
            coll.set_alpha(0.7)
    for txt in ax.texts:
        txt.set_fontsize(8)
        txt.set_color('#444444')
        txt.set_fontweight('bold')
    # Shrink node circles
    for coll in ax.collections:
        if not isinstance(coll, LineCollection):
            sizes = coll.get_sizes()
            if len(sizes) > 0:
                coll.set_sizes([200])
    legend = ax.get_legend()
    if legend:
        legend.remove()
    # Color pert nodes by their group color
    _pert_list = ["CP673451", "STK16IN1"]
    _node_colors = [pert_to_color.get(p, '#26547c') for p in _pert_list]
    for coll in ax.collections:
        offsets = coll.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue
        if np.all(offsets[:, 1] == 1.0) and len(offsets) == len(_pert_list):
            coll.set_facecolor(_node_colors)
            coll.set_edgecolor(_node_colors)
    plt.tight_layout(pad=0.3)
    save_name = f"{concentration}_cf_bipartite_CP673451_STK16IN1_{cond_label}"
    for ext in ('svg', 'png', 'pdf'):
        fig.savefig(os.path.join(figure_dir, f'{save_name}.{ext}'), dpi=300, bbox_inches='tight', transparent=True)
    print(f'Saved {save_name}.svg/png/pdf')
    plt.show()


In [ ]:

# Build group color legend entries dynamically from clustering
legend_elements = [
    Patch(facecolor=_g2c[grp], edgecolor='white', label=f'Group {grp}')
    for grp in sorted(_g2c)
] + [
    Patch(facecolor='#D0D0D0', edgecolor='white', label='Unique gene target'),
    Patch(facecolor='#1E90FF', edgecolor='white', label='Shared gene target'),
]

fig, ax = plt.subplots(figsize=(2.5, 1.5))
ax.legend(handles=legend_elements, loc='center', frameon=False, fontsize=9)
ax.axis('off')
plt.tight_layout(pad=0.2)

save_name = f"{concentration}_cf_bipartite_legend"
for ext in ('svg', 'png', 'pdf'):
    fig.savefig(os.path.join(figure_dir, f'{save_name}.{ext}'), dpi=300, bbox_inches='tight')
print(f'Saved {save_name}.svg/png/pdf')
plt.show()



In [ ]:
slk.pl.plot_cf_bipartite(
    data,
    cond_idx=1,          
    pert_name="CP673451",     
    top_n=15,            # top N target genes by effect size
)

In [ ]:
slk.pl.plot_cf_target_overlap(
    data,
    cond_idx=0,
    top_n=20,
    pert_names=["CP673451", "ALWII4127"],  # subset; None = all
    show_jaccard=False,      # pairwise Jaccard similarity heatmap
    show_membership=True,   # gene × drug binary matrix
)

In [ ]:
slk.pl.plot_cf_target_overlap(
    data,
    cond_idx=1,
    top_n=20,
    pert_names=["CP673451", "ALWII4127"],  # subset; None = all
    show_jaccard=False,      # pairwise Jaccard similarity heatmap
    show_membership=True,   # gene × drug binary matrix
)

In [ ]:
# --- Side-by-side comparison: CP673451 & ALWII4127 across concentrations ---
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

DRUGS = ["CP673451", "ALWII4127"]

# Load both concentrations — run save cell in 1um notebook first
path_1um  = os.path.join(figure_dir, '1um_pert_sensitivity.csv')
path_10um = os.path.join(figure_dir, '10um_pert_sensitivity.csv')
missing = [p for p in [path_1um, path_10um] if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        f'Missing CSV(s): {missing}\n'
        'Run the df_pert_sensitivity save cell in each notebook first.'
    )
df_1um  = pd.read_csv(path_1um)
df_10um = pd.read_csv(path_10um)

df_1um['concentration']  = '1 uM'
df_10um['concentration'] = '10 uM'
df_all = pd.concat([df_1um, df_10um], ignore_index=True)
# Ensure numeric and take absolute value for comparison
df_all['condition_effect_on_perturbation'] = pd.to_numeric(df_all['condition_effect_on_perturbation'], errors='coerce')
df_all['condition_effect_on_perturbation'] = np.abs(df_all['condition_effect_on_perturbation'])
df_plot = df_all[df_all['perturbation'].isin(DRUGS)].copy()

# Grouped bar plot
concs  = ['1 uM', '10 uM']
x      = np.arange(len(DRUGS))
width  = 0.2
colors = ['#00f59b', '#7014f2']

fig, ax = plt.subplots(figsize=(2, 2.8))
for j, (conc, color) in enumerate(zip(concs, colors)):
    vals = [
        df_plot.loc[(df_plot['perturbation'] == d) & (df_plot['concentration'] == conc),
                    'condition_effect_on_perturbation'].values
        for d in DRUGS
    ]
    vals = [v[0] if len(v) > 0 else 0.0 for v in vals]
    ax.bar(x + j * width - width/2, vals, width, label=conc, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(DRUGS, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Conditional perturbation response', fontsize=9)
ax.set_xlabel('Perturbations', fontsize=12)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.legend(fontsize=8, frameon=False)
sns.despine(ax=ax)
plt.tight_layout(pad=0.3)

save_name = 'CP673451_ALWII4127_sensitivity_comparison'
for ext in ('svg', 'png', 'pdf'):
    fig.savefig(os.path.join(figure_dir, f'{save_name}.{ext}'), dpi=300, bbox_inches='tight')
print(f'Saved {save_name}.svg/png/pdf')
plt.show()


## Expression of the blue (shared) genes in `10_cf_bipartite_CP673451_ALWII4127_0.5T`
DMSO vs CP673451 vs ALWII4127 in the with-T-cell (0.5T) cells, to see whether each drug increases or decreases these genes.
Note: the 45° labels in the bipartite figure sit diagonally offset from their nodes; counting nodes left to right, the blue ones are OLIG1, PHGDH, OLIG2, SLC26A2, GBP1 (the cell below recomputes them).

### Fig. S6: expression of the shared CP673451 / ALWII4127 targets under T-cell co-culture

In [ ]:
# ── Expression of the shared (blue) targets in 10_cf_bipartite_CP673451_ALWII4127_0.5T ──
# Blue genes = in the top 15 counterfactual targets (qval < 0.05) of BOTH CP673451 and ALWII4127,
# selected exactly as in the bipartite cell above. Direction vs DMSO comes from observed expression
# (library-size normalized + log1p) of the with-T-cell (0.5T) cells; Wilcoxon test vs DMSO.
import sherlock.plotting._simple as _slk_simple

AL, CP = 'ALWII4127', 'CP673451'
TCELL_COND = '0.5T'
TOP_N = 15
GROUP_ORDER = ['DMSO', CP, AL]
GROUP_COLORS = {'DMSO': '#BBBBBB', CP: drug_to_color[CP], AL: drug_to_color[AL]}  # palette2_very_light (bar-plot cell)

_cidx = list(data.uns['results']['conditions']).index(TCELL_COND)
_cf, _perts, _genes = _slk_simple._get_cf_data(data)
_cf_cond = _cf[_cidx].copy()
_cf_cond[np.asarray(data.uns['results']['ev_results'][_cidx]['qvals'], dtype=float) >= 0.05] = 0.0
_top = {}
for p in (CP, AL):
    ev = _cf_cond[int(np.where(_perts == p)[0][0])]
    _top[p] = {str(_genes[i]): ev[i] for i in np.argsort(ev)[::-1][:TOP_N] if ev[i] > 1e-10}
blue_genes = sorted(set(_top[CP]) & set(_top[AL]), key=lambda g: -(_top[CP][g] + _top[AL][g]))
print(f'Blue (shared) genes: {blue_genes}')

tc = data[(data.obs['tcell_treatment'] == TCELL_COND) & (data.obs['drug'].isin(GROUP_ORDER))].copy()
tc.obs['group'] = pd.Categorical(tc.obs['drug'].astype(str), categories=GROUP_ORDER)
sc.pp.normalize_total(tc, target_sum=1e4)
sc.pp.log1p(tc)
sc.tl.rank_genes_groups(tc, groupby='group', groups=[CP, AL], reference='DMSO',
                        method='wilcoxon', use_raw=False, key_added='drug_vs_dmso')

df_blue = pd.DataFrame(index=blue_genes)
for drug, tag in [(CP, 'CP'), (AL, 'AL')]:
    r = sc.get.rank_genes_groups_df(tc, group=drug, key='drug_vs_dmso').set_index('names')
    df_blue[f'log2FC_{tag}'] = r.loc[blue_genes, 'logfoldchanges']
    df_blue[f'padj_{tag}'] = r.loc[blue_genes, 'pvals_adj']
    df_blue[f'dir_{tag}'] = np.where(df_blue[f'log2FC_{tag}'] > 0, 'up', 'down')
X = tc[:, blue_genes].X
X = X.toarray() if hasattr(X, 'toarray') else np.asarray(X)
for g in GROUP_ORDER:
    df_blue[f'mean_{g}'] = X[(tc.obs['group'] == g).values].mean(axis=0)
print(df_blue.to_string())
df_blue.to_csv(os.path.join(figure_dir, f'{concentration}_cf_bipartite_blue_genes_expression_{TCELL_COND}.csv'))

def _stars(p):
    return '***' if p < 1e-3 else '**' if p < 1e-2 else '*' if p < 0.05 else 'ns'

fig, axes = plt.subplots(1, len(blue_genes), figsize=(1.15 * len(blue_genes), 1.7), sharey=False)
axes = np.atleast_1d(axes)
for ax, (gi, g) in zip(axes, enumerate(blue_genes)):
    d = pd.DataFrame({'expr': X[:, gi], 'group': tc.obs['group'].values})
    sns.violinplot(data=d, x='group', y='expr', order=GROUP_ORDER, hue='group', hue_order=GROUP_ORDER,
                   palette=GROUP_COLORS, saturation=1, legend=False, inner=None, cut=0, linewidth=0.5, ax=ax)
    ax.scatter(range(len(GROUP_ORDER)), df_blue.loc[g, [f'mean_{k}' for k in GROUP_ORDER]].values,
               color='black', s=6, zorder=3)
    ymax = d['expr'].max()
    for xi, tag in [(1, 'CP'), (2, 'AL')]:
        ax.text(xi, ymax * 1.04, _stars(df_blue.at[g, f'padj_{tag}']), ha='center', va='bottom', fontsize=7)
    ax.set_ylim(0, ymax * 1.18)
    ax.set_title(g, fontsize=9)
    ax.set_xlabel('')
    ax.set_ylabel('log1p norm. expr' if gi == 0 else '', fontsize=8)
    ax.set_xticks(range(len(GROUP_ORDER)))
    ax.set_xticklabels(['DMSO', 'CP', 'AL'], fontsize=7)
    ax.tick_params(axis='y', labelsize=7)
    sns.despine(ax=ax)
plt.tight_layout(pad=0.3)
save_name = f'{concentration}_cf_bipartite_blue_genes_violin_{TCELL_COND}'
for ext in ('svg', 'png', 'pdf'):
    plt.savefig(os.path.join(figure_dir, f'{save_name}.{ext}'), dpi=300, bbox_inches='tight')
print(f'Saved {save_name}.svg/png/pdf')
plt.show()

## Norman: reproducibility and held-out pairs

From [`scripts/norman/1_norman_analysis.ipynb`](../scripts/norman/1_norman_analysis.ipynb).

Loads the five SHERLOCK runs, the baseline runs and the GEARS runs from `results/norman_figure/`; any run that is missing is trained.

# Norman 2019 — Main Figure

Runs SHERLOCK (`l0_lambda=25, rank=6, synergy=True`) `N_RUNS` times on the held-out Norman split,
selects the best model by held-out combo ATE, and assembles a four-panel journal figure:

| Panel | Content |
|---|---|
| **(a)** | Counterfactual ATE across N runs — reproducibility & OOD generalisation |
| **(b)** | Single-perturbation latent UMAP (best model) |
| **(c)** | Parent-coefficient scatter (c₁ vs c₂) coloured by Norman GT class |
| **(d)** | GI classification AUC-ROC per class — SHERLOCK vs GEARS |

The notebook saves every intermediate result to `results/norman_figure/` and the final
composite figure to `results/norman_figure/main_figure.pdf`.

In [ ]:
import os, sys, math, random, textwrap
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr, bootstrap
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler

import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import torch

import sherlock as slk
from sherlock.tools._models import classify_gi

# ── Journal figure style ────────────────────────────────────────────────
RC = {
    'font.family':           'sans-serif',
    'font.sans-serif':       ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size':             8,
    'axes.labelsize':        8,
    'axes.titlesize':        9,
    'xtick.labelsize':       7,
    'ytick.labelsize':       7,
    'legend.fontsize':       6,
    'legend.title_fontsize': 7,
    'axes.spines.top':       False,
    'axes.spines.right':     False,
    'axes.linewidth':        0.75,
    'xtick.major.width':     0.75,
    'ytick.major.width':     0.75,
    'xtick.major.size':      3,
    'ytick.major.size':      3,
    'figure.dpi':            150,
    'savefig.dpi':           300,
    'pdf.fonttype':          42,
    'ps.fonttype':           42,
    'svg.fonttype': 'none',
}
mpl.rcParams.update(RC)

PPI = 96  # Figma/CSS px-per-inch — matches how pt-labeled SVGs get imported

# ── Colour palette ─────────────────────────────────────────────────────
CLR_SHERLOCK    = '#2166ac'
CLR_GEARS       = '#d73027'

# Keys match the exact strings in genetic_interactions.csv
PALETTE_GI = {
    'approximately additive':         '#4DAF4A',  # green
    'epistasis':                      '#FF7F00',  # orange
    'neomorphic':                     '#984EA3',  # purple
    'potentiation':                   '#F781BF',  # pink
    'redundant':                      '#A65628',  # brown
    'suppression':                    '#E41A1C',  # red
    'synergy (dissimilar phenotype)': '#377EB8',  # blue
    'synergy (similar phenotype)':    '#6BAED6',  # light blue
}

def _add_umap_axis_indicator(ax, x_label='UMAP1', y_label='UMAP2', length=0.15,
                              lw=1.0, color='0.15', fontsize=None):
    """Small L-shaped corner axis indicator (scanpy/Seurat style) — two short
    arrows from the bottom-left corner instead of full tick-labelled spines,
    used on UMAP panels where absolute coordinates are meaningless."""
    if fontsize is None:
        fontsize = mpl.rcParams['axes.labelsize']
    ax.annotate('', xy=(length, 0), xytext=(0, 0),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='-|>', color=color, lw=lw,
                                 shrinkA=0, shrinkB=0))
    ax.annotate('', xy=(0, length), xytext=(0, 0),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='-|>', color=color, lw=lw,
                                 shrinkA=0, shrinkB=0))
    ax.text(length / 2, -0.03, x_label, transform=ax.transAxes, fontsize=fontsize,
            ha='center', va='top', color=color)
    ax.text(-0.03, length / 2, y_label, transform=ax.transAxes, fontsize=fontsize,
            ha='right', va='center', color=color, rotation=90)


# ── Constants ──────────────────────────────────────────────────────────
N_RUNS    = 5
SEED_BASE = 0
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_DIR   = Path(f'{paths.RESULTS_DIR}/norman_figure')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}  |  N_RUNS: {N_RUNS}  |  Output: {OUT_DIR}')

## 1. Load and preprocess Norman 2019

In [ ]:
data = sc.read_h5ad(f'{paths.DATA_DIR}/norman/Norman_2019.h5ad')
data.X = data.layers['counts'].copy()

slk.pp.slk_prepare_data(
    data,
    pert_key='perturbation_name',
    ntc_label='control',
    treatment_key=None,
    inplace=True,
)

p_key = slk.configs.get_config('pert_key')
NTC   = slk.configs.get_config('ntc_label')

all_perts    = data.obs[p_key].unique()
single_perts = [p for p in all_perts if '+' not in str(p) and p != NTC]
combo_perts  = [p for p in all_perts if '+' in str(p)]
print(f'Cells: {data.n_obs}  |  Genes: {data.n_vars}')
print(f'Single perts: {len(single_perts)}  |  Combo perts: {len(combo_perts)}')

In [ ]:
# Gene filtering: keep genes with mean count > 0.5 UMI or that are pert targets
pert_genes = set()
for p in all_perts:
    if p != NTC:
        for g in str(p).split('+'):
            pert_genes.add(g.strip())

mean_expr = np.array(data.X.mean(axis=0)).flatten()
expr_mask = mean_expr > 0.5
pert_mask = data.var_names.isin(pert_genes)
keep_mask = expr_mask | pert_mask
data = data[:, keep_mask].copy()

print(f'Kept {data.n_vars} genes  '
      f'({expr_mask[keep_mask].sum()} expr + '
      f'{(pert_mask[keep_mask] & ~expr_mask[keep_mask]).sum()} pert-only)')

In [ ]:
# Observed treatment effects (log2-CPM vs NTC baseline)
slk.pp.treat_effect(
    data,
    label_col='pert',
    control_label='NTC',
    method='perturbseq',
    inplace=True,
    key='treat_effect',
)
te = data.uns['treat_effect']
print(f'treat_effect: {te.n_obs} perturbations × {te.n_vars} genes')

## 2. Train/val split — 20% of each Norman GT class held out

In [ ]:
def canon_pair(s):
    s = str(s)
    if '+' not in s:
        return s
    a, b = s.split('+', 1)
    return '+'.join(sorted([a.strip(), b.strip()]))

random.seed(42)
gi_labels = pd.read_csv(f'{paths.DATA_DIR}/norman/genetic_interactions.csv')
gi_labels['combination'] = gi_labels['combination'].apply(canon_pair)
gi_labels['genetic_interaction'] = gi_labels['genetic_interaction'].str.strip()

val_combos = set()
for cls, grp in gi_labels.groupby('genetic_interaction'):
    n_val = math.ceil(0.2 * len(grp))
    val_combos.update(random.sample(list(grp['combination']), n_val))

val_mask   = data.obs[p_key].apply(lambda p: ('+' in str(p)) and (canon_pair(p) in val_combos))
data_train = data[~val_mask].copy()

print(f'Total cells: {data.n_obs}  |  Training: {data_train.n_obs}')
print(f'GT pairs: {len(gi_labels)}  |  Held-out val combos: {len(val_combos)}')

## 3. Multi-run training

Trains SHERLOCK `N_RUNS` times with identical settings. After each run, computes ATE on
the full data (including held-out combos). The best run by held-out combo ATE is used for
UMAP, synergy analysis, and GI classification.

In [ ]:
def _ate_r(pred_df, obs_df, idx):
    idx = [p for p in idx if p in pred_df.index and p in obs_df.index]
    if len(idx) < 2:
        return float('nan')
    x = pred_df.loc[idx].values.ravel()
    y = obs_df.loc[idx].values.ravel()
    v = np.isfinite(x) & np.isfinite(y)
    return float(pearsonr(x[v], y[v])[0]) if v.sum() > 2 else float('nan')


def _effect_size_r(pred_df, obs_df):
    common = pred_df.index.intersection(obs_df.index)
    if len(common) < 3:
        return float('nan')
    p_norms = pred_df.loc[common].apply(np.linalg.norm, axis=1).values
    o_norms = obs_df.loc[common].apply(np.linalg.norm, axis=1).values
    v = np.isfinite(p_norms) & np.isfinite(o_norms)
    return float(pearsonr(p_norms[v], o_norms[v])[0]) if v.sum() > 2 else float('nan')


def _w_coact_matrix(W, threshold=0.5):
    """(P, P) cosine-normalized perturbation co-activation matrix. Invariant
    to permutation of the K gate dimensions, so it can be compared directly
    across independently trained runs (unlike the raw W)."""
    W_bin = (W > threshold).astype(float)
    dot   = W_bin @ W_bin.T
    norms = np.sqrt(np.diag(dot))
    denom = np.outer(norms, norms) + 1e-8
    return dot / denom


def _flat_triu(M):
    idx = np.triu_indices(M.shape[0], k=1)
    return M[idx]


def _loo_spearman_df(mats_by_run, metric_name):
    """Spearman r between each run's (P, P) matrix and the elementwise mean of
    every OTHER run's matrix (leave-one-out) — one row per run (n total).
    Unlike the C(n, 2) all-pairs comparisons, these n points each depend on a
    distinct held-out run, so they behave like an i.i.d. sample for
    bootstrapping instead of n(n-1)/2 non-independent pairs drawn from only
    n underlying runs."""
    rows = []
    keys = sorted(mats_by_run)
    for k in keys:
        others = [mats_by_run[j] for j in keys if j != k]
        if not others:
            continue
        mean_other = np.mean(others, axis=0)
        vi, vj = _flat_triu(mats_by_run[k]), _flat_triu(mean_other)
        valid  = np.isfinite(vi) & np.isfinite(vj)
        r = float(spearmanr(vi[valid], vj[valid])[0]) if valid.sum() > 2 else float('nan')
        rows.append({'metric': metric_name, 'run': k, 'r': r})
    return pd.DataFrame(rows)


def _compute_ate_metrics(model, data, val_combos):
    """Compute all ATE breakdown metrics for a model on full data."""
    ate  = model.predict_counterfactual_effects(
        data, n_centroids=25, observed_effect=data.uns['treat_effect'], approximate=False
    )
    pred = ate['predicted_df']
    obs  = ate['observed_df']
    singles = [p for p in pred.index if '+' not in str(p)]
    combos  = [p for p in pred.index if '+' in str(p) and canon_pair(p) not in val_combos]
    held    = [p for p in pred.index if canon_pair(p) in val_combos]
    return ate, pred, obs, {
        'ate_all':    ate['corr'],
        'ate_single': _ate_r(pred, obs, singles),
        'ate_combo':  _ate_r(pred, obs, combos),
        'ate_held':   _ate_r(pred, obs, held),
    }


def _cache_fresh(cache_path, *source_paths):
    """True if cache_path exists and is at least as new as every path in
    source_paths (files, or directories checked recursively). Retraining a
    model should invalidate every cache derived from it; comparing mtimes
    lets that happen automatically instead of a cache silently outliving the
    checkpoint(s) it was computed from."""
    cache_path = Path(cache_path)
    if not cache_path.exists():
        return False
    cache_mtime = cache_path.stat().st_mtime
    for src in source_paths:
        src = Path(src)
        if not src.exists():
            return False
        if src.is_dir():
            sub_paths = list(src.rglob('*'))
            if not sub_paths:
                continue
            src_mtime = max(p.stat().st_mtime for p in sub_paths)
        else:
            src_mtime = src.stat().st_mtime
        if src_mtime > cache_mtime:
            return False
    return True


# ── Paths ──────────────────────────────────────────────────────────────
MODELS_DIR        = OUT_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RUN_METRICS_PATH  = OUT_DIR / 'run_metrics.csv'
RUN_MATRICES_PATH = OUT_DIR / 'run_matrices.npz'
PAIRWISE_PATH     = OUT_DIR / 'pairwise_repro.csv'
checkpoints       = {i + 1: MODELS_DIR / f'run_{i + 1}.pth' for i in range(N_RUNS)}

if RUN_METRICS_PATH.exists():
    _saved = pd.read_csv(RUN_METRICS_PATH)
    existing_metrics = {int(r['run']): r for r in _saved.to_dict('records')}
else:
    existing_metrics = {}

if RUN_MATRICES_PATH.exists():
    _saved_mats  = np.load(RUN_MATRICES_PATH)
    rho_by_run   = {int(k.split('_')[-1]): _saved_mats[k] for k in _saved_mats.files if k.startswith('rho_corr_')}
    wcoact_by_run = {int(k.split('_')[-1]): _saved_mats[k] for k in _saved_mats.files if k.startswith('wcoact_')}
else:
    rho_by_run, wcoact_by_run = {}, {}

# ── Training / loading loop ────────────────────────────────────────────
for run_num in range(1, N_RUNS + 1):
    ckpt_path    = checkpoints[run_num]
    already_done = (run_num in existing_metrics) and ckpt_path.exists()

    if already_done:
        m = existing_metrics[run_num]
        print(f'Run {run_num}: loaded  '
              f"ate_all={m['ate_all']:.4f}  ate_single={m.get('ate_single', float('nan')):.4f}  "
              f"ate_combo={m.get('ate_combo', float('nan')):.4f}  ate_held={m['ate_held']:.4f}")
        if run_num not in rho_by_run or run_num not in wcoact_by_run:
            _model = slk.tl.load_results(ckpt_path)
            slk.tl.eval_single(_model, data_train, obsm_key='_z_tmp', uns_key='_tmp', device=DEVICE)
            _tmp = data_train.uns['_tmp']
            if _tmp.get('rho_corr') is not None:
                rho_by_run[run_num] = np.asarray(_tmp['rho_corr'], dtype=float)
            if _tmp.get('W') is not None:
                wcoact_by_run[run_num] = _w_coact_matrix(np.asarray(_tmp['W'], dtype=float))
        continue

    print(f'\n── Training run {run_num}/{N_RUNS} ──────────────────────────')
    out = slk.tl.run_single(
        data_train,
        model='vae',
        l0_lambda=1e-1, #20
        rank=16,
        combinatorial=True,
        synergy=True,
        synergy_rank=6,
        shuffle=True,
        num_workers=0,
        device=DEVICE,
        num_epochs=500,
        validate_every=50,
        patience=30, #20?
        n_epochs_l0_warmup=300,
        seed=SEED_BASE + run_num - 1,
        debug=False,
        use_de_align_loss=False,
        de_align_lambda=10,    
        de_align_direction_lambda=0,
        de_align_n_bg=512, #256 prob works
        de_align_exclude=[], 
        use_go_prior=False,
    )
    slk.tl.save_results(out, ckpt_path)

    model = out['model']
    _, pred, obs, ate_metrics = _compute_ate_metrics(model, data, val_combos)

    slk.tl.eval_single(model, data_train, obsm_key='_z_tmp', uns_key='_tmp', device=DEVICE)
    tmp   = data_train.uns['_tmp']
    W_mat = tmp.get('W')
    if tmp.get('rho_corr') is not None:
        rho_by_run[run_num] = np.asarray(tmp['rho_corr'], dtype=float)
    if W_mat is not None:
        wcoact_by_run[run_num] = _w_coact_matrix(np.asarray(W_mat, dtype=float))

    m = {
        'run':           run_num,
        **ate_metrics,
        'effect_size_r': _effect_size_r(pred, obs),
    }
    existing_metrics[run_num] = m
    pd.DataFrame(list(existing_metrics.values())).sort_values('run').to_csv(
        RUN_METRICS_PATH, index=False,
    )
    print(f"  ate_all={m['ate_all']:.4f}  ate_single={m['ate_single']:.4f}  "
          f"ate_combo={m['ate_combo']:.4f}  ate_held={m['ate_held']:.4f}")

np.savez(
    RUN_MATRICES_PATH,
    **{f'rho_corr_{k}': v for k, v in rho_by_run.items()},
    **{f'wcoact_{k}':   v for k, v in wcoact_by_run.items()},
)

# ── Cross-run reproducibility (rho_corr, W coactivation) ────────────────
# For each run, Spearman r between flattened upper-triangles of its rho_corr
# (learned perturbation covariance) / W-coactivation matrix and the mean of
# the OTHER runs' matrices (leave-one-out). This gives n independent points
# (one per run) instead of the C(n, 2) all-pairs comparisons, which reuse
# each run across multiple pairs and so are not i.i.d. even though there are
# more of them. effect_size_r stays a per-run metric since it measures
# within-run predictive accuracy, not cross-run reproducibility.
runs_df = pd.read_csv(RUN_METRICS_PATH)

pairwise_df = pd.concat([
    _loo_spearman_df(rho_by_run, 'rho_corr_r'),
    _loo_spearman_df(wcoact_by_run, 'w_coact_r'),
], ignore_index=True)
pairwise_df.to_csv(PAIRWISE_PATH, index=False)

for _metric, _grp in pairwise_df.groupby('metric'):
    _vals = _grp['r'].dropna().values
    if len(_vals) >= 3:
        _boot = bootstrap((_vals,), np.mean, confidence_level=0.95,
                           method='percentile', random_state=0)
        print(f'{_metric}: mean r = {_vals.mean():.4f}  '
              f'95% CI [{_boot.confidence_interval.low:.4f}, {_boot.confidence_interval.high:.4f}]  '
              f'(n_runs={len(_vals)})')
    elif len(_vals) == 1:
        print(f'{_metric}: r = {_vals[0]:.4f}  '
              f'(only 1 run — need >=3 runs for a bootstrap CI)')
    else:
        print(f'{_metric}: no valid runs.')

# ── Select and load best model ─────────────────────────────────────────
runs_df  = runs_df[runs_df['run'].map(lambda r: checkpoints.get(int(r), Path()).exists())]
best_run = int(runs_df.loc[runs_df['ate_held'].idxmax(), 'run'])
best_ate = float(runs_df['ate_held'].max())

print(f'\nBest run: {best_run}  (ate_held = {best_ate:.4f})')
best_model = slk.tl.load_results(checkpoints[best_run])
runs_df

## 3b. Other benchmark methods (cVAE, sVAE, contrastiveVI, scgen)

Same hyperparameters/model kwargs as `benchmarking_combinatorial.ipynb`'s `METHODS` list, trained
on the same held-out combinatorial split (`data_train`/`val_combos`). `sherlock_synergy` isn't
listed separately since the SHERLOCK run above already trains with `synergy=True`. Each
method/run is checkpointed independently (as in `benchmarking.ipynb`), so a re-executed notebook
reloads finished runs instead of retraining.

**Combos are atomic labels, not compositional.** Unlike SHERLOCK (`combinatorial=True`, which
splits `"A+B"` into components and sums their learned shifts — the mechanism that lets it
extrapolate to held-out combos at all), these baselines are run with `combinatorial=False`:
`"A+B"` gets its own independent embedding, exactly like a third, unrelated perturbation — no
sharing with `"A"` or `"B"`. That means a combo the model never saw in training has *no* learned
representation whatsoever, so held-out ATE is undefined for these methods by construction (not
just poor). Evaluation therefore uses `data_train` (not the full `data`) as the ATE prediction
input: `predict_counterfactual_effects` rebuilds its own perturbation index internally from
whatever `adata` it's given, so evaluating on the full `data` here would silently reintroduce the
held-out combos into that index and misalign every perturbation sorted after them relative to the
model's trained embedding table — evaluating on `data_train` keeps the eval-time and train-time
indices identical and simply leaves held-out combos absent (→ `ate_held` = NaN via the existing
`_ate_r` NaN-on-empty-index handling), correctly and safely. In Panel A below, that means the
`ATE (held-out)` box is populated by SHERLOCK only.

In [ ]:
# ── Other benchmark methods (cVAE, sVAE, contrastiveVI, scgen) ────────────
OTHER_METHODS = [
    ('cvae',          'cVAE',          {'combinatorial': False}),
    ('svae',          'sVAE',          {'sparse_mask_penalty': 10.0, 'combinatorial': False}),
    ('contrastivevi', 'contrastiveVI', {'combinatorial': False}),
    ('scgen',         'scgen',         {}),
]
OTHER_EPOCHS         = 200
OTHER_BATCH_SIZE     = 4096
OTHER_LATENT_DIM     = 16
OTHER_PATIENCE       = 20
OTHER_VALIDATE_EVERY = 50

BENCH_MODELS_DIR   = OUT_DIR / 'benchmark_models'
BENCH_MODELS_DIR.mkdir(parents=True, exist_ok=True)
BENCH_METRICS_PATH = OUT_DIR / 'benchmark_run_metrics.csv'

if BENCH_METRICS_PATH.exists():
    bench_records = pd.read_csv(BENCH_METRICS_PATH).to_dict('records')
else:
    bench_records = []
bench_done = {(r['method'], int(r['run'])) for r in bench_records}

for model_key, display_name, extra_kwargs in OTHER_METHODS:
    method_dir = BENCH_MODELS_DIR / display_name
    method_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n{'='*60}\n{display_name}  ({N_RUNS} runs, {OTHER_EPOCHS} epochs)\n{'='*60}")

    for run_num in range(1, N_RUNS + 1):
        if (display_name, run_num) in bench_done:
            print(f'  run {run_num}/{N_RUNS}: metrics already cached, skipping.')
            continue

        ckpt_path = method_dir / f'run_{run_num}.pth'
        if ckpt_path.exists():
            print(f'  run {run_num}/{N_RUNS}: loading checkpoint {ckpt_path}')
            model = slk.tl.load_results(ckpt_path)
        else:
            print(f'  run {run_num}/{N_RUNS}: training...')
            out = slk.tl.run(
                data_train,
                model=model_key,
                batch_size=OTHER_BATCH_SIZE,
                num_epochs=OTHER_EPOCHS,
                device=DEVICE,
                latent_dim=OTHER_LATENT_DIM,
                patience=OTHER_PATIENCE,
                validate_every=OTHER_VALIDATE_EVERY,
                seed=SEED_BASE + run_num - 1,
                **extra_kwargs,
            )
            model = out['model']
            slk.tl.save_results(out, ckpt_path)

        # data_train (not data) — see markdown above: these models treat every
        # combo as an atomic label, so evaluating on data_train keeps the
        # eval-time perturbation index identical to the trained one and
        # leaves held-out combos correctly absent (ate_held -> NaN) instead
        # of silently misaligning indices for everything else.
        _, _, _, ate_metrics = _compute_ate_metrics(model, data_train, val_combos)
        record = {'method': display_name, 'run': run_num, **ate_metrics}
        bench_records.append(record)
        bench_done.add((display_name, run_num))
        pd.DataFrame(bench_records).to_csv(BENCH_METRICS_PATH, index=False)
        print(f"    ate_all={record['ate_all']:.4f}  ate_single={record['ate_single']:.4f}  "
              f"ate_combo={record['ate_combo']:.4f}  ate_held={record['ate_held']:.4f}")

bench_runs_df = pd.DataFrame(bench_records)
print(f'\nOther-method benchmark done: {len(bench_runs_df)} (method, run) rows.')
bench_runs_df

## 4. Best model — latent embedding & GI scoring

In [ ]:
SYN_PATH = OUT_DIR / 'syn_best.csv'

# Always run eval to populate data.obsm['z'] (fast; needed for UMAP + get_gi)
model = best_model
slk.tl.eval_single(model, data, obsm_key='z', uns_key='results', device=DEVICE)

if _cache_fresh(SYN_PATH, checkpoints[best_run]):
    print(f'Loading cached GI scores from {SYN_PATH}')
    syn = pd.read_csv(SYN_PATH, index_col=0)
else:
    syn = model.get_gi(data, min_cells=5)

    # Canonicalise index and annotate with Norman GT labels
    syn.index = (syn['pert_a'].astype(str) + '+' + syn['pert_b'].astype(str)).map(canon_pair)
    gi_series = gi_labels.set_index('combination')['genetic_interaction']
    syn['gt_gi'] = syn.index.map(gi_series)

    gt_combos = set(gi_labels['combination'])
    fit_mask  = syn.index.to_series().isin(gt_combos)
    syn = classify_gi(syn, fit_mask=fit_mask, inplace=True)

    syn.to_csv(SYN_PATH)
    print(f'GI scores saved to {SYN_PATH}')

# Ensure gi_series is always defined for downstream cells
gi_series = gi_labels.set_index('combination')['genetic_interaction']

print(f'GI rows: {len(syn)}  |  GT-matched: {syn["gt_gi"].notna().sum()}  |  '
      f'Classified: {syn["predicted_gi"].notna().sum()}')

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def classify_and_report_gi(
    syn_df,
    gt_path=f"{paths.DATA_DIR}/norman/genetic_interactions.csv",
    pred_col="predicted_gi",
    inplace=True,
    figsize=(8, 6),
):
    def canon_pair(s):
        s = str(s)
        if "+" not in s:
            return s
        a, b = s.split("+", 1)
        return "+".join(sorted([a.strip(), b.strip()]))

    if not os.path.exists(gt_path):
        raise FileNotFoundError(f"Expected {gt_path} for Norman labels.")

    gt = pd.read_csv(gt_path)
    gt["combination"] = gt["combination"].map(canon_pair)
    gt["genetic_interaction"] = gt["genetic_interaction"].astype(str).str.strip()

    eval_combinations = set(gt["combination"])
    syn_combinations = (
        syn_df["pert_a"].astype(str) + "+" + syn_df["pert_b"].astype(str)
    ).map(canon_pair)

    classified = slk.tl.classify_gi(
        syn_df,
        pred_col=pred_col,
        # fit_mask=syn_combinations.isin(eval_combinations),
        inplace=inplace,
    )

    score_df = classified.copy()
    score_df["combination"] = syn_combinations
    score_df = score_df.merge(
        gt[["combination", "genetic_interaction"]],
        on="combination",
        how="inner",
    )
    score_df = score_df.loc[score_df[pred_col].notna()].copy()

    y_true = score_df["genetic_interaction"].astype(str)
    y_pred = score_df[pred_col].astype(str)
    labels = sorted(y_true.unique())

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=labels)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", labels=labels)

    print(f"Evaluated {len(score_df)} labeled interactions.")
    print(f"Accuracy:    {acc:.3f}")
    print(f"Macro F1:    {macro_f1:.3f}")
    print(f"Weighted F1: {weighted_f1:.3f}")
    print()
    print("Classification report:")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Norman label")
    ax.set_title("GI score confusion matrix")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    return classified

classify_and_report_gi(syn)
# gears_gi = classify_and_report_gi(gears_gi)


In [ ]:
UMAP_PATH     = OUT_DIR / 'umap_singlepert.npz'
UMAP_ALL_PATH = OUT_DIR / 'umap_allpert.npz'

single_mask = (
    (data.obs[p_key] != NTC) &
    (~data.obs['perturbation_name'].astype(str).str.contains(r'\+', regex=True))
)
sub = data[single_mask].copy()

# All non-NTC cells (singles + combos) for panel C
all_mask = data.obs[p_key] != NTC
all_sub  = data[all_mask].copy()
all_sub.obs['pert_type'] = np.where(
    all_sub.obs[p_key].astype(str).str.contains(r'\+', regex=True),
    'combination', 'single',
)

if _cache_fresh(UMAP_PATH, checkpoints[best_run]):
    print(f'Loading cached single-pert UMAP from {UMAP_PATH}')
    _u = np.load(UMAP_PATH, allow_pickle=True)
    sub.obsm['X_umap'] = _u['coords']
else:
    sc.pp.neighbors(sub, n_neighbors=15, use_rep='z')
    sc.tl.umap(sub)
    np.savez(UMAP_PATH, coords=sub.obsm['X_umap'])
    print(f'Single-pert UMAP saved to {UMAP_PATH}')

if _cache_fresh(UMAP_ALL_PATH, checkpoints[best_run]):
    print(f'Loading cached all-pert UMAP from {UMAP_ALL_PATH}')
    _u2 = np.load(UMAP_ALL_PATH, allow_pickle=True)
    all_sub.obsm['X_umap'] = _u2['coords']
else:
    sc.pp.neighbors(all_sub, n_neighbors=15, use_rep='z')
    sc.tl.umap(all_sub)
    np.savez(UMAP_ALL_PATH, coords=all_sub.obsm['X_umap'])
    print(f'All-pert UMAP saved to {UMAP_ALL_PATH}')

print(f'Single-pert cells: {sub.n_obs}  |  All-pert cells: {all_sub.n_obs}')

## 5. GEARS multi-run training & GI classification

In [ ]:
from sklearn.metrics import (
    f1_score, roc_auc_score, silhouette_score, silhouette_samples,
    adjusted_rand_score, normalized_mutual_info_score,
    adjusted_mutual_info_score,
)

GI_METRICS = [
    'additive_residual_norm', 'dominance',
    'parent_dcor_diff', 'equality_contribution', 'singles_similarity',
    'norm_ab_over_additive', 'norm_fit_over_ab', 'magnitude',
    'model_fit_dcor', 'norm_fit', 'signed_residual_alignment',
]


# ── Merge the two synergy sub-types into a single 'synergy' class ───────
# Norman splits synergy into (similar phenotype) / (dissimilar phenotype). For
# the GI classification figure we score them as ONE class. This must happen
# BEFORE metrics are computed: the per-class F1/AUC of two one-vs-rest classes
# cannot be averaged into a merged class after the fact.
# Only the *metrics* are merged — the returned `work` frame keeps the original
# 8 labels, so Panel D's UMAP colouring is unchanged.
MERGE_SYNERGY = True
_SYNERGY_ALIASES = {
    'synergy (similar phenotype)':    'synergy',
    'synergy (dissimilar phenotype)': 'synergy',
}

def _merge_syn(s):
    """Collapse syn-sim / syn-dis -> 'synergy' (no-op if MERGE_SYNERGY is False)."""
    s = s.astype(str)
    return s.replace(_SYNERGY_ALIASES) if MERGE_SYNERGY else s


def evaluate_gi_classification(df, model_name, gi_series):
    """Classify GI for all combinations; evaluate against GT where available."""
    work = df.copy()
    if 'pert_a' in work.columns and 'pert_b' in work.columns:
        work.index = (work['pert_a'].astype(str) + '+' + work['pert_b'].astype(str)).map(canon_pair)
    else:
        work.index = work.index.map(canon_pair)

    work['gt_gi'] = work.index.map(gi_series)

    # Classify ALL rows with finite metrics (no fit_mask) so unlabelled combos get predictions too
    work = classify_gi(work, inplace=True)

    eval_df = work[work['gt_gi'].notna() & work['predicted_gi'].notna()].copy()
    y_true  = _merge_syn(eval_df['gt_gi'])
    y_pred  = _merge_syn(eval_df['predicted_gi'])
    labels  = sorted(set(y_true))

    per_class = []
    for cls in labels:
        tb = (y_true == cls).astype(int)
        pb = (y_pred == cls).astype(int)
        n_pos = int(tb.sum())
        auc = roc_auc_score(tb, pb) if 0 < n_pos < len(tb) else np.nan
        per_class.append({
            'model': model_name, 'class': cls, 'n_positive': n_pos,
            'f1': f1_score(tb, pb, zero_division=0), 'auc_roc': auc,
        })

    summary = {
        'model':       model_name,
        'n_eval':      len(eval_df),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'ari':         adjusted_rand_score(y_true, y_pred),
        'ami':         adjusted_mutual_info_score(y_true, y_pred),
        'nmi':         normalized_mutual_info_score(y_true, y_pred),
    }
    return work, pd.DataFrame(per_class), summary


slk_classified, slk_class_df, slk_summary = evaluate_gi_classification(syn, 'SHERLOCK', gi_series)

In [ ]:
# ── GEARS multi-run training ─────────────────────────────────────────────
# Mirrors norman_run.ipynb: trains GEARS on data_train (held-out val_combos
# excluded, same split SHERLOCK uses) and predicts on gears_combos = all
# combinations observed in the full `data` (train-visible + held-out), i.e.
# train_gears(data_train, combos=get_observed_combinations(data, ...)).
# Runs N_RUNS times for CI in panel E; each run's model checkpoint, raw
# predictions, and classified metrics are cached independently so a run that
# already finished (even if a later run was interrupted) is never redone.
import sherlock.tools._gears as gears_tools
slk.tl.run_gears_gi              = gears_tools.run_gears_gi
slk.tl.get_observed_combinations = gears_tools.get_observed_combinations

GEARS_MODELS_DIR = OUT_DIR / 'gears_models'
GEARS_RUNS_DIR   = OUT_DIR / 'gears_runs'
GEARS_MODELS_DIR.mkdir(parents=True, exist_ok=True)
GEARS_RUNS_DIR.mkdir(parents=True, exist_ok=True)
GEARS_CLS_CACHE  = OUT_DIR / 'gears_per_run_class.csv'
GEARS_SUMM_CACHE = OUT_DIR / 'gears_per_run_summ.csv'

gears_combos = slk.tl.get_observed_combinations(data, pert_key=p_key, ntc_label=NTC)
print(f'GEARS combos (observed in full data, train + held-out): {len(gears_combos)}')
gears_singles = gears_tools.get_single_perturbations(data, pert_key=p_key, ntc_label=NTC)
print(f'GEARS singles (observed in full data): {len(gears_singles)}')

from scipy.sparse import csr_matrix, issparse

data_train_gears = data_train.copy()
_X_gears = data_train_gears.X
if issparse(_X_gears):
    _X_gears = _X_gears.toarray()
_X_gears = 1e4 * _X_gears / np.sum(_X_gears, axis=1, keepdims=True)
_X_gears = np.log2(_X_gears + 1)
data_train_gears.X = csr_matrix(_X_gears)  # GEARS assumes sparse .X, calls .toarray() internally

if GEARS_CLS_CACHE.exists() and GEARS_SUMM_CACHE.exists():
    _gears_run_class_df = pd.read_csv(GEARS_CLS_CACHE)
    _gears_run_summ_df  = pd.read_csv(GEARS_SUMM_CACHE)
else:
    _gears_run_class_df = pd.DataFrame(columns=['run'])
    _gears_run_summ_df  = pd.DataFrame(columns=['run'])
_gears_done_runs = set(_gears_run_summ_df['run'].unique().tolist())

gears_df = gears_classified = gears_class_df = gears_summary = None

for _rn in range(1, N_RUNS + 1):
    _pred_path = GEARS_RUNS_DIR / f'gears_gi_run{_rn}.csv'
    _ckpt_dir  = GEARS_MODELS_DIR / f'run_{_rn}'
    _has_ckpt  = (_ckpt_dir / 'config.pkl').exists()
    _preds_fresh = _has_ckpt and _cache_fresh(_pred_path, _ckpt_dir)

    if _rn in _gears_done_runs and _preds_fresh:
        print(f'GEARS run {_rn}/{N_RUNS}: metrics already cached and up to date, skipping.')
        gears_df = pd.read_csv(_pred_path)
        continue

    print(f'\n── GEARS run {_rn}/{N_RUNS} ──────────────────────────')
    _seed = SEED_BASE + _rn - 1
    random.seed(_seed)
    np.random.seed(_seed)
    torch.manual_seed(_seed)

    if _preds_fresh:
        print(f'  loading cached predictions from {_pred_path}')
        _gears_df_r = pd.read_csv(_pred_path)
        _ate_metrics_r = None
    else:
        _gears_run = slk.tl.run_gears_gi(
            data_train_gears,
            combos=gears_combos,
            data_dir=f'{paths.RESULTS_DIR}/gears_data',
            dataset_name='norman_figure_gears',
            pert_key=p_key,
            ntc_label=NTC,
            split='all',
            seed=_seed,
            batch_size=256,
            test_batch_size=1024,
            epochs=20,
            load_path=str(_ckpt_dir) if _has_ckpt else None,
            model_path=None if _has_ckpt else str(_ckpt_dir),
            verbose=True,
            quiet_steps=True,
            device=str(DEVICE),
        )
        _gears_df_r = _gears_run.gi if _gears_run.gi is not None else _gears_run.predictions
        _gears_df_r.to_csv(_pred_path, index=False)
        _ate_metrics_r = _gears_run.compute_ate(
            gears_singles + gears_combos, data.uns['treat_effect'], val_combos,
            already_normalized=True, verbose=False,
        )

    _classified_r, _cls_r, _summ_r = evaluate_gi_classification(_gears_df_r, 'GEARS', gi_series)
    _summ_r = {
        **_summ_r,
        **(_ate_metrics_r or {'ate_all': float('nan'), 'ate_single': float('nan'),
                               'ate_combo': float('nan'), 'ate_held': float('nan')}),
    }
    _cls_r = _cls_r.copy()
    _cls_r['run'] = _rn

    _gears_run_class_df = pd.concat([_gears_run_class_df, _cls_r], ignore_index=True)
    _gears_run_summ_df  = pd.concat(
        [_gears_run_summ_df, pd.DataFrame([{**_summ_r, 'run': _rn}])], ignore_index=True,
    )
    _gears_run_class_df.to_csv(GEARS_CLS_CACHE, index=False)
    _gears_run_summ_df.to_csv(GEARS_SUMM_CACHE, index=False)
    _gears_done_runs.add(_rn)

    gears_df, gears_classified, gears_class_df, gears_summary = (
        _gears_df_r, _classified_r, _cls_r, _summ_r,
    )
    print(f"  run {_rn}: f1_weighted={_summ_r['f1_weighted']:.4f}  "
          f"ate_all={_summ_r['ate_all']:.4f}  "
          f"ari={_summ_r['ari']:.4f}  ami={_summ_r['ami']:.4f}")

# If every run was already cached, reload the last run's predictions so
# downstream single-instance uses (UMAP fit, summary table) have a
# representative GEARS model.
if gears_classified is None:
    gears_df = pd.read_csv(GEARS_RUNS_DIR / f'gears_gi_run{N_RUNS}.csv')
    gears_classified, gears_class_df, gears_summary = evaluate_gi_classification(
        gears_df, 'GEARS', gi_series,
    )

print(f'\nGEARS per-run metrics:')
print(_gears_run_summ_df[['run', 'ate_all', 'f1_weighted', 'ari', 'ami']].to_string(index=False))

class_df   = pd.concat([slk_class_df, gears_class_df], ignore_index=True)
summary_df = pd.DataFrame([slk_summary, gears_summary])
class_df.to_csv(OUT_DIR / 'gi_classification.csv', index=False)

for name, clf in [('SHERLOCK', slk_classified), ('GEARS', gears_classified)]:
    n_pred = clf['predicted_gi'].notna().sum()
    n_nan  = clf['predicted_gi'].isna().sum()
    print(f'{name}: {n_pred} predicted, {n_nan} NaN (should be 0 if all metrics finite)')

print(summary_df.to_string(index=False))

In [ ]:
# ── Per-run GI classification for SHERLOCK — used for CI in panel E ─────
# Runs get_gi + classify_gi on every checkpoint and caches results.
# Cache filename encodes the labelling, so flipping MERGE_SYNERGY busts the
# cache automatically instead of silently reusing rows with the old classes.
_suffix     = '_synmerged' if MERGE_SYNERGY else ''
_cls_cache  = OUT_DIR / f'gi_per_run_class{_suffix}.csv'
_summ_cache = OUT_DIR / f'gi_per_run_summ{_suffix}.csv'

if _cache_fresh(_cls_cache, *checkpoints.values()) and _cache_fresh(_summ_cache, *checkpoints.values()):
    print('Loading cached per-run GI metrics...')
    _run_class_df = pd.read_csv(_cls_cache)
    _run_summ_df  = pd.read_csv(_summ_cache)
else:
    _cls_rows, _summ_rows = [], []
    for _rn in range(1, N_RUNS + 1):
        _ckpt = checkpoints[_rn]
        if not _ckpt.exists():
            print(f'  Run {_rn}: checkpoint not found, skipping.')
            continue
        print(f'  Run {_rn}/{N_RUNS}...', end=' ', flush=True)
        _m = slk.tl.load_results(_ckpt)
        slk.tl.eval_single(_m, data, obsm_key='z', uns_key='results', device=DEVICE)
        _syn_r = _m.get_gi(data, min_cells=5)
        _syn_r.index = (
            _syn_r['pert_a'].astype(str) + '+' + _syn_r['pert_b'].astype(str)
        ).map(canon_pair)
        _syn_r['gt_gi'] = _syn_r.index.map(gi_series)
        _fit_r = _syn_r.index.to_series().isin(set(gi_labels['combination']))
        _syn_r  = classify_gi(_syn_r, fit_mask=_fit_r, inplace=True)
        _, _cls_r, _summ_r = evaluate_gi_classification(_syn_r, 'SHERLOCK', gi_series)
        _cls_r['run']  = _rn
        _summ_r['run'] = _rn
        _cls_rows.append(_cls_r)
        _summ_rows.append(_summ_r)
        print('done')
    _run_class_df = pd.concat(_cls_rows, ignore_index=True)
    _run_summ_df  = pd.DataFrame(_summ_rows)
    _run_class_df.to_csv(_cls_cache, index=False)
    _run_summ_df.to_csv(_summ_cache, index=False)
    print(f'Per-run GI metrics cached.')

# ── Rebuild GEARS per-run metrics under the merged-synergy labelling ─────
# GEARS' cached per-class/summary CSVs were computed with the 8-class labelling.
# Recompute them from the cached RAW predictions (gears_runs/gears_gi_run*.csv)
# so both models are scored with the same merged classes. No GEARS retraining and
# no cache files are overwritten. The ate_* columns are carried over from the
# on-disk summary cache: they are unaffected by the GI relabelling, and
# compute_ate() would need the GEARS model object, which we do not reload.
_g_cls_rows, _g_summ_rows = [], []
for _rn in range(1, N_RUNS + 1):
    _g_raw = pd.read_csv(GEARS_RUNS_DIR / f'gears_gi_run{_rn}.csv')
    _, _gc_r, _gs_r = evaluate_gi_classification(_g_raw, 'GEARS', gi_series)
    _gc_r = _gc_r.copy()
    _gc_r['run'] = _rn
    _g_cls_rows.append(_gc_r)
    _g_summ_rows.append({**_gs_r, 'run': _rn})

_gears_run_class_df = pd.concat(_g_cls_rows, ignore_index=True)
_ate_cols  = ['ate_all', 'ate_single', 'ate_combo', 'ate_held']
_gears_ate = pd.read_csv(GEARS_SUMM_CACHE)[['run'] + _ate_cols]
_gears_run_summ_df = pd.DataFrame(_g_summ_rows).merge(_gears_ate, on='run', how='left')
print(f'GEARS per-run metrics rebuilt with merged synergy (ate_* preserved).')


# ── CI-ready long dataframes used by panel E ─────────────────────────────
# SHERLOCK and GEARS all have N_RUNS rows per class — all get CI bars
class_df_ci = pd.concat(
    [_run_class_df, _gears_run_class_df], ignore_index=True,
)

# For summary metrics
_SUMM_KEYS = ['f1_weighted', 'ari', 'ami']
_SUMM_NAMES = {'f1_weighted': 'F1', 'ari': 'ARI', 'ami': 'AMI'}
_slk_run_long = _run_summ_df[['run'] + _SUMM_KEYS].melt(
    id_vars='run', value_vars=_SUMM_KEYS, var_name='metric', value_name='score'
)
_slk_run_long['model'] = 'SHERLOCK'
_gears_run_long = _gears_run_summ_df[['run'] + _SUMM_KEYS].melt(
    id_vars='run', value_vars=_SUMM_KEYS, var_name='metric', value_name='score'
)
_gears_run_long['model'] = 'GEARS'
summ_long_ci = pd.concat(
    [_slk_run_long[['model', 'metric', 'score']],
     _gears_run_long[['model', 'metric', 'score']]],
    ignore_index=True,
)
summ_long_ci['Metric'] = pd.Categorical(
    summ_long_ci['metric'].map(_SUMM_NAMES),
    categories=list(_SUMM_NAMES.values()), ordered=True,
)
print(_run_summ_df[['run', 'f1_weighted', 'ari', 'ami']].to_string(index=False))

In [ ]:
# ── Synergy UMAP — 12-metric embedding for SHERLOCK and GEARS ────────────
from sklearn.preprocessing import StandardScaler
from umap import UMAP


def _fit_umap(df, metrics):
    avail  = [c for c in metrics if c in df.columns]
    X      = df[avail].values.astype(float)
    good   = np.all(np.isfinite(X), axis=1)
    work   = df[good].copy()
    X_s    = StandardScaler().fit_transform(X[good])
    reducer = UMAP(n_neighbors=10, min_dist=0.25, random_state=42)
    emb    = reducer.fit_transform(X_s)
    work['UMAP1'] = emb[:, 0]
    work['UMAP2'] = emb[:, 1]
    return work


# ── Pick the highest-weighted-F1 model per method for the UMAPs (fresh) ──
# The UMAPs (and the derived summary + Panel D) use the best GI *classifier*
# per method — not the best-ATE run or the last GEARS run. Recompute the chosen
# models' GI tables from source (SHERLOCK: reload checkpoint + get_gi; GEARS: raw
# per-run preds) so a stale syn_best.csv can never feed the figure.
_slk_best_run   = int(_run_summ_df.loc[_run_summ_df['f1_weighted'].idxmax(), 'run'])
_gears_best_run = int(_gears_run_summ_df.loc[_gears_run_summ_df['f1_weighted'].idxmax(), 'run'])
print(f'UMAP models by weighted-F1  —  SHERLOCK run {_slk_best_run}  |  GEARS run {_gears_best_run}')

_slk_best_model = slk.tl.load_results(checkpoints[_slk_best_run])
slk.tl.eval_single(_slk_best_model, data, obsm_key='z', uns_key='results', device=DEVICE)
_syn_bestf1 = _slk_best_model.get_gi(data, min_cells=5, mode='observed')
slk_classified, slk_class_df, slk_summary = evaluate_gi_classification(
    _syn_bestf1, 'SHERLOCK', gi_series)

_gears_bestf1_raw = pd.read_csv(GEARS_RUNS_DIR / f'gears_gi_run{_gears_best_run}.csv')
gears_classified, gears_class_df, gears_summary = evaluate_gi_classification(
    _gears_bestf1_raw, 'GEARS', gi_series)

syn_umap   = _fit_umap(slk_classified,   GI_METRICS)
gears_umap = _fit_umap(gears_classified, GI_METRICS)


def _cluster_metrics(umap_df, model_name, metrics):
    """Cluster quality metrics in the original metric space (not UMAP 2D)."""
    both  = umap_df[umap_df['gt_gi'].notna()].copy()
    avail = [c for c in metrics if c in both.columns]
    X_raw = both[avail].values.astype(float)
    finite = np.all(np.isfinite(X_raw), axis=1)
    both  = both[finite]
    X_s   = StandardScaler().fit_transform(X_raw[finite])
    y_gt  = both['gt_gi'].values.astype(str)
    y_pred = both['predicted_gi'].values.astype(str)
    if len(np.unique(y_gt)) > 1:
        _sil_s   = silhouette_samples(X_s, y_gt)
        _per_cls = pd.Series(_sil_s, index=y_gt).groupby(level=0).mean()
        sil      = float(_per_cls.mean())   # macro-average (equal weight per class)
    else:
        sil = float('nan')
    return {
        'model':      model_name,
        'silhouette': sil,
        'ari':        adjusted_rand_score(y_gt, y_pred),
        'nmi':        normalized_mutual_info_score(y_gt, y_pred),
    }


slk_clust   = _cluster_metrics(syn_umap,   'SHERLOCK', GI_METRICS)
gears_clust = _cluster_metrics(gears_umap, 'GEARS',    GI_METRICS)

for sd, cd in [(slk_summary, slk_clust), (gears_summary, gears_clust)]:
    sd['silhouette'] = cd['silhouette']
summary_df = pd.DataFrame([slk_summary, gears_summary])
print(summary_df[['model', 'f1_weighted', 'silhouette', 'ari', 'ami']].to_string(index=False))


# ── GI representative examples — used by panel D arrows & panel F heatmap ─
_cats_gi = ['approximately additive', 'epistasis', 'neomorphic',
            'potentiation', 'redundant', 'suppression', 'synergy']
_syn_all = {'synergy (similar phenotype)', 'synergy (dissimilar phenotype)', 'synergy'}
_val_set = set(val_combos)
_correct_gi = (
    (slk_classified['predicted_gi'] == slk_classified['gt_gi']) |
    (slk_classified['predicted_gi'].isin(_syn_all) & slk_classified['gt_gi'].isin(_syn_all))
)

_gi_examples = {}
_cbl_cnn1_idx = canon_pair('CBL+CNN1')
if _cbl_cnn1_idx in slk_classified.index:
    _gi_examples['synergy'] = slk_classified.loc[_cbl_cnn1_idx]
else:
    _p = slk_classified[slk_classified['gt_gi'].isin(_syn_all) &
                         slk_classified['predicted_gi'].isin(_syn_all)]
    _h = _p[_p.index.isin(_val_set)]
    _gi_examples['synergy'] = (_h if not _h.empty else _p).iloc[0]

for _cat in _cats_gi:
    if _cat == 'synergy':
        continue
    _p = slk_classified[_correct_gi & (slk_classified['gt_gi'] == _cat)]
    if _p.empty:
        continue
    _h = _p[_p.index.isin(_val_set)]
    _gi_examples[_cat] = (_h if not _h.empty else _p).iloc[0]

_gi_arrows = {}
for _cat, _row in _gi_examples.items():
    _na2, _nb2 = str(_row['pert_a']), str(_row['pert_b'])
    _cidx = canon_pair(f'{_na2}+{_nb2}')
    if _cidx in syn_umap.index:
        _gi_arrows[f'{_na2[:5]}/{_nb2[:5]}'] = _cidx

print(f'GI arrows ({len(_gi_arrows)}): {list(_gi_arrows.keys())}')

# ── Biologically motivated arrow annotations — GT-labelled pairs ──────────
HIGHLIGHT_COMBOS = {
    'DUSP9/MAPK1':   canon_pair('DUSP9+MAPK1'),
    'MAP2K6/SPI1':   canon_pair('MAP2K6+SPI1'),
    'CEBPA/B':       canon_pair('CEBPA+CEBPB'),
    'CBFA2T3/FEV':   canon_pair('FEV+CBFA2T3'),
    'CBL/PTPN12':    canon_pair('CBL+PTPN12'),
}

arrow_pairs = {}
for display, canonical in HIGHLIGHT_COMBOS.items():
    if canonical in syn_umap.index:
        row = syn_umap.loc[canonical]
        arrow_pairs[display] = canonical
        print(f'  GT  {display:18s} → GT: {row.get("gt_gi","N/A")}  Pred: {row.get("predicted_gi","N/A")}')
    else:
        print(f'  GT  {display:18s} not found in syn_umap')

# ── Novel annotations — unlabelled combinations of biological interest ────
NOVEL_COMBOS = {
    'BAK1/Bim':   canon_pair('BAK1+BCL2L11'),
    'p21/p27':    canon_pair('CDKN1A+CDKN1B'),
    'CEBPA/JUN':  canon_pair('CEBPA+JUN'),
    'UBASH3A/B':  canon_pair('UBASH3A+UBASH3B'),
}

novel_arrow_pairs = {}
for display, canonical in NOVEL_COMBOS.items():
    if canonical in syn_umap.index:
        row = syn_umap.loc[canonical]
        novel_arrow_pairs[display] = canonical
        print(f'  Novel {display:16s} → Pred: {row.get("predicted_gi","N/A")}')
    else:
        print(f'  Novel {display:16s} not found in syn_umap')

print(f'\n{len(arrow_pairs)} GT-labelled + {len(novel_arrow_pairs)} novel combinations annotated.')

In [ ]:
# ── GI-classification significance helper ────────────────────────────────
# SHERLOCK vs GEARS Welch t-test per summary metric (F1 / ARI / AMI), with a
# Benjamini-Hochberg FDR correction across the three metrics, drawn as a
# ggsignif-style bracket above each bar pair. Reused by every Panel-E summary.
def add_gi_significance(ax, long_df, model_a='SHERLOCK', model_b='GEARS',
                        metric_order=('F1', 'ARI', 'AMI'),
                        width=0.55, n_hue=2, y_pad=0.05, tick=0.018, fs=5):
    """`long_df` has columns model / Metric / score (one row per run). Bars are a
    seaborn barplot of x='Metric', hue='model' at the given `width`, so the two
    bar centres per metric sit at i +/- width/(2*n_hue)."""
    from scipy import stats as _st

    _p, _top = [], []
    for _m in metric_order:
        _a = long_df[(long_df['Metric'] == _m) & (long_df['model'] == model_a)]['score'].dropna().to_numpy()
        _b = long_df[(long_df['Metric'] == _m) & (long_df['model'] == model_b)]['score'].dropna().to_numpy()
        _p.append(_st.ttest_ind(_a, _b, equal_var=False).pvalue if len(_a) >= 2 and len(_b) >= 2 else np.nan)
        _cand = [np.nanmean(_a) + np.nanstd(_a) if len(_a) else np.nan,
                 np.nanmean(_b) + np.nanstd(_b) if len(_b) else np.nan]
        _top.append(np.nanmax(_cand) if np.any(np.isfinite(_cand)) else 1.0)

    # Benjamini-Hochberg across the three metrics
    _pa = np.array(_p, float); _adj = np.full_like(_pa, np.nan)
    _valid = np.where(np.isfinite(_pa))[0]
    _ordv = _valid[np.argsort(_pa[_valid])]; _mtot = len(_ordv); _prev = 1.0
    for _k in range(_mtot - 1, -1, -1):
        _ii = _ordv[_k]; _prev = min(_prev, _pa[_ii] * _mtot / (_k + 1)); _adj[_ii] = min(_prev, 1.0)

    def _stars(_q):
        if not np.isfinite(_q):
            return ''
        return '***' if _q < 1e-3 else '**' if _q < 1e-2 else '*' if _q < 0.05 else 'ns'

    # Asterisks sit high in the text box, 'ns' sits on the baseline; nudge each so
    # the ink lands at the same height above every bracket regardless of glyph.
    _lbl_dy = {'ns': 0.006}
    _x_a = -width / 2 + width / (2 * n_hue)                 # model_a bar offset within a group
    _x_b = _x_a + (n_hue - 1) * (width / n_hue)             # model_b bar offset
    _ytop = -np.inf
    for _i, _m in enumerate(metric_order):
        _lbl = _stars(_adj[_i])
        if not _lbl:
            continue
        _x1, _x2 = _i + _x_a, _i + _x_b
        _y = _top[_i] + y_pad
        ax.plot([_x1, _x1, _x2, _x2], [_y, _y + tick, _y + tick, _y],
                lw=0.5, c='0.35', clip_on=False)
        ax.text((_x1 + _x2) / 2, _y + tick + _lbl_dy.get(_lbl, -0.014), _lbl,
                ha='center', va='bottom', fontsize=fs, color='0.2', clip_on=False)
        _ytop = max(_ytop, _y + tick + 0.04)
    if np.isfinite(_ytop):
        _lo, _hi = ax.get_ylim()
        ax.set_ylim(_lo, max(_hi, _ytop))
    return ax


In [ ]:
# ── Panel A: Counterfactual ATE — in-distribution and OOD ────────────────
METRIC_LABELS = {
    'ate_all':    'ATE\n(all)',
    'ate_single': 'ATE\n(singles)',
    'ate_combo':  'ATE\n(combos)',
    'ate_held':   'ATE\n(held-out combos)',
}

METHOD_ORDER = ['SHERLOCK', 'GEARS', 'cVAE', 'sVAE', 'contrastiveVI']
METHOD_PALETTE = {
    'SHERLOCK':      CLR_SHERLOCK,
    'cVAE':          '#33a02c',
    'sVAE':          '#ff7f00',
    'contrastiveVI': '#6a3d9a',
    'GEARS':         CLR_GEARS,
}

sherlock_ate_df = runs_df[['run'] + list(METRIC_LABELS)].copy()
sherlock_ate_df['method'] = 'SHERLOCK'
gears_ate_df = _gears_run_summ_df[['run'] + list(METRIC_LABELS)].copy()
gears_ate_df['method'] = 'GEARS'
all_ate_df = pd.concat(
    [sherlock_ate_df, gears_ate_df,
     bench_runs_df[['method', 'run'] + list(METRIC_LABELS)]],
    ignore_index=True,
)

df_long = all_ate_df.melt(
    id_vars=['method', 'run'],
    value_vars=list(METRIC_LABELS),
    var_name='metric', value_name='Pearson r',
)
df_long['Metric'] = pd.Categorical(
    df_long['metric'].map(METRIC_LABELS),
    categories=[METRIC_LABELS[k] for k in METRIC_LABELS],
    ordered=True,
)

TEXT_SCALE_A = -2.  # +/- pt added to all Panel A text uniformly; 0.0 == RC defaults
TEXT_SCALE_B = -1.5

fig_a, ax_a = plt.subplots(figsize=(300/PPI, 150/PPI), constrained_layout=True)

sns.boxplot(
    data=df_long, x='Metric', y='Pearson r', hue='method',
    hue_order=METHOD_ORDER, palette=METHOD_PALETTE,
    width=0.75, linewidth=0.6, fliersize=0, ax=ax_a,
    boxprops=dict(alpha=0.55),
)
sns.stripplot(
    data=df_long, x='Metric', y='Pearson r', hue='method',
    hue_order=METHOD_ORDER, palette=METHOD_PALETTE,
    dodge=True, size=2.6 + TEXT_SCALE_A/3, jitter=0.1,
    linewidth=0.3, edgecolor='white', ax=ax_a, legend=False,
)

ax_a.axvline(2.5, color='0.55', linewidth=0.8, linestyle='--', alpha=0.7)
ax_a.set_ylim(bottom=0)

# ── Significance: SHERLOCK vs every other method, per column ──────────────
# Welch's t-test on the per-run values, Benjamini-Hochberg FDR across all
# comparisons; drawn as R/ggsignif-style brackets with ns / * / ** / ***.
from scipy import stats as _stats
_sig_metrics = list(METRIC_LABELS)                 # ate_all, ate_single, ate_combo, ate_held
_sig_others  = [m for m in METHOD_ORDER if m != 'SHERLOCK']

def _mvals(_mk, _meth):
    return df_long[(df_long['metric'] == _mk) &
                   (df_long['method'] == _meth)]['Pearson r'].dropna().to_numpy()

_comps = []                                        # [col_i, other_method, p]
for _i, _mk in enumerate(_sig_metrics):
    _s = _mvals(_mk, 'SHERLOCK')
    for _o in _sig_others:
        _ov = _mvals(_mk, _o)
        if len(_s) < 2 or len(_ov) < 2:
            continue
        _comps.append([_i, _o, _stats.ttest_ind(_s, _ov, equal_var=False).pvalue])

# Benjamini-Hochberg across the whole family
_pv = np.array([c[2] for c in _comps])
_ord = np.argsort(_pv); _mtot = len(_pv); _adj = np.empty(_mtot); _prev = 1.0
for _k in range(_mtot - 1, -1, -1):
    _ix = _ord[_k]; _prev = min(_prev, _pv[_ix] * _mtot / (_k + 1)); _adj[_ix] = min(_prev, 1.0)
for _c, _pa in zip(_comps, _adj):
    _c.append(_pa)

def _stars(_p):
    return '***' if _p < 1e-3 else '**' if _p < 1e-2 else '*' if _p < 0.05 else 'ns'

# dodge geometry (boxplot width=0.75, one box per method)
_n_hue, _box_w = len(METHOD_ORDER), 0.75
def _xpos(_i, _meth):
    _j = METHOD_ORDER.index(_meth)
    return _i - _box_w / 2 + _box_w / (2 * _n_hue) + _j * (_box_w / _n_hue)

_y0   = float(df_long['Pearson r'].max()) + 0.03   # bracket base, above the boxes
_step = 0.052                                      # gap between stacked brackets
_tick = 0.012                                      # bracket end-tick height
_sfs  = 6 + TEXT_SCALE_A
_ytop = _y0
for _i in range(len(_sig_metrics)):
    _col = sorted([c for c in _comps if c[0] == _i],
                  key=lambda c: METHOD_ORDER.index(c[1]))   # nearest (GEARS) lowest
    for _lvl, (_, _o, _p, _pa) in enumerate(_col):
        _x1, _x2 = _xpos(_i, 'SHERLOCK'), _xpos(_i, _o)
        _y = _y0 + _lvl * _step
        ax_a.plot([_x1, _x1, _x2, _x2], [_y, _y + _tick, _y + _tick, _y],
                  lw=0.5, c='0.35', clip_on=False)
        ax_a.text((_x1 + _x2) / 2, _y + _tick, _stars(_pa),
                  ha='center', va='bottom', fontsize=_sfs, color='0.2', clip_on=False)
        _ytop = max(_ytop, _y + _tick)
ax_a.set_ylim(0, _ytop + 0.05)
_ylim = ax_a.get_ylim()
_y_ann = _ylim[1] + 0.07 * (_ylim[1] - _ylim[0])
ax_a.text(1.0, _y_ann, 'in distribution', ha='center', va='top',
          fontsize=5.5 , color='0.45', style='italic')
ax_a.text(3.0, _y_ann, 'out of distribution', ha='center', va='top',
          fontsize=5.5 , color='0.45', style='italic')

ax_a.set_xlabel('')
ax_a.set_ylabel('Pearson $r$', fontsize=mpl.rcParams['axes.labelsize'] + TEXT_SCALE_B)
ax_a.set_title('Counterfactual ATE', pad=9, fontsize=mpl.rcParams['axes.titlesize'] )
ax_a.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.2f'))
ax_a.tick_params(axis='x', labelsize=mpl.rcParams['xtick.labelsize'] + TEXT_SCALE_B)
ax_a.tick_params(axis='y', labelsize=mpl.rcParams['ytick.labelsize'] + TEXT_SCALE_B)
ax_a.text(0.02, 0.04, f'$n$ = {N_RUNS} runs',
          transform=ax_a.transAxes, ha='left', va='bottom',
          fontsize=6 + TEXT_SCALE_A, color='0.4')
ax_a.legend(
    title='Method', frameon=False, loc='upper right',
    bbox_to_anchor=(1.15, 0.95),
    fontsize=mpl.rcParams['legend.fontsize'] + TEXT_SCALE_A,
    title_fontsize=mpl.rcParams['legend.title_fontsize'] + TEXT_SCALE_A,
)

fig_a.savefig(OUT_DIR / 'panel_a_ate.svg')
plt.show()
print('Panel A saved.')

### Fig. S7: reproducibility across five runs (Norman)

In [ ]:
# ── Panel B: reproducibility across runs + accuracy against the data ─────
# Four columns, two kinds of measure:
#   cross-run (leave-one-out Spearman: one run vs the mean of the others)
#     * perturbation correlation matrix
#     * gate co-activation matrix
#     * counterfactual effect profiles (perturbation x gene)
#   accuracy (within run, Pearson vs the measured effects)
#     * counterfactual effect sizes (per-perturbation effect magnitudes)

# Per-run counterfactual effect matrices, cached alongside the other matrices.
CF_PATH = OUT_DIR / 'sherlock_cf_effects.npz'
if CF_PATH.exists():
    _cf = np.load(CF_PATH, allow_pickle=True)
else:
    _store, _obs = {}, None
    for _rn in range(1, N_RUNS + 1):
        _m = slk.tl.load_results(checkpoints[_rn]).to(DEVICE)
        _ate = _m.predict_counterfactual_effects(
            data, n_centroids=25, observed_effect=data.uns['treat_effect'], approximate=False,
        )
        _p, _o = _ate['predicted_df'], _ate['observed_df']
        _common = _p.index.intersection(_o.index)
        _store[f'pred_{_rn}'] = _p.loc[_common].values.astype(np.float32)
        if _obs is None:
            _obs = _o.loc[_common].values.astype(np.float32)
            _store['perts'] = np.array(_common, dtype=object)
        del _m
    _store['obs'] = _obs
    np.savez_compressed(CF_PATH, **_store)
    _cf = np.load(CF_PATH, allow_pickle=True)

pred_by_run = {int(k.split('_')[-1]): _cf[k] for k in _cf.files if k.startswith('pred_')}
obs_mat     = _cf['obs']


def _loo_spearman_flat(mats_by_run, metric_name):
    """Leave-one-out Spearman for non-symmetric (perturbation x gene) matrices."""
    rows, keys = [], sorted(mats_by_run)
    for k in keys:
        mean_other = np.mean([mats_by_run[j] for j in keys if j != k], axis=0)
        a, b = mats_by_run[k].ravel(), mean_other.ravel()
        v = np.isfinite(a) & np.isfinite(b)
        rows.append({'metric': metric_name, 'run': k, 'r': float(spearmanr(a[v], b[v])[0])})
    return pd.DataFrame(rows)


L_RHO, L_GATE = 'Perturbation\ncorrelation', 'Gate\nco-activation'
L_CFP, L_CFS  = 'Counterfactual\neffect profile', 'Counterfactual\neffect size'
ROBUST_LABELS = {'rho_corr_r': L_RHO, 'w_coact_r': L_GATE,
                 'cf_effect_r': L_CFP, 'effect_size_r': L_CFS}

_cf_cross = _loo_spearman_flat(pred_by_run, 'cf_effect_r')

df_robust = pd.concat([
    pairwise_df.rename(columns={'r': 'Score'})[['metric', 'Score']],
    _cf_cross.rename(columns={'r': 'Score'})[['metric', 'Score']],
    runs_df[['effect_size_r']].rename(columns={'effect_size_r': 'Score'}).assign(metric='effect_size_r'),
], ignore_index=True)
df_robust['Metric'] = pd.Categorical(
    df_robust['metric'].map(ROBUST_LABELS),
    categories=[ROBUST_LABELS[k] for k in ROBUST_LABELS], ordered=True,
)

print(df_robust.groupby('metric', observed=True)['Score'].agg(['mean', 'std']).round(3).to_string())

CLR_ACC = '#7f7f7f'
_pal = {L_RHO: CLR_SHERLOCK, L_GATE: CLR_SHERLOCK, L_CFP: CLR_SHERLOCK, L_CFS: CLR_ACC}

fig_b2, ax_b2 = plt.subplots(figsize=(4.4, 3.0))
sns.stripplot(data=df_robust, x='Metric', y='Score', hue='Metric', palette=_pal, legend=False,
              size=5, jitter=0.15, linewidth=0.4, edgecolor='white', ax=ax_b2)
sns.boxplot(data=df_robust, x='Metric', y='Score', color='white', width=0.45, linewidth=0.75,
            fliersize=0, ax=ax_b2, boxprops=dict(edgecolor='0.3'),
            whiskerprops=dict(color='0.3'), capprops=dict(color='0.3'),
            medianprops=dict(color='0.15', linewidth=1.5))
ax_b2.set_ylim(0, 1.18)
ax_b2.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax_b2.set_xlabel('')
ax_b2.set_ylabel('Correlation')
ax_b2.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.2f'))
ax_b2.axvline(2.5, color='0.75', lw=0.6, ls='--')
ax_b2.plot([-0.4, 2.4], [1.12, 1.12], color=CLR_SHERLOCK, lw=1, clip_on=False)
ax_b2.text(1.0, 1.14, 'reproducibility across runs', ha='center', va='bottom',
           fontsize=7, color=CLR_SHERLOCK)
ax_b2.plot([2.6, 3.4], [1.12, 1.12], color=CLR_ACC, lw=1, clip_on=False)
ax_b2.text(3.0, 1.14, 'accuracy vs data', ha='center', va='bottom', fontsize=7, color=CLR_ACC)
ax_b2.text(0.99, 0.02, f'$n$ = {N_RUNS} runs', transform=ax_b2.transAxes,
           ha='right', va='bottom', fontsize=6, color='0.4')

plt.tight_layout()
fig_b2.savefig(OUT_DIR / 'panel_b_robustness.svg', bbox_inches='tight')
fig_b2.savefig(OUT_DIR / 'panel_b_robustness.png', dpi=300, bbox_inches='tight')
plt.show()
print('Panel B (robustness) saved.')


In [ ]:
slk_classified[slk_classified.index.str.contains('SET')]

In [ ]:
# ── Panel C: SHERLOCK latent space — Excel-associated perturbations ───────
# Three UMAPs of the SAME subset (all labeled "SHERLOCK latent space"):
#   1. coloured by perturbation name
#   2. coloured by group/cluster (wide figure so the legend fits)
#   3. coloured by single vs combination
# Genes + groups are hard-coded from aax4438_tables6.xlsx (both sheets combined;
# any gene appearing in >1 cluster was dropped, then clusters left with <2 genes
# were dropped). Subset = singles that are group genes + any combo with >=1 group
# gene (the partner need not be a group gene). One fresh UMAP is fit on the subset
# (no cache file, so nothing can go stale).
#
# Group label (fig 2 only): single-group clusters are shown; a combo whose two
# genes fall in DIFFERENT groups ("cluster / cluster") is dropped; a combo with
# only one gene in a group collapses into that gene's named group.
GENE_GROUPS = {
    'MAP2K':                   ['MAP2K3', 'MAP2K6'],
    'Ig domain containing':    ['IGDCC3', 'PRTG'],
    'Tyrosine phosphatase':    ['PTPN12', 'PTPN9'],
    'MAP4K':                   ['MAP4K3', 'MAP4K5'],
    'Brachyury TF':            ['CSRNP1', 'SET', 'TBX2', 'TBX3'],
    'Kinesin':                 ['KIF18B', 'KIF2C'],
    'bLZ TF':                  ['CEBPA', 'CEBPB', 'CEBPE', 'CELF2', 'FOSB', 'OSR2'],
    'Regulation of apoptosis': ['EGR1', 'FOXO4', 'IER5L', 'IKZF3', 'MEIS1', 'PRDM1',
                                'RHOXF2B', 'SAMD1', 'SGK1', 'TP73'],
    'Homeobox TF':             ['CBFA2T3', 'CITED1', 'DLX2', 'MIDN', 'NIT1', 'SNAI1', 'TMSB4X'],
}
_gene2group = {g: grp for grp, genes in GENE_GROUPS.items() for g in genes}
_EXCEL_GENES = set(_gene2group)

def _in_subset(name):
    """Single that is a group gene, or a combo with >=1 group gene."""
    return any(p in _EXCEL_GENES for p in str(name).split('+'))

def _group_of(name):
    """Group label: a single group; 'group A / group B' when the two genes
    fall in different groups; or None (no group gene). A combo with only one
    gene in a group collapses to that group."""
    parts = str(name).split('+')
    if len(parts) == 1:
        return _gene2group.get(parts[0])
    ga, gb = _gene2group.get(parts[0]), _gene2group.get(parts[1])
    if ga and gb:
        return ga if ga == gb else ' / '.join(sorted([ga, gb]))  # two groups
    return ga or gb                              # one gene in a group -> its group

# ── Build subset from the SHERLOCK latent space (same 'z' as the full panel) ──
_c_mask = all_sub.obs[p_key].astype(str).map(_in_subset).values
c_sub   = all_sub[_c_mask].copy()
c_sub.obs['pert_name']   = c_sub.obs[p_key].astype(str).values
c_sub.obs['group_label'] = c_sub.obs['pert_name'].map(_group_of).values
c_sub.obs['pert_type']   = np.where(
    c_sub.obs['pert_name'].str.contains(r'\+', regex=True), 'combination', 'single')

sc.pp.neighbors(c_sub, n_neighbors=15, use_rep='z')
sc.tl.umap(c_sub, random_state=42)
_coords = c_sub.obsm['X_umap']
print(f'Panel C subset: {c_sub.n_obs} cells | '
      f"{c_sub.obs['pert_name'].nunique()} perturbations | "
      f"{c_sub.obs['group_label'].nunique()} group categories (shown)")

def _palette(cats):
    """Distinct qualitative colours (tab20 + tab20b + tab20c = 60), cycled."""
    cats = list(cats)
    base = (list(plt.get_cmap('tab20').colors)
            + list(plt.get_cmap('tab20b').colors)
            + list(plt.get_cmap('tab20c').colors))
    return {c: base[i % len(base)] for i, c in enumerate(cats)}

def _style_umap(ax, coords, color_vals, palette, title):
    ax.scatter(coords[:, 0], coords[:, 1],
               c=[palette[v] for v in color_vals],
               s=0.8, alpha=0.4, linewidths=0, rasterized=True)
    ax.set_xlabel(''); ax.set_ylabel(''); ax.set_title(title, pad=4)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    _add_umap_axis_indicator(ax, length=1.0)

_TITLE = 'SHERLOCK latent space'

# ── 1. coloured by perturbation name (many perturbations -> no legend) ────
_name_cats = sorted(c_sub.obs['pert_name'].unique())
_name_pal  = _palette(_name_cats)
fig_c1, ax_c1 = plt.subplots(figsize=(196/PPI, 171/PPI), constrained_layout=True)
_style_umap(ax_c1, _coords, c_sub.obs['pert_name'].values, _name_pal, _TITLE)
fig_c1.savefig(OUT_DIR / 'panel_c_umap_byname.svg')
plt.show()

# ── 2. coloured by group/cluster (incl. two-group combos), subsampled ─────
SUBSAMPLE_FRAC = 0.3  # fraction of cells drawn in this panel (density control)
TEXT_SCALE_C   = -1.0  # +/- pt on the cluster legend text (shrink to fit the figsize)
_grp_valid = c_sub.obs['group_label'].notna().values
_grp_vals  = c_sub.obs['group_label'].values
_grp_cats  = sorted(set(_grp_vals[_grp_valid]))
_grp_pal   = _palette(_grp_cats)
# subsample cells so the embedding reads less dense (legend keeps all cats)
_grp_plot  = _grp_valid & (np.random.default_rng(0).random(c_sub.n_obs) < SUBSAMPLE_FRAC)
fig_c2 = plt.figure(figsize=(285/PPI, 185/PPI))
_gs_c2  = fig_c2.add_gridspec(1, 2, width_ratios=[2.5, 1.0], wspace=0.08)
ax_c2   = fig_c2.add_subplot(_gs_c2[0, 0])
_ax_leg = fig_c2.add_subplot(_gs_c2[0, 1]); _ax_leg.axis('off')
_style_umap(ax_c2, _coords[_grp_plot], _grp_vals[_grp_plot], _grp_pal, _TITLE)
_ax_leg.legend(
    handles=[Patch(color=_grp_pal[g], label=g.replace(' / ', ' /\n')) for g in _grp_cats],
    loc='center left', frameon=False, fontsize=5 + TEXT_SCALE_C,
    handlelength=0.9, handletextpad=0.4, labelspacing=0.4, borderaxespad=0,
    title='Cluster', title_fontsize=6 + TEXT_SCALE_C,
)
fig_c2.savefig(OUT_DIR / 'panel_c_umap_bygroup.svg')
plt.show()

# ── 3. coloured by single vs combination ─────────────────────────────────
_type_pal = {'single': CLR_SHERLOCK, 'combination': '#E8735A'}
fig_c3, ax_c3 = plt.subplots(figsize=(196/PPI, 171/PPI), constrained_layout=True)
_style_umap(ax_c3, _coords, c_sub.obs['pert_type'].values, _type_pal, _TITLE)
ax_c3.legend(
    handles=[Patch(color=v, label=k.capitalize()) for k, v in _type_pal.items()],
    frameon=False, loc='lower right', handlelength=0.8, borderaxespad=0,
)
fig_c3.savefig(OUT_DIR / 'panel_c_umap_bytype.svg')
plt.show()

print('Panel C (3 UMAPs) saved.')

In [ ]:
# ── SHERLOCK latent space — one figure per double (two-group) cluster ─────
# For each "A / B" dual grouping, colour its combos plus the two single-group
# clusters A and B; everything else is grey. Uses the single shared manifold
# (_coords) built in the Panel C cell above.
_dual_labels = sorted(l for l in c_sub.obs['group_label'].dropna().unique()
                      if ' / ' in str(l))
print(f'{len(_dual_labels)} double groupings: {_dual_labels}')

for _dual in _dual_labels:
    _a, _b  = _dual.split(' / ')
    _cats   = [_a, _b, _dual]
    _pal    = _palette(_cats)               # A, B, A/B -> three distinct colours
    _gl     = c_sub.obs['group_label'].values
    _mask   = np.isin(_gl, _cats)

    fig_d  = plt.figure(figsize=(420/PPI, 185/PPI))
    _gs_d   = fig_d.add_gridspec(1, 2, width_ratios=[1.4, 1.0], wspace=0.02)
    ax_d    = fig_d.add_subplot(_gs_d[0, 0])
    _ax_dl  = fig_d.add_subplot(_gs_d[0, 1]); _ax_dl.axis('off')

    # grey background (everything not in A, B, or A/B)
    ax_d.scatter(_coords[~_mask, 0], _coords[~_mask, 1],
                 c='0.85', s=0.8, alpha=0.4, linewidths=0, rasterized=True)
    # A, B and the A/B combos on top
    ax_d.scatter(_coords[_mask, 0], _coords[_mask, 1],
                 c=[_pal[v] for v in _gl[_mask]],
                 s=3.0, alpha=0.9, linewidths=0, rasterized=True)

    ax_d.set_xlabel(''); ax_d.set_ylabel('')
    ax_d.set_title('SHERLOCK latent space', pad=4)
    ax_d.set_xticks([]); ax_d.set_yticks([])
    for sp in ax_d.spines.values():
        sp.set_visible(False)
    _add_umap_axis_indicator(ax_d, length=1.0)

    _ax_dl.legend(
        handles=[Patch(color=_pal[c], label=c) for c in _cats],
        loc='center left', frameon=False, fontsize=5,
        handlelength=0.9, handletextpad=0.4, labelspacing=0.3, borderaxespad=0,
        title=_dual, title_fontsize=6,
    )
    _svg = _dual.replace(' / ', '__').replace(' ', '_')
    fig_d.savefig(OUT_DIR / f'panel_c_umap_dual_{_svg}.svg', bbox_inches='tight')
    plt.show()

print('Double-group UMAPs saved.')

In [ ]:
gi_labels[gi_labels['combination'].str.contains('PTPN12')]

In [ ]:
# ── SHERLOCK latent space — one combo + its two singles ──────────────────
# Colours a chosen combo and its two constituent single perturbations by name;
# everything else grey. Uses the shared manifold (_coords) from the Panel C cell.
HIGHLIGHT_COMBO = 'IKZF3+MAP2K6'

_ca, _cb = HIGHLIGHT_COMBO.split('+')
_pn = c_sub.obs['pert_name'].astype(str)
_is_combo = _pn.map(lambda n: '+' in n and sorted(n.split('+')) == sorted([_ca, _cb])).values

_lab = pd.Series(index=c_sub.obs.index, dtype=object)
_lab[(_pn == _ca).values] = _ca
_lab[(_pn == _cb).values] = _cb
_lab[_is_combo]           = HIGHLIGHT_COMBO
_show = _lab.notna().values

_cats = [_ca, _cb, HIGHLIGHT_COMBO]
_pal  = _palette(_cats)

fig_hc = plt.figure(figsize=(420/PPI, 185/PPI))
_gs_hc  = fig_hc.add_gridspec(1, 2, width_ratios=[1.4, 1.0], wspace=0.02)
ax_hc   = fig_hc.add_subplot(_gs_hc[0, 0])
_ax_hcl = fig_hc.add_subplot(_gs_hc[0, 1]); _ax_hcl.axis('off')

# grey background (everything not in the combo or its singles)
ax_hc.scatter(_coords[~_show, 0], _coords[~_show, 1],
              c='0.85', s=0.8, alpha=0.4, linewidths=0, rasterized=True)
# the combo + its two singles on top
ax_hc.scatter(_coords[_show, 0], _coords[_show, 1],
              c=[_pal[l] for l in _lab.values[_show]],
              s=3.0, alpha=0.9, linewidths=0, rasterized=True)

ax_hc.set_xlabel(''); ax_hc.set_ylabel('')
ax_hc.set_title('SHERLOCK latent space', pad=4)
ax_hc.set_xticks([]); ax_hc.set_yticks([])
for sp in ax_hc.spines.values():
    sp.set_visible(False)
_add_umap_axis_indicator(ax_hc, length=1.0)

_ax_hcl.legend(
    handles=[Patch(color=_pal[c], label=c) for c in _cats],
    loc='center left', frameon=False, fontsize=5,
    handlelength=0.9, handletextpad=0.4, labelspacing=0.3, borderaxespad=0,
    title=HIGHLIGHT_COMBO, title_fontsize=6,
)
_svg = HIGHLIGHT_COMBO.replace('+', '_')
fig_hc.savefig(OUT_DIR / f'panel_c_umap_combo_{_svg}.svg', bbox_inches='tight')
plt.show()
print(f'{HIGHLIGHT_COMBO}: {int(_is_combo.sum())} combo cells | '
      f'{int((_pn == _ca).sum())} {_ca} | {int((_pn == _cb).sum())} {_cb}')

In [ ]:
# ── SHERLOCK latent space — one combo + its two singles ──────────────────
# Colours a chosen combo and its two constituent single perturbations by name;
# everything else grey. Uses the shared manifold (_coords) from the Panel C cell.
HIGHLIGHT_COMBO = 'CEBPE+SET'

_ca, _cb = HIGHLIGHT_COMBO.split('+')
_pn = c_sub.obs['pert_name'].astype(str)
_is_combo = _pn.map(lambda n: '+' in n and sorted(n.split('+')) == sorted([_ca, _cb])).values

_lab = pd.Series(index=c_sub.obs.index, dtype=object)
_lab[(_pn == _ca).values] = _ca
_lab[(_pn == _cb).values] = _cb
_lab[_is_combo]           = HIGHLIGHT_COMBO
_show = _lab.notna().values

_cats = [_ca, _cb, HIGHLIGHT_COMBO]
_pal  = _palette(_cats)

fig_hc = plt.figure(figsize=(420/PPI, 185/PPI))
_gs_hc  = fig_hc.add_gridspec(1, 2, width_ratios=[1.4, 1.0], wspace=0.02)
ax_hc   = fig_hc.add_subplot(_gs_hc[0, 0])
_ax_hcl = fig_hc.add_subplot(_gs_hc[0, 1]); _ax_hcl.axis('off')

# grey background (everything not in the combo or its singles)
ax_hc.scatter(_coords[~_show, 0], _coords[~_show, 1],
              c='0.85', s=0.8, alpha=0.4, linewidths=0, rasterized=True)
# the combo + its two singles on top
ax_hc.scatter(_coords[_show, 0], _coords[_show, 1],
              c=[_pal[l] for l in _lab.values[_show]],
              s=3.0, alpha=0.9, linewidths=0, rasterized=True)

ax_hc.set_xlabel(''); ax_hc.set_ylabel('')
ax_hc.set_title('SHERLOCK latent space', pad=4)
ax_hc.set_xticks([]); ax_hc.set_yticks([])
for sp in ax_hc.spines.values():
    sp.set_visible(False)
_add_umap_axis_indicator(ax_hc, length=1.0)

_ax_hcl.legend(
    handles=[Patch(color=_pal[c], label=c) for c in _cats],
    loc='center left', frameon=False, fontsize=5,
    handlelength=0.9, handletextpad=0.4, labelspacing=0.3, borderaxespad=0,
    title=HIGHLIGHT_COMBO, title_fontsize=6,
)
_svg = HIGHLIGHT_COMBO.replace('+', '_')
fig_hc.savefig(OUT_DIR / f'panel_c_umap_combo_{_svg}.svg', bbox_inches='tight')
plt.show()
print(f'{HIGHLIGHT_COMBO}: {int(_is_combo.sum())} combo cells | '
      f'{int((_pn == _ca).sum())} {_ca} | {int((_pn == _cb).sum())} {_cb}')

In [ ]:
# ── Panel D: Synergy UMAP — GT class (left) vs predicted class (right) ──
# Display-only relabelling: the two synergy sub-classes are shown as one
# "synergy" and "approximately additive" is shown as "additive". Underlying
# gt_gi / predicted_gi columns (and all metrics) are unchanged.
_GI_DISPLAY = {
    'synergy (similar phenotype)':    'synergy',
    'synergy (dissimilar phenotype)': 'synergy',
    'approximately additive':         'additive',
}
PALETTE_GI_DISPLAY = {
    'additive':     '#4DAF4A',  # green
    'epistasis':    '#FF7F00',  # orange
    'neomorphic':   '#984EA3',  # purple
    'potentiation': '#F781BF',  # pink
    'redundant':    '#A65628',  # brown
    'suppression':  '#E41A1C',  # red
    'synergy':      '#377EB8',  # blue
}

def _draw_synergy_umap(ax, umap_df, colour_col, title,
                       val_combos=None, arrow_pairs=None, show_unlabelled=False,
                       right_labels=None, show_axis=True, text_scale=0.0, margin=0.1):
    """
    colour_col: column to colour by (gt_gi or predicted_gi).
    show_unlabelled: if True, draw NaN rows as gray (for GT panel only).
    arrow_pairs: {display_label: canonical_idx}
    """
    labelled   = umap_df[umap_df[colour_col].notna()].copy()
    unlabelled = umap_df[umap_df[colour_col].isna()].copy()
    # merge synergy sub-classes -> 'synergy', 'approximately additive' -> 'additive'
    labelled[colour_col] = labelled[colour_col].map(lambda _s: _GI_DISPLAY.get(str(_s), _s))

    if show_unlabelled and len(unlabelled) > 0:
        ax.scatter(
            unlabelled['UMAP1'], unlabelled['UMAP2'],
            color='0.75', s=10, alpha=0.45, linewidths=0, zorder=1,
        )

    if val_combos is not None:
        in_dist  = labelled[~labelled.index.isin(val_combos)]
        out_dist = labelled[labelled.index.isin(val_combos)]
    else:
        in_dist  = labelled
        out_dist = labelled.iloc[0:0]

    for cls in sorted(labelled[colour_col].unique()):
        color   = PALETTE_GI_DISPLAY.get(str(cls), '#888888')
        grp_id  = in_dist[in_dist[colour_col] == cls]
        grp_ood = out_dist[out_dist[colour_col] == cls]
        if len(grp_id) > 0:
            ax.scatter(
                grp_id['UMAP1'], grp_id['UMAP2'],
                color=color, marker='o', s=18, alpha=0.85,
                linewidths=0.3, edgecolors='white', zorder=2,
            )
        if len(grp_ood) > 0:
            ax.scatter(
                grp_ood['UMAP1'], grp_ood['UMAP2'],
                color=color, marker='^', s=22, alpha=0.85,
                linewidths=0.3, edgecolors='white', zorder=3,
            )

    if arrow_pairs:
        for display_label, idx in arrow_pairs.items():
            if idx not in umap_df.index:
                continue
            row  = umap_df.loc[idx]
            x, y = float(row['UMAP1']), float(row['UMAP2'])
            _right = right_labels and display_label in right_labels
            ax.annotate(
                display_label,
                xy=(x, y), xytext=(x + 1.5 if _right else x - 1.5, y + 0.8),
                fontsize=6.5 + text_scale, color='0.1',
                ha='left' if _right else 'right', va='bottom',
                arrowprops=dict(
                    arrowstyle='->', color='0.35',
                    lw=0.8, shrinkA=2, shrinkB=2,
                ),
            )

    ax.margins(margin)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(title, pad=3, y=1.1, fontsize=mpl.rcParams['axes.titlesize'] + text_scale)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    if show_axis:
        _add_umap_axis_indicator(ax, length=1.0, fontsize=mpl.rcParams['axes.labelsize'] + text_scale)


TEXT_SCALE = - 1.0  # +/- pt added to all Panel D text uniformly; 0.0 == RC defaults

fig_d, axes_d = plt.subplots(1, 2, figsize=(312/PPI, 172/PPI), constrained_layout=True)

_draw_synergy_umap(axes_d[0], syn_umap, 'gt_gi', 'Norman GI Labels',
                   val_combos=val_combos, arrow_pairs=None,
                   show_unlabelled=True, show_axis=True, text_scale=TEXT_SCALE, margin=0.1)
_draw_synergy_umap(axes_d[1], syn_umap, 'predicted_gi', 'Sherlock GI Prediction',
                   val_combos=val_combos, arrow_pairs=_gi_arrows,
                   show_unlabelled=False, show_axis=True, text_scale=TEXT_SCALE, margin=0.1,
                   right_labels={'CBL/CNN1', 'CEBPB/PTPN1'})

# Legend
SHOW_LEGEND = True  # temporarily off to check panel sizing on the 371x172px canvas
class_handles = [Patch(color=PALETTE_GI_DISPLAY.get(cls, '#888888'), label=cls)
                 for cls in sorted(PALETTE_GI_DISPLAY)]
extra_handles = [
    Patch(color='0.75', label='No GT label'),
    Patch(color='none', label=''),
    Line2D([0], [0], marker='o', color='0.4', linestyle='None', ms=5, label='In-distribution'),
    Line2D([0], [0], marker='^', color='0.4', linestyle='None', ms=5, label='Held-out (OOD)'),
]
if SHOW_LEGEND:
    axes_d[1].legend(
        handles=class_handles + extra_handles,
        title='GI class', frameon=False,
        fontsize=mpl.rcParams['legend.fontsize'] + TEXT_SCALE,
        title_fontsize=mpl.rcParams['legend.title_fontsize'] + TEXT_SCALE,
        bbox_to_anchor=(1.15, 1), loc='upper left',
        handlelength=0.8, handleheight=0.8,
    )

fig_d.savefig(OUT_DIR / 'panel_d_synergy_umap.svg')
plt.show()
print('Panel D (synergy UMAP) saved.')

In [ ]:
# ── Panel D (GEARS): Synergy UMAP — GT class (left) vs GEARS predicted class (right) ──
fig_d_gears, axes_d_gears = plt.subplots(1, 2, figsize=(312/PPI, 172/PPI), constrained_layout=True)

_draw_synergy_umap(axes_d_gears[0], gears_umap, 'gt_gi', 'Norman GI Labels',
                   val_combos=val_combos, arrow_pairs=None,
                   show_unlabelled=True, show_axis=True, text_scale=TEXT_SCALE, margin=0.1)
_draw_synergy_umap(axes_d_gears[1], gears_umap, 'predicted_gi', 'GEARS GI Prediction',
                   val_combos=val_combos, #arrow_pairs=_gi_arrows,
                   show_unlabelled=False, show_axis=True, text_scale=TEXT_SCALE, margin=0.1,
                   right_labels={'CBL/CNN1', 'CEBPB/PTPN1'})

if SHOW_LEGEND:
    axes_d_gears[1].legend(
        handles=class_handles + extra_handles,
        title='GI class', frameon=False,
        fontsize=mpl.rcParams['legend.fontsize'] + TEXT_SCALE,
        title_fontsize=mpl.rcParams['legend.title_fontsize'] + TEXT_SCALE,
        bbox_to_anchor=(1.15, 1), loc='upper left',
        handlelength=0.8, handleheight=0.8,
    )

fig_d_gears.savefig(OUT_DIR / 'panel_d_synergy_umap_gears.svg')
plt.show()
print('Panel D (GEARS synergy UMAP) saved.')


In [ ]:
# ── Panel E: GI classification — per-class AUC (left) + summary (right) ──
MODEL_ORDER = ['SHERLOCK', 'GEARS']
class_order = (
    class_df_ci.groupby('class')['auc_roc']
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
bar_pal = {'SHERLOCK': CLR_SHERLOCK, 'GEARS': CLR_GEARS}

TEXT_SCALE_E = -1.0  # +/- pt added to all Panel E text uniformly; 0.0 == RC defaults

fig_e, (ax_e1, ax_e2) = plt.subplots(
    1, 2, figsize=(300/PPI, 150/PPI), constrained_layout=True,
    gridspec_kw={'width_ratios': [2.2, 1.8]},
)

# Left: per-class AUC-ROC
sns.barplot(
    data=class_df_ci.dropna(subset=['auc_roc']),
    y='class', x='auc_roc',
    hue='model', hue_order=MODEL_ORDER,
    palette=bar_pal, order=class_order,
    width=0.62, errorbar='sd', capsize=0.15, err_kws={'linewidth': 0.4}, ax=ax_e1, legend=False,
)
ax_e1.axvline(0.5, color='0.4', linewidth=0.4, linestyle='--')
ax_e1.set_xlabel('AUC-ROC (one-vs-rest)', fontsize=mpl.rcParams['axes.labelsize'] + TEXT_SCALE_E - 1)
ax_e1.set_ylabel('')
_CLASS_LABEL_MAP = {
    'approximately additive': 'additive',
    # syn-sim / syn-dis are merged into a single 'synergy' class upstream
    # (MERGE_SYNERGY in the evaluate_gi_classification cell).
}
ax_e1.set_yticklabels([_CLASS_LABEL_MAP.get(c, c) for c in class_order])
ax_e1.set_xlim(0, 1)
ax_e1.set_xticks([0, 0.5, 1.0])
ax_e1.tick_params(axis='x', labelsize=mpl.rcParams['xtick.labelsize'] + TEXT_SCALE_E)
ax_e1.tick_params(axis='y', labelsize=mpl.rcParams['ytick.labelsize'] + TEXT_SCALE_E)
ax_e1.spines['left'].set_linewidth(0.4)
ax_e1.spines['bottom'].set_linewidth(0.4)
ax_e1.tick_params(axis='both', width=0.4)
ax_e1.spines['top'].set_visible(False)
ax_e1.spines['right'].set_visible(False)
h1 = [Patch(color=bar_pal[m], label=m) for m in MODEL_ORDER]
l1 = list(MODEL_ORDER)

# Right: F1, ARI, AMI with SHERLOCK CIs across runs
sns.barplot(
    data=summ_long_ci, x='Metric', y='score',
    hue='model', hue_order=MODEL_ORDER,
    palette=bar_pal, width=0.55, errorbar='sd', capsize=0.15, err_kws={'linewidth': 0.4}, ax=ax_e2,
)
ax_e2.set_xlabel('')
ax_e2.set_ylabel('Score', fontsize=mpl.rcParams['axes.labelsize'] + TEXT_SCALE_E)
ax_e2.set_ylim(-0.05, 1.05)
ax_e2.tick_params(axis='x', labelsize=mpl.rcParams['xtick.labelsize'] + TEXT_SCALE_E)
ax_e2.tick_params(axis='y', labelsize=mpl.rcParams['ytick.labelsize'] + TEXT_SCALE_E)
ax_e2.spines['left'].set_linewidth(0.4)
ax_e2.spines['bottom'].set_linewidth(0.4)
ax_e2.tick_params(axis='both', width=0.4)
ax_e2.spines['top'].set_visible(False)
ax_e2.spines['right'].set_visible(False)
ax_e2.get_legend().remove()
ax_e2.legend(h1, l1, title='Model', frameon=False,
             fontsize=mpl.rcParams['legend.fontsize'] + TEXT_SCALE_E,
             title_fontsize=mpl.rcParams['legend.title_fontsize'] + TEXT_SCALE_E,
             loc='upper right', bbox_to_anchor=(1, 1.5))

fig_e.suptitle('GI classification', fontsize=mpl.rcParams['axes.titlesize'] + TEXT_SCALE_E)
# SHERLOCK vs GEARS significance on the F1/ARI/AMI bars (Welch t, BH-FDR)
add_gi_significance(ax_e2, summ_long_ci, fs=mpl.rcParams['legend.fontsize'] + TEXT_SCALE_E)

fig_e.savefig(OUT_DIR / 'panel_e_gi_classification.svg')
plt.show()
print('Panel E saved.')

In [ ]:
# ── Per-run GI classification on the HELD-OUT TEST SET (val_combos) ──────
# Mirror of the per-run cell above, but every metric is restricted to the
# held-out combos only (val_combos == the test set). SHERLOCK reloads each
# checkpoint and re-scores GI; GEARS reuses its cached raw per-run predictions.
# evaluate_gi_classification already returns the full per-combo `work` frame
# (classified, no fit_mask), so we just subset it to the test combos.
_test_suffix     = '_synmerged' if MERGE_SYNERGY else ''
_test_cls_cache  = OUT_DIR / f'gi_test_per_run_class{_test_suffix}.csv'
_test_summ_cache = OUT_DIR / f'gi_test_per_run_summ{_test_suffix}.csv'
_test_combos     = set(val_combos)   # held-out combos == test set

def _subset_gi_metrics(work, model_name, keep_combos, run):
    """Per-class + summary GI metrics on `keep_combos` only (merged synergy)."""
    sub     = work[work.index.isin(keep_combos)]
    eval_df = sub[sub['gt_gi'].notna() & sub['predicted_gi'].notna()].copy()
    y_true  = _merge_syn(eval_df['gt_gi'])
    y_pred  = _merge_syn(eval_df['predicted_gi'])
    labels  = sorted(set(y_true))
    per_class = []
    for cls in labels:
        tb = (y_true == cls).astype(int)
        pb = (y_pred == cls).astype(int)
        n_pos = int(tb.sum())
        auc = roc_auc_score(tb, pb) if 0 < n_pos < len(tb) else np.nan
        per_class.append({
            'model': model_name, 'class': cls, 'n_positive': n_pos,
            'f1': f1_score(tb, pb, zero_division=0), 'auc_roc': auc, 'run': run,
        })
    summary = {
        'model': model_name, 'run': run, 'n_eval': len(eval_df),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'ari': adjusted_rand_score(y_true, y_pred),
        'ami': adjusted_mutual_info_score(y_true, y_pred),
        'nmi': normalized_mutual_info_score(y_true, y_pred),
    }
    return pd.DataFrame(per_class), summary

if (_cache_fresh(_test_cls_cache, *checkpoints.values())
        and _cache_fresh(_test_summ_cache, *checkpoints.values())):
    print('Loading cached per-run TEST-set GI metrics...')
    _test_run_class_df = pd.read_csv(_test_cls_cache)
    _test_run_summ_df  = pd.read_csv(_test_summ_cache)
else:
    _t_cls_rows, _t_summ_rows = [], []
    # SHERLOCK — reload each checkpoint, score GI, restrict to the test combos
    for _rn in range(1, N_RUNS + 1):
        _ckpt = checkpoints[_rn]
        if not _ckpt.exists():
            print(f'  SHERLOCK run {_rn}: checkpoint not found, skipping.')
            continue
        print(f'  SHERLOCK run {_rn}/{N_RUNS}...', end=' ', flush=True)
        _m = slk.tl.load_results(_ckpt)
        slk.tl.eval_single(_m, data, obsm_key='z', uns_key='results', device=DEVICE)
        _syn_r = _m.get_gi(data, min_cells=5)
        _work, _, _ = evaluate_gi_classification(_syn_r, 'SHERLOCK', gi_series)
        _cls_r, _summ_r = _subset_gi_metrics(_work, 'SHERLOCK', _test_combos, _rn)
        _t_cls_rows.append(_cls_r)
        _t_summ_rows.append(_summ_r)
        print('done')
    # GEARS — reuse cached raw per-run predictions
    for _rn in range(1, N_RUNS + 1):
        _g_raw = pd.read_csv(GEARS_RUNS_DIR / f'gears_gi_run{_rn}.csv')
        _work, _, _ = evaluate_gi_classification(_g_raw, 'GEARS', gi_series)
        _cls_r, _summ_r = _subset_gi_metrics(_work, 'GEARS', _test_combos, _rn)
        _t_cls_rows.append(_cls_r)
        _t_summ_rows.append(_summ_r)
    _test_run_class_df = pd.concat(_t_cls_rows, ignore_index=True)
    _test_run_summ_df  = pd.DataFrame(_t_summ_rows)
    _test_run_class_df.to_csv(_test_cls_cache, index=False)
    _test_run_summ_df.to_csv(_test_summ_cache, index=False)
    print('Per-run TEST-set GI metrics cached.')

# CI-ready frames for the test-set panel (same shape as class_df_ci / summ_long_ci)
class_df_ci_test = _test_run_class_df.copy()
_slk_test_long = _test_run_summ_df[_test_run_summ_df['model'] == 'SHERLOCK'][['run'] + _SUMM_KEYS].melt(
    id_vars='run', value_vars=_SUMM_KEYS, var_name='metric', value_name='score'
)
_slk_test_long['model'] = 'SHERLOCK'
_gears_test_long = _test_run_summ_df[_test_run_summ_df['model'] == 'GEARS'][['run'] + _SUMM_KEYS].melt(
    id_vars='run', value_vars=_SUMM_KEYS, var_name='metric', value_name='score'
)
_gears_test_long['model'] = 'GEARS'
summ_long_ci_test = pd.concat(
    [_slk_test_long[['model', 'metric', 'score']],
     _gears_test_long[['model', 'metric', 'score']]],
    ignore_index=True,
)
summ_long_ci_test['Metric'] = pd.Categorical(
    summ_long_ci_test['metric'].map(_SUMM_NAMES),
    categories=list(_SUMM_NAMES.values()), ordered=True,
)
print(f'Test set: {len(_test_combos)} held-out combos')
print(_test_run_summ_df[['model', 'run', 'n_eval', 'f1_weighted', 'ari', 'ami']].to_string(index=False))

### Fig. S8: genetic-interaction classification on the 22 held-out pairs

In [ ]:
# ── Panel E (test set): GI classification on held-out combos only ────────
# Same layout as Panel E above, but scored on the held-out test combos.
MODEL_ORDER = ['SHERLOCK', 'GEARS']
class_order_test = (
    class_df_ci_test.groupby('class')['auc_roc']
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
bar_pal = {'SHERLOCK': CLR_SHERLOCK, 'GEARS': CLR_GEARS}

TEXT_SCALE_E = -1.0  # +/- pt added to all Panel E text uniformly; 0.0 == RC defaults

fig_et, (ax_et1, ax_et2) = plt.subplots(
    1, 2, figsize=(300/PPI, 150/PPI), constrained_layout=True,
    gridspec_kw={'width_ratios': [2.2, 1.8]},
)

# Left: per-class AUC-ROC (test set)
sns.barplot(
    data=class_df_ci_test.dropna(subset=['auc_roc']),
    y='class', x='auc_roc',
    hue='model', hue_order=MODEL_ORDER,
    palette=bar_pal, order=class_order_test,
    width=0.62, errorbar='sd', capsize=0.15, err_kws={'linewidth': 0.4}, ax=ax_et1, legend=False,
)
ax_et1.axvline(0.5, color='0.4', linewidth=0.4, linestyle='--')
ax_et1.set_xlabel('AUC-ROC (one-vs-rest)', fontsize=mpl.rcParams['axes.labelsize'] + TEXT_SCALE_E - 1)
ax_et1.set_ylabel('')
ax_et1.set_yticklabels([_CLASS_LABEL_MAP.get(c, c) for c in class_order_test])
ax_et1.set_xlim(0, 1)
ax_et1.set_xticks([0, 0.5, 1.0])
ax_et1.tick_params(axis='x', labelsize=mpl.rcParams['xtick.labelsize'] + TEXT_SCALE_E)
ax_et1.tick_params(axis='y', labelsize=mpl.rcParams['ytick.labelsize'] + TEXT_SCALE_E)
ax_et1.spines['left'].set_linewidth(0.4)
ax_et1.spines['bottom'].set_linewidth(0.4)
ax_et1.tick_params(axis='both', width=0.4)
ax_et1.spines['top'].set_visible(False)
ax_et1.spines['right'].set_visible(False)
h1 = [Patch(color=bar_pal[m], label=m) for m in MODEL_ORDER]
l1 = list(MODEL_ORDER)

# Right: F1, ARI, AMI with CIs across runs (test set)
sns.barplot(
    data=summ_long_ci_test, x='Metric', y='score',
    hue='model', hue_order=MODEL_ORDER,
    palette=bar_pal, width=0.55, errorbar='sd', capsize=0.15, err_kws={'linewidth': 0.4}, ax=ax_et2,
)
ax_et2.set_xlabel('')
ax_et2.set_ylabel('Score', fontsize=mpl.rcParams['axes.labelsize'] + TEXT_SCALE_E)
ax_et2.set_ylim(-0.05, 1.05)
ax_et2.tick_params(axis='x', labelsize=mpl.rcParams['xtick.labelsize'] + TEXT_SCALE_E)
ax_et2.tick_params(axis='y', labelsize=mpl.rcParams['ytick.labelsize'] + TEXT_SCALE_E)
ax_et2.spines['left'].set_linewidth(0.4)
ax_et2.spines['bottom'].set_linewidth(0.4)
ax_et2.tick_params(axis='both', width=0.4)
ax_et2.spines['top'].set_visible(False)
ax_et2.spines['right'].set_visible(False)
ax_et2.get_legend().remove()
ax_et2.legend(h1, l1, title='Model', frameon=False,
              fontsize=mpl.rcParams['legend.fontsize'] + TEXT_SCALE_E,
              title_fontsize=mpl.rcParams['legend.title_fontsize'] + TEXT_SCALE_E,
              loc='upper right', bbox_to_anchor=(1, 1.5))

fig_et.suptitle('GI classification (held-out test set)',
                fontsize=mpl.rcParams['axes.titlesize'] + TEXT_SCALE_E)
# SHERLOCK vs GEARS significance on the F1/ARI/AMI bars (Welch t, BH-FDR)
add_gi_significance(ax_et2, summ_long_ci_test, fs=mpl.rcParams['legend.fontsize'] + TEXT_SCALE_E)

fig_et.savefig(OUT_DIR / 'panel_e_gi_classification_test.svg')
plt.show()
print('Panel E (test set) saved.')

## Norman: classification from measured expression

From [`scripts/norman/2_raw_expression_gi.ipynb`](../scripts/norman/2_raw_expression_gi.ipynb).

Reads the per-run scores cached by `scripts/norman/1_norman_analysis.ipynb`.

# Norman 2019 — GI classification from raw expression

Scores genetic interactions directly from **observed expression**, with exactly the routine used for
SHERLOCK and GEARS in `norman_figure.ipynb` (Fig. 6d):

1. effect vectors = mean log2-CPM of each perturbation's cells minus the NTC mean (same formula as
   `get_gi(..., observed_singles='raw')`), on the same 3,270 filtered genes;
2. `regress_gi_params` — Theil–Sen regression of the double on its two singles, `min_cells=5`;
3. `classify_gi` — label-free equal-weight composite scores, argmax over classes;
4. `evaluate_gi_classification` — synergy subtypes merged, all labelled pairs (in distribution).

Raw expression is deterministic, so the five runs differ only in the Theil–Sen random seed
(1000–1004; 1000 is the seed the models use). SHERLOCK (observed mode) and GEARS per-run scores are
rebuilt from the cached `norman_figure` results. Output: one SVG in `results/module_interaction/`.

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import sklearn.linear_model as sklm
from sklearn.metrics import f1_score, roc_auc_score, adjusted_rand_score, adjusted_mutual_info_score

import sherlock as slk
from sherlock.tools._models import regress_gi_params, classify_gi
warnings.filterwarnings('ignore')

mpl.rcParams.update({'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 9,
                     'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
                     'axes.spines.top': False, 'axes.spines.right': False, 'svg.fonttype': 'none'})

FIG_DIR = Path(f'{paths.RESULTS_DIR}/norman_figure')
SVG = Path(f'{paths.RESULTS_DIR}/module_interaction/supp_raw_expression_gi_per_class.svg')
SEEDS = [1000, 1001, 1002, 1003, 1004]
COLORS = {'Raw expression': '#4d4d4d', 'SHERLOCK': '#2166ac', 'GEARS': '#d73027'}

def canon_pair(s):
    s = str(s)
    if '+' not in s:
        return s
    a, b = s.split('+', 1)
    return '+'.join(sorted([a.strip(), b.strip()]))

## Data (same loading and gene filter as `norman_figure.ipynb`)

In [ ]:
data = sc.read_h5ad(f'{paths.DATA_DIR}/norman/Norman_2019.h5ad')
data.X = data.layers['counts'].copy()
slk.pp.slk_prepare_data(data, pert_key='perturbation_name', ntc_label='control',
                        treatment_key=None, inplace=True)
p_key, NTC = slk.configs.get_config('pert_key'), slk.configs.get_config('ntc_label')

all_perts = data.obs[p_key].unique()
pert_genes = {g.strip() for p in all_perts if p != NTC for g in str(p).split('+')}
mean_expr = np.asarray(data.X.mean(axis=0)).ravel()
data = data[:, (mean_expr > 0.5) | data.var_names.isin(pert_genes)].copy()

gi_labels = pd.read_csv(f'{paths.DATA_DIR}/norman/genetic_interactions.csv')
gi_labels['combination'] = gi_labels['combination'].map(canon_pair)
gi_labels['genetic_interaction'] = gi_labels['genetic_interaction'].str.strip()
gi_series = gi_labels.set_index('combination')['genetic_interaction']
print(f'{data.n_obs} cells, {data.n_vars} genes, {len(gi_labels)} labelled pairs')

## Raw effect vectors (log2-CPM, perturbation mean minus NTC mean)

In [ ]:
X = sp.csr_matrix(data.X, dtype=np.float64)
lib = np.asarray(X.sum(1)).ravel()
Xn = sp.diags(1e4 / np.maximum(lib, 1e-8)) @ X
Xn.data = np.log2(Xn.data + 1.0)                          # log2(1e4 * x / lib + 1), zeros stay zero

labels = data.obs[p_key].astype(str).map(canon_pair).values
uniq = pd.Index(sorted(set(labels)))
code_ = uniq.get_indexer(labels)
G = sp.csr_matrix((np.ones(len(labels)), (code_, np.arange(len(labels)))), shape=(len(uniq), len(labels)))
counts = np.asarray(G.sum(1)).ravel()
means = np.asarray((G @ Xn).todense()) / counts[:, None]

base = means[uniq.get_loc(str(NTC))]
effects = {u: means[i] - base for i, u in enumerate(uniq) if u != str(NTC)}
n_obs = {u: int(counts[i]) for i, u in enumerate(uniq) if u != str(NTC)}
combos = [c for c in effects if '+' in c and all(p in effects for p in c.split('+'))]
print(f'{len(effects) - len(combos)} singles, {len(combos)} combinations')

## Evaluation (copied from `norman_figure.ipynb`)

In [ ]:
_SYN = {'synergy (similar phenotype)': 'synergy', 'synergy (dissimilar phenotype)': 'synergy'}
merge_syn = lambda s: s.astype(str).replace(_SYN)

def evaluate_gi_classification(df, model_name):
    work = df.copy()
    work.index = (work['pert_a'].astype(str) + '+' + work['pert_b'].astype(str)).map(canon_pair)
    work['gt_gi'] = work.index.map(gi_series)
    work = classify_gi(work, inplace=True)
    ev = work[work['gt_gi'].notna() & work['predicted_gi'].notna()]
    y_true, y_pred = merge_syn(ev['gt_gi']), merge_syn(ev['predicted_gi'])
    per_class = []
    for cls in sorted(set(y_true)):
        tb, pb = (y_true == cls).astype(int), (y_pred == cls).astype(int)
        per_class.append({'model': model_name, 'class': cls, 'n_positive': int(tb.sum()),
                          'f1': f1_score(tb, pb, zero_division=0),
                          'auc_roc': roc_auc_score(tb, pb) if 0 < tb.sum() < len(tb) else np.nan})
    summary = {'model': model_name, 'n_eval': len(ev),
               'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
               'ari': adjusted_rand_score(y_true, y_pred),
               'ami': adjusted_mutual_info_score(y_true, y_pred)}
    return pd.DataFrame(per_class), summary

## Five Theil–Sen regressions on raw expression

In [ ]:
_TS = sklm.TheilSenRegressor
cls_rows, summ_rows = [], []
for run, seed in enumerate(SEEDS, 1):
    sklm.TheilSenRegressor = lambda *a, _s=seed, **k: _TS(*a, **{**k, 'random_state': _s})
    try:
        gi = regress_gi_params(effects, combos=combos, n_obs=n_obs, min_cells=5, progress=False)
    finally:
        sklm.TheilSenRegressor = _TS
    gi['gt_gi'] = gi.index.map(canon_pair).map(gi_series)
    gi = classify_gi(gi, fit_mask=gi.index.map(canon_pair).isin(set(gi_labels['combination'])), inplace=True)
    c, s = evaluate_gi_classification(gi, 'Raw expression')
    c['run'] = s['run'] = run
    cls_rows.append(c); summ_rows.append(s)
raw_cls, raw_summ = pd.concat(cls_rows), pd.DataFrame(summ_rows)
raw_summ.round(3)

## SHERLOCK and GEARS per-run scores (cached `norman_figure` results)

In [ ]:
slk_cls = pd.read_csv(FIG_DIR / 'gi_per_run_class_synmerged_observed.csv')
slk_summ = pd.read_csv(FIG_DIR / 'gi_per_run_summ_synmerged_observed.csv')
g_cls, g_summ = [], []
for run in range(1, 6):
    c, s = evaluate_gi_classification(pd.read_csv(FIG_DIR / 'gears_runs' / f'gears_gi_run{run}.csv'), 'GEARS')
    c['run'] = s['run'] = run
    g_cls.append(c); g_summ.append(s)

cls_all = pd.concat([raw_cls, slk_cls, *g_cls], ignore_index=True)
summ_all = pd.concat([raw_summ, slk_summ, pd.DataFrame(g_summ)], ignore_index=True)
overall = summ_all.groupby('model')[['f1_weighted', 'ari', 'ami', 'n_eval']].agg(['mean', 'std'])
overall = overall.loc[list(COLORS)]
print('Overall (mean ± SD over 5 runs, 86 labelled pairs, synergy merged):')
for m, r in overall.iterrows():
    print(f"  {m:15s} weighted F1 {r[('f1_weighted','mean')]:.3f} ± {r[('f1_weighted','std')]:.3f}   "
          f"ARI {r[('ari','mean')]:.3f} ± {r[('ari','std')]:.3f}   AMI {r[('ami','mean')]:.3f} ± {r[('ami','std')]:.3f}")
per_class = cls_all.groupby(['class', 'model'])[['f1', 'auc_roc']].agg(['mean', 'std']).round(3)
per_class

## Supplementary figure

### Fig. S9: genetic-interaction classification from measured expression, SHERLOCK and GEARS

In [ ]:
classes = sorted(cls_all['class'].unique())
models = list(COLORS)
fig, axes = plt.subplots(1, 3, figsize=(10, 2.8), gridspec_kw={'width_ratios': [3, 3, 1.1]})
w = 0.26
for ax, metric, title in [(axes[0], 'f1', 'Per-class F1'), (axes[1], 'auc_roc', 'Per-class AUC-ROC')]:
    for j, m in enumerate(models):
        sub = cls_all[cls_all['model'] == m]
        mu = [sub.loc[sub['class'] == c, metric].mean() for c in classes]
        sd = [sub.loc[sub['class'] == c, metric].std() for c in classes]
        x = np.arange(len(classes)) + (j - 1) * w
        ax.bar(x, mu, w, yerr=sd, color=COLORS[m], label=m, capsize=1.5, error_kw={'lw': 0.6})
    ax.set_xticks(np.arange(len(classes)))
    ax.set_xticklabels([c.replace('approximately ', 'approx. ') for c in classes], rotation=35, ha='right')
    ax.set_ylim(0, 1.05); ax.set_title(title)
axes[1].axhline(0.5, color='0.6', lw=0.6, ls='--')
axes[0].legend(frameon=False, loc='upper left', ncol=3, bbox_to_anchor=(0, 1.02))

ax = axes[2]
for j, m in enumerate(models):
    v = summ_all.loc[summ_all['model'] == m, 'f1_weighted']
    ax.bar(j, v.mean(), 0.7, yerr=v.std(), color=COLORS[m], capsize=2, error_kw={'lw': 0.6})
    ax.scatter(np.full(len(v), j) + np.linspace(-0.15, 0.15, len(v)), v, s=5, color='k', zorder=3)
    ax.text(j, v.mean() + v.std() + 0.03, f'{v.mean():.2f}', ha='center', fontsize=6.5)
ax.set_xticks(range(len(models))); ax.set_xticklabels(['Raw', 'SHERLOCK', 'GEARS'], rotation=35, ha='right')
ax.set_ylim(0, 1.05); ax.set_title('Weighted F1')
plt.tight_layout()
fig.savefig(SVG, bbox_inches='tight')
plt.show()
print('saved', SVG)

## Norman: consensus modules

From [`scripts/norman/3_module_interaction.ipynb`](../scripts/norman/3_module_interaction.ipynb).

Builds `results/module_interaction/consensus_modules.csv`, used by Figs. S10–S15.

# Norman 2019 — consensus perturbation modules vs. genetic interactions

The 5 SHERLOCK combinatorial runs in `norman_figure.ipynb` each learn a 105 × 105 perturbation
correlation matrix `rho_corr` (single-gene perturbations, saved in
`results/norman_figure/run_matrices.npz`). The leave-one-out Spearman r of these matrices is
≈ 0.79, so the runs should agree on which perturbations group together.

This notebook:

1. **Checks cross-run agreement** of `rho_corr`, both entrywise and after hard clustering.
2. **Builds the expected grouping** as a consensus over runs. Each run is clustered at many
   resolutions (k), and the fraction of (run, k) clusterings that put genes *i* and *j* together
   is their co-assignment probability. Consensus modules are clusters of that matrix.
3. **Relates the modules to Norman's ground-truth interaction classes**
   (`genetic_interactions.csv`), using pair-level ρ, co-assignment and same-module membership, plus
   module-level enrichment.
4. **Robustness checks.** It repeats the tests per run and on the held-out test combos only, and
   controls for the similarity of the observed single-perturbation expression effects.

No models are retrained. Only the saved matrices are used, plus the raw data for the
observed-effect control in §6.

In [ ]:
import sys, math, random
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from scipy.cluster.hierarchy import linkage, fcluster, leaves_list
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr, kruskal, mannwhitneyu, fisher_exact, hypergeom
from sklearn.metrics import adjusted_rand_score, silhouette_score
from statsmodels.stats.multitest import multipletests

mpl.rcParams.update({
    'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 9,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 150, 'savefig.dpi': 300, 'svg.fonttype': 'none',
})

N_RUNS   = 5
FIG_DIR  = Path(f'{paths.RESULTS_DIR}/norman_figure')
OUT_DIR  = Path(f'{paths.RESULTS_DIR}/module_interaction')
OUT_DIR.mkdir(parents=True, exist_ok=True)

K_RANGE        = list(range(5, 21))  # resolutions averaged into the consensus
LINKAGE_RUN    = 'ward'              # linkage for per-run clusterings
LINKAGE_CONS   = 'average'           # linkage for clustering the consensus matrix
K_FINAL_RANGE  = K_RANGE             # candidate numbers of consensus modules
N_PERM         = 5000                # permutations for module-pair × interaction tests

# How modules are defined:
#   'consensus' — cluster the co-assignment matrix C (expected grouping over runs)
#   'mean_rho'  — Ward clustering of the expected (run-averaged) rho directly
# Section 5d compares both; pair-level rho results always use the expected rho.
MODULE_SOURCE  = 'consensus'

# Ground-truth classes from genetic_interactions.csv, with the two synergy classes merged
MERGE_GI = {'synergy (dissimilar phenotype)': 'synergy',
            'synergy (similar phenotype)':    'synergy'}
PALETTE_GI = {
    'approximately additive': '#4DAF4A',
    'epistasis':              '#FF7F00',
    'neomorphic':             '#984EA3',
    'potentiation':           '#F781BF',
    'redundant':              '#A65628',
    'suppression':            '#E41A1C',
    'synergy':                '#377EB8',
}
GI_ORDER = list(PALETTE_GI)

## 1. Load per-run `rho_corr` matrices

In [ ]:
# rho_corr is indexed by PerturbMatchingDataset(combinatorial=True).perturbation_dict:
# the unique single-gene components of all non-NTC labels, sorted alphabetically.
# All 105 genes also appear as singles, so the held-out split does not change this order.
obs = ad.read_h5ad(f'{paths.DATA_DIR}/norman/Norman_2019.h5ad', backed='r').obs
labels = obs['perturbation_name'].astype(str).unique()
genes  = sorted({g.strip() for l in labels if l != 'control' for g in l.split('+')})
g2i    = {g: i for i, g in enumerate(genes)}
P      = len(genes)

_mats = np.load(FIG_DIR / 'run_matrices.npz')
R = np.stack([_mats[f'rho_corr_{k}'] for k in range(1, N_RUNS + 1)]).astype(float)
assert R.shape == (N_RUNS, P, P), R.shape
R = (R + R.transpose(0, 2, 1)) / 2
for r in R:
    np.fill_diagonal(r, 1.0)

R_mean = R.mean(0)
TRI = np.triu_indices(P, k=1)
print(f'{N_RUNS} runs × {P} × {P}   mean off-diagonal rho = {R_mean[TRI].mean():.3f}')

## 2. How similar are the runs?

Entrywise agreement of ρ is high. Hard clusterings of individual runs agree much less (ARI).
Many ρ values sit in the middle of the range, where small shifts move a gene across a cluster
boundary. This is why the grouping below is built as a consensus rather than from any single run.

In [ ]:
def cluster_corr(M, k, method='ward'):
    D = 1.0 - M
    np.fill_diagonal(D, 0.0)
    D = np.clip((D + D.T) / 2, 0, None)
    return fcluster(linkage(squareform(D, checks=False), method=method), k, criterion='maxclust')

pairs = list(combinations(range(N_RUNS), 2))
rho_pair_sp = pd.DataFrame(
    [(f'run{i+1}', f'run{j+1}', spearmanr(R[i][TRI], R[j][TRI])[0]) for i, j in pairs],
    columns=['run_a', 'run_b', 'spearman_r'])
loo = pd.read_csv(FIG_DIR / 'pairwise_repro.csv').query("metric == 'rho_corr_r'")
print(f"rho_corr LOO Spearman (from norman_figure): {loo['r'].mean():.3f}")
print(f"rho_corr pairwise Spearman:                 {rho_pair_sp['spearman_r'].mean():.3f} "
      f"(range {rho_pair_sp['spearman_r'].min():.3f}–{rho_pair_sp['spearman_r'].max():.3f})")

per_run_labels = {k: [cluster_corr(R[i], k, LINKAGE_RUN) for i in range(N_RUNS)] for k in K_RANGE}
ari_scan = pd.DataFrame([
    {'k': k, 'pairwise_ARI': np.mean([adjusted_rand_score(per_run_labels[k][i], per_run_labels[k][j])
                                     for i, j in pairs])}
    for k in K_RANGE])
print('\nPer-run hard clustering agreement (mean pairwise ARI):')
print(ari_scan.round(3).to_string(index=False))

## 3. Expected grouping: consensus co-assignment

$C_{ij} = \frac{1}{N_{runs}\,|K|}\sum_{run}\sum_{k \in K} \mathbb{1}[\text{gene } i \text{ and } j
\text{ share a cluster}]$, with Ward clustering on $1-\rho$ for $k = 5..20$.

The consensus matrix is then clustered (average linkage on $1-C$). To choose the number of
modules, two quantities are computed at each k: the mean ARI between the consensus modules and
each run's own clustering, and the mean run-to-run ARI. K is the value that maximises their
ratio, i.e. where the consensus agrees with the runs most beyond how much they agree with each
other.

In [ ]:
C_run = np.zeros((N_RUNS, P, P))
for i in range(N_RUNS):
    for k in K_RANGE:
        l = per_run_labels[k][i]
        C_run[i] += (l[:, None] == l[None, :])
C_run /= len(K_RANGE)
C = C_run.mean(0)

D_cons = 1.0 - C
np.fill_diagonal(D_cons, 0.0)

sel_rows = []
for k in K_FINAL_RANGE:
    lab = cluster_corr(C, k, LINKAGE_CONS)
    run_ari = [adjusted_rand_score(lab, cluster_corr(R[i], k, LINKAGE_RUN)) for i in range(N_RUNS)]
    sizes = np.bincount(lab)[1:]
    sel_rows.append({'k': k,
                     'ARI_vs_runs': np.mean(run_ari),
                     'run_run_ARI': ari_scan.set_index('k').loc[k, 'pairwise_ARI'],
                     'silhouette_consensus': silhouette_score(D_cons, lab, metric='precomputed'),
                     'n_singletons': int((sizes == 1).sum()),
                     'largest': int(sizes.max())})
sel_df = pd.DataFrame(sel_rows)
# Raw ARI_vs_runs keeps rising with k simply because individual runs also agree more at high k,
# so its maximum sits at the edge of the range. Instead pick the k where the consensus gains
# most over run-to-run agreement.
sel_df['consensus_gain'] = sel_df['ARI_vs_runs'] / sel_df['run_run_ARI']
K_FINAL = int(sel_df.loc[sel_df['consensus_gain'].idxmax(), 'k'])
print(sel_df.round(3).to_string(index=False))
print(f'\nSelected K_FINAL = {K_FINAL}')

fig, ax = plt.subplots(figsize=(4, 2.6))
ax.plot(sel_df['k'], sel_df['ARI_vs_runs'], marker='o', ms=4, lw=1.5, color='#2166ac', label='ARI vs individual runs')
ax.plot(sel_df['k'], sel_df['silhouette_consensus'], marker='s', ms=4, lw=1.5, color='0.45', label='Silhouette (1 − C)')
ax.plot(ari_scan['k'], ari_scan['pairwise_ARI'], ls='--', lw=1.2, color='0.7', label='Run–run ARI (no consensus)')
ax.axvline(K_FINAL, color='0.2', lw=0.75, ls=':')
ax.set_xlabel('Number of modules (k)')
ax.set_ylabel('Score')
ax.legend(frameon=False, loc='best')
ax.set_title('Choosing the number of consensus modules')
plt.tight_layout()
fig.savefig(OUT_DIR / 'module_k_selection.svg', bbox_inches='tight')
plt.show()

In [ ]:
def module_labels(source, k=K_FINAL):
    raw = cluster_corr(C, k, LINKAGE_CONS) if source == 'consensus' else cluster_corr(R_mean, k, 'ward')
    # Relabel modules M1..MK by decreasing size
    relab = {old: f'M{n+1}' for n, old in enumerate(pd.Series(raw).value_counts().index)}
    return pd.Series([relab[m] for m in raw], index=genes, name='module')

module_of = module_labels(MODULE_SOURCE)
MODULES = [f'M{n+1}' for n in range(K_FINAL)]
print(f'Module source: {MODULE_SOURCE}   K = {K_FINAL}')

# Per-gene confidence: mean co-assignment with the other members of its module
conf = {}
for g in genes:
    mates = [g2i[h] for h in module_of.index[module_of == module_of[g]] if h != g]
    conf[g] = C[g2i[g], mates].mean() if mates else np.nan

# Reference: Norman et al. perturbation-map groups (same cleaned table S6 groups as norman_figure Panel C)
GENE_GROUPS = {
    'MAP2K':                   ['MAP2K3', 'MAP2K6'],
    'Ig domain containing':    ['IGDCC3', 'PRTG'],
    'Tyrosine phosphatase':    ['PTPN12', 'PTPN9'],
    'MAP4K':                   ['MAP4K3', 'MAP4K5'],
    'Brachyury TF':            ['CSRNP1', 'SET', 'TBX2', 'TBX3'],
    'Kinesin':                 ['KIF18B', 'KIF2C'],
    'bLZ TF':                  ['CEBPA', 'CEBPB', 'CEBPE', 'CELF2', 'FOSB', 'OSR2'],
    'Regulation of apoptosis': ['EGR1', 'FOXO4', 'IER5L', 'IKZF3', 'MEIS1', 'PRDM1',
                                'RHOXF2B', 'SAMD1', 'SGK1', 'TP73'],
    'Homeobox TF':             ['CBFA2T3', 'CITED1', 'DLX2', 'MIDN', 'NIT1', 'SNAI1', 'TMSB4X'],
}
norman_group = pd.Series({g: grp for grp, gs in GENE_GROUPS.items() for g in gs if g in g2i})

module_df = pd.DataFrame({
    'module': module_of,
    'confidence': pd.Series(conf),
    'norman_group': norman_group.reindex(genes),
})
module_df.index.name = 'gene'
module_df.to_csv(OUT_DIR / 'consensus_modules.csv')

_ng = module_df.dropna(subset=['norman_group'])
print(f'ARI(consensus modules, Norman groups) on {len(_ng)} annotated genes: '
      f"{adjusted_rand_score(_ng['module'], _ng['norman_group']):.3f}")
_null = [adjusted_rand_score(np.random.default_rng(s).permutation(_ng['module'].values), _ng['norman_group'])
         for s in range(1000)]
print(f"  permutation p = {(np.sum(np.array(_null) >= adjusted_rand_score(_ng['module'], _ng['norman_group'])) + 1) / 1001:.4f}")

summary = (module_df.reset_index()
           .groupby('module')
           .agg(n=('gene', 'size'),
                mean_confidence=('confidence', 'mean'),
                genes=('gene', lambda s: ', '.join(sorted(s))),
                norman_groups=('norman_group', lambda s: '; '.join(f'{k}×{v}' for k, v in s.value_counts().items())))
           .reindex(MODULES))
pd.set_option('display.max_colwidth', 200)
summary

## Norman: interaction rules, modules and the erythroid axis

From [`scripts/norman/4_interaction_biology.ipynb`](../scripts/norman/4_interaction_biology.ipynb).

# Norman 2019 — interaction rules and the erythroid axis

K562 is a chronic myeloid leukaemia (CML) blast-crisis line that can be pushed toward erythroid,
megakaryocytic or granulocytic states; Norman et al. describe erythroid, megakaryocyte (CD41⁺) and
granulocyte clusters in this screen. Using the 5 trained SHERLOCK runs from `norman_figure.ipynb`, this
notebook:

1. rebuilds the seven model-parameter descriptors and the rule-based interaction classifier;
2. finds which latent (W) axes carry the erythroid and granulocytic programmes;
3. defines the fate axis from canonical lineage markers and places the modules on it;
4. tests how the positions of a pair's two modules relate to its interaction type.

Only the **measured, labelled panel** is analysed — a curated selection of combinations, so nothing is
extrapolated to unmeasured pairs. Nothing is written to disk. Modules are the consensus modules from
`module_interaction.ipynb`; ground truth is Norman's `genetic_interactions.csv` with the two synergy
classes merged.

In [ ]:
import sys, warnings
from pathlib import Path
from itertools import combinations
import numpy as np, pandas as pd, torch, scanpy as sc
import matplotlib as mpl, matplotlib.pyplot as plt, seaborn as sns
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from scipy.stats import mannwhitneyu
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
warnings.filterwarnings('ignore')
import sherlock as slk

mpl.rcParams.update({'font.size': 8, 'axes.titlesize': 9, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 150, 'svg.fonttype': 'none'})
N_RUNS, GATE_THR, N_BG = 5, 0.5, 256
FIG = Path(f'{paths.RESULTS_DIR}/norman_figure')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PALETTE = {'approximately additive': '#4DAF4A', 'epistasis': '#FF7F00', 'neomorphic': '#984EA3',
           'potentiation': '#F781BF', 'redundant': '#A65628', 'suppression': '#E41A1C', 'synergy': '#377EB8'}
CLASSES = list(PALETTE)
short = lambda c: c.replace('approximately additive', 'additive')
NAMES = {'rho': 'parent similarity', 'private_frac': 'private latent factors',
         'c_sum': 'parent scaling', 'syn_rel': 'interaction term size',
         'joint_over_add': 'combination vs sum of parents', 'dom_gap': 'parent dominance',
         'syn_offgate_dec': 'interaction on unused factors'}

# data, with the gene filter the models were trained on
data = sc.read_h5ad(f'{paths.DATA_DIR}/norman/Norman_2019.h5ad')
data.X = data.layers['counts'].copy()
slk.pp.slk_prepare_data(data, pert_key='perturbation_name', ntc_label='control', treatment_key=None, inplace=True)
labels = data.obs[slk.configs.get_config('pert_key')].astype(str).values
NTC = slk.configs.get_config('ntc_label')
genes = sorted({g for l in set(labels) if l != NTC for g in l.split('+')})
g2i = {g: i for i, g in enumerate(genes)}
P = len(genes)
_lib = np.asarray(data.X.sum(1)).ravel()
def _pb(mask):                                      # observed pseudobulk log2 CPM of the current gene set
    return np.log2(np.asarray(data.X[mask].multiply(1e4 / _lib[mask][:, None]).mean(0)).ravel() + 1)

# Observed fate scores from CANONICAL markers, on all genes (before the model's gene filter).
# Per marker: log2 fold change vs NTC for each single perturbation, z-scored across perturbations;
# a fate score is the mean over that fate's markers. CRISPRa targets are never used as markers.
MARKERS = {
    'erythroid': ['HBG1', 'HBG2', 'HBZ', 'HBA1', 'HBA2', 'HBE1', 'HBD', 'GYPA', 'GYPB', 'ALAS2', 'SLC25A37',
                  'TFRC', 'KLF1', 'GATA1', 'EPOR', 'AHSP', 'SLC4A1'],
    'megakaryocytic': ['ITGA2B', 'ITGB3', 'GP9', 'GP1BA', 'GP1BB', 'GP6', 'PF4', 'PPBP', 'VWF', 'TUBB1', 'SELP',
                       'CD9', 'MYL9', 'TREML1', 'F13A1', 'PLEK', 'MPL', 'LTBP1', 'CD36', 'PECAM1'],
    'granulocytic': ['MPO', 'ELANE', 'AZU1', 'PRTN3', 'CTSG', 'LYZ', 'S100A8', 'S100A9', 'CSF3R', 'ITGAM', 'CD33',
                     'MNDA', 'FCGR1A', 'LST1', 'FCER1G', 'CST3', 'CEBPE', 'LGALS3', 'ANXA1', 'SRGN'],
}
FATES = list(MARKERS)
MARKERS = {f: [g for g in gs if g in set(data.var_names) and g not in set(genes)] for f, gs in MARKERS.items()}
_mg = sorted({g for gs in MARKERS.values() for g in gs})
_Xm = data[:, _mg].X.tocsr()
_cpm = lambda mask: np.asarray(_Xm[mask].multiply(1e4 / _lib[mask][:, None]).mean(0)).ravel()
_ntc_m = _cpm(labels == NTC)
_lfc = pd.DataFrame(np.stack([np.log2(_cpm(labels == g) + .05) - np.log2(_ntc_m + .05) for g in genes]),
                    index=genes, columns=_mg)
_z = (_lfc - _lfc.mean()) / (_lfc.std() + 1e-9)
fate = pd.DataFrame({f: _z[gs].mean(1) for f, gs in MARKERS.items()})
_det = np.asarray((_Xm[labels == NTC] > 0).mean(0)).ravel()
print('canonical markers used:', {f: len(gs) for f, gs in MARKERS.items()},
      '| detected in control cells (median):', {f: round(float(np.median(_det[[_mg.index(g) for g in gs]])), 3)
                                                 for f, gs in MARKERS.items()})
data = data[:, (np.asarray(data.X.mean(0)).ravel() > 0.5) | data.var_names.isin(set(genes))].copy()
var = np.array(data.var_names)
readout = ~np.isin(var, genes)                      # exclude CRISPRa targets from any readout
bg_idx = np.random.default_rng(0).choice(np.where(labels == NTC)[0], N_BG, replace=False)
x_bg = torch.tensor(data.X[bg_idx].toarray(), dtype=torch.float32, device=DEVICE)

# learned rho (5 runs), consensus modules, ground truth
_m = np.load(FIG / 'run_matrices.npz')
R = np.mean([(_m[f'rho_corr_{k}'] + _m[f'rho_corr_{k}'].T) / 2 for k in range(1, N_RUNS + 1)], 0)
module = pd.read_csv(f'{paths.RESULTS_DIR}/module_interaction/consensus_modules.csv', index_col=0)['module']
MODULES = sorted(module.unique(), key=lambda m: int(m[1:]))
gi = pd.read_csv(f'{paths.DATA_DIR}/norman/genetic_interactions.csv')
gi['combination'] = gi['combination'].map(lambda s: '+'.join(sorted(s.split('+'))))
gi['gi'] = gi['genetic_interaction'].str.strip().replace({'synergy (dissimilar phenotype)': 'synergy',
                                                          'synergy (similar phenotype)': 'synergy'})
gi = gi[gi['combination'].map(lambda c: all(g in g2i for g in c.split('+')))].reset_index(drop=True)
print(f'{P} perturbations, {readout.sum()} readout genes, {len(gi)} labelled pairs, {len(MODULES)} modules')

## 1. Seven descriptors from the model's parameters

Computed for the labelled pairs only. The combined latent state is
**z_ab = z₀ + c_a·S_a + c_b·S_b + syn(ρ_a, ρ_b)** with S = A⊙W.

| Descriptor | From | Meaning |
|---|---|---|
| `rho` | covariance Σ_P | do the two perturbations act in the same direction? |
| `private_frac` | gates W, A⊙W | share of each shift on factors the partner does not gate |
| `c_sum` | parent-scale layer | c_a + c_b: damped (< 2) or amplified |
| `syn_rel` | synergy MLP | ‖syn‖ / ‖S_a + S_b‖ — how big the synergy term is |
| `joint_over_add` | decoder | ‖decoded combination‖ / ‖D_a + D_b‖ |
| `dom_gap` | decoder | \|cos(combination, D_a) − cos(combination, D_b)\| — dominance |
| `syn_offgate_dec` | syn × W × decoder | decoded effect of syn on factors **neither** parent gates — where it acts |

In [ ]:
lab = gi[['combination', 'gi', 'genetic_interaction']].copy()
lab['a'] = lab['combination'].str.split('+').str[0]
lab['b'] = lab['combination'].str.split('+').str[1]
IA, IB = lab['a'].map(g2i).values, lab['b'].map(g2i).values
DESC = ['private_frac', 'c_sum', 'syn_rel', 'joint_over_add', 'dom_gap', 'syn_offgate_dec']
runs = []
for run in range(1, N_RUNS + 1):
    m = slk.tl.load_results(FIG / 'models' / f'run_{run}.pth').to(DEVICE)
    with torch.no_grad():
        sh = m._get_all_pert_shifts(DEVICE)[1:]
        S, W, q = sh[:, 0], sh[:, 1], sh[:, 2]
        u = m._abduct_ntc(x_bg)
        ln = lambda z: torch.log1p(m._decode_to_expr(z, 1.) * 1e4)
        def dec(shift):                                            # (b, d) -> (b, G), mean over backgrounds
            z = (u[None] + shift[:, None]).reshape(-1, u.shape[-1])
            return ln(z).reshape(shift.shape[0], N_BG, -1).mean(1)
        base = ln(u).mean(0)
        D = dec(S) - base                                          # single-perturbation effects
        act = W > GATE_THR
        delta = float(S.abs()[act].median())                       # decoded signature of each latent factor
        E = torch.eye(m.latent_dim, device=DEVICE) * delta
        V = ((dec(E) - dec(-E)) / 2).cpu().numpy()
        a = torch.as_tensor(IA, device=DEVICE); b = torch.as_tensor(IB, device=DEVICE)
        ca, cb = m._parent_scales_from_rho(q[a], q[b])
        syn = m._synergy_shift_from_rho(q[a], q[b])
        lin = ca * S[a] + cb * S[b]
        g_lin = dec(lin)
        d_ab = dec(lin + syn) - base
        off = ~(act[a] | act[b])
        add = D[a] + D[b]; addn = add.norm(dim=1) + 1e-8
        cos = lambda x, y: (x * y).sum(1) / (x.norm(dim=1) * y.norm(dim=1) + 1e-8)
        pa = ((~act[b]) * S[a] ** 2).sum(1) / ((S[a] ** 2).sum(1) + 1e-8)
        pb = ((~act[a]) * S[b] ** 2).sum(1) / ((S[b] ** 2).sum(1) + 1e-8)
        desc = {'private_frac': (pa + pb) / 2, 'c_sum': (ca + cb).squeeze(-1),
                'syn_rel': syn.norm(dim=1) / ((S[a] + S[b]).norm(dim=1) + 1e-8),
                'joint_over_add': d_ab.norm(dim=1) / addn,
                'dom_gap': (cos(d_ab, D[a]) - cos(d_ab, D[b])).abs(),
                'syn_offgate_dec': (dec(lin + syn * off) - g_lin).norm(dim=1) / addn}
        runs.append({'desc': pd.DataFrame({k: v.cpu().numpy() for k, v in desc.items()}),
                     'D': D.cpu().numpy(), 'S': S.cpu().numpy(), 'W': W.cpu().numpy(), 'V': V})
    del m; torch.cuda.empty_cache()

for k in DESC:
    lab[k] = np.mean([r['desc'][k].values for r in runs], 0)
lab['rho'] = R[IA, IB]
lab['module_a'] = lab['a'].map(module); lab['module_b'] = lab['b'].map(module)
DESC7 = ['rho'] + DESC
print(f'descriptors computed for the {len(lab)} labelled pairs')

## 2. Rule-based classification

A decision list, applied top to bottom; unclaimed pairs are called additive. The cut points are the
fixed values found by rule mining in `module_interaction.ipynb`.

### Fig. S11a: rule-classifier confusion matrix and per-parameter AUROC

In [ ]:
RULES = [  # (class, [(descriptor, op, threshold)])
    ('approximately additive', [('c_sum', '<=', 1.8769652366638183), ('syn_rel', '<=', 0.1731259450316429)]),
    ('redundant',    [('rho', '>', 0.36017714440822585), ('joint_over_add', '<=', 0.9664641320705414)]),
    ('synergy',      [('rho', '>', 0.36017714440822585), ('joint_over_add', '>', 1.1006439685821534)]),
    ('potentiation', [('private_frac', '>', 0.24869043231010438), ('c_sum', '>', 1.502985668182373)]),
    ('epistasis',    [('joint_over_add', '<=', 1.0306628942489624), ('dom_gap', '>', 0.6432075917720794)]),
    ('suppression',  [('rho', '<=', 0.13503594752401113), ('joint_over_add', '<=', 1.1355479955673218)]),
    ('neomorphic',   [('syn_offgate_dec', '>', 0.06492092683911324), ('rho', '<=', 0.3258450113236904)]),
]

def classify(df):
    pred = np.array(['approximately additive'] * len(df), dtype=object)
    free = np.ones(len(df), bool)
    for cls, conds in RULES:
        hit = free.copy()
        for f, op, thr in conds:
            hit &= (df[f] > thr).values if op == '>' else (df[f] <= thr).values
        pred[hit] = cls; free &= ~hit
    return pred

lab['rule_class'] = classify(lab)
acc = np.mean(lab['rule_class'] == lab['gi']); bal = balanced_accuracy_score(lab['gi'], lab['rule_class'])
print(f'Rule-based classifier on {len(lab)} labelled pairs: accuracy {acc:.3f}, balanced {bal:.3f}')
for cls, conds in RULES:
    print(f"  {short(cls):12s}: " + '  AND  '.join(f'{f} {op} {thr:.3g}' for f, op, thr in conds))

auc = pd.DataFrame({f: {c: mannwhitneyu(lab.loc[lab.gi == c, f], lab.loc[lab.gi != c, f]).statistic /
                        ((lab.gi == c).sum() * (lab.gi != c).sum()) for c in CLASSES} for f in DESC7})
fig, ax = plt.subplots(1, 2, figsize=(13, 3.9), gridspec_kw={'width_ratios': [1, 1.3]})
sns.heatmap(confusion_matrix(lab['gi'], lab['rule_class'], labels=CLASSES), annot=True, fmt='d', cmap='Oranges',
            cbar=False, linewidths=1, linecolor='white', ax=ax[0],
            xticklabels=[short(c) for c in CLASSES], yticklabels=[short(c) for c in CLASSES])
ax[0].set(xlabel='class assigned by the rules', ylabel='Norman label',
          title=f'Rule classifier, 86 labelled pairs: accuracy {acc:.2f}, balanced accuracy {bal:.2f}')
sns.heatmap(auc.rename(columns=NAMES), annot=True, fmt='.2f', cmap='PuOr_r', center=.5, vmin=0, vmax=1, linewidths=.5,
            linecolor='white', ax=ax[1], yticklabels=[short(c) for c in CLASSES], cbar_kws={'label': 'AUROC'})
ax[1].set_title('Separation of each class by each model parameter\n(AUROC; 0.5 = no separation, > 0.5 = higher in that class)')
for a_ in ax:
    a_.tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

## 3. Which W axes carry the erythroid and granulocytic programmes?

Each latent factor's decoded signature (the expression change from moving along that axis) is matched
across runs (Hungarian assignment on |cos|, sign-corrected) and scored on two programmes, using only
canonical markers the model decodes:

- **erythroid** — globins, glycophorins, haem synthesis, GATA1, NFE2, TFRC (20 genes);
- **granulocytic** — CD33, LST1, FCER1G, CST3 (the only decodable canonical granulocyte markers).

Canonical megakaryocytic markers (ITGA2B/CD41, ITGB3, GP9, PF4, …) are all below the model's gene
filter, so megakaryocytic priming is assessed from the observed data in section 4, not from model axes.

In [ ]:
ERY = ['HBG1', 'HBG2', 'HBZ', 'HBA1', 'HBE1', 'HBD', 'HBQ1', 'GYPA', 'GYPB', 'GYPC', 'ALAS2', 'HMBS', 'UROD',
       'BLVRB', 'SLC25A37', 'GATA1', 'NFE2', 'TFRC', 'ANK1', 'PRDX2']
GRAN = [g for g in MARKERS['granulocytic'] if g in set(var[readout])]
PROGS = {'erythroid': np.isin(var, ERY) & readout, 'granulocytic': np.isin(var, GRAN) & readout}
P2 = list(PROGS)
print('decodable programme genes:', {k: int(v.sum()) for k, v in PROGS.items()}, '| granulocytic:', GRAN)
def prog_score(M):                                  # gene-space rows -> programme scores (per-row z units)
    sd = M[:, readout].std(1) + 1e-9
    return pd.DataFrame({k: M[:, msk].mean(1) / sd for k, msk in PROGS.items()})

nrm = lambda X: X[:, readout] / (np.linalg.norm(X[:, readout], axis=1, keepdims=True) + 1e-9)
L = runs[0]['V'].shape[0]
perm, sign, match = [np.arange(L)], [np.ones(L)], []
for r in runs[1:]:
    C = nrm(runs[0]['V']) @ nrm(r['V']).T
    ri, ci = linear_sum_assignment(-np.abs(C))
    perm.append(ci); sign.append(np.sign(C[ri, ci])); match.append(np.abs(C[ri, ci]))
V_al = np.stack([r['V'][p] * sg[:, None] for r, p, sg in zip(runs, perm, sign)])
S_al = np.stack([r['S'][:, p] * sg for r, p, sg in zip(runs, perm, sign)])
W_al = np.stack([r['W'][:, p] for r, p in zip(runs, perm)])

per_run = [prog_score(V) for V in V_al]
fac = sum(per_run) / N_RUNS
fac.index = [f'F{k}' for k in range(L)]
for p_ in P2:
    fac[f'sd_{p_}'] = np.std([s_[p_] for s_ in per_run], 0)
fac['cross-run similarity'] = np.mean(match, 0)
fac['similarity sd'] = np.std(match, 0)
_nuse = (W_al > GATE_THR).sum(1)                    # perturbations using each factor, per run
fac['perts using'] = _nuse.mean(0)
fac['perts using sd'] = _nuse.std(0)
used = fac['perts using'] >= 1
dom = fac[P2].abs().idxmax(1)
val = fac[P2].to_numpy()[np.arange(L), [P2.index(d) for d in dom]]
fac['call'] = np.where(~used, 'unused', np.where(np.abs(val) < 0.5, 'none', dom + np.where(val > 0, ' +', ' −')))
load = pd.DataFrame(S_al.mean(0), index=genes, columns=fac.index)
fac['drives +'] = [', '.join(load[f].nlargest(4).index) for f in fac.index]
fac['drives −'] = [', '.join(load[f].nsmallest(4).index) for f in fac.index]
print(f"{(~used).sum()} of {L} latent factors are used by no perturbation and are excluded: {', '.join(fac.index[~used])}")
fac[used].sort_values('erythroid', ascending=False)[P2 + ['cross-run similarity', 'similarity sd',
    'perts using', 'perts using sd', 'call', 'drives +', 'drives −']].round(2)

### Can the latent axes be matched across runs?

Latent dimensions are arbitrary within a run, so the axis names above rely on matching. **(a)** For
runs 2–5, the |cosine| between every run-1 axis signature and every signature of that run, with columns
reordered by the matching: a bright, clean diagonal means each axis has one clear counterpart.
**(b)** For every axis, the |cosine| of its match in each run (dots), the best alternative in the same run
(grey ×), and the range of |cosine| between unmatched axes (shaded; 95th percentile marked). Unused axes
are shown in grey. The permutation test compares the total matched |cosine| with 5,000 random pairings.

### Fig. S14: latent factors matched across runs

In [ ]:
Cs = []
for r, p_ in zip(runs[1:], perm[1:]):
    Cs.append(np.abs(nrm(runs[0]['V']) @ nrm(r['V']).T)[:, p_])     # matched partners on the diagonal
eye = np.eye(L, dtype=bool)
diag = np.stack([np.diag(Cm) for Cm in Cs])                          # (runs 2-5, axes)
runner = np.stack([np.where(eye, -1, Cm).max(1) for Cm in Cs])       # best alternative for each run-1 axis
offd = np.concatenate([Cm[~eye] for Cm in Cs])
null95 = np.quantile(offd, .95)
rng_m = np.random.default_rng(0)
perm_p = []
for Cm in Cs:
    tot = np.trace(Cm)
    null_tot = np.array([Cm[np.arange(L), rng_m.permutation(L)].sum() for _ in range(5000)])
    perm_p.append((np.sum(null_tot >= tot) + 1) / 5001)
match_tab = pd.DataFrame({'mean matched |cos|': diag.mean(0), 'lowest matched |cos|': diag.min(0),
                          'mean best alternative': runner.mean(0), 'mean margin': (diag - runner).mean(0),
                          'runs above null 95th pct': (diag > null95).sum(0), 'used': used.values},
                         index=fac.index)
print(f'Unmatched pairs: median |cos| {np.median(offd):.2f}, 95th percentile {null95:.2f}')
print('Total matched |cos| vs random pairings, per run:',
      ', '.join(f'run {k + 2}: p = {pv:.4f}' for k, pv in enumerate(perm_p)))

fig = plt.figure(figsize=(15, 8.2))
gsm = fig.add_gridspec(2, 5, height_ratios=[1, 1.05], width_ratios=[1, 1, 1, 1, .06], hspace=.45, wspace=.12)
cax = fig.add_subplot(gsm[0, 4])
for k, Cm in enumerate(Cs):
    axm = fig.add_subplot(gsm[0, k])
    sns.heatmap(Cm, cmap='viridis', vmin=0, vmax=1, square=True, cbar=k == len(Cs) - 1, ax=axm,
                cbar_ax=cax if k == len(Cs) - 1 else None,
                xticklabels=False, yticklabels=list(fac.index) if k == 0 else False,
                cbar_kws={'label': '|cos|'} if k == len(Cs) - 1 else None)
    axm.set_title(f'run 1 vs run {k + 2}\n(total matched vs random: p = {perm_p[k]:.4f})', fontsize=8)
    axm.set_xlabel(f'run {k + 2} latent factors (matched order)', fontsize=7)
    axm.tick_params(axis='y', labelsize=6)
fig.axes[1].set_ylabel('run 1 latent factors', fontsize=8)
axb = fig.add_subplot(gsm[1, :4])
xs = np.arange(L)
axb.axhspan(offd.min(), null95, color='0.9', zorder=0)
axb.axhline(null95, color='0.5', ls='--', lw=.8)
axb.text(L - .5, null95 + .01, 'unmatched pairs, 95th percentile', ha='right', va='bottom', fontsize=7, color='0.35')
for j in range(L):
    col = '#2166ac' if used.iloc[j] else '0.65'
    axb.scatter(np.full(diag.shape[0], j) + np.linspace(-.15, .15, diag.shape[0]), diag[:, j], s=26, color=col, zorder=3)
    axb.scatter(np.full(diag.shape[0], j) + np.linspace(-.15, .15, diag.shape[0]), runner[:, j], s=22, marker='x',
                color='0.55', lw=1, zorder=2)
    axb.hlines(diag[:, j].mean(), j - .3, j + .3, color=col, lw=2, zorder=4)
axb.set_xticks(xs)
axb.set_xticklabels([f"{f}{'' if u else ' (unused)'}" for f, u in zip(fac.index, used)], rotation=40, ha='right', fontsize=7)
axb.set_ylim(0, 1.02); axb.set_ylabel('signature similarity with the run-1 factor (|cos|)')
axb.legend(handles=[mpl.lines.Line2D([], [], marker='o', ls='', color='#2166ac', label='matched factor, one dot per run'),
                    mpl.lines.Line2D([], [], marker='x', ls='', color='0.55', label='closest other factor in that run'),
                    mpl.lines.Line2D([], [], marker='o', ls='', color='0.65', label='factor used by no perturbation')],
           frameon=False, fontsize=7, loc='lower left', ncol=3)
axb.set_title('Match quality per latent factor', fontsize=9, loc='left')
axb.text(-0.045, 1.02, 'b', transform=axb.transAxes, fontsize=15, va='bottom', ha='right')
fig.axes[1].text(-0.25, 1.06, 'a', transform=fig.axes[1].transAxes, fontsize=15, va='bottom', ha='right')
plt.show()
match_tab.round(2)

## 4. Defining the fate axis

Each perturbation gets an **observed** fate score per lineage from canonical markers (section 1 setup).
These give the axis a biological definition:

- **erythroid programme up vs down** is the main axis; the model's decoded erythroid score agrees with it;
- **megakaryocytic** priming (canonical markers) is examined separately, because it does not oppose
  the erythroid programme in K562. It is a **weak signal**: its markers are detected in ~1% of control
  cells and are absent from the model's genes, so the code below checks that the score is reproducible
  (split-half reliability) and puts a confidence interval on its relation to the erythroid score;
- **granulocytic** priming is what opposes it.

Modules are assigned to a **pole** by their members' mean observed erythroid score: *erythroid up*
(> +0.2), *erythroid down* (< −0.2) or *neutral*. Within the *down* pole, a module is called
*granulocytic* if its granulocytic score exceeds 0.5; within the *up* pole, *erythroid + megakaryocytic*
if its megakaryocytic score exceeds 0.3.

In [ ]:
from scipy.stats import pearsonr, spearmanr, fisher_exact
print('Observed canonical fate scores: correlation across the 105 single perturbations')
print(fate.corr().round(2).to_string())

# How trustworthy is each score? Split every perturbation's cells (and the controls) into two random halves,
# score each half separately, and correlate the halves across perturbations (10 random splits).
rng_f = np.random.default_rng(0)
def _fate_scores(rows_by_label):
    ntc_ = _cpm(rows_by_label[NTC])
    lf = pd.DataFrame(np.stack([np.log2(_cpm(rows_by_label[g]) + .05) - np.log2(ntc_ + .05) for g in genes]),
                      index=genes, columns=_mg)
    zz = (lf - lf.mean()) / (lf.std() + 1e-9)
    return pd.DataFrame({f: zz[gs].mean(1) for f, gs in MARKERS.items()})
_rows = {g: np.where(labels == g)[0] for g in genes + [NTC]}
half_r = {f: [] for f in FATES}
for _ in range(10):
    split = {g: rng_f.permutation(ix) for g, ix in _rows.items()}
    h1 = _fate_scores({g: ix[: len(ix) // 2] for g, ix in split.items()})
    h2 = _fate_scores({g: ix[len(ix) // 2:] for g, ix in split.items()})
    for f in FATES:
        half_r[f].append(pearsonr(h1[f], h2[f])[0])
rel = pd.DataFrame({'split-half r': {f: np.mean(v) for f, v in half_r.items()}})
rel['full-data reliability'] = 2 * rel['split-half r'] / (1 + rel['split-half r'])     # Spearman-Brown
print('\nScore reliability:'); print(rel.round(2).to_string())

x_, y_ = fate['erythroid'].values, fate['megakaryocytic'].values
r_em = pearsonr(x_, y_)[0]
boot = [pearsonr(x_[i], y_[i])[0] for i in rng_f.integers(0, len(x_), (5000, len(x_)))]
permr = np.array([pearsonr(x_, rng_f.permutation(y_))[0] for _ in range(5000)])
keep = fate.index != 'FEV'
print(f"\nErythroid vs megakaryocytic: r = {r_em:.2f}, 95% CI [{np.quantile(boot, .025):.2f}, "
      f"{np.quantile(boot, .975):.2f}], permutation p = {(np.sum(np.abs(permr) >= abs(r_em)) + 1) / 5001:.4f}; "
      f"without FEV r = {pearsonr(x_[keep], y_[keep])[0]:.2f}")

D_mean = np.mean([r['D'] for r in runs], 0)
pert = prog_score(D_mean); pert.index = genes
print(f"\nModel-decoded vs observed erythroid score: r = {pearsonr(pert['erythroid'], fate['erythroid'])[0]:.2f}"
      f" (Spearman {spearmanr(pert['erythroid'], fate['erythroid'])[0]:.2f})")

NORMAN = {'erythroid': ['CBL', 'CNN1', 'UBASH3A', 'UBASH3B', 'PTPN9', 'PTPN12'],
          'granulocyte': ['CEBPA', 'CEBPB', 'CEBPE', 'SPI1'],
          'G1 arrest': ['CDKN1A', 'CDKN1B', 'CDKN1C']}
print('\nNorman’s named groups (observed fate scores, mean over members):')
print(pd.DataFrame({k: fate.loc[v].mean() for k, v in NORMAN.items()}).T.round(2).to_string())
print('\nStrongest canonical megakaryocytic priming (Norman’s CD41⁺ group has no named members):')
print(fate.sort_values('megakaryocytic', ascending=False).head(6).round(2).to_string())

mods = fate.groupby(module.reindex(genes).values).mean().reindex(MODULES)
mods['decoded erythroid'] = pert.groupby(module.reindex(genes).values)['erythroid'].mean()
mods['n'] = module.value_counts().reindex(MODULES)
mods['pole'] = np.select([mods['erythroid'] > .2, mods['erythroid'] < -.2], ['erythroid up', 'erythroid down'], 'neutral')
mods['identity'] = np.select(
    [(mods['pole'] == 'erythroid down') & (mods['granulocytic'] > .5), mods['pole'] == 'erythroid down',
     (mods['pole'] == 'erythroid up') & (mods['megakaryocytic'] > .3), mods['pole'] == 'erythroid up'],
    ['granulocytic', 'erythroid repressed only', 'erythroid + megakaryocytic', 'erythroid'], 'neutral')
for k, v in NORMAN.items():
    mods[f'Norman {k}'] = [', '.join(sorted(set(module.index[module == m]) & set(v))) for m in MODULES]
s6_groups = {}
for df in pd.read_excel(Path(paths.DATA_DIR) / 'norman' / 'aax4438_tables6.xlsx', sheet_name=None, header=None).values():
    rows = df.values.tolist()
    for i, rr in enumerate(rows[:-1]):
        if str(rr[0]).strip().isdigit():
            s6_groups[str(rr[1]).strip()] = [g.strip() for g in str(rows[i + 1][0]).split(',')]
mods['Norman Table S6 clusters (≥ 2 members)'] = [
    '; '.join(sorted({lbl for lbl, gs in s6_groups.items() if len(set(gs) & set(module.index[module == m])) >= 2}))
    for m in MODULES]
mods['members'] = [', '.join(module.index[module == m]) for m in MODULES]
pd.set_option('display.max_colwidth', 80)
mods.sort_values('erythroid', ascending=False).round(2)

## 5. How a pair's position on the axis relates to its interaction

Two views of the labelled pairs: the **poles of the two modules** (erythroid up / neutral / down,
abbreviated U / 0 / D) and the **observed erythroid scores of the two parents**. Enrichment is tested
with Fisher's exact test against the other labelled pairs, and the eight tests are adjusted together
(Benjamini–Hochberg q). These statistics describe this panel only.

In [ ]:
PS = {'erythroid up': 'U', 'neutral': '0', 'erythroid down': 'D'}
lab['pole_pair'] = ['×'.join(sorted([PS[mods.loc[module[a], 'pole']], PS[mods.loc[module[b], 'pole']]],
                                    key=['U', '0', 'D'].index)) for a, b in zip(lab['a'], lab['b'])]
lab['within_module'] = lab['module_a'] == lab['module_b']
lab['ery_lo'] = [min(fate.loc[a, 'erythroid'], fate.loc[b, 'erythroid']) for a, b in zip(lab['a'], lab['b'])]
lab['ery_hi'] = [max(fate.loc[a, 'erythroid'], fate.loc[b, 'erythroid']) for a, b in zip(lab['a'], lab['b'])]
lab['ery_gap'] = lab['ery_hi'] - lab['ery_lo']
lab['opposite_parents'] = (lab['ery_lo'] < 0) & (lab['ery_hi'] > 0)
PP = [p for p in ['U×U', 'U×0', 'U×D', '0×0', '0×D', 'D×D'] if p in set(lab['pole_pair'])]
ct_pole = pd.crosstab(lab['pole_pair'], lab['gi']).reindex(index=PP, columns=CLASSES, fill_value=0)
print('Interactions by the poles of the two modules (U = erythroid up, D = erythroid down):')
print(ct_pole.assign(n=ct_pole.sum(1)).to_string())

def enrich(cls, mask, label):
    k = int(((lab['gi'] == cls) & mask).sum()); n = int((lab['gi'] == cls).sum())
    t = [[k, n - k], [int(mask.sum()) - k, len(lab) - int(mask.sum()) - (n - k)]]
    orr, p = fisher_exact(t)
    return {'class': short(cls), 'where': label, 'in': f'{k}/{n}', 'base rate': f'{mask.mean():.0%}',
            'odds ratio': orr, 'p': p}
claims = pd.DataFrame([
    enrich('redundant', lab['within_module'], 'same module'),
    enrich('suppression', lab['within_module'], 'same module'),
    enrich('suppression', lab['pole_pair'] == 'U×D', 'opposite poles (U×D)'),
    enrich('suppression', lab['opposite_parents'], 'parents push erythroid opposite ways'),
    enrich('redundant', lab['pole_pair'] == 'U×D', 'opposite poles (U×D)'),
    enrich('approximately additive', lab['pole_pair'] == 'U×D', 'opposite poles (U×D)'),
    enrich('synergy', lab['pole_pair'] == 'U×U', 'both erythroid up (U×U)'),
    enrich('redundant', lab['pole_pair'] == 'D×D', 'both erythroid down (D×D)'),
])
from statsmodels.stats.multitest import multipletests
claims['q (BH)'] = multipletests(claims['p'], method='fdr_bh')[1]
claims = claims.drop(columns='p')
print('\n' + claims.round(4).to_string(index=False))

opp = lab[(lab['pole_pair'] == 'U×D') & lab['gi'].isin(['synergy', 'suppression'])]
cmp_ = opp.groupby('gi')[['ery_gap', 'rho', 'joint_over_add', 'c_sum']].mean()
cmp_['n'] = opp['gi'].value_counts()
cmp_['Norman dissimilar-phenotype'] = opp.groupby('gi')['genetic_interaction'].apply(
    lambda s: int((s == 'synergy (dissimilar phenotype)').sum()))
print('\nOpposite-pole pairs: what separates synergy from suppression')
cmp_.round(2)

## Biological findings

All statements concern the measured, labelled panel (86 pairs). It is a curated selection, so none is
claimed to extend to unmeasured combinations.

**1. Interaction type is readable from seven model parameters** (accuracy 0.77, balanced 0.78). ρ
separates co-directional from opposing pairs; the size of the combination relative to the sum separates
redundant (smaller), additive (equal) and synergy (larger); private latent factors with undamped parent
scales mark potentiation; decoder dominance marks epistasis; and synergy acting on latent factors that
neither parent gates marks neomorphism.

**2. One dominant latent axis carries the erythroid programme, and the C/EBPs oppose it.** F13
(erythroid score 5.1; reproduced across runs, |cos| 0.86; opened by 87 of 105 perturbations) is pushed
up by IKZF3, DUSP9, TP73 and PRDM1 and down by CEBPA, CEBPE, CEBPB and OSR2. F5 (erythroid 2.8) is
driven by FEV, ELMSAN1, IKZF3 and LHX1. **F12 is the latent axis of the lineage conflict** (granulocytic
+1.6, erythroid −1.3): the C/EBPs, MAPK1 and ETS2 load on one end and CNN1, PRTG, IGDCC3 and KLF1 on the
other — the same genes that form the suppression pairs in point 6.

**3. The fate axis is erythroid up versus erythroid down.** With canonical markers, granulocytic
priming opposes the erythroid programme (r = −0.31). Megakaryocytic priming is a **weak signal** — its
markers are detected in ~1% of control cells and the model cannot decode them — but its score is
reproducible (split-half reliability 0.84, versus 0.99 erythroid and 0.97 granulocytic), and it does not
oppose the erythroid programme: the two co-vary weakly (r = +0.29, 95% CI 0.04–0.50, permutation
p = 0.005; unchanged without FEV). The strongest megakaryocytic inducers (FEV, TP73, IKZF3) also raise
erythroid genes, consistent with a bipotent megakaryocyte–erythroid pattern. The score also co-varies
with granulocytic markers (r = +0.35; CEBPA is among the top megakaryocytic scorers), so megakaryocytic priming is not cleanly
separable from either fate in this screen. The model's decoded erythroid score matches the observed one
(r = 0.80). Norman's erythroid group raises the erythroid programme (+0.39), his granulocyte group
lowers it and raises granulocytic markers (−0.81 / +2.37), and the G1-arrest group (CDKN1A/B/C) sits on
the erythroid side (+0.65).

**4. The learned modules recover Norman's functional clusters.** 9 of his 20 perturbation-map clusters
and phenotype groups fall into a single module more often than random modules of the same sizes would
allow (79% of members in the best module on average, versus 41% by chance). The tyrosine-phosphatase
cluster (6 of 8 in M2), the bZIP TFs (5 of 5 in M3), the granulocyte group (4 of 4 in M3), the CDK
inhibitors (3 of 3 in M7), the RNA-pol-II TFs (7 of 12 in M1), the forkhead TFs (4 of 6 in M1),
regulation of apoptosis (5 of 9 in M5) and Norman's erythroid group (4 of 6 in M2) are all concentrated
(BH q between 0.004 and 0.032 across the 20 clusters). Most unconcentrated clusters have only two members, which cannot
reach significance. The
clusters' own programme scores follow the modules: the phosphatase, erythrocytosis, CDK-inhibitor,
apoptosis and Ig-domain clusters raise the erythroid programme (+0.6 to +0.9), while the bZIP TFs and
granulocyte group raise granulocytic markers (+2.0, +2.4) and MAPK1–ETS2 lowers the erythroid programme
(−1.2).

**5. Modules split into three poles, and the "down" pole is not one fate.** *Erythroid up:* M5
(erythroid + megakaryocytic: FEV, LHX1, IKZF3), M11, M7 (G1 arrest), M2 (Norman's erythroid core CNN1,
PTPN9, PTPN12, UBASH3B) and M1 (CBL). *Neutral:* M4 and M6 (MAP2K3/MAP2K6). *Erythroid down:* M3 is
granulocytic (all four C/EBP/PU.1 members of Norman's granulocyte group); M8 (MAPK1, ETS2), M9, M10 and
M12 lower the erythroid programme without adopting a granulocytic or megakaryocytic signature.

**6. Where two modules sit constrains the interaction; the model's parameters decide among the options.**
- *Redundancy is within a module:* all 8 redundant pairs join two genes of the same module
  (27% of labelled pairs are within-module; q = 4×10⁻⁵), and 5 of the 8 lie in the erythroid-down pole,
  including the C/EBP paralogs (q = 0.003).
- *Suppression needs opposite poles:* no suppression pair is within a module (q = 0.035), and all 12
  join an erythroid-up module to an erythroid-down module (38% base rate; q = 2×10⁻⁵). At the gene level,
  11 of 12 suppression pairs have parents that push the erythroid programme in opposite directions
  (48% of labelled pairs; q = 0.003).
  The antagonists are the C/EBPs and the MAPK1/ETS2 module against the erythroid phosphatase module
  (CNN1, PTPN12, UBASH3B), PRTG/IGDCC3 and LYL1.
- *Synergy concentrates in the erythroid-up pole* (11 of 24; q = 0.035). Opposite-pole pairs that
  synergise instead of suppressing (9, seven of them Norman's dissimilar-phenotype type) have parents much
  closer together on the erythroid axis (gap 0.61 versus 1.74) and are the ones the model still aligns: ρ +0.36 versus −0.13, a combination 1.21× the sum versus 0.91×, and undamped
  versus damped parent scales (1.76 versus 1.29).
- *Additivity avoids opposite poles* (2 of 15; q = 0.04).

All q-values are Benjamini–Hochberg adjusted within their family of tests (the 8 pair-level tests
above; the 20 cluster tests in point 4).

*Caveats.* Latent axes match clearly across runs (every run's total matched similarity beats random pairings, p = 0.0002), but F3 and F15 cannot be matched unambiguously — their best alternative is as similar as their assigned partner — so they are not interpreted. Canonical megakaryocytic markers are detected in only ~1% of control cells, so that score is
noisy; the model cannot decode them at all, and it decodes only four canonical granulocyte genes. Rule cut
points were fit on these labelled pairs.

## Figures

Labelled pairs only. The erythroid axis runs from *erythroid up* (brown) to *erythroid down* (blue);
interaction classes keep their own colours.

In [ ]:
from matplotlib.path import Path as MPath
ERY_CMAP = mpl.colors.LinearSegmentedColormap.from_list('ery', ['#2166ac', '#f7f7f7', '#8c2d04'])
enorm = mpl.colors.TwoSlopeNorm(vmin=fate['erythroid'].min(), vcenter=0, vmax=fate['erythroid'].max())
POLE_C = {'erythroid up': '#8c2d04', 'neutral': '#9e9e9e', 'erythroid down': '#2166ac'}
ID_C = {'granulocytic': '#7a7a00', 'erythroid + megakaryocytic': '#01665e'}
PROG_C = {'erythroid': '#8c2d04', 'granulocytic': '#7a7a00'}
GROUP_C = {'erythroid': '#8c2d04', 'granulocyte': '#7a7a00', 'G1 arrest': '#5e3c99'}

# re-decode the exemplar pairs (figure 6)
EXAMPLES = [('CEBPA', 'CEBPB'), ('CBL', 'PTPN12'), ('CEBPE', 'CNN1'), ('DUSP9', 'MAPK1'), ('FEV', 'ISL2')]
ex_pred = {e: [] for e in EXAMPLES}
for run in range(1, N_RUNS + 1):
    m = slk.tl.load_results(FIG / 'models' / f'run_{run}.pth').to(DEVICE)
    with torch.no_grad():
        sh = m._get_all_pert_shifts(DEVICE)[1:]
        S_, q_ = sh[:, 0], sh[:, 2]
        u = m._abduct_ntc(x_bg)
        def dec(shift):
            z = (u[None] + shift[:, None]).reshape(-1, u.shape[-1])
            return torch.log1p(m._decode_to_expr(z, 1.) * 1e4).reshape(shift.shape[0], N_BG, -1).mean(1)
        base = torch.log1p(m._decode_to_expr(u, 1.) * 1e4).mean(0)
        ia = torch.as_tensor([g2i[a] for a, _ in EXAMPLES], device=DEVICE)
        ib = torch.as_tensor([g2i[b] for _, b in EXAMPLES], device=DEVICE)
        ca, cb = m._parent_scales_from_rho(q_[ia], q_[ib]); syn = m._synergy_shift_from_rho(q_[ia], q_[ib])
        d_a, d_b = dec(S_[ia]) - base, dec(S_[ib]) - base
        d_ab = dec(ca * S_[ia] + cb * S_[ib] + syn) - base
        for k, e in enumerate(EXAMPLES):
            ex_pred[e].append(np.stack([d_a[k].cpu().numpy(), d_b[k].cpu().numpy(), d_ab[k].cpu().numpy()]))
    del m; torch.cuda.empty_cache()
print('decoded', len(EXAMPLES), 'exemplar pairs')

### 1. Position on the erythroid axis and interaction type

**(a)** Modules ordered by their members' observed erythroid score (number under each module). Arcs are
labelled interactions: cooperative classes above the line, suppression / epistasis / neomorphic below;
loops are pairs within one module. Background: brown = *erythroid up* pole, blue = *erythroid down*
pole, white = neutral. A coloured ring marks a module's identity within its pole (olive = granulocytic,
teal = erythroid + megakaryocytic; megakaryocytic is a weak signal, see section 4). **(b)** Every labelled
pair placed by its two parents' observed
erythroid scores. The shaded corner holds pairs whose parents push the erythroid programme in opposite
directions.

### Fig. S13a–b: labelled interactions on the erythroid axis

In [ ]:
order_m = mods.sort_values('erythroid').index.tolist()     # erythroid down (left) -> up (right)
ery_sem = fate.groupby(module.reindex(genes).values)['erythroid'].sem()     # spread of the module mean
xr = {m_: i for i, m_ in enumerate(order_m)}
COOP = ['approximately additive', 'synergy', 'redundant', 'potentiation']
fig = plt.figure(figsize=(14, 11))
gs_ = fig.add_gridspec(2, 2, height_ratios=[1.1, 1], width_ratios=[1, .42], hspace=.1, wspace=.05)
ax = fig.add_subplot(gs_[0, :])
for pole, col in POLE_C.items():
    idx = [xr[m_] for m_ in order_m if mods.loc[m_, 'pole'] == pole]
    if idx and pole != 'neutral':
        ax.axvspan(min(idx) - .5, max(idx) + .5, color=col, alpha=.07)
        ax.text((min(idx) + max(idx)) / 2, 6.4, pole, color=col, fontsize=12.2, fontweight='bold', ha='center')
off = dict(zip(CLASSES, np.linspace(-.18, .18, len(CLASSES))))
for (ma, mb, cls), n in lab.groupby(['module_a', 'module_b', 'gi']).size().items():
    xa, xb = sorted([xr[ma], xr[mb]])
    sgn = 1 if cls in COOP else -1
    if xa == xb:
        h = sgn * (0.5 + 0.12 * CLASSES.index(cls))
        verts = [(xa - .12, 0), (xa - .32, h), (xa + .32, h), (xa + .12, 0)]
    else:
        h = sgn * (0.55 * (xb - xa) + 0.3) * (1 + off[cls])
        verts = [(xa, 0), (xa, h), (xb, h), (xb, 0)]
    ax.add_patch(mpl.patches.PathPatch(MPath(verts, [MPath.MOVETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4]),
                                       fc='none', ec=PALETTE[cls], lw=.8 + 1.4 * n, alpha=.45))
for m_ in order_m:
    r_ = mods.loc[m_]
    ax.scatter(xr[m_], 0, s=230 + 30 * r_['n'], color=ERY_CMAP(enorm(r_['erythroid'])), edgecolor='0.3',
               lw=.6, zorder=5)
    ax.text(xr[m_], 0, m_, ha='center', va='center', fontsize=10.1, fontweight='bold', zorder=6, color='k')
    ax.text(xr[m_], -1.4, f"{r_['erythroid']:+.2f}\n± {ery_sem[m_]:.2f}", ha='center', fontsize=8.1,
            color='0.3', zorder=6, linespacing=.95,
            bbox=dict(fc='white', ec='none', pad=.5, alpha=.8))
ax.axhline(0, color='0.5', lw=.6)
ax.set_xlim(-.7, len(order_m) - .3); ax.set_ylim(-6.9, 6.9); ax.set_xticks([]); ax.set_yticks([])
for s_ in ['left', 'bottom']:
    ax.spines[s_].set_visible(False)
h_cls = [mpl.lines.Line2D([], [], color=PALETTE[c], lw=3, label=short(c)) for c in CLASSES]
ax.legend(handles=h_cls, frameon=False, fontsize=9.5, loc='upper left', bbox_to_anchor=(1.0, 1.0), title='interaction')
_cb = fig.colorbar(mpl.cm.ScalarMappable(norm=enorm, cmap=ERY_CMAP), ax=ax, fraction=.018, pad=.16,
                   location='right', shrink=.5)
_cb.set_label('module mean erythroid score', fontsize=9.5)
_cb.ax.tick_params(labelsize=8.1)
ax.text(-0.02, 1.02, 'a', transform=ax.transAxes, fontsize=15, va='bottom', ha='right')

ax2 = fig.add_subplot(gs_[1, 0])
lo, hi = lab['ery_lo'].min() - .2, lab['ery_hi'].max() + .2
ax2.add_patch(mpl.patches.Rectangle((lo, 0), 0 - lo, hi, color='0.9', zorder=0))
ax2.plot([lo, hi], [lo, hi], color='0.75', lw=.6)
ax2.axvline(0, color='0.6', ls='--', lw=.6); ax2.axhline(0, color='0.6', ls='--', lw=.6)
for c in CLASSES:
    s_ = lab[lab['gi'] == c]
    ax2.scatter(s_['ery_lo'], s_['ery_hi'], s=40, color=PALETTE[c], edgecolor='white', lw=.5, label=short(c), zorder=3)
ax2.text(lo + .05, hi - .05, 'parents push the erythroid\nprogramme in opposite directions', fontsize=9.5, va='top', color='0.3')
ax2.text(.05, hi - .05, 'both raise it', fontsize=9.5, va='top', color=POLE_C['erythroid up'])
xhi = max(lab['ery_lo'].max(), 0) + .35
ax2.text(-.05, lo + .05, 'both lower it', fontsize=9.5, color=POLE_C['erythroid down'], ha='right',
         bbox=dict(fc='white', ec='none', pad=1, alpha=.8))
ax2.set(xlim=(lo, xhi), ylim=(lo, hi), xlabel='observed erythroid score, lower parent',
        ylabel='observed erythroid score, higher parent')
ax2.text(-0.08, 1.02, 'b', transform=ax2.transAxes, fontsize=15, va='bottom', ha='right')
ax2.set_title('Each labelled pair, placed by its two parents', fontsize=12.2, loc='left')

ax3 = fig.add_subplot(gs_[1, 1]); ax3.axis('off')
txt = ['Enrichment on this panel', '(Fisher; BH-adjusted q, 8 tests)']
for _, r_ in claims.iterrows():
    txt.append(f"{r_['class']}: {r_['in']} {r_['where']}\n   (base {r_['base rate']}, q = {r_['q (BH)']:.1g})")
ax3.text(0, 1, '\n'.join(txt), va='top', fontsize=9.9, family='monospace')
plt.show()

### 2. Modules, Norman's functional clusters, fate programmes and latent axes

One row per module, ordered from erythroid up to erythroid down. **(a)** Overlap with Norman's
perturbation-map clusters (Table S6B) and named phenotype groups: dot size = shared genes, dot colour =
the module's fate identity. The strip above gives each Norman cluster's own observed erythroid and
granulocytic scores; the percentage is the share of the cluster that falls in its best module, and *
marks clusters more concentrated than in random modules of the same sizes (5,000 permutations,
BH-adjusted q < 0.05 across the 20 clusters).
**(b)** The module's observed programme scores (canonical markers; the megakaryocytic column is a weak
signal — reliability 0.84 versus ≥ 0.97 for the other two, see section 4). **(c)** The module's mean loading on
each latent axis that at least one perturbation opens; the strip above is each axis's programme
content (decoded erythroid and granulocytic genes).

### Fig. S12a–c: consensus modules vs Norman clusters, measured programs and latent factors

In [ ]:
from statsmodels.stats.multitest import multipletests
# Norman's perturbation-map clusters (Table S6B) plus his named phenotype groups
_s6b = pd.read_excel(Path(paths.DATA_DIR) / 'norman' / 'aax4438_tables6.xlsx', sheet_name='Table S6B Perturb map annot', header=None).values.tolist()
norman_fx = {}
for i, rr in enumerate(_s6b[:-1]):
    if str(rr[0]).strip().isdigit():
        mem = [g.strip() for g in str(_s6b[i + 1][0]).split(',') if g.strip() in g2i]
        if len(mem) >= 2:
            norman_fx[str(rr[1]).strip()] = mem
for k, v in NORMAN.items():
    norman_fx[f'{k} group'] = v

# correspondence: share of each cluster in its best module, against random modules of the same sizes
order_m = mods.sort_values('erythroid', ascending=False).index.tolist()
mcode = module.reindex(genes).map({m_: i for i, m_ in enumerate(MODULES)}).values
rng_ = np.random.default_rng(0)
perm_codes = np.stack([rng_.permutation(mcode) for _ in range(5000)])
rows = []
for name, mem in norman_fx.items():
    idx = [g2i[g] for g in mem]
    counts = module[mem].value_counts()
    frac = counts.iloc[0] / len(mem)
    null = (perm_codes[:, idx][..., None] == np.arange(len(MODULES))).sum(1).max(1) / len(mem)
    rows.append({'Norman cluster': name, 'n': len(mem), 'best module': counts.index[0], 'in best module': frac,
                 'expected (random modules)': null.mean(), 'p': (np.sum(null >= frac) + 1) / (len(null) + 1),
                 'erythroid': fate.loc[mem, 'erythroid'].mean(), 'granulocytic': fate.loc[mem, 'granulocytic'].mean()})
fx = pd.DataFrame(rows)
fx['q (BH)'] = multipletests(fx['p'], method='fdr_bh')[1]
fx = fx.drop(columns='p')
fx['_order'] = fx['best module'].map({m_: i for i, m_ in enumerate(order_m)})
fx = fx.sort_values(['_order', 'in best module'], ascending=[True, False]).drop(columns='_order').reset_index(drop=True)
print(f"Clusters more concentrated than random modules (BH q < 0.05): {(fx['q (BH)'] < .05).sum()} / {len(fx)};  "
      f"mean share in best module {fx['in best module'].mean():.2f} vs {fx['expected (random modules)'].mean():.2f} by chance")

ID_COL = {'erythroid': '#8c2d04', 'erythroid + megakaryocytic': '#01665e', 'granulocytic': '#7a7a00',
          'erythroid repressed only': '#2166ac', 'neutral': '#9e9e9e'}
ax_order = fac[used].sort_values('erythroid', ascending=False).index.tolist()
mod_load = load.groupby(module.reindex(genes).values).mean().reindex(order_m)[ax_order]
nC, nA, nM = len(fx), len(ax_order), len(order_m)

fig = plt.figure(figsize=(22, 9))
gs2 = fig.add_gridspec(2, 3, width_ratios=[nC, 3.6, nA], height_ratios=[2.2, nM], wspace=.05, hspace=.04)
# (a) strip: each Norman cluster's own programme scores
axs_ = fig.add_subplot(gs2[0, 0])
sns.heatmap(fx[['erythroid', 'granulocytic']].T.values, cmap='RdBu_r', center=0, vmin=-2, vmax=2, cbar=False,
            annot=True, fmt='.1f', annot_kws={'fontsize': 8.1}, linewidths=.5, linecolor='white', ax=axs_,
            xticklabels=False, yticklabels=['erythroid', 'granulocytic'])
axs_.tick_params(axis='y', labelsize=9.5, rotation=0)
axs_.text(-0.02, 1.25, 'a', transform=axs_.transAxes, fontsize=15, va='bottom', ha='right')
axs_.set_title('Norman’s clusters: measured programme scores (top) and overlap with our modules (below)',
               fontsize=12.2, loc='left')
# (a) dot matrix
axd = fig.add_subplot(gs2[1, 0])
for j, r_ in fx.iterrows():
    mem = set(norman_fx[r_['Norman cluster']])
    for i, m_ in enumerate(order_m):
        k = len(mem & set(module.index[module == m_]))
        if k:
            axd.scatter(j + .5, i + .5, s=30 + 70 * k, color=ID_COL[mods.loc[m_, 'identity']], edgecolor='white',
                        lw=.6, zorder=3)
            axd.text(j + .5, i + .5, k, ha='center', va='center', fontsize=8.1, color='white', zorder=4)
axd.set_xlim(0, nC); axd.set_ylim(nM, 0)
axd.set_xticks(np.arange(nC) + .5)
axd.set_xticklabels([f"{c}  ({v:.0%}{'*' if q_ < .05 else ''})" for c, v, q_ in
                     zip(fx['Norman cluster'], fx['in best module'], fx['q (BH)'])], rotation=60, ha='right', fontsize=9.5)
axd.set_yticks(np.arange(nM) + .5); axd.set_yticklabels(order_m, fontsize=10.8)
axd.grid(color='0.93', lw=.6); axd.set_axisbelow(True)
for s_ in axd.spines.values():
    s_.set_visible(False)
axd.legend(handles=[mpl.lines.Line2D([], [], marker='o', ls='', color=c, ms=8, label=k) for k, c in ID_COL.items()],
           title='module identity', frameon=False, fontsize=9.5, loc='upper left', bbox_to_anchor=(0, -.42), ncol=5)
# (b) module programme scores
axp = fig.add_subplot(gs2[1, 1])
sns.heatmap(mods.loc[order_m, ['erythroid', 'granulocytic', 'megakaryocytic']], cmap='RdBu_r', center=0,
            vmin=-1.2, vmax=1.2, annot=True, fmt='.2f', annot_kws={'fontsize': 8.8}, cbar=False, linewidths=.5,
            linecolor='white', ax=axp, yticklabels=False)
axp.set_xticklabels(['erythroid', 'granulocytic', 'megakaryocytic'], rotation=60, ha='right', fontsize=9.5)
axp.text(-0.15, 1.02, 'b', transform=axp.transAxes, fontsize=15, va='bottom', ha='right')
axp.set_title('measured\nprogrammes', fontsize=12.2)
# (c) strip: axis programme content, then module loadings
axt = fig.add_subplot(gs2[0, 2])
sns.heatmap(fac.loc[ax_order, P2].T.values, cmap='RdBu_r', center=0, vmin=-4, vmax=4, cbar=False, annot=True,
            fmt='.1f', annot_kws={'fontsize': 8.8}, linewidths=.5, linecolor='white', ax=axt, xticklabels=False,
            yticklabels=P2)
axt.tick_params(axis='y', labelsize=9.5, rotation=0)
axt.text(-0.02, 1.25, 'c', transform=axt.transAxes, fontsize=15, va='bottom', ha='right')
axt.set_title('latent factors: programme content (top) and mean shift per module (below)', fontsize=12.2, loc='left')
axl = fig.add_subplot(gs2[1, 2])
lim = np.abs(mod_load.values).max()
sns.heatmap(mod_load, cmap='PuOr_r', center=0, vmin=-lim, vmax=lim, annot=True, fmt='.1f', annot_kws={'fontsize': 8.1},
            linewidths=.5, linecolor='white', ax=axl, yticklabels=False,
            cbar_kws={'label': 'mean latent shift of the module', 'shrink': .6})
axl.set_xticklabels([f"{f} ({fac.loc[f, 'cross-run similarity']:.2f})" for f in ax_order], rotation=60, ha='right', fontsize=9.5)
axl.set_xlabel('latent factor (cross-run similarity)', fontsize=10.1)
plt.show()
fx.round(3)

### 3. Who drives the three most interpretable axes

Loadings on F13 (dominant erythroid axis), F5 (erythroid with megakaryocytic-leaning genes) and F12
(granulocytic up, erythroid down). **Each bar is one perturbed gene** (CRISPRa target), coloured by
Norman group; the most extreme genes on each side are named.

### Fig. S15: perturbations driving F13, F5 and F12

In [ ]:
SHOW = [f for f in ['F13', 'F5', 'F12'] if f in fac.index and used[f]]
group_of = {g: k for k, v in NORMAN.items() for g in v}
fig, axs = plt.subplots(len(SHOW), 1, figsize=(15, 3.9 * len(SHOW)))
for ax, f in zip(np.atleast_1d(axs), SHOW):
    lo_ = load[f].sort_values()
    ax.bar(range(len(lo_)), lo_.values, color=[GROUP_C.get(group_of.get(g, ''), '0.82') for g in lo_.index], width=.85)
    span = np.abs(lo_.values).max()
    for i in list(range(7)) + list(range(len(lo_) - 7, len(lo_))):
        ax.text(i, lo_.values[i] + np.sign(lo_.values[i]) * .02 * span, lo_.index[i], rotation=90, fontsize=8.8,
                ha='center', va='bottom' if lo_.values[i] > 0 else 'top')
    ax.axhline(0, color='k', lw=.6); ax.set_xticks([])
    ax.set_ylim(lo_.min() * 1.45, lo_.max() * 1.45)
    ax.set_ylabel(f'latent shift along {f}')
    ax.set_title(f"{f}  |  erythroid programme {fac.loc[f, 'erythroid']:+.1f}, granulocytic "
                 f"{fac.loc[f, 'granulocytic']:+.1f}  |  cross-run similarity "
                 f"{fac.loc[f, 'cross-run similarity']:.2f} ± {fac.loc[f, 'similarity sd']:.2f}  |  used by "
                 f"{fac.loc[f, 'perts using']:.0f} ± {fac.loc[f, 'perts using sd']:.0f} of {P} perturbations",
                 fontsize=12.2, loc='left')
np.atleast_1d(axs)[0].legend(handles=[mpl.patches.Patch(color=c, label=f'Norman {k}') for k, c in GROUP_C.items()] +
                             [mpl.patches.Patch(color='0.82', label='other')], frameon=False, fontsize=9.5, loc='upper left')
np.atleast_1d(axs)[-1].set_xlabel('CRISPRa target genes, ordered by their shift along the factor')
plt.tight_layout(); plt.show()

### 4. The neomorphic rule

### Fig. S11c: the neomorphic rule

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
for c in CLASSES:
    s_ = lab[lab['gi'] == c]
    ax.scatter(s_['syn_offgate_dec'], s_['rho'], s=46 if c == 'neomorphic' else 24, color=PALETTE[c],
               edgecolor='k' if c == 'neomorphic' else 'white', lw=.6, label=short(c), zorder=3)
(_, _, x0), (_, _, y1) = dict(RULES)['neomorphic']
xmax = lab['syn_offgate_dec'].max() * 1.08
ax.add_patch(mpl.patches.Rectangle((x0, lab['rho'].min() - .05), xmax - x0, y1 - lab['rho'].min() + .05,
                                   color=PALETTE['neomorphic'], alpha=.08, zorder=0))
ax.text(xmax, y1 + .01, 'region the rule calls neomorphic', ha='right', va='bottom', fontsize=7,
        color=PALETTE['neomorphic'])
ax.set(xlabel='interaction term acting on latent factors neither parent uses\n(decoded, relative to the parents\u2019 additive effect)',
       ylabel='parent similarity (ρ)', xlim=(0, xmax))
ax.legend(frameon=False, fontsize=7, ncol=2, loc='lower right')
ax.set_title('Neomorphic pairs combine dissimilar parents with an interaction term\non latent factors neither parent uses', fontsize=9)
plt.tight_layout(); plt.show()

### 5. What each interaction looks like in the model

Drawn from real data — the **mean descriptors of each class's labelled pairs** — placed on a schematic
geometry: the angle between parents A and B is arccos(mean ρ); B is shortened by the mean dominance
(`dom_gap`); the solid arrow is the sum scaled by the mean `joint_over_add`; the dotted arrow's length is
proportional to the mean `syn_offgate_dec`. Only the dotted arrow's direction is arbitrary.

### Fig. S11d: class-average schematics

In [ ]:
cm_ = lab.groupby('gi')[DESC7].mean().reindex(CLASSES)
fig, axs = plt.subplots(2, 4, figsize=(13, 7))       # two rows so the vectors are large enough to read
axs = axs.ravel()
axs[-1].axis('off')                                   # seven classes in an eight-panel grid
for ax, c in zip(axs, CLASSES):
    r_ = cm_.loc[c]
    th = np.arccos(np.clip(r_['rho'], -1, 1))               # angle between parents from rho
    a = np.array([1.0, 0.0])
    b = (1 - 0.6 * r_['dom_gap']) * np.array([np.cos(th), np.sin(th)])   # shorter partner = dominance
    comb = (a + b) * r_['joint_over_add']                   # size relative to the sum
    syn_v = np.array([-0.35, 0.9]) * r_['syn_offgate_dec'] * 4
    for v, col, lab_ in [(a, '0.35', 'A'), (b, '0.6', 'B')]:
        ax.annotate('', xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color=col, lw=1.6))
        ax.text(*(v * 1.08), lab_, fontsize=9, color=col)
    ax.annotate('', xy=a + b, xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color='0.75', lw=1, ls='--'))
    ax.annotate('', xy=comb, xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color=PALETTE[c], lw=2.6))
    ax.annotate('', xy=comb + syn_v, xytext=comb, arrowprops=dict(arrowstyle='-|>', color=PALETTE[c], lw=1.2, ls=':'))
    ax.set_xlim(-1.3, 2.4); ax.set_ylim(-1.1, 2.2); ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(short(c), color=PALETTE[c], fontsize=12, fontweight='bold')
    ax.text(-1.25, -1.05, f"parent similarity {r_['rho']:.2f}   combination/sum {r_['joint_over_add']:.2f}\n"
                          f"dominance {r_['dom_gap']:.2f}   interaction on unused factors "
                          f"{r_['syn_offgate_dec']:.2f}", fontsize=8)
fig.suptitle('Class averages: grey arrows = the two parents, dashed = their sum, '
             'solid = the combination, dotted = the interaction term', fontsize=11, y=1.0)
plt.tight_layout(); plt.show()

### 6. Worked examples: parents, predicted and observed combination

Programme scores (erythroid, granulocytic) of each parent, the model's predicted combination and the
measured combination. A score is the mean change of the programme's genes relative to control, in units
of that profile's own gene-wise spread: **positive = the programme is switched on, negative = it is
pushed below control** (e.g. the C/EBPs lower the erythroid genes). All five pairs were in the model's
training set, so this shows fit rather than prediction.

### Fig. S10: worked examples of five labelled pairs

In [ ]:
canon_lbl = np.array(['+'.join(sorted(l.split('+'))) if '+' in l else l for l in labels])
ntc_f = _pb(labels == NTC)
obs_vec = lambda lbl: _pb(canon_lbl == lbl) - ntc_f
EX_CLASS = dict(zip(lab['combination'], lab['gi']))
fig, axs = plt.subplots(1, len(EXAMPLES), figsize=(17, 3.6))
for ax, e in zip(axs, EXAMPLES):
    A_, B_ = e
    pred = np.mean(ex_pred[e], 0)
    obs = np.stack([obs_vec(A_), obs_vec(B_), obs_vec('+'.join(sorted(e)))])
    sc_p = prog_score(pred).values; sc_o = prog_score(obs).values
    rows = [(A_, sc_p[0]), (B_, sc_p[1]), ('A+B predicted', sc_p[2]), ('A+B observed', sc_o[2])]
    x = np.arange(len(rows)); w = .36
    for j, pname in enumerate(P2):
        ax.bar(x + (j - .5) * w, [r[1][j] for r in rows], width=w, color=PROG_C[pname], label=pname)
    ax.axhline(0, color='k', lw=.5)
    ax.set_xticks(x); ax.set_xticklabels([r[0] for r in rows], rotation=30, ha='right', fontsize=7)
    cls = EX_CLASS['+'.join(sorted(e))]
    ax.set_title(f"{A_}+{B_}\n{short(cls)}", color=PALETTE[cls], fontsize=9, fontweight='bold')
axs[0].set_ylabel('programme score'); axs[0].legend(frameon=False, fontsize=7)
plt.tight_layout(); plt.show()

### 7. Descriptor fingerprint

### Fig. S11b: descriptor fingerprint per class

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
for i, c in enumerate(CLASSES):
    for j, f in enumerate(DESC7):
        v = auc.loc[c, f]
        ax.scatter(j, i, s=1400 * abs(v - .5) ** 1.2 + 4, color=plt.cm.PuOr_r(v), edgecolor='0.3', lw=.3)
ax.set_xticks(range(len(DESC7))); ax.set_xticklabels([NAMES[f] for f in DESC7], rotation=30, ha='right')
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels([short(c) for c in CLASSES])
ax.invert_yaxis(); ax.set_xlim(-.6, len(DESC7) - .4); ax.set_ylim(len(CLASSES) - .4, -.6)
sm = mpl.cm.ScalarMappable(cmap='PuOr_r', norm=mpl.colors.Normalize(0, 1)); sm.set_array([])
plt.colorbar(sm, ax=ax, shrink=.8, label='AUROC (orange = class is higher)')
ax.set_title('Which model parameter marks which interaction class\n(dot size = strength of separation)')
for s_ in ax.spines.values():
    s_.set_visible(False)
plt.tight_layout(); plt.show()